# GoA Run #15 / Run-21 (v18.0.0) — OBJECT VISION (core audit fixes 1-4)
Deep-mode trigger + IL v1.6 + Transport v26.3.4 + dual-server + Object Vision

**Certified:** my_agent v1.0.21 (Object Vision) | CM v1.6 | artifact adb470eae95e94d557fd110420676096
**Core fixes:** (1) OBJECTS block — model ko seedha id/color/rows/cols dikhta
hai, RLE column-arithmetic khatam; (2) extract_objects data ab prompt mein
jaata hai (pehle compute karke phenk dete the); (3) coordinate targeting
direct; (4) _obs_fallback dual-format fix (grid_text "W8" RLE pehle
<malformed> deta tha). Obs advisory cap 1500->2500 (noise fix).
**Budget:** 17 min/game, 7.5h global (25x17=7.08h + infra < 7.5h; Kaggle 9h)
**Cells:** 36 (coder-conformance runs BEFORE THE RUN — FIX-V)


In [ ]:
# PREFLIGHT v8 (FINAL): paths, GPU(96GB hard), host RAM, parquet, disk — v8: VRAM GiB unit fix (external audit round 2)
# Locked facts (Chief-confirmed):
#   - RTX PRO 6000 = 96 GB GDDR7 hard limit → VRAM < 90 GB = FAIL
#   - Offline eval, single GPU, host RAM ~30-60 GB
#   - Python 3.12 mandatory, output = submission.parquet
#   - ARC framework import test is DEFERRED to Cell 2 (post-pip-install)
import os, sys, shutil, subprocess, json, glob
import importlib, importlib.metadata

# ---------- helpers ----------
def check(label, path, min_entries=1):
    if not os.path.isdir(path):
        print(f"MISS {label}: {path}")
        return False
    try:
        entries = os.listdir(path)
    except Exception as e:
        print(f"ERR  {label}: {path} — {e}")
        return False
    ok = len(entries) >= min_entries
    print(("OK   " if ok else "MISS ") + f"{label}: {path} ({len(entries)} entries)")
    return ok

def _norm(v):
    return str(v).split("+")[0] if v else None

def _mod_version(modname):
    try:
        return importlib.metadata.version(modname), "metadata"
    except Exception:
        try:
            m = importlib.import_module(modname)
            v = getattr(m, "__version__", None)
            return (v, "attribute") if v else (None, None)
        except Exception:
            return None, None

# ---------- /kaggle/input discovery ----------
print("=== /kaggle/input ===")
try:
    for p in sorted(os.listdir("/kaggle/input")):
        print(" ", p)
        full = "/kaggle/input/" + p
        if os.path.isdir(full):
            try:
                for sub in sorted(os.listdir(full))[:5]:
                    print("    └─", sub)
            except PermissionError:
                print("    └─ <permission denied>")
except Exception as e:
    print("  ERR listing /kaggle/input:", e)

# ---------- Python version ----------
print(f"INFO python: {sys.version.split()[0]} | exec: {sys.executable}")
if sys.version_info[:2] != (3, 12):
    print(f"WARN Python 3.12 expected, mila {sys.version.split()[0]} — ARC runtime check karo")

# ---------- Inputs ----------
REASONER_PATH = "/kaggle/input/datasets/gray2488/qwen38-27b-fp8"
CODER_PATH    = "/kaggle/input/datasets/gray2488/qwen3-coder-30b-a3b-instruct"
COMP          = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
FRAMEWORK_DIR = COMP + "/ARC-AGI-3-Agents"
WHEELS        = COMP + "/arc_agi_3_wheels"
ENVFILES      = COMP + "/environment_files"

checks = [
    check("reasoner weights", REASONER_PATH),
    check("coder weights",    CODER_PATH),
    check("framework",        FRAMEWORK_DIR),
    check("wheels",           WHEELS),
    check("env files",        ENVFILES),
]
# BUG-1 FIX: hard assert wapas — inputs ke bina aage jana = 9-hour session waste
assert all(checks), "Input missing — Add Input panel check karo (upar /kaggle/input listing dekho)"

# ---------- Working dir writability ----------
if os.access("/kaggle/working", os.W_OK):
    print("OK   /kaggle/working writable")
else:
    print("FAIL /kaggle/working not writable — submission save nahi hoga")

SUBMISSION_PATH = "/kaggle/working/submission.parquet"
print(f"INFO submission target: {SUBMISSION_PATH}")

# ---------- nvidia-smi ----------
out = None
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    if out.returncode == 0 and out.stdout.strip():
        print(f"OK   nvidia-smi: {out.stdout.strip()}")
    else:
        print(f"WARN nvidia-smi exit={out.returncode} | "
              f"stdout={out.stdout.strip()!r} | stderr={out.stderr.strip()!r}")
except FileNotFoundError:
    print("WARN nvidia-smi not found — CPU-only environment?")
except Exception as e:
    print(f"WARN nvidia-smi failed: {e}")

# ---------- Host RAM ----------
ram_gb = None
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                ram_gb = int(line.split()[1]) / 1e6
                print(f"INFO host RAM: {ram_gb:.1f} GB")
                if ram_gb < 60:
                    print(f"WARN host RAM {ram_gb:.1f} GB < 60 — direct safetensors load "
                          f"fail ho sakta hai. low_cpu_mem_usage=True + device_map='auto' use karo")
                else:
                    print("OK   host RAM sufficient for streaming safetensors load")
                break
except Exception as e:
    print(f"WARN host RAM check failed: {e}")

# ---------- torch + GPU (96 GB hard target) ----------
vram_gb = None
gpu_name = None
try:
    import torch as torch_mod  # noqa
    if torch_mod.cuda.is_available():
        props = torch_mod.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024 ** 3)   # GiB (v8: /1e9 decimal tha -> 102.7, locked-fact 90-100 GiB range fail karta tha; 97887 MiB = 95.6 GiB)
        gpu_name = props.name
        print(f"OK   torch {torch_mod.__version__} | {gpu_name} | {vram_gb:.0f} GB")

        # Chief-confirmed: RTX PRO 6000 = 96 GB GDDR7 hard limit
        if vram_gb < 90:
            print(f"FAIL VRAM {vram_gb:.0f} GB < 90 — RTX PRO 6000 (96 GB) expect tha. "
                  f"Actual: {gpu_name}. Accelerator='rtx6000' select karo")
        elif vram_gb > 100:
            print(f"WARN VRAM {vram_gb:.0f} GB > 100 — unexpected GPU: {gpu_name}")
        else:
            # COSMETIC FIX: 27B FP8 (~27GB) + 30B MoE FP8 (~31GB) = ~58 GB baseline
            print(f"OK   VRAM {vram_gb:.0f} GB — RTX PRO 6000 confirmed "
                  f"(~{vram_gb - 58:.0f} GB headroom after 27B FP8 + 30B MoE)")

        if "RTX" in gpu_name and "6000" in gpu_name:
            print(f"OK   GPU name matches RTX PRO 6000: {gpu_name}")
        else:
            print(f"WARN GPU name '{gpu_name}' — RTX PRO 6000 expect tha")

        # nvidia-smi vs torch cross-check
        if out is not None and out.returncode == 0:
            try:
                smi_mem = out.stdout.split(",")[1].strip()
                smi_mb = int("".join(c for c in smi_mem if c.isdigit()))
                smi_gb = smi_mb / 1024
                if abs(smi_gb - vram_gb) > 5:
                    print(f"WARN nvidia-smi VRAM {smi_gb:.1f} GB vs torch VRAM "
                          f"{vram_gb:.1f} GB mismatch (>5 GB)")
            except Exception:
                pass
    else:
        print(f"WARN torch {torch_mod.__version__} | CUDA unavailable — GPU enable karo")

    # ---------- wheels manifest match ----------
    _m = (glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/manifest.json")
          or glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/wheels_manifest.json")
          or glob.glob("/kaggle/input/models/*/goa-vllm-wheels/manifest.json")
          or glob.glob("/kaggle/input/*/goa-vllm-wheels/manifest.json"))
    if _m:
        with open(_m[0]) as f:
            _manifest = json.load(f)
        _wt = _manifest.get("torch_version")
        if _wt is None:
            print("WARN manifest mila par 'torch_version' key nahi")
        elif not isinstance(_wt, str):
            print(f"WARN manifest torch_version non-string: {type(_wt).__name__}")
        elif _norm(_wt) == _norm(torch_mod.__version__):
            print(f"OK   wheels torch {_wt} MATCH (normalized: {_norm(_wt)})")
        else:
            print(f"WARN wheels torch {_wt} != image torch {torch_mod.__version__} — LEAN risk")
    else:
        print("WARN wheels manifest nahi mila — torch check skip")
except Exception as e:
    print(f"WARN torch/GPU check skipped: {e}")

# ---------- vLLM / httpx ----------
# COSMETIC FIX: 'v == "?"' dead branch hataya (importlib.metadata "?" nahi return karta)
for mod in ("vllm", "httpx"):
    v, src = _mod_version(mod)
    if v:
        print(f"OK   {mod} {v} (via {src})")
    else:
        print(f"WARN {mod} missing — bootstrap cell se install hoga")

# BUG-2 FIX: ARC framework import test REMOVED from preflight.
# Reasons:
#   (a) arc-agi ka pip install cell 2 mein hota hai — preflight us se pehle chalta hai
#   (b) fallback 'import agents' unrelated pip package ko pick kar sakta hai (false positive)
#   (c) FRAMEWORK_DIR se import → sys.modules cache → cell 2 ka pip-installed version SHADOW
# Preflight sirf structure verify karta hai; actual import test Cell 2 (post-install) mein.
if os.path.isdir(FRAMEWORK_DIR):
    try:
        _fw_entries = sorted(os.listdir(FRAMEWORK_DIR))
        print(f"OK   FRAMEWORK_DIR structure: {_fw_entries[:10]}"
              + (" ..." if len(_fw_entries) > 10 else ""))
    except Exception as e:
        print(f"WARN FRAMEWORK_DIR listing failed: {e}")
else:
    print(f"FAIL FRAMEWORK_DIR exist nahi karta: {FRAMEWORK_DIR}")

# BUG-3 FIX: env vars section — sirf MAX_ACTIONS rakha (verified = 80).
# GAME_TIME_LIMIT_S / MESSAGE_LIMIT expected values unverified the — hata diye.
_ma = os.environ.get("MAX_ACTIONS")
if _ma is None:
    print("INFO env MAX_ACTIONS: <unset> — framework default 80 use hoga")
else:
    try:
        print(f"INFO env MAX_ACTIONS: {int(_ma)} (framework cap = 80)")
    except ValueError:
        print(f"WARN env MAX_ACTIONS: '{_ma}' non-numeric — int() fail hoga")

# ---------- Parquet deps + roundtrip ----------
_parquet_deps_ok = True
for mod in ("pandas", "pyarrow"):
    v, src = _mod_version(mod)
    if v:
        print(f"OK   {mod} {v}")
    else:
        print(f"WARN {mod} missing — bootstrap cell install karega (submission.parquet ke liye)")
        _parquet_deps_ok = False

_parquet_roundtrip_ok = False
if _parquet_deps_ok:
    _test_path = "/kaggle/working/.preflight_write_test.parquet"
    try:
        import pandas as pd
        _test_df = pd.DataFrame({"a": [1, 2, 3], "b": ["x", "y", "z"]})
        _test_df.to_parquet(_test_path, index=False)
        _back = pd.read_parquet(_test_path)
        assert _back.shape == (3, 2), f"roundtrip shape mismatch: {_back.shape}"
        _parquet_roundtrip_ok = True
        print("OK   parquet write+read roundtrip PASSED")
    except Exception as e:
        print(f"FAIL parquet write test: {e}")
    finally:
        try:
            if os.path.exists(_test_path):
                os.remove(_test_path)
        except Exception:
            pass

# ---------- Disk ----------
try:
    du = shutil.disk_usage("/kaggle/working")
    print(f"OK   disk /kaggle/working: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")
except Exception as e:
    print(f"WARN disk check failed: {e}")

# ---------- Final summary ----------
# BUG-1 FIX: 'Inputs attached' fact add; COSMETIC FIX: Host RAM check ab >= 60
print("\n=== LOCKED-FACTS COMPLIANCE ===")
_facts = [
    ("Inputs attached",          all(checks)),
    ("Python 3.12",              sys.version_info[:2] == (3, 12)),
    ("Working writable",         os.access("/kaggle/working", os.W_OK)),
    ("Parquet deps",             _parquet_deps_ok),
    ("Parquet roundtrip",        _parquet_roundtrip_ok),
    ("GPU = RTX PRO 6000 96GB",  vram_gb is not None and 90 <= vram_gb <= 100),
    ("Host RAM >= 60 GB",        ram_gb is not None and ram_gb >= 60),
]
for name, ok in _facts:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

# ARC framework reachability + symbols ab Cell 2 verify karega (post pip-install)
print("INFO ARC framework/symbol import test Cell 2 mein hoga (post-install)")

In [ ]:
# Cell 2 v4: Offline installs (verified spec) + games cache + env vars
# Verified spec (DIR/067 W-4): vllm + httpx (bundle) + arc-agi
import subprocess, os, shutil, sys, glob, importlib

# ---------- Offline env vars ----------
# vLLM subprocesses tak propagate honge — real improvement
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("OK   offline env vars set")

# ---------- Resolve ALL wheels sources ----------
WHEELS_SOURCES = []
if 'WHEELS' in globals() and os.path.isdir(WHEELS):
    WHEELS_SOURCES.append(WHEELS)
WHEELS_SOURCES += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/wheels")
WHEELS_SOURCES += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels")
WHEELS_SOURCES += glob.glob("/kaggle/input/goa-vllm-wheels/wheels")
WHEELS_SOURCES += glob.glob("/kaggle/input/*/goa-vllm-wheels/wheels")
WHEELS_SOURCES = list(dict.fromkeys(WHEELS_SOURCES))  # dedupe, preserve order
WHEELS_SOURCES = [d for d in WHEELS_SOURCES if os.path.isdir(d)]
print(f"OK   wheel sources ({len(WHEELS_SOURCES)}): {WHEELS_SOURCES}")
if not WHEELS_SOURCES:
    raise RuntimeError("Koi wheels source nahi mila")

# Inventory: debug aid
_all_wheels = []
for d in WHEELS_SOURCES:
    _all_wheels += glob.glob(os.path.join(d, "*.whl"))
print(f"INFO total wheel files: {len(_all_wheels)}")

# ---------- BUG-D FIX: verified spec only ----------
# Verified: vllm + httpx (bundle) + arc-agi + python-dotenv (bundle).
# transformers/safetensors/tokenizers/accelerate vllm ke transitive deps hain — auto aayenge.
# 'arcengine' pip se framework ke engine ko shadow karega — REMOVED.
# 'flask', 'verifiers' bundle mein nahi hain → bulk-FAIL trigger karte the — REMOVED.
PACKAGES_WANTED = ["vllm", "httpx", "arc-agi", "python-dotenv"]

_find_links_args = []
for d in WHEELS_SOURCES:
    _find_links_args += ["--find-links", d]

# ---------- Bulk install ----------
def _pip_install(pkgs):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-index", *_find_links_args, *pkgs],
        capture_output=True, text=True, check=False,
    )

print(f"INFO installing {len(PACKAGES_WANTED)} packages: {PACKAGES_WANTED}")
r = _pip_install(PACKAGES_WANTED)

if r.returncode == 0:
    print("OK   bulk install complete")
else:
    print(f"WARN bulk install exit={r.returncode} — per-package fallback chalu")
    print("  STDOUT tail:", r.stdout[-800:])
    print("  STDERR tail:", r.stderr[-800:])
    _ok, _fail = [], []
    for pkg in PACKAGES_WANTED:
        rr = _pip_install([pkg])
        if rr.returncode == 0:
            _ok.append(pkg)
            print(f"  OK   {pkg}")
        else:
            _fail.append(pkg)
            _err = rr.stderr.strip().splitlines()[-1] if rr.stderr.strip() else "?"
            print(f"  FAIL {pkg} — {_err}")
    print(f"\nSummary: {len(_ok)} installed, {len(_fail)} failed")
    if _fail:
        print(f"WARN ye packages install nahi hue: {_fail}")

# ---------- BUG-C FIX: vllm kernel import avoid ----------
# vllm kernel import libcudart error se FAIL hota hai (verified).
# Subprocess check Cell 3 (bootstrap v5) mein authoritative hai — wahan verify hoga.
_verify_required = ["pandas", "pyarrow", "httpx"]  # submission + HTTP — must-have
_failed_required = []
for mod in _verify_required:
    try:
        _m = importlib.import_module(mod)
        print(f"OK   {mod} {getattr(_m, '__version__', '?')}")
    except ImportError as e:
        print(f"FAIL {mod} (required): {e}")
        _failed_required.append(mod)
if _failed_required:
    raise RuntimeError(f"Required imports fail: {_failed_required}")

# vllm installed hai ya nahi — pip metadata se check (kernel import nahi)
try:
    _vllm_v = importlib.metadata.version("vllm")
    print(f"OK   vllm {_vllm_v} (metadata only — kernel import Cell 3 subprocess mein)")
except Exception:
    print("WARN vllm metadata nahi mila — bootstrap cell check karega")

# ---------- BUG-B FIX: sirf 'arc_agi' import (koi fallback nahi) ----------
# Verified: import name poori chain mein sirf 'arc_agi' hai.
# Fallbacks ('agents', 'arc_agi_agents') unrelated pip packages ko pick kar sakte the
# → false-positive "OK ARC package". Wahi risk jo preflight v6 BUG-2b tha.
try:
    _arc_module = importlib.import_module("arc_agi")
    _v = getattr(_arc_module, "__version__", None)
    print(f"OK   arc_agi @ {getattr(_arc_module, '__file__', '?')}"
          + (f" (v{_v})" if _v else ""))
except ImportError as e:
    # Fallback: FRAMEWORK_DIR ko sys.path mein daal ke try karo (framework source path)
    _fw = globals().get("FRAMEWORK_DIR")
    if _fw and os.path.isdir(_fw):
        if _fw not in sys.path:
            sys.path.insert(0, _fw)
        try:
            _arc_module = importlib.import_module("arc_agi")
            print(f"OK   arc_agi @ {getattr(_arc_module, '__file__', '?')} (via FRAMEWORK_DIR)")
        except ImportError as e2:
            raise RuntimeError(f"arc_agi import fail (pip + framework dir dono se): {e2}")
    else:
        raise RuntimeError(f"arc_agi import fail: {e}")

# ---------- Games cache ----------
# BUG-A FIX: hard gate hata diya. Asli verified gate run cell ka `assert len(game_ids) == 25` hai.
# Public set = 25 games (verified). environment_files ka file-count games se alag metric hai
# (per-game multiple files) — koi verified number nahi. Isliye yahan sirf WARN/INFO.
CACHE_DIR = "/kaggle/working/environment_files"

if os.path.exists(CACHE_DIR):
    try:
        _n = len(os.listdir(CACHE_DIR))
    except Exception:
        _n = 0
    # Incomplete-cache re-copy logic rakha — bas hard number-gate nahi
    if _n == 0:
        print(f"WARN cache exists but empty (0 entries) — re-copy")
        shutil.rmtree(CACHE_DIR, ignore_errors=True)
        shutil.copytree(ENVFILES, CACHE_DIR)
    else:
        print(f"OK   cache exists ({_n} entries)")
else:
    print(f"INFO copying env files → {CACHE_DIR} (may take minutes)...")
    shutil.copytree(ENVFILES, CACHE_DIR)

_n = len(os.listdir(CACHE_DIR))
if _n == 0:
    raise RuntimeError(f"games cache empty after copy — ENVFILES check karo: {ENVFILES}")
print(f"OK   games cached: {_n} entries (game-count verify run cell karega: assert == 25)")

_free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"{'WARN' if _free_gb < 5 else 'OK'}   disk free after copy: {_free_gb:.1f} GB")

# ---------- Working dirs ----------
for d in ("/kaggle/working/goa/agent", "/kaggle/working/logs"):
    try:
        os.makedirs(d, exist_ok=True)
        print(f"OK   mkdir {d}")
    except Exception as e:
        print(f"FAIL mkdir {d}: {e}")
        raise

print("OK   Cell 2 complete — ready for vLLM bootstrap")

In [ ]:
# vLLM BOOTSTRAP v7: skip-if-installed + exact vllm wheel match + LD_LIBRARY_PATH propagated

import glob, subprocess, sys, os, shutil
import importlib.metadata as _im

_free = shutil.disk_usage("/").free / 1e9
if _free < 5:
    raise SystemExit(f"Disk free {_free:.1f} GB < 5 GB — vLLM install fail ho sakta hai")
print(f"OK   disk free: {_free:.1f} GB")

_vllm_preinstalled = None
try:
    _vllm_preinstalled = _im.version("vllm")
    print(f"OK   vllm {_vllm_preinstalled} already installed — install skip")
except Exception:
    print("INFO vllm not installed yet — wheels se install hoga")

_cands = []
if 'WHEELS_SOURCES' in globals():
    _cands += list(WHEELS_SOURCES)
_cands += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/wheels")
_cands += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels")
_cands += glob.glob("/kaggle/input/datasets/*/wheels/wheels")
_cands += glob.glob("/kaggle/input/goa-vllm-wheels/wheels")
_cands += glob.glob("/kaggle/input/*/goa-vllm-wheels/wheels")
_cands = list(dict.fromkeys(_cands))

_valid = []
for d in _cands:
    if not os.path.isdir(d):
        continue
    try:
        whls = glob.glob(os.path.join(d, "*.whl"))
        vllm_whls = [w for w in whls if os.path.basename(w).lower().startswith("vllm-")]
        if vllm_whls:
            _valid.append((d, len(whls), len(vllm_whls), vllm_whls))
    except Exception:
        pass

if _vllm_preinstalled is None:
    if _valid:
        _valid.sort(key=lambda x: (-x[2], -x[1]))
        W = _valid[0][0]
        _vllm_wheel = os.path.basename(sorted(_valid[0][3])[0])
        print(f"OK   wheels: {W} ({_valid[0][1]} total, {_valid[0][2]} vllm)")
        print(f"     picked: {_vllm_wheel}")

        _vllm_ver = None
        _parts = _vllm_wheel.split("-")
        if len(_parts) >= 2 and _parts[0].lower() == "vllm":
            _vllm_ver = _parts[1]
        _pin = f"vllm=={_vllm_ver}" if _vllm_ver else "vllm"
        print(f"INFO pip install {_pin} httpx (LIVE, 2-5 min):", flush=True)

        rc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index",
             "--find-links", W, _pin, "httpx"],
        ).returncode

        if rc != 0 and _vllm_ver:
            print(f"WARN pinned install fail (rc={rc}) — unpinned retry:", flush=True)
            rc = subprocess.run(
                [sys.executable, "-m", "pip", "install", "--no-index",
                 "--find-links", W, "vllm", "httpx"],
            ).returncode

        if rc != 0:
            raise SystemExit(
                "pip install FAIL — torch mismatch (S-3)? ya wheels incomplete. "
                f"Wheels dir: {W}"
            )
        print("OK   pip install complete")
    else:
        print("WARN wheels dataset nahi mila (koi candidate mein 'vllm-*.whl' nahi tha)")
        print(f"     checked dirs: {_cands[:5]}{' ...' if len(_cands) > 5 else ''}")
        print("     preinstalled vLLM pe bharosa — missing ho to subprocess import fail karega")

try:
    _hv = _im.version("httpx")
    print(f"OK   httpx {_hv}")
except Exception as e:
    raise SystemExit(f"httpx missing — wheels dataset attach karo: {e}")

import sysconfig, site
_site_dirs = set()
for _p in ("purelib", "platlib"):
    _d = sysconfig.get_paths().get(_p)
    if _d and os.path.isdir(_d):
        _site_dirs.add(_d)
try:
    for _d in site.getsitepackages():
        if _d and os.path.isdir(_d):
            _site_dirs.add(_d)
except Exception:
    pass

_libs = []
for base in _site_dirs:
    _libs += glob.glob(f"{base}/nvidia/*/lib")
    _libs += glob.glob(f"{base}/cuda/*/lib")
_libs += glob.glob("/usr/local/cuda*/lib64")
_libs += glob.glob("/usr/local/cuda*/lib")
_libs = list(dict.fromkeys(_libs))

print(f"INFO site dirs: {sorted(_site_dirs)}")
print(f"INFO nvidia/cuda lib dirs found: {len(_libs)}")
if not _libs:
    print("WARN koi nvidia/cuda lib dir nahi mila — subprocess import fail ho sakta hai")

_existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
_new_ld = ":".join(_libs) + (":" + _existing_ld if _existing_ld else "")
os.environ["LD_LIBRARY_PATH"] = _new_ld
_env = dict(os.environ)

_r = subprocess.run(
    [sys.executable, "-c", "import vllm; print(vllm.__version__)"],
    env=_env, capture_output=True, text=True,
)
if _r.returncode == 0:
    print(f"OK   vLLM ready (subprocess): {_r.stdout.strip()}")
else:
    print("FAIL vllm subprocess import")
    print("  LD_LIBRARY_PATH:", _new_ld[:500] + ("..." if len(_new_ld) > 500 else ""))
    print("  STDOUT:", _r.stdout.strip() or "(empty)")
    print("  STDERR tail:", _r.stderr.strip()[-1200:] if _r.stderr.strip() else "(empty)")
    raise SystemExit("vllm subprocess import FAIL — upar ka error dekho")

print("OK   BOOTSTRAP complete: vllm (subprocess) + httpx ready")


In [ ]:
%%writefile goa/agent/context_manager.py
"""agent/context_manager.py — 3-Zone Context Manager + Grid Diff Encoder. v1.6.

v1.6 — Martial-law core audit (Chief order, 2026-09-30): (1) OBJECTS
        rendering — set_observation(objects=...) structured object list
        (id/color/size/bbox) observation mein render karta hai, taaki model
        ko RLE rows se column-arithmetic karke target nikalna na pade.
        (2) _obs_fallback dual-format — grid_text ka "char+count" RLE
        ("W8") pehle NONZERO_CELLS=<malformed> deta tha; ab dono formats
        chalete hain. + T-29/T-30.

INTEGRATED BUILD — module code plus an inline self-test suite under
``if __name__ == "__main__":``.  Import this file as a normal module; run
it directly to execute the full v1.4 test battery.

Pure data-structure module.  No LLM calls, no network, no I/O, no threads,
no globals mutated across instances.  Stdlib only.

Design (per DIR/2026-028, research facts F-1..F-7): full grids are too
big for the window (F-1) so diffs are used; static-first ordering (F-2,
F-4); ~2400-token working target is a quality decision (F-3); Zone 2 is
written by CODER output routed through the harness, the Reasoner never
writes it (F-5); warn at 70%, flush at 100% (F-6); silent truncation is
the worst failure mode (F-7) — fail loud.

Per-zone token caps (``zone1_max_tokens``, ``zone2_max_tokens``,
``zone3_max_tokens``, ``observation_max_tokens``) are ADVISORY: exceeding
them logs a WARNING at build time but does not raise.  The sole hard
enforcement gate is ``total_hard_cap``, which runs the eviction cascade
and raises ``ContextOverflowError`` if the packet cannot be reduced.
Zone 2 additionally enforces four per-field sub-caps via
``_ZONE2_FIELD_CAPS`` (see ``update_zone2``).

JSON snapshot encoding
----------------------
``snapshot()`` returns a dict that ``json.dumps`` can serialise.  Values
that JSON cannot represent natively are wrapped in single-key marker
dicts (``__set__``, ``__frozenset__``, ``__bytes__``, ``__tuple__``).
``load_snapshot()`` inverts these via ``_json_restore()`` so sets,
frozensets, bytes, and tuples round-trip.  Reserved marker keys: a user
dict with exactly one key equal to one of those markers will be
misinterpreted.  No such dict is expected in the ARC-AGI-3
action/observation domain.

Changelog
---------
v1.0  Initial (DIR/2026-028).
v1.1  Post-audit fixes (F-01..F-06 first independent audit).
v1.2  Post-v1.1 audit fixes (F-01..F-05 v1.1 audit).
v1.3  Post-v1.2 audit fixes (F-01..F-03 v1.2 audit) + inline test suite.
        F-01  _rle_decode_rows validates 0 <= value <= GRID_MAX_VALUE.
        F-02  apply_delta verifies ch["old"] against current cell.
        F-03  Per-zone advisory caps emit WARNINGs at build time.
        INT   Self-test suite integrated under __main__.
v1.5  Per DIR/2026-068 (Memory & Coder Awakening, Chief directive):
        Field caps raised after Run #13 evidence — reasoner ~250-450
        tokens likhta tha, sirf 80 yaad rehte the (memory starvation,
        81-action blind reflex). confirmed_rules 120->200,
        hypotheses 80->350, world_model_status 60->200,
        goal_notes 40->100. Zone budgets / total_hard_cap module
        defaults UNCHANGED (runtime injection ke through badhte hain,
        T-suite determinism ke liye). T-3 version-proof ban gaya
        (cap constant se derive hota hai). Server budget: actual
        context_window 16384; truncation ab bhi advisory-warning only.
v1.4  Per DIR/2026-030:
        T-19 tuple roundtrip: __tuple__ marker added; contract and test
        aligned.  Previously tuples were serialised to lists and not
        restored, so an (x, y) tuple in available_actions rendered as
        [x, y] after a mid-run snapshot/restore — the Reasoner saw a
        different prompt than it had before.  With the marker, tuple
        round-trip is byte-stable.  Reserved marker count is now four.
"""
from __future__ import annotations

import hashlib
import json
import logging
import random
from collections import deque
from dataclasses import dataclass
from typing import Any, Callable, Deque, Dict, List, Optional, Tuple

logger = logging.getLogger(__name__)

GRID_SIZE = 64
GRID_MAX_VALUE = 15
DEFAULT_REFRESH_EVERY = 10
MAX_DELTA_LIST = 400
_MAX_SUMMARY_LINES = 20
_MAX_DEAD_SIGS_PER_LEVEL = 200
__version__ = "1.6"
_MAX_FALLBACK_CELLS = 500
# v1.6: OBJECTS rendering cap (observation budget protection)
_MAX_OBS_OBJECTS = 24
# v1.6: grid_text "char+count" RLE decode table — PALETTE order (digits 0..15).
# my_agent T-suite isko grid_text.CHAR_TO_DIGIT ke against cross-check
# karta hai (integration drift guard).
_FALLBACK_PALETTE = "WwgGcBMPRbSYOrNp"
_FALLBACK_CHAR_TO_DIGIT = {c: i for i, c in enumerate(_FALLBACK_PALETTE)}

_ZONE2_FIELD_CAPS: Dict[str, int] = {
    "confirmed_rules": 200,
    "hypotheses": 350,
    "world_model_status": 200,
    "goal_notes": 100,
}
_ZONE2_FIELDS: Tuple[str, ...] = tuple(_ZONE2_FIELD_CAPS.keys())

__all__ = [
    "GridDiffEncoder", "ContextManager", "PromptPacket",
    "ContextOverflowError", "Zone2OverflowError", "default_token_counter",
    "GRID_SIZE",
]


def default_token_counter(text: str) -> int:
    """Conservative upper bound: UTF-8 byte count.  Never undercounts."""
    return len(text.encode("utf-8"))


class ContextOverflowError(RuntimeError):
    """Raised when the packet cannot fit under total_hard_cap even after
    the full eviction cascade."""


class Zone2OverflowError(ValueError):
    """Raised when a Zone 2 update exceeds its per-field cap and the
    caller did not opt into truncation via ``truncate=True``."""


def _json_safe(v: Any) -> Any:
    """Coerce ``v`` to a JSON-serialisable form, recursively.

    Sets/frozensets/bytes/tuples are wrapped in single-key marker dicts
    so they survive round-trip; other objects become reprs.
    """
    if v is None or isinstance(v, (bool, int, float, str)):
        return v
    if isinstance(v, list):
        return [_json_safe(x) for x in v]
    if isinstance(v, tuple):
        return {"__tuple__": [_json_safe(x) for x in v]}
    if isinstance(v, dict):
        return {str(k): _json_safe(val) for k, val in v.items()}
    if isinstance(v, frozenset):
        return {"__frozenset__":
                sorted((_json_safe(x) for x in v), key=repr)}
    if isinstance(v, set):
        return {"__set__": sorted((_json_safe(x) for x in v), key=repr)}
    if isinstance(v, bytes):
        return {"__bytes__": v.hex()}
    return repr(v)


def _json_restore(v: Any) -> Any:
    """Inverse of :func:`_json_safe`.

    Marker dicts (``__set__`` / ``__frozenset__`` / ``__bytes__`` /
    ``__tuple__``) are decoded back into their original Python types.
    Ordinary dicts and lists are recursed.
    """
    if isinstance(v, dict):
        if set(v.keys()) == {"__set__"}:
            return set(_json_restore(x) for x in v["__set__"])
        if set(v.keys()) == {"__frozenset__"}:
            return frozenset(_json_restore(x) for x in v["__frozenset__"])
        if set(v.keys()) == {"__bytes__"}:
            return bytes.fromhex(v["__bytes__"])
        if set(v.keys()) == {"__tuple__"}:
            return tuple(_json_restore(x) for x in v["__tuple__"])
        return {k: _json_restore(val) for k, val in v.items()}
    if isinstance(v, list):
        return [_json_restore(x) for x in v]
    return v


# --------------------------------------------------------------------------- #
# GridDiffEncoder
# --------------------------------------------------------------------------- #

class GridDiffEncoder:
    """Compact encoder for 64x64 integer grids (values 0-15).

    Full encoding: one row per list element, each row RLE-encoded as
    ``<value_hex>:<count_decimal>`` pairs joined by ``,``.  Delta
    encoding: a list of ``{"r","c","old","new"}`` cell changes plus
    base_checksum, new checksum, new version.

    ``apply_delta`` is atomic: all changes are staged on a candidate
    copy and committed only after every change validates (coordinate
    range, value range, old-value match) and the post-apply checksum
    matches.  On any rejection the internal grid is left byte-identical
    and ``needs_refresh`` is set True.

    ``decode`` is a reconstruction routine.  It always validates
    coordinate range and value range (both in the full RLE path and in
    every delta change).  Pass ``verify=True`` to also verify base and
    post-delta checksums against the recorded checksum fields.
    """

    def __init__(self, refresh_every: int = DEFAULT_REFRESH_EVERY) -> None:
        self.refresh_every = int(refresh_every)
        self.needs_refresh: bool = True
        self._version: int = 0
        self._frame_count: int = 0
        self._delta_count: int = 0
        self._checksum: Optional[str] = None
        self._grid: Optional[List[List[int]]] = None

    def encode_full(self, grid: List[List[int]]) -> Dict[str, Any]:
        """Encode ``grid`` as a full snapshot; clears refresh state."""
        self._validate_grid(grid)
        cs = self._compute_checksum(grid)
        self._version += 1
        self._checksum = cs
        self._grid = [list(r) for r in grid]
        self._frame_count = 0
        self._delta_count = 0
        self.needs_refresh = False
        return {"type": "full", "version": self._version,
                "checksum": cs, "rows": [self._rle_row(r) for r in grid]}

    def encode_delta(self, old_grid: List[List[int]],
                     new_grid: List[List[int]]) -> Dict[str, Any]:
        """Encode the cell-level diff between two same-shape grids."""
        self._validate_grid(old_grid)
        self._validate_grid(new_grid)
        changes: List[Dict[str, int]] = []
        for r in range(GRID_SIZE):
            orow = old_grid[r]
            nrow = new_grid[r]
            for c in range(GRID_SIZE):
                if orow[c] != nrow[c]:
                    changes.append({"r": r, "c": c,
                                    "old": orow[c], "new": nrow[c]})
        base_cs = self._compute_checksum(old_grid)
        new_cs = self._compute_checksum(new_grid)
        self._version += 1
        self._checksum = new_cs
        self._grid = [list(r) for r in new_grid]
        self._delta_count += 1
        if self._delta_count > MAX_DELTA_LIST:
            self.needs_refresh = True
        return {"type": "delta", "version": self._version,
                "base_checksum": base_cs, "checksum": new_cs,
                "changes": changes}

    def apply_delta(self, delta: Dict[str, Any]) -> bool:
        """Apply a delta to internal state; return False on any failure.

        Atomic.  Validates base checksum, coordinate range, cell-value
        range, old-value match (F-02 v1.3), and post-apply checksum.
        On any rejection the internal grid is left byte-identical.
        """
        if self.needs_refresh:
            return False
        if delta.get("type") != "delta":
            self.needs_refresh = True
            return False
        if delta.get("base_checksum") != self._checksum:
            logger.warning("grid drift: base checksum mismatch")
            self.needs_refresh = True
            return False
        if self._grid is None:
            self.needs_refresh = True
            return False

        candidate = [row[:] for row in self._grid]
        for ch in delta.get("changes", []):
            r = int(ch["r"])
            c = int(ch["c"])
            nv = int(ch["new"])
            if not (0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE):
                logger.warning(
                    "grid drift: change out of bounds (%d,%d)", r, c)
                self.needs_refresh = True
                return False
            if not (0 <= nv <= GRID_MAX_VALUE):
                logger.warning(
                    "grid drift: change value out of range %d", nv)
                self.needs_refresh = True
                return False
            # F-02 (v1.3): verify the delta's stated old value matches reality.
            ov = int(ch["old"])
            if candidate[r][c] != ov:
                logger.warning(
                    "grid drift: old value mismatch at (%d,%d): "
                    "delta says %d, grid has %d",
                    r, c, ov, candidate[r][c])
                self.needs_refresh = True
                return False
            candidate[r][c] = nv

        if self._compute_checksum(candidate) != delta.get("checksum"):
            logger.warning("grid drift: post-apply checksum mismatch")
            self.needs_refresh = True
            return False

        self._grid = candidate
        self._checksum = delta["checksum"]
        self._version = int(delta["version"])
        self._delta_count += 1
        if self._delta_count > MAX_DELTA_LIST:
            self.needs_refresh = True
        return True

    @classmethod
    def decode(cls, state: Dict[str, Any], verify: bool = False
               ) -> List[List[int]]:
        """Reconstruct a 64x64 grid from full or {full, deltas} bundle.

        Always validates coordinate and value range (including the
        values embedded in the full RLE).  ``verify=True`` additionally
        checks base and post-delta checksums.
        """
        if "full" in state:
            full = state["full"]
            grid = cls._rle_decode_rows(full["rows"])
            if verify and cls._compute_checksum(grid) != full.get("checksum"):
                raise ValueError("decode: full checksum mismatch")
            for d in state.get("deltas", []):
                if verify:
                    if cls._compute_checksum(grid) != d.get("base_checksum"):
                        raise ValueError("decode: base checksum mismatch")
                for ch in d.get("changes", []):
                    r = int(ch["r"])
                    c = int(ch["c"])
                    nv = int(ch["new"])
                    if not (0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE):
                        raise ValueError(
                            f"decode: coordinate out of range ({r},{c})")
                    if not (0 <= nv <= GRID_MAX_VALUE):
                        raise ValueError(
                            f"decode: value out of range ({nv})")
                    grid[r][c] = nv
                if verify and cls._compute_checksum(grid) != d.get("checksum"):
                    raise ValueError("decode: post-delta checksum mismatch")
            return grid
        if state.get("type") == "full":
            grid = cls._rle_decode_rows(state["rows"])
            if verify and cls._compute_checksum(grid) != state.get("checksum"):
                raise ValueError("decode: full checksum mismatch")
            return grid
        raise ValueError(
            "decode expects a full-encoding dict or a {full,deltas} bundle")

    def mark_new_game(self) -> None:
        self.needs_refresh = True

    def mark_new_level(self) -> None:
        self.needs_refresh = True

    def mark_death(self) -> None:
        self.needs_refresh = True

    def tick(self) -> None:
        self._frame_count += 1
        if self.refresh_every > 0 and self._frame_count >= self.refresh_every:
            self.needs_refresh = True

    def snapshot_state(self) -> Dict[str, Any]:
        return {"refresh_every": self.refresh_every,
                "needs_refresh": self.needs_refresh,
                "version": self._version,
                "frame_count": self._frame_count,
                "delta_count": self._delta_count,
                "checksum": self._checksum,
                "grid": self._grid}

    def load_state(self, snap: Dict[str, Any]) -> None:
        self.refresh_every = int(snap.get("refresh_every",
                                          DEFAULT_REFRESH_EVERY))
        self.needs_refresh = bool(snap.get("needs_refresh", True))
        self._version = int(snap.get("version", 0))
        self._frame_count = int(snap.get("frame_count", 0))
        self._delta_count = int(snap.get("delta_count", 0))
        self._checksum = snap.get("checksum")
        grid = snap.get("grid")
        self._grid = [list(r) for r in grid] if grid else None

    @staticmethod
    def _validate_grid(grid: List[List[int]]) -> None:
        if len(grid) != GRID_SIZE:
            raise ValueError(f"grid must have {GRID_SIZE} rows")
        for r, row in enumerate(grid):
            if len(row) != GRID_SIZE:
                raise ValueError(f"row {r}: must have {GRID_SIZE} cells")
            for c, v in enumerate(row):
                if not isinstance(v, int) or not (0 <= v <= GRID_MAX_VALUE):
                    raise ValueError(
                        f"cell ({r},{c}) = {v!r} not an int 0-{GRID_MAX_VALUE}")

    @staticmethod
    def _compute_checksum(grid: List[List[int]]) -> str:
        h = hashlib.md5()
        for row in grid:
            h.update(bytes(row))
        return h.hexdigest()

    @staticmethod
    def _rle_row(row: List[int]) -> str:
        parts: List[str] = []
        cur = row[0]
        n = 1
        for v in row[1:]:
            if v == cur:
                n += 1
            else:
                parts.append(f"{cur:x}:{n}")
                cur = v
                n = 1
        parts.append(f"{cur:x}:{n}")
        return ",".join(parts)

    @staticmethod
    def _rle_decode_rows(rows: List[str]) -> List[List[int]]:
        grid: List[List[int]] = []
        for s in rows:
            out: List[int] = []
            for chunk in s.split(","):
                if not chunk:
                    continue
                vhex, sep, cdec = chunk.partition(":")
                if not sep:
                    raise ValueError(f"malformed RLE chunk: {chunk!r}")
                v = int(vhex, 16)
                n = int(cdec)
                # F-01 (v1.3): enforce the grid value domain in the full path.
                if not (0 <= v <= GRID_MAX_VALUE):
                    raise ValueError(
                        f"RLE value out of range: {v} (chunk {chunk!r})")
                out.extend([v] * n)
            if len(out) != GRID_SIZE:
                raise ValueError("decoded row wrong length")
            grid.append(out)
        if len(grid) != GRID_SIZE:
            raise ValueError("decoded grid wrong number of rows")
        return grid


# --------------------------------------------------------------------------- #
# PromptPacket
# --------------------------------------------------------------------------- #

@dataclass
class PromptPacket:
    """One assembled prompt for the inference layer."""

    text: str
    token_estimate: int
    per_zone_tokens: Dict[str, int]
    needs_refresh: bool
    version: int


# --------------------------------------------------------------------------- #
# ContextManager
# --------------------------------------------------------------------------- #

class ContextManager:
    """Three-zone context manager for the ARC-AGI-3 agent loop.

    Zone 1 (system) set once, never evicted.  Zone 2 is per-game
    knowledge: four named fields, each with its own token cap, written
    WHOLESALE.  Zone 3 is a rolling FIFO of the last ten actions plus a
    bounded summary slot.  Observation carries the grid encoding,
    available actions, levels completed, and game state.

    ``build_prompt`` counts tokens BEFORE returning; if the total exceeds
    ``total_hard_cap`` it runs an eviction cascade: (1) drop oldest Zone 3
    entries, (2) force observation coordinate-only fallback, (3) drop
    Zone 2 hypotheses then goal_notes.  If still over cap, raises
    ``ContextOverflowError`` — never silently truncates.

    Per-zone caps are ADVISORY: exceeding them logs a WARNING at build
    time but does not raise.  The sole hard gate is ``total_hard_cap``.
    Zone 2 additionally enforces four per-field sub-caps via
    ``_ZONE2_FIELD_CAPS``.

    Contract (F-5): Zone 2 is written via update_zone2 / promote_to_zone2
    calls the HARNESS routes from CODER output.
    """

    def __init__(
        self,
        zone1_max_tokens: int = 200,
        zone2_max_tokens: int = 300,
        zone3_max_tokens: int = 200,
        observation_max_tokens: int = 2500,
        total_hard_cap: int = 7800,
        refresh_every: int = DEFAULT_REFRESH_EVERY,
        token_counter: Optional[Callable[[str], int]] = None,
    ) -> None:
        self.zone1_max_tokens = int(zone1_max_tokens)
        self.zone2_max_tokens = int(zone2_max_tokens)
        self.zone3_max_tokens = int(zone3_max_tokens)
        self.observation_max_tokens = int(observation_max_tokens)
        self.total_hard_cap = int(total_hard_cap)
        self.refresh_every = int(refresh_every)
        self._tc: Callable[[str], int] = token_counter or default_token_counter

        self._zone1: str = ""
        self._zone2: Dict[str, str] = {f: "" for f in _ZONE2_FIELDS}
        self._zone3_summary: Deque[str] = deque(maxlen=_MAX_SUMMARY_LINES)
        self._zone3_entries: Deque[Dict[str, Any]] = deque(maxlen=10)

        self._obs_encoding: Optional[Dict[str, Any]] = None
        self._obs_actions: List[Any] = []
        self._obs_levels: int = 0
        self._obs_state: str = "unknown"
        self._obs_objects: List[Any] = []
        self._obs_force_fallback: bool = False
        self._obs_fallback_truncated: int = 0

        self._dead_sigs: Deque[str] = deque(maxlen=_MAX_DEAD_SIGS_PER_LEVEL)
        self._level_num: int = 0

        self._build_version: int = 0
        self._last_total_tokens: int = 0
        self.needs_refresh: bool = True

    # ---- Zone 1 ---- #

    def set_system_prompt(self, text: str) -> None:
        """Set Zone 1.  Never evicted, never rewritten by build_prompt."""
        self._zone1 = text or ""
        t = self._tc(self._zone1)
        if t > self.zone1_max_tokens:
            logger.warning(
                "zone1 set to %d tokens (declared cap %d); zone1 is "
                "never evicted and this may force overflow at build time",
                t, self.zone1_max_tokens)

    # ---- Zone 2 ---- #

    def update_zone2(self, field: str, new_text: str,
                     truncate: bool = False) -> None:
        """Wholesale-replace a Zone 2 field; enforce its per-field cap.

        Raises ``Zone2OverflowError`` by default.  Pass ``truncate=True``
        to opt into prefix-keep truncation (logs a WARNING).
        """
        if field not in self._zone2:
            raise ValueError(f"unknown zone2 field: {field!r}")
        cap = _ZONE2_FIELD_CAPS[field]
        text = new_text or ""
        if self._tc(text) <= cap:
            self._zone2[field] = text
            return
        if not truncate:
            raise Zone2OverflowError(
                f"zone2 field {field!r}: {self._tc(text)} tokens > cap "
                f"{cap}; pass truncate=True to accept truncation")
        truncated = self._truncate_prefix_to_cap(text, cap)
        logger.warning(
            "zone2 field %r truncated %d -> %d tokens (cap %d)",
            field, self._tc(text), self._tc(truncated), cap)
        self._zone2[field] = truncated

    def reset_zone2(self) -> None:
        """Clear all four Zone 2 fields (call on game change)."""
        for f in _ZONE2_FIELDS:
            self._zone2[f] = ""

    def promote_to_zone2(self, line: str, field: str,
                         truncate: bool = False) -> None:
        """Promote one distilled line into a Zone 2 field immediately.

        F-02 (v1.2): the ``truncate`` flag is caller-controlled and
        defaults to False — an over-cap composed field raises
        ``Zone2OverflowError`` rather than silently dropping the tail.
        """
        if field not in self._zone2:
            raise ValueError(f"unknown zone2 field: {field!r}")
        existing = self._zone2.get(field, "")
        composed = f"{line}\n{existing}".strip() if existing else line
        self.update_zone2(field, composed, truncate=truncate)

    # ---- Zone 3 ---- #

    def append_action(self, action: Any, result: str,
                      delta_count: int) -> None:
        """Append one executed action to the Zone 3 FIFO (max 10)."""
        dc = int(delta_count)
        self._zone3_entries.append({
            "action": action, "result": str(result),
            "delta_count": dc, "changed": dc > 0})

    def add_summary_line(self, line: str) -> None:
        """Append a distilled failing-attempt summary line.

        The deque has maxlen=_MAX_SUMMARY_LINES; oldest evicted at cap.
        """
        self._zone3_summary.append(str(line))

    # ---- Observation ---- #

    def set_observation(
        self,
        encoding_dict: Optional[Dict[str, Any]],
        available_actions: Optional[List[Any]] = None,
        levels_completed: int = 0,
        state: str = "unknown",
        objects: Optional[List[Any]] = None,
    ) -> None:
        """Set the observation slot (grid encoding + metadata).

        v1.6: ``objects`` = list of dicts {id, color_char, color_digit,
        size, bbox[r0, r1, c0, c1]}.  Observation block mein "OBJECTS:"
        section render hota hai (cap _MAX_OBS_OBJECTS).
        """
        self._obs_encoding = encoding_dict
        self._obs_actions = list(available_actions or [])
        self._obs_objects = list(objects or [])
        self._obs_levels = int(levels_completed)
        self._obs_state = str(state)
        self._obs_force_fallback = False
        self._obs_fallback_truncated = 0

    # ---- Dead signatures ---- #

    def register_dead_signature(self, sig: str) -> None:
        """Record an object signature that never changed the grid.

        FIFO-bounded at _MAX_DEAD_SIGS_PER_LEVEL.
        """
        s = str(sig)
        if s and s not in self._dead_sigs:
            self._dead_sigs.append(s)

    def clear_dead_signatures(self) -> None:
        self._dead_sigs.clear()

    def set_level(self, level_num: int) -> None:
        """Notify a level transition; clears the dead-signature registry."""
        n = int(level_num)
        if n != self._level_num:
            self._dead_sigs.clear()
            self._level_num = n

    # ---- Memory pressure (F-6) ---- #

    def memory_pressure(self) -> str:
        """Return 'ok' / 'warn' / 'flush' from the last build total."""
        if self.total_hard_cap <= 0:
            return "flush"
        r = self._last_total_tokens / self.total_hard_cap
        if r >= 1.0:
            return "flush"
        if r >= 0.7:
            return "warn"
        return "ok"

    # ---- build_prompt + cascade ---- #

    def build_prompt(self) -> PromptPacket:
        """Assemble the packet; enforce total_hard_cap via cascade."""
        text, tokens, total = self._assemble_with_tokens()
        if total > self.total_hard_cap:
            text, tokens, total = self._evict_and_rebuild()
        if total > self.total_hard_cap:
            self._last_total_tokens = total
            self._build_version += 1
            raise ContextOverflowError(
                f"cannot fit under total_hard_cap={self.total_hard_cap}; "
                f"packet={total} tokens")
        self._last_total_tokens = total
        self._build_version += 1
        return PromptPacket(
            text=text, token_estimate=total, per_zone_tokens=tokens,
            needs_refresh=self.needs_refresh, version=self._build_version)

    def _evict_and_rebuild(self) -> Tuple[str, Dict[str, int], int]:
        """4-step eviction cascade; return (text, tokens, total)."""
        _, _, total = self._assemble_with_tokens()
        while total > self.total_hard_cap and len(self._zone3_entries) > 3:
            self._zone3_entries.popleft()
            _, _, total = self._assemble_with_tokens()
        if total > self.total_hard_cap and not self._obs_force_fallback:
            self._obs_force_fallback = True
            _, _, total = self._assemble_with_tokens()
        if total > self.total_hard_cap and self._zone2.get("hypotheses"):
            self._zone2["hypotheses"] = ""
            _, _, total = self._assemble_with_tokens()
        if total > self.total_hard_cap and self._zone2.get("goal_notes"):
            self._zone2["goal_notes"] = ""
            _, _, total = self._assemble_with_tokens()
        return self._assemble_with_tokens()

    def _assemble_with_tokens(self) -> Tuple[str, Dict[str, int], int]:
        sections = (
            ("zone1", "=== SYSTEM ===", self._zone1),
            ("zone2", "=== GAME KNOWLEDGE ===", self._zone2_block()),
            ("zone3", "=== RECENT ACTIONS ===", self._zone3_block()),
            ("observation", "=== OBSERVATION ===", self._obs_block()),
        )
        chunks: List[str] = []
        tokens: Dict[str, int] = {}
        for key, header, body in sections:
            chunk = f"{header}\n{body}"
            chunks.append(chunk)
            tokens[key] = self._tc(chunk)
        text = "\n\n".join(chunks)
        total = self._tc(text)
        sum_zones = sum(tokens.values())
        if total > sum_zones:
            tokens["overhead"] = total - sum_zones

        # F-03 (v1.3): advisory per-zone cap warnings.
        for key, cap in (
            ("zone1", self.zone1_max_tokens),
            ("zone2", self.zone2_max_tokens),
            ("zone3", self.zone3_max_tokens),
            ("observation", self.observation_max_tokens),
        ):
            if tokens.get(key, 0) > cap:
                logger.warning(
                    "zone %s at %d tokens exceeds advisory cap %d",
                    key, tokens[key], cap)

        return text, tokens, total

    def _zone2_block(self) -> str:
        parts: List[str] = []
        for f in _ZONE2_FIELDS:
            v = self._zone2.get(f, "")
            if v:
                parts.append(f"[{f}]\n{v}")
        return "\n".join(parts)

    def _zone3_block(self) -> str:
        parts: List[str] = []
        if self._zone3_summary:
            parts.append("SUMMARY:\n" + "\n".join(self._zone3_summary))
        for e in self._zone3_entries:
            tag = "changed" if e.get("changed") else "no-change"
            parts.append(
                f"- {e.get('action')} -> {e.get('result')} ({tag})")
        return "\n".join(parts)

    def _objects_lines(self) -> List[str]:
        """v1.6: structured OBJECTS rendering — model ko seedha
        rows/cols dikhta hai, RLE column-arithmetic nahi karna padta."""
        out: List[str] = []
        if not self._obs_objects:
            return out
        out.append("OBJECTS (0-indexed rows/cols; ACTION r/c targets):")
        for ob in self._obs_objects[:_MAX_OBS_OBJECTS]:
            try:
                r0, r1, c0, c1 = ob["bbox"]
                out.append(
                    "  [{}] color={} size={} rows={}-{} cols={}-{}".format(
                        ob["id"], ob["color_char"], ob["size"],
                        int(r0), int(r1), int(c0), int(c1)))
            except (KeyError, TypeError, ValueError, IndexError):
                continue
        if len(self._obs_objects) > _MAX_OBS_OBJECTS:
            out.append("  <{} more objects hidden>".format(
                len(self._obs_objects) - _MAX_OBS_OBJECTS))
        return out

    def _obs_block(self) -> str:
        if self._obs_force_fallback:
            return self._obs_fallback()
        lines = [
            f"levels_completed={self._obs_levels}",
            f"state={self._obs_state}",
            f"available_actions={self._obs_actions}",
        ]
        if self._dead_sigs:
            lines.append(f"do_not_retry={list(self._dead_sigs)}")
        lines.extend(self._objects_lines())
        enc = self._obs_encoding
        if enc is None:
            lines.append("grid=none")
        elif enc.get("type") == "full":
            lines.append("GRID_FULL:")
            for i, r in enumerate(enc.get("rows", [])):
                lines.append(f"  {i:02d}|{r}")
        elif enc.get("type") == "delta":
            lines.append(f"GRID_DELTA v{enc.get('version')}:")
            for ch in enc.get("changes", []):
                lines.append(
                    f"  ({ch['r']},{ch['c']}) {ch['old']}->{ch['new']}")
        else:
            lines.append("grid=unknown")
        return "\n".join(lines)

    @staticmethod
    def _decode_rle_chunk(chunk: str):
        """v1.6: dono RLE formats — legacy "hex:dec" ("5:12") aur
        grid_text "char+count" ("W8").  Returns (value, count) ya
        (None, None) on malformed input."""
        if ":" in chunk:
            vhex, _, cdec = chunk.partition(":")
            try:
                return int(vhex, 16), int(cdec)
            except ValueError:
                return None, None
        if len(chunk) < 2 or chunk[0] not in _FALLBACK_CHAR_TO_DIGIT:
            return None, None
        try:
            return _FALLBACK_CHAR_TO_DIGIT[chunk[0]], int(chunk[1:])
        except ValueError:
            return None, None

    def _obs_fallback(self) -> str:
        """Coordinate-only summary.  Handles full and delta encodings."""
        self._obs_fallback_truncated = 0
        lines = [
            f"levels_completed={self._obs_levels}",
            f"state={self._obs_state}",
            f"available_actions={self._obs_actions}",
        ]
        if self._dead_sigs:
            lines.append(f"do_not_retry={list(self._dead_sigs)}")
        lines.extend(self._objects_lines())

        enc = self._obs_encoding or {}
        etype = enc.get("type")

        if etype == "full":
            total_nonzero = 0
            pending: List[str] = []
            for r, s in enumerate(enc.get("rows", [])):
                c = 0
                for chunk in s.split(","):
                    if not chunk:
                        continue
                    v, n = self._decode_rle_chunk(chunk)
                    if v is None:
                        logger.warning(
                            "fallback: malformed RLE chunk %r at row %d",
                            chunk, r)
                        lines.append("NONZERO_CELLS=<malformed>")
                        return "\n".join(lines)
                    if v != 0:
                        for k in range(n):
                            total_nonzero += 1
                            if total_nonzero <= _MAX_FALLBACK_CELLS:
                                pending.append(f"  ({r},{c + k},{v})")
                    c += n
            if total_nonzero == 0:
                lines.append("NONZERO_CELLS=0")
                return "\n".join(lines)
            lines.append("NONZERO_CELLS (r,c,v):")
            lines.extend(pending)
            if total_nonzero > _MAX_FALLBACK_CELLS:
                hidden = total_nonzero - _MAX_FALLBACK_CELLS
                self._obs_fallback_truncated = hidden
                lines.append(
                    f"  <TRUNCATED: {hidden} more non-zero cells>")
                logger.warning(
                    "observation fallback truncated: %d non-zero cells "
                    "hidden (cap %d)", hidden, _MAX_FALLBACK_CELLS)
            return "\n".join(lines)

        if etype == "delta":
            changes = enc.get("changes", [])
            if not changes:
                lines.append("GRID_DELTA (no changes)")
                return "\n".join(lines)
            lines.append(f"GRID_DELTA v{enc.get('version')} "
                         f"({len(changes)} changes):")
            shown = changes[:_MAX_FALLBACK_CELLS]
            for ch in shown:
                lines.append(f"  ({ch['r']},{ch['c']},{ch['new']})")
            if len(changes) > _MAX_FALLBACK_CELLS:
                hidden = len(changes) - _MAX_FALLBACK_CELLS
                self._obs_fallback_truncated = hidden
                lines.append(f"  <TRUNCATED: {hidden} more changes>")
                logger.warning(
                    "observation fallback truncated: %d delta changes "
                    "hidden (cap %d)", hidden, _MAX_FALLBACK_CELLS)
            return "\n".join(lines)

        lines.append("grid=unknown")
        return "\n".join(lines)

    # ---- snapshot / load ---- #

    def snapshot(self) -> Dict[str, Any]:
        """Return a JSON-serialisable snapshot of all internal state."""
        return {
            "schema": "context_manager/1.4",
            "zone1": self._zone1,
            "zone2": _json_safe(dict(self._zone2)),
            "zone3_summary": list(self._zone3_summary),
            "zone3_entries": _json_safe(list(self._zone3_entries)),
            "obs_encoding": _json_safe(self._obs_encoding),
            "obs_actions": _json_safe(self._obs_actions),
            "obs_levels": self._obs_levels,
            "obs_state": self._obs_state,
            "obs_force_fallback": self._obs_force_fallback,
            "obs_fallback_truncated": self._obs_fallback_truncated,
            "dead_sigs": list(self._dead_sigs),
            "level_num": self._level_num,
            "build_version": self._build_version,
            "last_total_tokens": self._last_total_tokens,
            "needs_refresh": self.needs_refresh,
        }

    def load_snapshot(self, snap: Dict[str, Any]) -> None:
        """Restore state from a snapshot produced by :meth:`snapshot`."""
        self._zone1 = snap.get("zone1", "")
        z2 = _json_restore(snap.get("zone2", {}))
        self._zone2 = {f: z2.get(f, "") for f in _ZONE2_FIELDS}
        self._zone3_summary = deque(
            snap.get("zone3_summary", []), maxlen=_MAX_SUMMARY_LINES)
        self._zone3_entries = deque(
            _json_restore(snap.get("zone3_entries", [])), maxlen=10)
        self._obs_encoding = _json_restore(snap.get("obs_encoding"))
        self._obs_actions = _json_restore(list(snap.get("obs_actions", [])))
        self._obs_levels = int(snap.get("obs_levels", 0))
        self._obs_state = snap.get("obs_state", "unknown")
        self._obs_force_fallback = bool(
            snap.get("obs_force_fallback", False))
        self._obs_fallback_truncated = int(
            snap.get("obs_fallback_truncated", 0))
        self._dead_sigs = deque(
            snap.get("dead_sigs", []), maxlen=_MAX_DEAD_SIGS_PER_LEVEL)
        self._level_num = int(snap.get("level_num", 0))
        self._build_version = int(snap.get("build_version", 0))
        self._last_total_tokens = int(snap.get("last_total_tokens", 0))
        self.needs_refresh = bool(snap.get("needs_refresh", True))

    # ---- helpers ---- #

    def _truncate_prefix_to_cap(self, text: str, cap: int) -> str:
        """Keep the longest prefix of ``text`` whose token count <= cap."""
        if self._tc(text) <= cap:
            return text
        lo, hi = 0, len(text)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self._tc(text[:mid]) <= cap:
                lo = mid
            else:
                hi = mid - 1
        return text[:lo]


# =========================================================================== #
# INLINE TEST SUITE (v1.4) — runs only under __main__
# =========================================================================== #

def _grid(value: int = 0) -> List[List[int]]:
    return [[value] * GRID_SIZE for _ in range(GRID_SIZE)]


def _random_grid(rng: random.Random) -> List[List[int]]:
    return [[rng.randrange(0, 16) for _ in range(GRID_SIZE)]
            for _ in range(GRID_SIZE)]


class _Capture(logging.Handler):
    def __init__(self) -> None:
        super().__init__()
        self.records: List[logging.LogRecord] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.records.append(record)


def _t1_zone2_reset():
    cm = ContextManager()
    cm.set_system_prompt("sys")
    cm.update_zone2("confirmed_rules", "rule A")
    cm.update_zone2("hypotheses", "h1")
    cm.update_zone2("world_model_status", "wip")
    cm.update_zone2("goal_notes", "goal")
    cm.reset_zone2()
    for f in ("confirmed_rules", "hypotheses",
              "world_model_status", "goal_notes"):
        assert cm._zone2[f] == "", f"{f} not cleared"
    assert cm._zone1 == "sys", "zone1 lost on reset"


def _t2_whole_field_replacement():
    cm = ContextManager()
    cm.update_zone2("confirmed_rules", "old text here")
    cm.update_zone2("confirmed_rules", "new")
    assert cm._zone2["confirmed_rules"] == "new"
    assert "old" not in cm._zone2["confirmed_rules"]


def _t3_field_cap_and_warning():
    # v1.5: version-proof — cap constant se derive hota hai (80 tha, ab 350)
    _cap = _ZONE2_FIELD_CAPS["hypotheses"]
    cm = ContextManager(token_counter=lambda s: len(s))
    try:
        cm.update_zone2("hypotheses", "X" * (_cap + 120))
    except Zone2OverflowError:
        pass
    else:
        raise AssertionError("expected Zone2OverflowError")
    cap = _Capture()
    logger.addHandler(cap)
    try:
        cm.update_zone2("hypotheses", "X" * (_cap + 120), truncate=True)
    finally:
        logger.removeHandler(cap)
    assert cm._tc(cm._zone2["hypotheses"]) <= _cap
    assert any("truncated" in r.getMessage() for r in cap.records)


def _t4_grid_diff_roundtrip():
    rng = random.Random(1)
    enc = GridDiffEncoder()
    g0 = _random_grid(rng)
    full = enc.encode_full(g0)
    bundle = {"full": full, "deltas": []}
    for _ in range(5):
        g1 = [row[:] for row in g0]
        for _ in range(20):
            r = rng.randrange(GRID_SIZE)
            c = rng.randrange(GRID_SIZE)
            g1[r][c] = rng.randrange(16)
        bundle["deltas"].append(enc.encode_delta(g0, g1))
        g0 = g1
    decoded = GridDiffEncoder.decode(bundle)
    assert decoded == g0, "roundtrip mismatch"


def _t5_drift_guard():
    enc = GridDiffEncoder()
    g0 = _grid(0)
    enc.encode_full(g0)
    g1 = [row[:] for row in g0]
    g1[0][0] = 5
    delta = enc.encode_delta(g0, g1)
    g3 = _grid(3)
    enc.encode_full(g3)
    before = [r[:] for r in enc._grid]
    ok = enc.apply_delta(delta)
    assert ok is False
    assert enc.needs_refresh is True
    assert enc._grid == before, "internal state mutated on drift"
    assert enc._grid == g3


def _t6_refresh_triggers():
    enc = GridDiffEncoder(refresh_every=10)
    enc.encode_full(_grid(0))
    assert enc.needs_refresh is False
    enc.mark_new_game()
    assert enc.needs_refresh is True
    enc.encode_full(_grid(0))
    enc.mark_new_level()
    assert enc.needs_refresh is True
    enc.encode_full(_grid(0))
    enc.mark_death()
    assert enc.needs_refresh is True
    enc.encode_full(_grid(0))
    for _ in range(9):
        enc.tick()
    assert enc.needs_refresh is False
    enc.tick()
    assert enc.needs_refresh is True


def _t7_build_order():
    cm = ContextManager()
    cm.set_system_prompt("SYS")
    cm.update_zone2("confirmed_rules", "R")
    cm.append_action("A1", "ok", 1)
    cm.set_observation(None, ["a"], 0, "RUNNING")
    t = cm.build_prompt().text
    i1 = t.index("=== SYSTEM ===")
    i2 = t.index("=== GAME KNOWLEDGE ===")
    i3 = t.index("=== RECENT ACTIONS ===")
    i4 = t.index("=== OBSERVATION ===")
    assert i1 < i2 < i3 < i4, f"order wrong: {i1},{i2},{i3},{i4}"


def _t8_hard_gate_cascade():
    cm = ContextManager(
        zone1_max_tokens=100, zone2_max_tokens=200, zone3_max_tokens=200,
        observation_max_tokens=500, total_hard_cap=800,
        token_counter=lambda s: len(s))
    cm.set_system_prompt("S" * 60)
    cm.update_zone2("confirmed_rules", "R" * 60)
    cm.update_zone2("hypotheses", "H" * 60)
    cm.update_zone2("goal_notes", "G" * 30)
    for i in range(10):
        cm.append_action(f"a{i}", "res" * 10, delta_count=1)
    cm.set_observation(
        {"type": "full", "rows": ["0:64"] * GRID_SIZE}, [], 0, "RUNNING")
    n_before = len(cm._zone3_entries)
    pkt = cm.build_prompt()
    n_after = len(cm._zone3_entries)
    assert n_after <= 3, f"zone3 not shrunk to <=3: {n_after}"
    assert n_after < n_before
    assert pkt.token_estimate <= 800


def _t9_cascade_exhaustion():
    cm = ContextManager(
        zone1_max_tokens=200, zone2_max_tokens=300,
        zone3_max_tokens=200, observation_max_tokens=500,
        total_hard_cap=100, token_counter=lambda s: len(s))
    cm.set_system_prompt("A" * 300)
    cm.update_zone2("confirmed_rules", "R" * 100)
    try:
        cm.build_prompt()
    except ContextOverflowError:
        return
    raise AssertionError("expected ContextOverflowError")


def _t10_promote_to_zone2():
    cm = ContextManager()
    for i in range(10):
        cm.append_action(f"a{i}", "r", 1)
    assert len(cm._zone3_entries) == 10
    cm.promote_to_zone2("hypothesis violated!", "hypotheses")
    assert "hypothesis violated" in cm._zone2["hypotheses"]


def _t11_dead_signature():
    cm = ContextManager()
    cm.register_dead_signature("obj-42")
    cm.set_observation(None, ["a"], 0, "RUNNING")
    pkt1 = cm.build_prompt()
    assert "obj-42" in pkt1.text
    cm.set_level(1)
    pkt2 = cm.build_prompt()
    assert "obj-42" not in pkt2.text


def _t12_token_counting():
    cm = ContextManager(token_counter=lambda s: len(s))
    cm.set_system_prompt("hello world")
    cm.append_action("a", "b", 1)
    cm.set_observation(None, [], 0, "RUNNING")
    pkt = cm.build_prompt()
    assert pkt.token_estimate == len(pkt.text)
    samples = ["hello world foo", "a", "", "   ", "hi\nthere",
               "unicode: हिन्दी"]
    for s in samples:
        assert default_token_counter(s) >= len(s.split()), s


def _t13_snapshot_roundtrip():
    cm = ContextManager()
    cm.set_system_prompt("SYS")
    cm.update_zone2("confirmed_rules", "R")
    cm.append_action("a1", "ok", 1)
    cm.add_summary_line("attempt 1: failed because X")
    cm.set_observation(
        {"type": "full", "rows": ["0:64"] * GRID_SIZE}, [], 0, "RUNNING")
    cm.register_dead_signature("d1")
    pkt1 = cm.build_prompt()
    snap = cm.snapshot()
    js = json.dumps(snap)
    cm2 = ContextManager()
    cm2.load_snapshot(json.loads(js))
    pkt2 = cm2.build_prompt()
    assert pkt1.text == pkt2.text, "text mismatch after roundtrip"
    assert pkt1.token_estimate == pkt2.token_estimate


def _t14_fuzz():
    rng = random.Random(99)
    fields = ("confirmed_rules", "hypotheses",
              "world_model_status", "goal_notes")
    for _ in range(200):
        cm = ContextManager(
            zone1_max_tokens=100, zone2_max_tokens=300,
            zone3_max_tokens=200, observation_max_tokens=500,
            total_hard_cap=3000, token_counter=lambda s: len(s))
        cm.set_system_prompt("S" * 50)
        try:
            for _ in range(20):
                op = rng.randrange(6)
                if op == 0:
                    cm.append_action(
                        f"a{rng.randrange(10)}",
                        "r" * rng.randrange(20),
                        delta_count=rng.randrange(3))
                elif op == 1:
                    try:
                        cm.update_zone2(rng.choice(fields),
                                        "X" * rng.randrange(100))
                    except Zone2OverflowError:
                        pass
                elif op == 2:
                    try:
                        cm.promote_to_zone2(
                            f"evt{rng.randrange(100)}",
                            rng.choice(fields))
                    except Zone2OverflowError:
                        pass
                elif op == 3:
                    if rng.random() < 0.5:
                        enc = {"type": "delta", "version": 1,
                               "changes": [{"r": 0, "c": 0,
                                            "old": 0, "new": 1}]}
                    else:
                        enc = {"type": "full",
                               "rows": ["0:64"] * GRID_SIZE}
                    cm.set_observation(enc, ["a"], 0, "RUNNING")
                elif op == 4:
                    cm.register_dead_signature(
                        f"sig{rng.randrange(20)}")
                else:
                    cm.set_level(rng.randrange(5))
            try:
                cm.build_prompt()
            except ContextOverflowError:
                pass
            assert cm._zone1 == "S" * 50, "zone1 mutated"
        except Exception as e:
            raise AssertionError(f"fuzz raised: {e!r}")


def _t15_apply_delta_partial_reject():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [
            {"r": 0, "c": 0, "old": 0, "new": 7},
            {"r": 999, "c": 0, "old": 0, "new": 7},
        ],
    }
    before = [row[:] for row in enc._grid]
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated despite rejection"
    assert enc._grid[0][0] == 0


def _t16_apply_delta_bad_checksum():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [{"r": 5, "c": 5, "old": 0, "new": 9}],
    }
    before = [row[:] for row in enc._grid]
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated on checksum reject"
    assert enc._grid[5][5] == 0


def _t17_zone3_summary_bounded():
    cm = ContextManager()
    for i in range(1000):
        cm.add_summary_line(f"line {i}")
    assert len(cm._zone3_summary) <= _MAX_SUMMARY_LINES
    assert "line 999" in list(cm._zone3_summary)


def _t18_dead_sigs_bounded():
    cm = ContextManager()
    for i in range(5000):
        cm.register_dead_signature(f"sig {i}")
    assert len(cm._dead_sigs) <= _MAX_DEAD_SIGS_PER_LEVEL
    assert "sig 4999" in list(cm._dead_sigs)


def _t19_snapshot_arbitrary_any():
    cm = ContextManager()
    cm.set_system_prompt("SYS")
    cm.append_action({"a", "b"}, "ok", 1)
    cm.append_action(b"\x00\x01", "ok", 0)
    cm.set_observation(None, [{"x": 1}, (1, 2)], 0, "RUNNING")
    snap = cm.snapshot()
    js = json.dumps(snap)

    entry0 = snap["zone3_entries"][0]["action"]
    assert entry0 == {"__set__": ["a", "b"]} or \
           entry0 == {"__set__": ["b", "a"]}, \
        f"set encoding lost: {entry0!r}"
    entry1 = snap["zone3_entries"][1]["action"]
    assert entry1 == {"__bytes__": "0001"}, \
        f"bytes encoding lost: {entry1!r}"
    assert snap["obs_actions"][0] == {"x": 1}, \
        f"dict obs_action corrupted: {snap['obs_actions'][0]!r}"
    # v1.4: tuples are now preserved via the __tuple__ marker.
    assert snap["obs_actions"][1] == {"__tuple__": [1, 2]}, \
        f"tuple marker missing: {snap['obs_actions'][1]!r}"

    cm2 = ContextManager()
    cm2.load_snapshot(json.loads(js))
    # Round-trip: the tuple must come back as a tuple, not a list.
    assert cm2._obs_actions[1] == (1, 2), \
        f"tuple not restored: {cm2._obs_actions[1]!r}"
    assert isinstance(cm2._obs_actions[1], tuple), \
        f"restored value is {type(cm2._obs_actions[1]).__name__}, not tuple"

    p1 = cm.build_prompt()
    p2 = cm2.build_prompt()
    assert p1.text == p2.text, "text mismatch after roundtrip"


def _t20_fallback_truncation_marker():
    cm = ContextManager()
    rows = ["1:64"] * 30 + ["0:64"] * (GRID_SIZE - 30)
    cm.set_observation({"type": "full", "rows": rows}, [], 0, "RUNNING")
    cm._obs_force_fallback = True
    text = cm.build_prompt().text
    assert "TRUNCATED" in text, "no truncation marker emitted"
    expected_hidden = 30 * GRID_SIZE - _MAX_FALLBACK_CELLS
    assert cm._obs_fallback_truncated == expected_hidden, \
        (f"counter wrong: got {cm._obs_fallback_truncated}, "
         f"expected {expected_hidden}")
    assert f"<TRUNCATED: {expected_hidden} more non-zero cells>" in text, \
        "marker does not report exact hidden count"


def _t21_promote_fail_loud():
    # v1.5: version-proof — cap constant se derive (80 tha, ab 350)
    _cap = _ZONE2_FIELD_CAPS["hypotheses"]
    cm = ContextManager(token_counter=lambda s: len(s))
    cm.update_zone2("hypotheses", "x" * (_cap - 10), truncate=True)
    try:
        cm.promote_to_zone2("surprising event!", "hypotheses")
    except Zone2OverflowError:
        pass
    else:
        raise AssertionError("expected Zone2OverflowError from promote")
    assert cm._zone2["hypotheses"] == "x" * (_cap - 10)
    cm.promote_to_zone2("surprising event!", "hypotheses", truncate=True)
    assert cm._tc(cm._zone2["hypotheses"]) <= _cap


def _t22_fallback_handles_delta():
    cm = ContextManager()
    delta = {"type": "delta", "version": 3,
             "base_checksum": "x", "checksum": "y",
             "changes": [{"r": 1, "c": 2, "old": 0, "new": 5},
                         {"r": 3, "c": 4, "old": 0, "new": 7}]}
    cm.set_observation(delta, [], 0, "RUNNING")
    cm._obs_force_fallback = True
    text = cm.build_prompt().text
    assert "GRID_DELTA" in text, "delta not surfaced in fallback"
    assert "(1,2,5)" in text and "(3,4,7)" in text
    assert "NONZERO_CELLS=0" not in text, \
        "delta silently lost as NONZERO_CELLS=0"


def _t23_apply_delta_bad_value():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    before = [row[:] for row in enc._grid]
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [{"r": 0, "c": 0, "old": 0, "new": 99}],
    }
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated on value-range reject"


def _t24_decode_verify():
    enc = GridDiffEncoder()
    g0 = _grid(0)
    full = enc.encode_full(g0)
    g1 = [row[:] for row in g0]
    g1[0][0] = 5
    d = enc.encode_delta(g0, g1)
    d_bad = dict(d)
    d_bad["checksum"] = "deadbeef"
    bundle = {"full": full, "deltas": [d_bad]}
    g = GridDiffEncoder.decode(bundle, verify=False)
    assert g[0][0] == 5
    try:
        GridDiffEncoder.decode(bundle, verify=True)
    except ValueError as exc:
        assert "checksum" in str(exc).lower()
    else:
        raise AssertionError("expected ValueError with verify=True")


def _t25_decode_range_validation():
    enc = GridDiffEncoder()
    full = enc.encode_full(_grid(0))
    bad_bundle = {"full": full, "deltas": [
        {"type": "delta", "version": 2,
         "base_checksum": full["checksum"], "checksum": "x",
         "changes": [{"r": 999, "c": 0, "old": 0, "new": 1}]}]}
    try:
        GridDiffEncoder.decode(bad_bundle, verify=False)
    except ValueError as exc:
        assert "range" in str(exc).lower() or "coordinate" in str(exc).lower()
    else:
        raise AssertionError("expected range ValueError without verify")


def _t26_decode_bad_rle_value():
    bad_full = {"type": "full", "version": 1, "checksum": "x",
                "rows": ["10:64"] + ["0:64"] * (GRID_SIZE - 1)}
    try:
        GridDiffEncoder.decode(bad_full, verify=False)
    except ValueError as exc:
        assert "out of range" in str(exc).lower(), str(exc)
    else:
        raise AssertionError(
            "decode accepted RLE value 16 (GRID_MAX_VALUE is 15)")


def _t27_apply_delta_bad_old():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    before = [row[:] for row in enc._grid]
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [{"r": 0, "c": 0, "old": 14, "new": 5}],
    }
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated on old-value mismatch"


def _t28_per_zone_cap_warns():
    cm = ContextManager(
        zone1_max_tokens=10, total_hard_cap=7800,
        token_counter=lambda s: len(s))
    cap = _Capture()
    logger.addHandler(cap)
    try:
        cm.set_system_prompt("S" * 50)
        cm.build_prompt()
    finally:
        logger.removeHandler(cap)
    assert any("advisory cap" in r.getMessage() for r in cap.records), \
        "no advisory-cap warning emitted"
    assert len(cm._zone1) == 50


def _t29_objects_rendered():
    cm = ContextManager()
    cm.set_system_prompt("s")
    obs = [{"id": 1, "color_char": "g", "color_digit": 2, "size": 24,
            "bbox": [5, 10, 30, 33]},
           {"id": 2, "color_char": "B", "color_digit": 5, "size": 25,
            "bbox": [30, 34, 8, 12]}]
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": ["W64"] * 64},
                       ["ACTION1"], 0, "RUNNING", objects=obs)
    text = cm.build_prompt().text
    assert "OBJECTS" in text, "objects block missing from prompt"
    assert "[1] color=g size=24 rows=5-10 cols=30-33" in text, \
        f"object 1 render wrong"
    assert "[2] color=B size=25 rows=30-34 cols=8-12" in text, \
        f"object 2 render wrong"
    # objects=None => no block (backward compat, v1.5 behavior)
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": ["W64"] * 64},
                       ["ACTION1"], 0, "RUNNING")
    assert "OBJECTS" not in cm.build_prompt().text, \
        "objects=None par bhi block render hua"
    # cap: 30 objects => sirf 24 render + hidden marker
    many = [{"id": i, "color_char": "g", "size": 1, "bbox": [0, 0, i, i]}
            for i in range(1, 31)]
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": ["W64"] * 64},
                       [], 0, "RUNNING", objects=many)
    t2 = cm.build_prompt().text
    assert "<6 more objects hidden>" in t2, "cap marker missing"
    # fallback mode mein bhi objects render hone chahiye
    cm._obs_force_fallback = True
    t3 = cm.build_prompt().text
    assert "OBJECTS" in t3 and "[1] color=g" in t3, \
        "fallback mode mein objects missing"


def _t30_fallback_accepts_char_format():
    # v1.6 regression: grid_text ka "char+count" RLE pehle <malformed> deta tha
    cm = ContextManager()
    rows = ["W8,g4,W52"] * 30 + ["W64"] * 34
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": rows}, [], 0, "RUNNING")
    cm._obs_force_fallback = True
    text = cm.build_prompt().text
    assert "<malformed>" not in text, "char-format RLE malformed!"
    assert "NONZERO_CELLS (r,c,v):" in text, "nonzero listing missing"
    assert "(0,8,2)" in text and "(0,9,2)" in text and "(0,11,2)" in text, \
        "char-format coordinates wrong"
    expected_nonzero = 30 * 4
    assert cm._obs_fallback_truncated == max(
        0, expected_nonzero - _MAX_FALLBACK_CELLS), \
        cm._obs_fallback_truncated
    # legacy hex:dec format ab bhi chale (v1.4 contract preserved)
    cm2 = ContextManager()
    cm2.set_observation({"type": "full", "dim": [64, 64],
                         "checksum": "x",
                         "rows": ["1:64"] * 30 + ["0:64"] * 34},
                        [], 0, "RUNNING")
    cm2._obs_force_fallback = True
    t2 = cm2.build_prompt().text
    assert "<malformed>" not in t2, "legacy format broken!"
    assert "NONZERO_CELLS (r,c,v):" in t2
    assert "(0,0,1)" in t2, "legacy coordinates wrong"
    # malformed chunk ab bhi fail-loud
    cm3 = ContextManager()
    cm3.set_observation({"type": "full", "dim": [64, 64],
                         "checksum": "x", "rows": ["zz9"]},
                        [], 0, "RUNNING")
    cm3._obs_force_fallback = True
    assert "NONZERO_CELLS=<malformed>" in cm3.build_prompt().text


def _run_self_tests() -> int:
    """Execute the full v1.6 test battery.  Returns process exit code."""
    tests = [
        ("T-1  zone2 reset", _t1_zone2_reset),
        ("T-2  whole-field replacement", _t2_whole_field_replacement),
        ("T-3  field cap + warning", _t3_field_cap_and_warning),
        ("T-4  grid diff roundtrip", _t4_grid_diff_roundtrip),
        ("T-5  drift guard", _t5_drift_guard),
        ("T-6  refresh triggers", _t6_refresh_triggers),
        ("T-7  build order", _t7_build_order),
        ("T-8  hard gate cascade", _t8_hard_gate_cascade),
        ("T-9  cascade exhaustion", _t9_cascade_exhaustion),
        ("T-10 promote to zone2", _t10_promote_to_zone2),
        ("T-11 dead signature", _t11_dead_signature),
        ("T-12 token counting", _t12_token_counting),
        ("T-13 snapshot roundtrip", _t13_snapshot_roundtrip),
        ("T-14 fuzz", _t14_fuzz),
        ("T-15 apply_delta partial-reject atomic",
         _t15_apply_delta_partial_reject),
        ("T-16 apply_delta bad-checksum atomic",
         _t16_apply_delta_bad_checksum),
        ("T-17 zone3 summary bounded", _t17_zone3_summary_bounded),
        ("T-18 dead sigs bounded", _t18_dead_sigs_bounded),
        ("T-19 snapshot arbitrary Any", _t19_snapshot_arbitrary_any),
        ("T-20 fallback truncation marker",
         _t20_fallback_truncation_marker),
        ("T-21 promote fail-loud", _t21_promote_fail_loud),
        ("T-22 fallback handles delta", _t22_fallback_handles_delta),
        ("T-23 apply_delta value-range reject", _t23_apply_delta_bad_value),
        ("T-24 decode verify", _t24_decode_verify),
        ("T-25 decode range validation", _t25_decode_range_validation),
        ("T-26 decode rejects bad RLE value", _t26_decode_bad_rle_value),
        ("T-27 apply_delta rejects bad old", _t27_apply_delta_bad_old),
        ("T-28 per-zone advisory cap warns", _t28_per_zone_cap_warns),
        ("T-29 objects rendered in observation", _t29_objects_rendered),
        ("T-30 fallback accepts char+count RLE", _t30_fallback_accepts_char_format),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"PASS  {name}")
        except Exception as e:
            failed += 1
            print(f"FAIL  {name}: {e}")
    print()
    print(f"{len(tests) - failed}/{len(tests)} passed")
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    logging.basicConfig(level=logging.WARNING,
                        format="%(levelname)s %(name)s %(message)s")
    import sys as _sys
    _sys.exit(_run_self_tests())

In [ ]:
%%writefile goa/agent/inference_layer.py
"""agent/inference_layer.py — Inference Layer (IL). v1.5.

Changelog:
  v1.5 — DIR/2026/070 (D-3 single source of truth):
         _call_transport forwards `reasoning_mode=mode` to the transport.
         Transport is sole decision point for DEEP/FAST wire params.
  v1.4 — DIR/2026/060 lineage.
  v1.3 — DIR/2026/055 lineage.

  F-2 (P2, INDUS-DIR-070 final-audit closure):
    `ask_reasoner` signature default changed from `mode="deep"` (v1.0.17
    Cell 5 L55, S-3) to `mode=IL_MODE_FAST` (== "fast") in v1.5.  This is
    a second change beyond DIR-070 Task 2's minimal-diff scope (which was
    only `_call_transport` + `reasoning_mode`).  Harmless in production:
    my_agent always passes `mode=` explicitly at the call site.  The new
    default reflects the intended fail-closed FAST posture when a caller
    omits `mode`.  Covered by new test T-IL-DEFAULT below.
"""
from __future__ import annotations
import json, logging, os
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Sequence

logger = logging.getLogger(__name__)
__version__ = "1.6"  # v1.6 certified upgrade (DIR-078): +call_log, CODER_CODE_CAP, build_spec

REASONER_ANSWER_CAP      = 200
REASONER_THINKING_BUDGET = 1000
IL_MODE_FAST = "fast"
IL_MODE_DEEP = "deep"

_DEFAULT_PARAMS = {"temperature": 0.0, "top_p": 1.0, "top_k": -1,
                   "presence_penalty": 0.0, "repetition_penalty": 1.0}


@dataclass
class ReasonerResult:
    observation_summary: str
    hypothesis_updates: List[str]
    instruction: str
    prediction: str
    raw: Optional[Dict[str, Any]] = None

@dataclass
class WorldModelResult:
    code: str
    notes: str = ""

@dataclass
class DiagnoseResult:
    root_cause: str
    confidence: float = 0.0
    suggested_fix: str = ""

@dataclass
class PromptPacket:
    text: str
    meta: Dict[str, Any] = field(default_factory=dict)


class InferenceLayer:
    def __init__(self, transport, *, reasoner_model=None,
                 coder_model=None, context_window=16384,
                 default_mode=IL_MODE_FAST):
        self._transport = transport
        self._reasoner_model = (reasoner_model
                                or os.environ.get("VLLM_REASONER_NAME")
                                or "qwen3.8-27b")
        self._coder_model = (coder_model
                             or os.environ.get("VLLM_CODER_NAME")
                             or "qwen3-coder-30b-a3b")
        self._context_window = int(context_window)
        self._default_mode = default_mode

    @property
    def version(self): return __version__

    def call_log(self):
        """LA-3b FIX (Indus, 2026-09-29 final logic audit): THE RUN reads
        telemetry via the IL *instance* (_il.call_log()); call_log was
        module-level only, so hasattr(_il, "call_log") was False and the
        run's fast/deep/coder counts silently read 0. Instance passthrough
        to the shared ring buffer."""
        return call_log()

    def ask_reasoner(self, prompt_packet, mode=IL_MODE_FAST, *, params=None):
        # F-2 (P2): default is IL_MODE_FAST ("fast") as of v1.5.
        # v1.0.17 Cell 5 used mode="deep" — the change is intentional and
        # documented in the module changelog.  my_agent always passes
        # `mode=` explicitly; this default governs only direct callers.
        text = getattr(prompt_packet, "text", None)
        if text is None and isinstance(prompt_packet, dict):
            text = prompt_packet.get("text", "")
        if text is None: text = str(prompt_packet)
        merged = dict(_DEFAULT_PARAMS)
        if params: merged.update(params)
        rj = self._call_transport(
            model=self._reasoner_model, system=None,
            messages=[{"role": "user", "content": text}],
            max_tokens=REASONER_ANSWER_CAP, mode=mode, params=merged,
            chat_template_kwargs={})
        return self._parse_reasoner_result(rj)

    def write_world_model(self, brief):
        rj = self._call_transport(
            model=self._coder_model, system=None,
            messages=[{"role": "user", "content": brief}],
            max_tokens=REASONER_ANSWER_CAP, mode=None,
            params=dict(_DEFAULT_PARAMS), chat_template_kwargs={})
        return WorldModelResult(code=self._extract_content(rj), notes="")

    def diagnose_failure(self, function_name, failure_trace, current_code=""):
        prompt = (f"Root-cause diagnosis for function {function_name!r}.\n"
                  f"Trace:\n{failure_trace}\nCurrent code:\n{current_code}\n"
                  "Return a short root-cause sentence.")
        rj = self._call_transport(
            model=self._coder_model, system=None,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=REASONER_ANSWER_CAP, mode=None,
            params=dict(_DEFAULT_PARAMS), chat_template_kwargs={})
        return DiagnoseResult(root_cause=self._extract_content(rj).strip())

    def _call_transport(self, *, model, system, messages, max_tokens,
                        mode, params, chat_template_kwargs):
        ctk = dict(chat_template_kwargs or {})
        call_kwargs = {
            "model": model, "system": system, "messages": list(messages),
            "max_tokens": max_tokens,
            "temperature": params["temperature"],
            "top_p": params["top_p"], "top_k": params["top_k"],
            "stop_sequences": [], "chat_template_kwargs": ctk,
            "presence_penalty": params.get("presence_penalty", 0.0),
            "repetition_penalty": params.get("repetition_penalty", 1.0),
            "reasoning_mode": mode,
        }
        _t0 = _time_mod.time()
        rj = self._transport(**call_kwargs)
        # LA-3 FIX (Indus, 2026-09-29 final logic audit): PR-1 §3.2
        # telemetry — _log_call existed but was never wired, so call_log()
        # was always empty and THE RUN's fast/deep/coder telemetry read 0.
        try:
            _u = rj.get("usage") if isinstance(rj, dict) else None
            _u = _u or {}
            _ch = rj.get("choices") if isinstance(rj, dict) else None
            _fr = None
            if _ch:
                _m0 = _ch[0]
                _fr = (_m0.get("finish_reason") if isinstance(_m0, dict)
                       else getattr(_m0, "finish_reason", None))
            _log_call(model=model, mode=mode,
                      prompt_tokens=_u.get("prompt_tokens", 0),
                      completion_tokens=_u.get("completion_tokens", 0),
                      latency_ms=(_time_mod.time() - _t0) * 1000.0,
                      finish_reason=_fr)
        except Exception:
            pass  # fail-open telemetry
        return rj

    @staticmethod
    def _extract_content(rj):
        try:
            choices = rj.get("choices") if isinstance(rj, dict) else rj.choices
            msg = choices[0].get("message") if isinstance(choices[0], dict) \
                  else choices[0].message
            return msg.get("content", "") if isinstance(msg, dict) \
                   else getattr(msg, "content", "")
        except Exception:
            return ""

    def _parse_reasoner_result(self, rj):
        content = self._extract_content(rj)
        try: obj = json.loads(content)
        except Exception: obj = {}
        return ReasonerResult(
            observation_summary=str(obj.get("observation_summary", "")),
            hypothesis_updates=list(obj.get("hypothesis_updates", []) or []),
            instruction=str(obj.get("instruction", content.strip())),
            prediction=str(obj.get("prediction", "")),
            raw=rj if isinstance(rj, dict) else None)


def _t_il_rm_1():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.ask_reasoner({"text": "x"}, mode="deep")
    assert spy and spy[-1] == "deep"

def _t_il_rm_2():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.ask_reasoner({"text": "x"}, mode="fast")
    assert spy and spy[-1] == "fast"

def _t_il_rm_3():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "```python\npass\n```"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.write_world_model("dummy"); assert spy and spy[-1] is None
    il.diagnose_failure("plan", "trace", "code"); assert spy[-1] is None

def _t_il_env_coder():
    os.environ["VLLM_CODER_NAME"] = "custom-coder-x"
    try:
        il = InferenceLayer(transport=lambda **kw: {})
        assert il._coder_model == "custom-coder-x"
    finally:
        os.environ.pop("VLLM_CODER_NAME", None)


def _t_il_default():
    """F-2 (P2) explicit coverage: signature default is IL_MODE_FAST."""
    import inspect
    sig = inspect.signature(InferenceLayer.ask_reasoner)
    assert sig.parameters["mode"].default == IL_MODE_FAST
    assert IL_MODE_FAST == "fast"
    # Behavioural check: caller omits mode → transport sees "fast"
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.ask_reasoner({"text": "x"})   # no mode kwarg
    assert spy and spy[-1] == "fast", spy


def _run_self_tests():
    tests = [("T-IL-RM-1", _t_il_rm_1), ("T-IL-RM-2", _t_il_rm_2),
             ("T-IL-RM-3", _t_il_rm_3), ("T-IL-ENV", _t_il_env_coder),
             ("T-IL-DEFAULT", _t_il_default)]
    failed = 0
    for name, fn in tests:
        try: fn(); print(f"PASS  {name}")
        except Exception as e:
            failed += 1; print(f"FAIL  {name}: {type(e).__name__}: {e}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed")
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    import sys
    sys.exit(_run_self_tests())


# =========================================================================
# v1.6 ADDITIONS — PR-1 (DIR-2026-076 §3): call_log + CODER_CODE_CAP +
# spec-builder. Base v1.5 above is UNCHANGED (certified).
# =========================================================================
import threading as _threading
import time as _time_mod

_CALL_LOG = []
_CALL_LOG_CAP = 5000
_CALL_LOG_LOCK = _threading.Lock()

# PR-1 §3.3: hard generation cap for coder output.
CODER_CODE_CAP = 2048
CODER_CODE_CAP_EXCEEDED = "coder_code_cap_exceeded"

_SPEC_TEMPLATE = """You are writing Python that will be executed in a sandbox.
TASK: {task}
HYPOTHESIS: {hypothesis}
REPLAY_EVIDENCE:
{replay_evidence}
IO_CONTRACT:
{io_contract}
CONSTRAINTS:
{constraints}
Write the function {entry_signature}. Be conservative: prefer
conservative-interpretation. List your assumptions as comments.
Return ONLY fenced Python code."""


def _log_call(model, mode, prompt_tokens, completion_tokens,
              latency_ms, finish_reason):
    """Fail-open telemetry append (PR-1 §3.2). Never breaks inference."""
    try:
        with _CALL_LOG_LOCK:
            _CALL_LOG.append({
                "ts": _time_mod.time(), "model": model, "mode": mode,
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "latency_ms": latency_ms, "finish_reason": finish_reason,
            })
            if len(_CALL_LOG) > _CALL_LOG_CAP:
                del _CALL_LOG[: len(_CALL_LOG) - _CALL_LOG_CAP]
    except Exception:
        pass


def call_log() -> list:
    """Shallow copy of the telemetry ring buffer (caller-safe)."""
    with _CALL_LOG_LOCK:
        return list(_CALL_LOG)


def build_spec(task_ctx: dict) -> str:
    """PR-1 §3.4: typed SPEC prompt from task_ctx (frozen template §4).

    Keys: task, hypothesis, replay_evidence, io_contract, constraints,
    alternatives (2-4), mode ("action"|"world"), entry_signature.
    """
    entry = task_ctx.get("entry_signature",
                         "next_action(frame, history)" if task_ctx.get("mode") != "world"
                         else "next_grid(state, action)")
    alts = task_ctx.get("alternatives") or []
    ctx = dict(task_ctx)
    ctx.setdefault("task", "")
    ctx.setdefault("hypothesis", "")
    ctx.setdefault("replay_evidence", "(none)")
    ctx.setdefault("io_contract", "pure function; no IO")
    ctx.setdefault("constraints", "stdlib + numpy only; deterministic")
    ctx.setdefault("entry_signature", entry)
    spec = _SPEC_TEMPLATE.format(**ctx)
    if alts:
        spec += "\nADMISSIBLE_ALTERNATIVES:\n" + "\n".join(
            f"- {a}" for a in alts[:4])
    return spec


In [ ]:
%%writefile goa/agent/grid_text.py
"""agent/grid_text.py — Grid → Text converter. v1.0.8.

Pure module: stdlib only, no I/O, no threads, no print(), deterministic.
Converts ARC-AGI-3 engine frames (list of 64x64 int grids) into compact
observation text for the Reasoner.

Changelog: v1.0 initial (DIR/2026/033); v1.0.1 (DIR/035); v1.0.2-v1.0.4
audit F-01..F-08, H-2, M-2, L-2; v1.0.5 R-1..R-7; v1.0.6 mask_scope;
v1.0.7 T-29 recolor -> move fixture; v1.0.8 T-29 assertions corrected
to match the actual code contract (r5c-only for intersects; multi-row
bbox form for contains).

Interop: rle_anchor(grid)["checksum"] == md5(concat(row bytes)) — same
algorithm as context_manager.GridDiffEncoder._compute_checksum.
"""
from __future__ import annotations
import hashlib, json, logging, operator
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

logger = logging.getLogger(__name__)

__all__ = [
    "PALETTE", "DIGIT_TO_CHAR", "DIGIT_TO_NAME", "DIGIT_TO_HEX",
    "CHAR_TO_DIGIT", "GRID_DIM", "MAX_REGION_CELLS", "DIFF_CELL_CAP",
    "MAX_TEXT_CHARS", "LEGEND_MAX_CHARS", "MAX_SUMMARY_OBJECTS",
    "GridTextError", "RegionTooLargeError", "DimensionMismatchError",
    "InvalidGridError", "InvalidArgumentError",
    "ObjectRec", "GridTextConverter",
    "legend_text", "extract_objects", "adjacency", "cell_diff",
    "object_delta", "auto_hud_mask", "rle_anchor", "decode_rle_anchor",
    "render_region", "object_summary_text", "diff_text",
]

PALETTE = (
    (0, "white", "#FFFFFF", "W"), (1, "off-white", "#CCCCCC", "w"),
    (2, "light gray", "#999999", "g"), (3, "gray", "#666666", "G"),
    (4, "charcoal", "#333333", "c"), (5, "black", "#000000", "B"),
    (6, "magenta", "#E53AA3", "M"), (7, "pink", "#FF7BCC", "P"),
    (8, "red", "#F93C31", "R"), (9, "blue", "#1E93FF", "b"),
    (10, "light blue", "#88D8F1", "S"), (11, "yellow", "#FFDC00", "Y"),
    (12, "orange", "#FF851B", "O"), (13, "maroon", "#921231", "r"),
    (14, "green", "#4FCC30", "N"), (15, "purple", "#A356D6", "p"),
)
DIGIT_TO_CHAR = {d: c for d, _, _, c in PALETTE}
DIGIT_TO_NAME = {d: n for d, n, _, _ in PALETTE}
DIGIT_TO_HEX = {d: h for d, _, h, _ in PALETTE}
CHAR_TO_DIGIT = {c: d for d, _, _, c in PALETTE}

GRID_DIM = 64
MAX_REGION_CELLS = 576
DIFF_CELL_CAP = 400
MAX_TEXT_CHARS = 4000
LEGEND_MAX_CHARS = 400
MAX_SUMMARY_OBJECTS = 40
_MASK_SCOPES = ("intersects", "contains")
_TRUNC = "\n...[renderer capped]"


class GridTextError(RuntimeError):
    """Base for typed failures."""


class RegionTooLargeError(GridTextError):
    """Crop exceeds MAX_REGION_CELLS."""


class DimensionMismatchError(GridTextError):
    """Shape mismatch."""


class InvalidGridError(GridTextError):
    """Non-rectangular / zero-width / OOR grid, or malformed anchor."""


class InvalidArgumentError(GridTextError):
    """Bad caller argument (connectivity, OOB region, scope)."""


def _validate_grid(grid: Any, name: str = "grid") -> None:
    if not isinstance(grid, (list, tuple)) or len(grid) == 0:
        raise InvalidGridError(f"{name}: non-empty sequence required")
    w0 = len(grid[0])
    if w0 == 0:
        raise InvalidGridError(f"{name}: zero-width rows invalid")
    for r, row in enumerate(grid):
        if len(row) != w0:
            raise InvalidGridError(
                f"{name}: row {r} len {len(row)} != {w0}")
        for c, v in enumerate(row):
            if isinstance(v, bool):
                raise InvalidGridError(
                    f"{name}: ({r},{c}) bool not int 0..15")
            try:
                iv = operator.index(v)
            except TypeError:
                raise InvalidGridError(
                    f"{name}: ({r},{c}) {v!r} not integer")
            if not 0 <= iv <= 15:
                raise InvalidGridError(
                    f"{name}: ({r},{c})={iv} not in 0..15")


def _md5_grid(grid) -> str:
    h = hashlib.md5()
    for row in grid:
        h.update(bytes(row))
    return h.hexdigest()


def _detect_background(grid) -> int:
    c: Counter = Counter()
    for row in grid:
        for v in row:
            c[v] += 1
    if not c:
        return 0
    best_d, best_c = 0, -1
    for d in sorted(c.keys()):
        if c[d] > best_c:
            best_c, best_d = c[d], d
    return int(best_d)


def _canonical_cells(cells, color_digit: int) -> str:
    if not cells:
        return f"{color_digit}\n"
    mr = min(r for r, _ in cells)
    mc = min(c for _, c in cells)
    pts = sorted((r - mr, c - mc) for r, c in cells)
    return f"{color_digit}\n" + "\n".join(f"{r},{c}" for r, c in pts)


def _compute_ohash(cells, color_digit: int) -> str:
    return hashlib.md5(_canonical_cells(cells, color_digit)
                       .encode("utf-8")).hexdigest()


def _bbox_str(b) -> str:
    r0, r1, c0, c1 = b
    if r0 == r1 and c0 == c1:
        return f"r{r0}c{c0}"
    if r0 == r1:
        return f"r{r0}c{c0}-{c1}"
    if c0 == c1:
        return f"r{r0}-{r1}c{c0}"
    return f"r{r0}-{r1}c{c0}-{c1}"


def _key(ohash: str, b) -> str:
    return f"{ohash}|{int(b[0])},{int(b[1])},{int(b[2])},{int(b[3])}"


def _hard_cap(text: str) -> str:
    if len(text) <= MAX_TEXT_CHARS:
        return text
    keep = MAX_TEXT_CHARS - len(_TRUNC)
    return (text[:keep] + _TRUNC) if keep > 0 else _TRUNC[:MAX_TEXT_CHARS]


def _bbox_center(b) -> Tuple[float, float]:
    r0, r1, c0, c1 = b
    return ((r0 + r1) / 2.0, (c0 + c1) / 2.0)


def _bbox_fully_masked(b, mask_rows, mask_cols) -> bool:
    r0, r1, c0, c1 = b
    mr, mc = set(mask_rows), set(mask_cols)
    return (all(r in mr for r in range(r0, r1 + 1))
            or all(c in mc for c in range(c0, c1 + 1)))


def _bbox_intersects_mask(b, mask_rows, mask_cols) -> bool:
    r0, r1, c0, c1 = b
    mr, mc = set(mask_rows), set(mask_cols)
    return (any(r in mr for r in range(r0, r1 + 1))
            or any(c in mc for c in range(c0, c1 + 1)))


def _entry_is_masked(b, mr, mc, scope: str) -> bool:
    if not mr and not mc:
        return False
    if scope == "intersects":
        return _bbox_intersects_mask(b, mr, mc)
    return _bbox_fully_masked(b, mr, mc)


def legend_text() -> str:
    parts = " ".join(f"{c}={n}" for _, n, _, c in PALETTE)
    out = (f"colors: {parts} "
           "(names are palette labels only, not game meanings)")
    return out[:LEGEND_MAX_CHARS]


@dataclass
class ObjectRec:
    id: int
    color_digit: int
    color_char: str
    size: int
    bbox: Tuple[int, int, int, int]
    ohash: str
    pixels: Optional[List[List[int]]] = None
    cells: List[Tuple[int, int]] = field(default_factory=list, repr=False)

    def to_dict(self) -> Dict[str, Any]:
        return {"id": self.id, "color_digit": self.color_digit,
                "color_char": self.color_char, "size": self.size,
                "bbox": list(self.bbox), "ohash": self.ohash,
                "pixels": self.pixels}


def extract_objects(grid, background=None, connectivity=4):
    if connectivity not in (4, 8):
        raise InvalidArgumentError(f"connectivity {connectivity} not 4/8")
    _validate_grid(grid)
    if background is None:
        background = _detect_background(grid)
    h, w = len(grid), len(grid[0])
    neigh = (((-1, 0), (1, 0), (0, -1), (0, 1)) if connectivity == 4
             else ((-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1),
                   (1, -1), (1, 0), (1, 1)))
    seen = [[False] * w for _ in range(h)]
    comps = []
    for r0 in range(h):
        for c0 in range(w):
            v = grid[r0][c0]
            if v == background or seen[r0][c0]:
                continue
            stack = [(r0, c0)]; seen[r0][c0] = True; cells = []
            while stack:
                r, c = stack.pop(); cells.append((r, c))
                for dr, dc in neigh:
                    nr, nc = r + dr, c + dc
                    if (0 <= nr < h and 0 <= nc < w
                            and not seen[nr][nc]
                            and grid[nr][nc] == v):
                        seen[nr][nc] = True; stack.append((nr, nc))
            comps.append((v, cells))
    comps.sort(key=lambda vc: (min(r for r, _ in vc[1]),
                               min(c for _, c in vc[1])))
    objs = []
    for i, (v, cells) in enumerate(comps, start=1):
        rs = [r for r, _ in cells]; cs = [c for _, c in cells]
        bbox = (min(rs), max(rs), min(cs), max(cs))
        pix = ([[r, c] for r, c in sorted(cells)]
               if len(cells) <= 16 else None)
        objs.append(ObjectRec(i, int(v), DIGIT_TO_CHAR[int(v)],
                              len(cells), bbox,
                              _compute_ohash(cells, int(v)), pix,
                              list(cells)))
    return objs


def adjacency(objects, grid):
    _validate_grid(grid, "adjacency.grid")
    h, w = len(grid), len(grid[0])
    owner = [[0] * w for _ in range(h)]
    for o in objects:
        for r, c in o.cells:
            owner[r][c] = o.id
    pairs = set()
    for r in range(h):
        for c in range(w):
            a = owner[r][c]
            if a == 0:
                continue
            if c + 1 < w:
                b = owner[r][c + 1]
                if b and b != a:
                    pairs.add((min(a, b), max(a, b)))
            if r + 1 < h:
                b = owner[r + 1][c]
                if b and b != a:
                    pairs.add((min(a, b), max(a, b)))
    return [list(p) for p in sorted(pairs)]


def cell_diff(old_grid, new_grid, mask_rows=(), mask_cols=()):
    _validate_grid(old_grid, "cell_diff.old_grid")
    _validate_grid(new_grid, "cell_diff.new_grid")
    if (len(old_grid) != len(new_grid)
            or len(old_grid[0]) != len(new_grid[0])):
        raise DimensionMismatchError(
            f"cell_diff: {len(old_grid)}x{len(old_grid[0])} vs "
            f"{len(new_grid)}x{len(new_grid[0])}")
    h, w = len(old_grid), len(old_grid[0])
    mr, mc = set(mask_rows), set(mask_cols)
    out = []
    for r in range(h):
        if r in mr:
            continue
        for c in range(w):
            if c in mc:
                continue
            if old_grid[r][c] != new_grid[r][c]:
                out.append((r, c, int(old_grid[r][c]),
                            int(new_grid[r][c])))
    return out


def _render_cell_runs(changes) -> List[str]:
    bp = defaultdict(lambda: defaultdict(list))
    for r, c, o, n in changes:
        bp[(o, n)][r].append(c)
    lines = []
    for (o, n) in sorted(bp.keys()):
        parts = []
        for r in sorted(bp[(o, n)].keys()):
            cols = sorted(bp[(o, n)][r]); i = 0
            while i < len(cols):
                j = i
                while j + 1 < len(cols) and cols[j + 1] == cols[j] + 1:
                    j += 1
                a, b = cols[i], cols[j]
                parts.append(f"r{r}c{a}" if a == b else f"r{r}c{a}-{b}")
                i = j + 1
        lines.append(f"{DIGIT_TO_CHAR[o]}->{DIGIT_TO_CHAR[n]}: "
                     + ",".join(parts))
    return lines


def _pair_nearest(olds, news):
    pairs, used_o, used_n = [], set(), set()
    for n in sorted(news, key=lambda x: x.id):
        best_o, best_d = None, None
        nc = _bbox_center(n.bbox)
        for o in sorted(olds, key=lambda x: x.id):
            if o.id in used_o:
                continue
            oc = _bbox_center(o.bbox)
            d = (oc[0] - nc[0]) ** 2 + (oc[1] - nc[1]) ** 2
            if best_d is None or d < best_d:
                best_d, best_o = d, o
        if best_o is not None:
            used_o.add(best_o.id); used_n.add(n.id)
            pairs.append((best_o, n))
    return pairs


def object_delta(old_objects, new_objects):
    old_h = defaultdict(list); new_h = defaultdict(list)
    for o in old_objects:
        old_h[o.ohash].append(o)
    for n in new_objects:
        new_h[n.ohash].append(n)
    used_o, used_n, moved = set(), set(), []
    for h, news in new_h.items():
        for o, n in _pair_nearest(old_h.get(h, []), news):
            used_o.add(o.id); used_n.add(n.id)
            if o.bbox != n.bbox:
                moved.append([n.id, n.ohash,
                              list(o.bbox), list(n.bbox)])
    rem_o = [o for o in old_objects if o.id not in used_o]
    rem_n = [n for n in new_objects if n.id not in used_n]
    by_bbox = defaultdict(list)
    for o in rem_o:
        by_bbox[o.bbox].append(o)
    recolored = []
    for n in rem_n:
        for o in by_bbox.get(n.bbox, []):
            if set(o.cells) == set(n.cells) and \
                    o.color_digit != n.color_digit:
                used_o.add(o.id); used_n.add(n.id)
                recolored.append([n.id, n.ohash,
                                  o.color_digit, n.color_digit,
                                  list(n.bbox)])
                break
    new_l = [[n.id, n.ohash, list(n.bbox), n.color_char, n.size]
             for n in new_objects if n.id not in used_n]
    gone_l = [[o.id, o.ohash, list(o.bbox)]
              for o in old_objects if o.id not in used_o]
    return {"moved": moved, "recolored": recolored,
            "new": new_l, "gone": gone_l}


def auto_hud_mask(frame_history, threshold=0.5):
    if not frame_history or len(frame_history) < 2:
        return (), ()
    for i, f in enumerate(frame_history):
        _validate_grid(f, f"frame_history[{i}]")
    fh, fw = len(frame_history[0]), len(frame_history[0][0])
    for i in range(1, len(frame_history)):
        fi = frame_history[i]
        if len(fi) != fh or len(fi[0]) != fw:
            raise DimensionMismatchError(
                f"frame {i} shape {len(fi)}x{len(fi[0])} != {fh}x{fw}")
    total = len(frame_history) - 1
    rh, ch = Counter(), Counter()
    for i in range(total):
        a, b = frame_history[i], frame_history[i + 1]
        rc = [False] * fh; cc = [False] * fw
        for r in range(fh):
            for c in range(fw):
                if a[r][c] != b[r][c]:
                    rc[r] = True; cc[c] = True
        for r in range(fh):
            if rc[r]:
                rh[r] += 1
        for c in range(fw):
            if cc[c]:
                ch[c] += 1
    return (tuple(sorted(r for r, k in rh.items() if k / total > threshold)),
            tuple(sorted(c for c, k in ch.items() if k / total > threshold)))


def rle_anchor(grid) -> Dict[str, Any]:
    _validate_grid(grid)
    rows = []
    for row in grid:
        parts = []; cur, cnt = row[0], 1
        for v in row[1:]:
            if v == cur:
                cnt += 1
            else:
                parts.append(f"{DIGIT_TO_CHAR[int(cur)]}{cnt}")
                cur, cnt = v, 1
        parts.append(f"{DIGIT_TO_CHAR[int(cur)]}{cnt}")
        rows.append(",".join(parts))
    return {"type": "full", "dim": [len(grid), len(grid[0])],
            "checksum": _md5_grid(grid), "rows": rows}


def decode_rle_anchor(anchor) -> List[List[int]]:
    grid = []
    try:
        for s in anchor["rows"]:
            row = []
            if s:
                for ch in s.split(","):
                    if not ch:
                        raise InvalidGridError("empty chunk")
                    if ch[0] not in CHAR_TO_DIGIT:
                        raise InvalidGridError(f"char {ch[0]!r}")
                    row.extend([CHAR_TO_DIGIT[ch[0]]] * int(ch[1:]))
            grid.append(row)
    except (KeyError, ValueError, IndexError) as e:
        raise InvalidGridError(f"malformed anchor ({e})") from e
    _validate_grid(grid, "decode.result")
    if "dim" in anchor:
        got = [len(grid), len(grid[0]) if grid else 0]
        if got != list(anchor["dim"]):
            raise InvalidGridError(
                f"dim={anchor['dim']} but decoded {got}")
    return grid


def render_region(grid, r0, r1, c0, c1):
    _validate_grid(grid)
    h, w = len(grid), len(grid[0])
    if not (0 <= r0 <= r1 < h) or not (0 <= c0 <= c1 < w):
        raise InvalidArgumentError(
            f"region r{r0}-{r1} c{c0}-{c1} OOB in {h}x{w}")
    rh, rw = r1 - r0 + 1, c1 - c0 + 1
    if rh * rw > MAX_REGION_CELLS:
        raise RegionTooLargeError(f"{rh*rw} > {MAX_REGION_CELLS}")
    sub = [list(row[c0:c1 + 1]) for row in grid[r0:r1 + 1]]
    lines = [f"region r{r0}-{r1} c{c0}-{c1} dim={rh}x{rw} "
             f"checksum={_md5_grid(sub)}"]
    for i, row in enumerate(sub):
        lines.append(f"{r0 + i} " + "".join(DIGIT_TO_CHAR[v] for v in row))
    return _hard_cap("\n".join(lines))


def object_summary_text(packet) -> str:
    objs = packet.get("objects", [])
    top = sorted(objs, key=lambda o: (-o["size"], o["id"]))[
        :MAX_SUMMARY_OBJECTS]
    lines = []
    for o in top:
        r0, r1, c0, c1 = o["bbox"]
        lines.append(f"#{o['id']} {DIGIT_TO_NAME[o['color_digit']]} "
                     f"{c1-c0+1}x{r1-r0+1} "
                     f"r{r0}-{r1} c{c0}-{c1} h={o['ohash'][:6]}")
    if len(objs) > MAX_SUMMARY_OBJECTS:
        k = len(objs) - MAX_SUMMARY_OBJECTS
        lines.append(f"+{k} more objects "
                     f"(total {len(objs)}; needs_full)")
    return _hard_cap("\n".join(lines))


def _object_delta_lines(delta, mr=(), mc=(), scope="intersects"):
    out = []
    for _id, _h, fb, tb in delta.get("moved", []):
        if _entry_is_masked(fb, mr, mc, scope) \
                or _entry_is_masked(tb, mr, mc, scope):
            continue
        out.append(f"obj#{_id} moved {_bbox_str(tuple(fb))} -> "
                   f"{_bbox_str(tuple(tb))}")
    for _id, _h, od, nd, bb in delta.get("recolored", []):
        if _entry_is_masked(bb, mr, mc, scope):
            continue
        out.append(f"obj#{_id} recolored "
                   f"{DIGIT_TO_CHAR[od]}->{DIGIT_TO_CHAR[nd]}")
    for _id, _h, bb, ch, _sz in delta.get("new", []):
        if _entry_is_masked(bb, mr, mc, scope):
            continue
        r0, r1, c0, c1 = bb
        out.append(f"new obj#{_id} ({ch}, {c1-c0+1}x{r1-r0+1}, "
                   f"r{r0}c{c0})")
    for _id, _h, bb in delta.get("gone", []):
        if _entry_is_masked(bb, mr, mc, scope):
            continue
        out.append(f"gone: obj#{_id}")
    return out


def _explained_cells(delta, cur, prev):
    s = set()
    for _id, h, fb, tb in delta.get("moved", []):
        for cell in prev.get(_key(h, fb), []):
            s.add((cell[0], cell[1]))
        for cell in cur.get(_key(h, tb), []):
            s.add((cell[0], cell[1]))
    for _id, h, _od, _nd, bb in delta.get("recolored", []):
        for cell in cur.get(_key(h, bb), []):
            s.add((cell[0], cell[1]))
    for _id, h, bb, _ch, _sz in delta.get("new", []):
        for cell in cur.get(_key(h, bb), []):
            s.add((cell[0], cell[1]))
    for _id, h, bb in delta.get("gone", []):
        for cell in prev.get(_key(h, bb), []):
            s.add((cell[0], cell[1]))
    return s


def diff_text(packet) -> str:
    delta = packet.get("object_delta", {})
    mr = packet.get("_mask_rows", ())
    mc = packet.get("_mask_cols", ())
    scope = packet.get("_mask_scope", "intersects")
    obj_lines = _object_delta_lines(delta, mr, mc, scope)
    if len(obj_lines) > MAX_SUMMARY_OBJECTS:
        total = len(obj_lines)
        k = total - MAX_SUMMARY_OBJECTS
        obj_lines = obj_lines[:MAX_SUMMARY_OBJECTS]
        obj_lines.append(
            f"+{k} more objects (total {total}; needs_full)")
    lines = list(obj_lines)
    raw = packet.get("cell_changes_raw")
    if obj_lines and raw is not None:
        expl = _explained_cells(
            delta, packet.get("_obj_cells_cur", {}) or {},
            packet.get("_obj_cells_prev", {}) or {})
        res = [tuple(ch) for ch in raw
               if (ch[0], ch[1]) not in expl]
        lines.extend(_render_cell_runs(res))
    elif packet.get("cell_diff"):
        lines.extend(packet["cell_diff"])
    return _hard_cap("\n".join(lines))


class GridTextConverter:
    """Stateful converter.  mask_scope: "intersects" (default, suppresses
    any entry whose bbox touches a mask) or "contains" (suppresses only
    fully-masked bboxes)."""

    def __init__(self, mask_rows=(), mask_cols=(), connectivity=4,
                 mask_scope="intersects"):
        if mask_scope not in _MASK_SCOPES:
            raise InvalidArgumentError(
                f"mask_scope {mask_scope!r} not in {_MASK_SCOPES}")
        self._mask_rows = tuple(int(r) for r in mask_rows)
        self._mask_cols = tuple(int(c) for c in mask_cols)
        self._connectivity = int(connectivity)
        self._mask_scope = mask_scope
        self._version = 0
        self._prev_grid = None
        self._prev_objects = None
        self._prev_cells: Dict[str, List[List[int]]] = {}
        self._last_packet = None

    def set_frame(self, frame):
        if not frame:
            raise InvalidArgumentError("frame must contain a grid")
        last = [list(row) for row in frame[-1]]
        _validate_grid(last, "frame[-1]")
        self._version += 1
        bg = _detect_background(last)
        objects = extract_objects(
            last, background=bg, connectivity=self._connectivity)
        adj = adjacency(objects, last)
        cells_cur: Dict[str, List[List[int]]] = {}
        for o in objects:
            cells_cur[_key(o.ohash, o.bbox)] = [[r, c]
                                                for r, c in o.cells]
        needs_full = False
        raw, cdl = [], []
        delta = {"moved": [], "recolored": [], "new": [], "gone": []}
        if self._prev_grid is not None:
            if (len(self._prev_grid) != len(last)
                    or len(self._prev_grid[0]) != len(last[0])):
                needs_full = True
            else:
                raw = cell_diff(self._prev_grid, last,
                                self._mask_rows, self._mask_cols)
                if len(raw) > DIFF_CELL_CAP:
                    needs_full = True; raw = []
                else:
                    cdl = _render_cell_runs(raw)
                if self._prev_objects is not None:
                    delta = object_delta(self._prev_objects, objects)
        if len(objects) > MAX_SUMMARY_OBJECTS:
            needs_full = True
        dc = (len(delta["moved"]) + len(delta["recolored"])
              + len(delta["new"]) + len(delta["gone"]))
        if dc > MAX_SUMMARY_OBJECTS:
            needs_full = True
        if needs_full:
            raw, cdl = [], []
        packet = {
            "version": self._version, "checksum": _md5_grid(last),
            "dim": [len(last), len(last[0])], "bg": bg,
            "masked": f"rows={len(self._mask_rows)} "
                      f"cols={len(self._mask_cols)}",
            "objects": [o.to_dict() for o in objects],
            "adjacency": adj, "cell_diff": cdl, "object_delta": delta,
            "cell_changes_raw": [list(ch) for ch in raw],
            "needs_full": needs_full, "legend": legend_text(),
            "_obj_cells_cur": cells_cur,
            "_obj_cells_prev": self._prev_cells,
            "_mask_rows": list(self._mask_rows),
            "_mask_cols": list(self._mask_cols),
            "_mask_scope": self._mask_scope,
        }
        self._prev_grid = last; self._prev_objects = objects
        self._prev_cells = cells_cur; self._last_packet = packet
        return packet

    def object_summary_text(self):
        return object_summary_text(self._last_packet or {})

    def diff_text(self):
        return diff_text(self._last_packet or {})

    def last_packet(self):
        return self._last_packet


# ==========================================================================
# Inline self-tests (v1.0.8, 29 tests)
# ==========================================================================

def _eg(v=0):
    return [[v] * GRID_DIM for _ in range(GRID_DIM)]


def _pb(g, r0, c0, h, w, v):
    for r in range(r0, r0 + h):
        for c in range(c0, c0 + w):
            g[r][c] = v


def _t1():
    assert len(PALETTE) == 16
    assert [p[0] for p in PALETTE] == list(range(16))
    assert len({p[3] for p in PALETTE}) == 16
    assert PALETTE[0] == (0, "white", "#FFFFFF", "W")
    assert PALETTE[15] == (15, "purple", "#A356D6", "p")


def _t2():
    L = legend_text()
    assert len(L) <= LEGEND_MAX_CHARS
    for _d, n, _hx, ch in PALETTE:
        assert f"{ch}={n}" in L
    assert "palette labels only" in L


def _t3():
    g = _eg(0)
    _pb(g, 1, 1, 1, 3, 5); _pb(g, 10, 10, 2, 2, 8)
    _pb(g, 20, 20, 1, 1, 15)
    o = extract_objects(g, 0)
    assert [x.id for x in o] == [1, 2, 3]
    assert o[0].bbox == (1, 1, 1, 3) and o[0].size == 3
    assert o[1].bbox == (10, 11, 10, 11) and o[1].size == 4
    assert o[2].bbox == (20, 20, 20, 20)


def _t4():
    g = [[0] * 4 for _ in range(4)]
    g[1][1] = 5; g[2][2] = 5
    assert _detect_background(g) == 0
    assert _detect_background([[0, 0, 5, 5]] * 4) == 0


def _t5():
    g = _eg(0); g[1][1] = 5; g[2][2] = 5
    assert len(extract_objects(g, 0, 4)) == 2
    assert len(extract_objects(g, 0, 8)) == 1


def _t6():
    g1 = _eg(0); _pb(g1, 1, 1, 1, 3, 5)
    o1 = extract_objects(g1, 0)[0]
    g2 = _eg(0); _pb(g2, 40, 40, 1, 3, 5)
    assert extract_objects(g2, 0)[0].ohash == o1.ohash
    g3 = _eg(0); _pb(g3, 1, 1, 1, 3, 8)
    assert extract_objects(g3, 0)[0].ohash != o1.ohash
    g4 = _eg(0); _pb(g4, 1, 1, 1, 4, 5)
    assert extract_objects(g4, 0)[0].ohash != o1.ohash


def _t7():
    g = _eg(0)
    _pb(g, 1, 1, 1, 3, 5); _pb(g, 10, 10, 5, 5, 8)
    o = extract_objects(g, 0)
    assert [x for x in o if x.size == 3][0].pixels is not None
    assert [x for x in o if x.size == 25][0].pixels is None


def _t8():
    g = _eg(0)
    _pb(g, 1, 1, 1, 2, 5); _pb(g, 1, 3, 1, 2, 8)
    _pb(g, 5, 5, 1, 1, 15)
    adj = adjacency(extract_objects(g, 0), g)
    assert [1, 2] in adj and [1, 3] not in adj and [2, 3] not in adj


def _t9():
    a = _eg(0); b = _eg(0)
    for c in range(4):
        b[0][c] = 5
    b[0][7] = 5
    for c in range(3, 6):
        b[2][c] = 5
    lines = _render_cell_runs(cell_diff(a, b))
    assert lines[0] == "W->B: r0c0-3,r0c7,r2c3-5", lines[0]
    assert "r2" not in _render_cell_runs(cell_diff(a, b, [2]))[0]
    assert "r0c7" not in _render_cell_runs(cell_diff(a, b, (), [7]))[0]


def _t10():
    a = _eg(0); _pb(a, 1, 1, 1, 3, 5)
    b = _eg(0); _pb(b, 10, 10, 1, 3, 5)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert d["moved"][0][2] == [1, 1, 1, 3]
    assert d["moved"][0][3] == [10, 10, 10, 12]
    a = _eg(0); _pb(a, 1, 1, 1, 3, 5)
    b = _eg(0); _pb(b, 1, 1, 1, 3, 8)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert d["recolored"][0][2] == 5 and d["recolored"][0][3] == 8
    a = _eg(0); b = _eg(0); _pb(b, 5, 5, 2, 2, 15)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert d["new"][0][3] == "p" and d["new"][0][4] == 4
    a = _eg(0); _pb(a, 5, 5, 2, 2, 15); b = _eg(0)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert len(d["gone"]) == 1


def _t11():
    a = _eg(0); _pb(a, 1, 1, 1, 3, 5)
    b = _eg(0); _pb(b, 10, 10, 1, 3, 5); b[40][40] = 8
    c = GridTextConverter(); c.set_frame([a]); pkt = c.set_frame([b])
    txt = c.diff_text()
    assert txt.startswith("obj#")
    assert "obj#1 moved r1c1-3 -> r10c10-12" in txt
    assert "new obj#2 (R, 1x1, r40c40)" in txt
    for p in ("W->B:", "B->W:", "W->R:"):
        assert p not in txt
    assert any(l == "B->W: r1c1-3" for l in pkt["cell_diff"])
    assert any(l == "W->B: r10c10-12" for l in pkt["cell_diff"])
    assert any(l == "W->R: r40c40" for l in pkt["cell_diff"])


def _t12():
    a = _eg(0); b = _eg(0)
    for i in range(500):
        b[i // 64][i % 64] = 5
    c = GridTextConverter(); c.set_frame([a])
    assert c.set_frame([b])["needs_full"] is True


def _t13():
    base = _eg(0); frames = []
    for i in range(10):
        g = [row[:] for row in base]; g[0][i] = 5; frames.append(g)
    rows, _ = auto_hud_mask(frames, 0.5)
    assert 0 in rows and 30 not in rows


def _t14():
    import random as _r
    rng = _r.Random(20260918)
    g = [[rng.randrange(16) for _ in range(GRID_DIM)]
         for _ in range(GRID_DIM)]
    anc = rle_anchor(g)
    assert anc["dim"] == [GRID_DIM, GRID_DIM]
    assert anc["checksum"] == _md5_grid(g)
    assert decode_rle_anchor(anc) == g


def _t15():
    g = _eg(0); _pb(g, 5, 5, 3, 4, 8)
    lines = render_region(g, 4, 8, 4, 9).split("\n")
    assert lines[0].startswith("region r4-8 c4-9 dim=5x6 checksum=")
    assert lines[1][2:] == "W" * 6
    assert lines[2][2:] == "WRRRRW"
    try:
        render_region(g, 0, 24, 0, 24)
    except RegionTooLargeError:
        pass
    else:
        raise AssertionError("expected RegionTooLargeError")


def _t16():
    a = _eg(0); _pb(a, 1, 1, 2, 2, 5)
    b = _eg(0); _pb(b, 10, 10, 2, 2, 5)
    c = GridTextConverter(); p1 = c.set_frame([a]); p2 = c.set_frame([b])
    assert p1["version"] == 1 and p2["version"] == 2
    for k in ("version", "checksum", "dim", "bg", "masked", "objects",
              "adjacency", "cell_diff", "object_delta",
              "cell_changes_raw", "needs_full", "legend"):
        assert k in p2
    assert json.loads(json.dumps(p2))["version"] == 2


def _t17():
    g = _eg(0)
    _pb(g, 1, 1, 1, 3, 15); _pb(g, 5, 5, 2, 2, 8)
    c = GridTextConverter(); c.set_frame([g])
    lines = c.object_summary_text().split("\n")
    assert lines[0].startswith("#2 red 2x2 r5-6 c5-6 h=")
    assert lines[1].startswith("#1 purple 3x1 r1-1 c1-3 h=")


def _t18():
    g = _eg(0)
    for r in range(GRID_DIM):
        for c in range(GRID_DIM):
            g[r][c] = 0 if (r + c) % 2 == 0 else 5
    try:
        render_region(g, 0, 63, 0, 63)
    except RegionTooLargeError:
        pass
    else:
        raise AssertionError("full-board must raise")
    assert len(render_region(g, 0, 11, 0, 11)) < MAX_TEXT_CHARS


def _t19():
    a = _eg(0); _pb(a, 1, 1, 2, 2, 5)
    b = _eg(0); _pb(b, 10, 10, 2, 2, 5); b[40][40] = 8
    c1 = GridTextConverter(); c1.set_frame([a]); p1 = c1.set_frame([b])
    c2 = GridTextConverter(); c2.set_frame([a]); p2 = c2.set_frame([b])
    assert json.dumps(p1, sort_keys=True) == json.dumps(p2, sort_keys=True)


def _t20():
    import random as _r
    rng = _r.Random(33)
    for _ in range(200):
        a = [[rng.randrange(16) for _ in range(32)] for _ in range(32)]
        b = [[rng.randrange(16) for _ in range(32)] for _ in range(32)]
        c = GridTextConverter(); c.set_frame([a]); pkt = c.set_frame([b])
        for o in pkt["objects"]:
            r0, r1, c0, c1 = o["bbox"]
            assert 0 <= r0 <= r1 < 32 and 0 <= c0 <= c1 < 32
        if not pkt["needs_full"]:
            for ch in pkt["cell_changes_raw"]:
                assert a[ch[0]][ch[1]] != b[ch[0]][ch[1]]


def _t21():
    import random as _r
    rng = _r.Random(35)
    g = [[rng.randrange(16) for _ in range(GRID_DIM)]
         for _ in range(GRID_DIM)]
    c = GridTextConverter(); pkt = c.set_frame([g])
    assert len(pkt["objects"]) > MAX_SUMMARY_OBJECTS
    txt = c.object_summary_text()
    assert len(txt) <= MAX_TEXT_CHARS
    assert "more objects" in txt and "needs_full" in txt
    assert pkt["needs_full"] is True


def _t22():
    import random as _r
    rng = _r.Random(41)
    dense = [[rng.randrange(16) for _ in range(GRID_DIM)]
             for _ in range(GRID_DIM)]
    pkt = GridTextConverter().set_frame([dense])
    assert pkt["needs_full"] and pkt["cell_changes_raw"] == []
    assert pkt["cell_diff"] == []
    a = _eg(0)
    b = [[rng.randrange(16) for _ in range(GRID_DIM)]
         for _ in range(GRID_DIM)]
    c2 = GridTextConverter(); c2.set_frame([a]); p2 = c2.set_frame([b])
    assert p2["needs_full"] and p2["cell_changes_raw"] == []


def _t23():
    g0 = [[0] * 4 for _ in range(3)]
    g1 = [row[:] for row in g0]; g1[0][0] = 5; g1[0][1] = 5
    g2 = [row[:] for row in g1]
    rows, _ = auto_hud_mask([g0, g1, g2], 0.5)
    assert 0 not in rows
    rows2, _ = auto_hud_mask([g0, g1, g2], 0.4)
    assert 0 in rows2


def _t24():
    a = _eg(0)
    a[1][1] = 5; a[2][1] = 5; a[2][2] = 5
    b = _eg(0)
    b[10][10] = 5; b[11][10] = 5; b[11][11] = 5; b[1][2] = 8
    c = GridTextConverter(); c.set_frame([a]); c.set_frame([b])
    assert "r1c2" in c.diff_text()


def _t25():
    a = [[0] * 4 for _ in range(4)]
    for bad in ([[0] * 5 for _ in range(4)],
                [[0] * 4 for _ in range(5)]):
        try:
            cell_diff(a, bad)
        except DimensionMismatchError:
            pass
        else:
            raise AssertionError("expected DimensionMismatchError")


def _t26():
    a = _eg(0)
    for r in (1, 2):
        for c in (1, 2):
            a[r][c] = 5
    for r in (10, 11):
        for c in (10, 11):
            a[r][c] = 5
    b = _eg(0)
    for r in (40, 41):
        for c in (40, 41):
            b[r][c] = 5
    b[10][10] = 8
    b[10][11] = 5; b[11][10] = 5; b[11][11] = 5
    c = GridTextConverter(); c.set_frame([a]); c.set_frame([b])
    assert "r10c10" in c.diff_text()


def _t27():
    for bad in ([[]], [[0, 0], [0]], [[0, 99]], [[0, -1]], "x"):
        try:
            _validate_grid(bad)
        except (InvalidGridError, TypeError):
            pass
        else:
            raise AssertionError(f"expected error on {bad!r}")
    for bad in ([[True, 0], [0, 0]], [[1.0, 0], [0, 0]]):
        try:
            _validate_grid(bad)
        except InvalidGridError:
            pass
        else:
            raise AssertionError("bool/float cells must be rejected")
    good = [[0, 0], [0, 0]]
    cases = [
        lambda: cell_diff(good, [[0, 0], [0]]),
        lambda: cell_diff([[0, 99], [0, 0]], good),
        lambda: adjacency([], [[0, 0], [0]]),
        lambda: auto_hud_mask([[[0, 0]], [[]]], 0.5),
        lambda: decode_rle_anchor({"type": "full", "dim": [1, 2],
                                   "checksum": "x", "rows": ["W1"]}),
        lambda: extract_objects([[0, 0], [0, 0]], connectivity=5),
        lambda: render_region([[0, 0], [0, 0]], 5, 6, 0, 1),
    ]
    for call in cases:
        try:
            call()
        except GridTextError:
            pass
        else:
            raise AssertionError("expected typed GridTextError")
    try:
        auto_hud_mask([[[0, 0]], [[0, 0, 0]]], 0.5)
    except DimensionMismatchError:
        pass
    else:
        raise AssertionError("shape mismatch must raise")


def _t28():
    a = _eg(0); a[1][1] = 5; a[1][2] = 5
    c = GridTextConverter(mask_rows=[1]); c.set_frame([a])
    b = [row[:] for row in a]; b[1][1] = 8
    c.set_frame([b])
    assert "r1c" not in c.diff_text()


def _t29():
    """R-2 v1.0.8: two-part test corrected.
       Part 1: 'intersects' scope suppresses the OBJECT line, but rows
       outside the mask (r4, r6) legitimately appear as residual cell
       runs.  Only the masked row coordinate (r5c) must be absent.
       Part 2: 'contains' scope renders the moved line; the fixture
       bbox spans 3 rows so the multi-row form 'r4-6c10-11' is what
       _bbox_str actually returns.
    """
    a = _eg(0)
    for r in (4, 5, 6):
        for c in (10, 11):
            a[r][c] = 5
    b = _eg(0)
    for r in (20, 21, 22):
        for c in (10, 11):
            b[r][c] = 5

    # Part 1: default 'intersects'.
    c1 = GridTextConverter(mask_rows=[5])
    c1.set_frame([a]); c1.set_frame([b])
    t1 = c1.diff_text()
    assert "r5c" not in t1, f"masked row leaked: {t1}"
    assert "obj#" not in t1, f"object line should be suppressed: {t1}"

    # Part 2: 'contains'.
    c2 = GridTextConverter(mask_rows=[5], mask_scope="contains")
    c2.set_frame([a]); c2.set_frame([b])
    t2 = c2.diff_text()
    assert "moved" in t2, t2
    assert "r4-6c10-11" in t2, t2
    assert "r20-22c10-11" in t2, t2


def _run_self_tests() -> int:
    tests = [
        ("T-1 palette", _t1), ("T-2 legend", _t2),
        ("T-3 objects", _t3), ("T-4 background", _t4),
        ("T-5 connectivity", _t5), ("T-6 ohash", _t6),
        ("T-7 pixels", _t7), ("T-8 adjacency", _t8),
        ("T-9 cell_diff", _t9), ("T-10 object_delta", _t10),
        ("T-11 priority", _t11), ("T-12 diff cap", _t12),
        ("T-13 HUD mask", _t13), ("T-14 RLE", _t14),
        ("T-15 region", _t15), ("T-16 pipeline", _t16),
        ("T-17 formats", _t17), ("T-18 H-5", _t18),
        ("T-19 determinism", _t19), ("T-20 fuzz", _t20),
        ("T-21 obj cap", _t21), ("T-22 needs_full clears raw", _t22),
        ("T-23 HUD transitions", _t23), ("T-24 bbox residual", _t24),
        ("T-25 dim mismatch", _t25),
        ("T-26 dup-ohash", _t26), ("T-27 validation", _t27),
        ("T-28 mask contains", _t28),
        ("T-29 mask intersects / contains (v1.0.8)", _t29),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            logging.info("PASS  %s", name)
        except Exception as e:
            failed += 1
            logging.info("FAIL  %s: %s: %s", name, type(e).__name__, e)
    logging.info("")
    logging.info("%d/%d passed", len(tests) - failed, len(tests))
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    import sys as _sys
    logging.basicConfig(level=logging.INFO,
                        format="%(message)s", stream=_sys.stdout)
    _sys.exit(_run_self_tests())

In [ ]:
%%writefile goa/agent/action_pipeline.py
"""agent/action_pipeline.py — Action Pipeline + Delta Comparator. v1.0.5.

Two-tier watchdog: no-op gate + debounced object-level divergence, with
patch trigger and premise-failure abort.

Sources: DIR/2026/046 (directive), 043/044/045 (frozen constants),
DIR/2026/047 (R-series audit).

Changelog:
  v1.0    initial — DIR/2026/046.
  v1.0.1..v1.0.4  static-audit closure (see prior changelogs).
  v1.0.5  DIR/2026/047 R-series closure:
     R-F1 (HIGH) Delta packet was diffed against _prev_objects,
          which during a persistent divergence is already the
          diverging state.  Both frames identical -> empty packet.
          New field _last_stable_objects tracks the last matching
          frame's objects; survival and premise-failure branches
          diff against it (fallback _prev_objects).  New T-22
          exercises a persistent stable-jump divergence.
     L-1 (LOW)  step() resets _retry_count when the incoming action
          differs from the previously dispatched one.  _retry_history
          capped at RETRY_HISTORY_CAP (50) entries.

Determinism: no threads, no I/O, no print(), no wall-clock, no
unseeded random.  Double run produces byte-equal event-log JSON.

Deviations from DIR/2026/046:
  D-1  §3.1 per-game values not in directive text; clean/reject
       globals (21+4).
  D-2  §4.3 retry-budget phrasing self-contradictory; parenthetical
       reading chosen (retries after original: clean=2, reject=3).
  D-3  §4.4 fingerprint includes a redundant total-object count.
  D-4  observe() raises DivergenceError on first survival.
  D-5  §4.8 diagnose-then-patch implemented as injectable
       patch_hook; harness supplies the wire.
  D-6  T-14 try/except wrapper added in v1.0.1.
  D-7  Debounce ticks on held frames per §4.5 "hold for
       debounce_steps".
  D-8  Post-patch reset semantics: PASS resets the tracker (fix
       worked); FAIL / no-prediction keeps it open so persistent
       divergence escalates.
  D-9  §4.8 delta baseline was unspecified; v1.0.5 uses the last
       stable frame's objects (R-F1 fix).
"""
from __future__ import annotations
import hashlib, json, logging, statistics
from collections import Counter
from dataclasses import dataclass
from types import MappingProxyType
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

logger = logging.getLogger(__name__)

try:
    from agent import grid_text as _gt
except ImportError:
    try:
        import grid_text as _gt
    except ImportError:
        _gt = None

try:
    from agent import inference_layer as _il
except ImportError:
    _il = None

try:
    from agent import context_manager as _cm
except ImportError:
    _cm = None

__all__ = [
    "PipelineError", "NoOpDetected", "DivergenceError",
    "PremiseFailureError", "CalibrationError", "StaleObservationError",
    "ActionResult", "ObserveResult", "CalibrationRow", "EventRecord",
    "DeltaPacket", "CLEAN_SET", "REJECT_LIST", "REJECT_LIST_IDS",
    "GLOBAL_DYNAMICS", "ALL_GAME_IDS", "DEFAULT_CALIBRATION",
    "LiveCalibration", "NoOpPredictor", "ActionPipeline",
    "MAX_DELTA_PACKET_CHARS", "RETRY_HISTORY_CAP",
]

CLEAN_SET: Mapping[str, Any] = MappingProxyType({
    "games_count": 21, "noop_pct": 3.78, "exact_repeat_pct": 1.00,
    "jitter_pct": 2.35, "retry_success_pct": 59.1,
    "switch_after_double_fail_pct": 3.9, "pause_retry_s": 1.10,
    "pause_fresh_s": 0.57, "median_revert_distance": 5,
})

REJECT_LIST_IDS: Tuple[str, ...] = (
    "lp85-305b61c3", "s5i5-18d95033", "vc33-5430563c",
    "dc22-fdcac232")

REJECT_LIST: Mapping[str, Any] = MappingProxyType({
    "games_count": 4, "noop_pct": 10.1, "exact_repeat_pct": 45.3,
    "retry_success_pct": 68.2, "retry_budget": 3,
})

# p90=200 verbatim from DIR/2026/046 §2.3.
GLOBAL_DYNAMICS: Mapping[str, Any] = MappingProxyType({
    "oscillation_pct": 57.8, "changeset_median": 54,
    "changeset_p90": 200,
})

ALL_GAME_IDS: Tuple[str, ...] = (
    "ar25-0c556536", "bp35-0a0ad940", "cd82-fb555c5d",
    "cn04-2fe56bfb", "dc22-fdcac232", "ft09-0d8bbf25",
    "g50t-5849a774", "ka59-38d34dbb", "lf52-271a04aa",
    "lp85-305b61c3", "ls20-9607627b", "m0r0-492f87ba",
    "r11l-495a7899", "re86-8af5384d", "s5i5-18d95033",
    "sb26-7fbdac44", "sc25-635fd71a", "sk48-d8078629",
    "sp80-589a99af", "su15-1944f8ab", "tn36-ef4dde99",
    "tr87-cd924810", "tu93-0768757b", "vc33-5430563c",
    "wa30-ee6fef47")

MAX_DELTA_PACKET_CHARS = 4000
LIVE_CALIB_N_TARGET = 30
# REVERT_WINDOW_MAX=15 verbatim from DIR/2026/046 §3.2 ("k in 1..15").
REVERT_WINDOW_MAX = 15
RETRY_HISTORY_CAP = 50


class PipelineError(RuntimeError):
    """Base typed failure for the action pipeline."""


class NoOpDetected(PipelineError):
    """A dispatched action produced zero cell change."""


class DivergenceError(PipelineError):
    """First surviving divergence; carries the DeltaPacket as .packet."""

    def __init__(self, message: str, packet: "DeltaPacket") -> None:
        super().__init__(message)
        self.packet = packet


class PremiseFailureError(PipelineError):
    """Two consecutive surviving divergences — abort the plan."""


class CalibrationError(PipelineError):
    """Calibration table lookup or live estimator failure."""


class StaleObservationError(PipelineError):
    """Checksum mismatch in the patch packet reference."""


@dataclass(frozen=True)
class CalibrationRow:
    game_id: str
    noop_rate: float
    retry_pct: float
    debounce_steps: int
    oscillation_pct: float
    is_reject: bool
    retry_budget: int

    def to_dict(self) -> Dict[str, Any]:
        return {"game_id": self.game_id, "noop_rate": self.noop_rate,
                "retry_pct": self.retry_pct,
                "debounce_steps": self.debounce_steps,
                "oscillation_pct": self.oscillation_pct,
                "is_reject": self.is_reject,
                "retry_budget": self.retry_budget}


def _build_default_calibration() -> Dict[str, CalibrationRow]:
    rows: Dict[str, CalibrationRow] = {}
    for gid in ALL_GAME_IDS:
        rj = gid in REJECT_LIST_IDS
        rows[gid] = CalibrationRow(
            game_id=gid,
            noop_rate=(REJECT_LIST["noop_pct"] if rj
                       else CLEAN_SET["noop_pct"]) / 100.0,
            retry_pct=(REJECT_LIST["retry_success_pct"] if rj
                       else CLEAN_SET["retry_success_pct"]) / 100.0,
            debounce_steps=CLEAN_SET["median_revert_distance"],
            oscillation_pct=GLOBAL_DYNAMICS["oscillation_pct"] / 100.0,
            is_reject=rj,
            retry_budget=REJECT_LIST["retry_budget"] if rj else 2)
    return rows


DEFAULT_CALIBRATION: Mapping[str, CalibrationRow] = MappingProxyType(
    _build_default_calibration())


class LiveCalibration:
    """Running estimator over the first N=30 transitions (§3.2)."""

    def __init__(self, n_target: int = LIVE_CALIB_N_TARGET) -> None:
        self.n_target = int(n_target)
        self.transitions = 0
        self.noop_count = 0
        self.revert_distances: List[int] = []
        self._history: List[List[List[int]]] = []

    def observe(self, old_frame, new_frame) -> None:
        self.transitions += 1
        if old_frame == new_frame:
            self.noop_count += 1
        if self._history:
            for k in range(1, min(REVERT_WINDOW_MAX,
                                  len(self._history)) + 1):
                ref = self._history[-k]
                if len(ref) != len(new_frame):
                    continue
                if any(ref[r] == new_frame[r]
                       for r in range(len(new_frame))):
                    self.revert_distances.append(k)
                    break
        self._history.append([list(r) for r in new_frame])
        if len(self._history) > REVERT_WINDOW_MAX:
            self._history.pop(0)

    @property
    def is_ready(self) -> bool:
        return self.transitions >= self.n_target

    @property
    def noop_rate(self) -> Optional[float]:
        return self.noop_count / self.transitions \
            if self.transitions else None

    @property
    def median_revert_distance(self) -> Optional[int]:
        return (int(statistics.median(self.revert_distances))
                if self.revert_distances else None)

    def to_dict(self) -> Dict[str, Any]:
        return {"transitions": self.transitions,
                "n_target": self.n_target,
                "is_ready": self.is_ready,
                "noop_rate": self.noop_rate,
                "median_revert_distance": self.median_revert_distance}


@dataclass
class EventRecord:
    kind: str
    tick: int
    data: Dict[str, Any]

    def to_dict(self) -> Dict[str, Any]:
        return {"kind": self.kind, "tick": self.tick, "data": self.data}


@dataclass
class ActionResult:
    action: Any
    attempt: int
    no_op: bool
    switch_flag: bool
    tick: int
    frame_before_checksum: Optional[str]

    def to_dict(self) -> Dict[str, Any]:
        return {"action": repr(self.action), "attempt": self.attempt,
                "no_op": self.no_op, "switch_flag": self.switch_flag,
                "tick": self.tick,
                "frame_before_checksum": self.frame_before_checksum}


@dataclass
class ObserveResult:
    first: bool = False
    no_op: bool = False
    retry_queued: bool = False
    switch_flag_set: bool = False
    debounce_started: bool = False
    debounce_decremented: bool = False
    divergence: bool = False
    premise_failure: bool = False
    match: bool = False
    prediction_pass: bool = False
    prediction_fail: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return dict(self.__dict__)


@dataclass
class DeltaPacket:
    game_id: str
    moved: List[list]
    recolored: List[list]
    new: List[list]
    gone: List[list]
    retry_history: List[Dict[str, Any]]
    calibration_row: Dict[str, Any]
    previous_checksums: Dict[str, str]
    needs_full: bool
    char_count: int
    predicted_post_state: Optional[Dict[str, Any]] = None

    def to_dict(self) -> Dict[str, Any]:
        return {"game_id": self.game_id, "moved": self.moved,
                "recolored": self.recolored, "new": self.new,
                "gone": self.gone, "retry_history": self.retry_history,
                "calibration_row": self.calibration_row,
                "previous_checksums": self.previous_checksums,
                "needs_full": self.needs_full,
                "char_count": self.char_count,
                "predicted_post_state": self.predicted_post_state}


class NoOpPredictor:
    """Default hook — no prediction; comparator runs in drift mode."""

    def __call__(self, state, action):
        return None


def _require_grid_text() -> None:
    if _gt is None:
        raise PipelineError(
            "grid_text module unavailable — cannot extract objects "
            "or compute checksums")


def _extract(frame):
    _require_grid_text()
    return _gt.extract_objects(frame, background=0)


def _rle_checksum(frame) -> str:
    _require_grid_text()
    return _gt.rle_anchor(frame)["checksum"]


def _fingerprint(objects) -> Tuple[Tuple[Tuple[str, int], ...], int]:
    counts = Counter(o.ohash for o in objects)
    return (tuple(sorted(counts.items())), len(objects))


def _fp_to_dict(fp) -> Dict[str, Any]:
    return {"ohash_counts": [list(x) for x in fp[0]], "total": fp[1]}


def _dict_to_fp(d) -> Tuple[Tuple[Tuple[str, int], ...], int]:
    return (tuple((str(k), int(v)) for k, v in d["ohash_counts"]),
            int(d["total"]))


def _diff_objects(prev_objs, curr_objs) -> Dict[str, List[list]]:
    if _gt is None:
        return {"moved": [], "recolored": [], "new": [], "gone": []}
    return _gt.object_delta(prev_objs, curr_objs)


class ActionPipeline:
    """Two-tier action watchdog.  Lifecycle:
        submit_plan(actions); step(); observe(settled_frame)

    No-op gate and divergence check run in parallel.  On survival the
    DivergenceError fires with a DeltaPacket diffed against the last
    STABLE frame (not the previous observation).  The next observation
    resets (prediction PASS) or keeps the tracker open (FAIL / none).
    """

    def __init__(
        self,
        game_id: str,
        dispatch: Optional[Callable[[Any], None]] = None,
        predict_hook: Optional[Callable[[Any, Any], Any]] = None,
        patch_hook: Optional[Callable[["DeltaPacket"], Any]] = None,
        calibration: Optional[Mapping[str, CalibrationRow]] = None,
    ) -> None:
        self.game_id = game_id
        if calibration is not None:
            self._calib: Dict[str, CalibrationRow] = dict(calibration)
        else:
            self._calib = dict(DEFAULT_CALIBRATION)
        row = self._calib.get(game_id)
        if row is None:
            row = CalibrationRow(
                game_id=game_id,
                noop_rate=CLEAN_SET["noop_pct"] / 100.0,
                retry_pct=CLEAN_SET["retry_success_pct"] / 100.0,
                debounce_steps=CLEAN_SET["median_revert_distance"],
                oscillation_pct=GLOBAL_DYNAMICS["oscillation_pct"] / 100.0,
                is_reject=False, retry_budget=2)
        self._row = row
        self._dispatch = dispatch
        self._predict = predict_hook or NoOpPredictor()
        self._patch_hook = patch_hook
        self._live = LiveCalibration()
        self._queue: List[Any] = []
        self._prev_settled: Optional[List[List[int]]] = None
        self._prev_checksum: Optional[str] = None
        self._prev_objects: Optional[List] = None
        # R-F1: baseline for the delta packet — objects of the last
        # frame whose fingerprint matched.  Kept alongside
        # _last_stable_fp so survival can diff against it.
        self._last_stable_objects: Optional[List] = None
        self._last_stable_fp: Optional[Tuple] = None
        self._events: List[EventRecord] = []
        self._tick = 0
        self._retry_count = 0
        self._current_action: Optional[Any] = None
        self._current_predicted_fp: Optional[Tuple] = None
        self._switch_flag = False
        self._debounce_remaining = 0
        self._consecutive_divergences = 0
        self._retry_history: List[Dict[str, Any]] = []
        self._last_patch: Optional[DeltaPacket] = None
        self._pending_prediction: Optional[Tuple] = None
        self._post_patch_pending = False

    def check_integration_deps(self) -> Dict[str, bool]:
        return {
            "grid_text": _gt is not None,
            "context_manager": _cm is not None,
            "inference_layer": _il is not None,
        }

    def _record(self, kind: str,
                data: Optional[Dict[str, Any]] = None) -> None:
        self._events.append(EventRecord(kind=kind, tick=self._tick,
                                        data=dict(data or {})))

    def event_log(self) -> List[EventRecord]:
        return list(self._events)

    def event_log_json(self) -> str:
        return json.dumps([e.to_dict() for e in self._events],
                          sort_keys=True)

    def last_patch(self) -> Optional[DeltaPacket]:
        return self._last_patch

    @property
    def switch_flag(self) -> bool:
        return self._switch_flag

    def submit_plan(self, actions: Sequence[Any]) -> None:
        self._queue = list(actions)
        self._record("plan_submitted", {"count": len(actions)})

    def step(self) -> Optional[ActionResult]:
        if not self._queue:
            return None
        action = self._queue.pop(0)
        # L-1: reset the retry counter when a NEW action is
        # dispatched.  A quiescent streak that previously exhausted
        # its retry budget would otherwise deny a freshly-issued
        # action its retry budget.
        if self._current_action is not None and action != self._current_action:
            self._retry_count = 0
        self._current_action = action
        self._tick += 1
        if self._prev_settled is not None:
            self._current_predicted_fp = self._predict(
                self._prev_settled, action)
        if self._dispatch is not None:
            self._dispatch(action)
        self._record("action_dispatched",
                     {"action": repr(action),
                      "queue_remaining": len(self._queue)})
        return ActionResult(
            action=action, attempt=self._retry_count + 1, no_op=False,
            switch_flag=self._switch_flag, tick=self._tick,
            frame_before_checksum=self._prev_checksum)

    def observe(self, settled_frame) -> ObserveResult:
        last = [list(row) for row in settled_frame[-1]]
        checksum = _rle_checksum(last)

        if self._prev_settled is None:
            self._prev_settled = last
            self._prev_checksum = checksum
            self._prev_objects = _extract(last)
            self._last_stable_fp = _fingerprint(self._prev_objects)
            self._last_stable_objects = self._prev_objects
            self._record("first_observation", {"checksum": checksum})
            return ObserveResult(first=True)

        pred_result = self._verify_pending_prediction(last)

        if self._post_patch_pending:
            self._post_patch_pending = False
            if pred_result == "PASS":
                actual_fp = _fingerprint(_extract(last))
                self._last_stable_fp = actual_fp
                self._last_stable_objects = _extract(last)
                self._debounce_remaining = 0
                self._consecutive_divergences = 0
                self._record("post_patch_reset",
                             {"kept_open": False, "prediction": "PASS"})
            else:
                self._record("post_patch_reset", {
                    "kept_open": True,
                    "prediction": pred_result or "none"})

        state_changed = (checksum != self._prev_checksum)
        retry_queued = False
        switch_set = False
        if state_changed:
            self._retry_count = 0
            self._switch_flag = False
        else:
            retry_queued, switch_set = self._handle_no_op(checksum)

        new_objects = _extract(last)
        div_res = self._check_divergence(new_objects)

        self._live.observe(self._prev_settled, last)
        self._prev_settled = last
        self._prev_checksum = checksum
        self._prev_objects = new_objects

        div_res.no_op = (not state_changed)
        div_res.retry_queued = retry_queued
        div_res.switch_flag_set = switch_set
        if pred_result is not None:
            div_res.prediction_pass = (pred_result == "PASS")
            div_res.prediction_fail = (pred_result == "FAIL")

        if div_res.premise_failure:
            self._clear_queue()
            raise PremiseFailureError(
                f"two consecutive surviving divergences on "
                f"{self.game_id!r}")
        if div_res.divergence:
            raise DivergenceError(
                f"divergence survived debounce on {self.game_id!r}",
                self._last_patch)
        return div_res

    def _verify_pending_prediction(self, frame) -> Optional[str]:
        if self._pending_prediction is None:
            return None
        actual_fp = _fingerprint(_extract(frame))
        passed = (actual_fp == self._pending_prediction)
        result = "PASS" if passed else "FAIL"
        self._record("prediction_verified", {
            "result": result,
            "expected": _fp_to_dict(self._pending_prediction),
            "actual": _fp_to_dict(actual_fp),
        })
        self._pending_prediction = None
        return result

    def _handle_no_op(self, checksum: str) -> Tuple[bool, bool]:
        self._retry_count += 1
        self._retry_history.append({
            "action": repr(self._current_action),
            "attempt": self._retry_count, "no_op": True,
            "checksum": checksum})
        # L-1: cap the history.  The packet carries the tail.
        if len(self._retry_history) > RETRY_HISTORY_CAP:
            self._retry_history = \
                self._retry_history[-RETRY_HISTORY_CAP:]
        self._record("no_op_detected",
                     {"attempt": self._retry_count,
                      "action": repr(self._current_action)})
        if self._retry_count > self._row.retry_budget:
            self._switch_flag = True
            self._record("switch_flag_set",
                         {"after_attempts": self._retry_count})
            return False, True
        self._queue.insert(0, self._current_action)
        return True, False

    def _delta_baseline_objects(self):
        """R-F1: the objects the patch delta should be measured
        against.  Prefer the last stable frame; fall back to the
        previous observation when no stable baseline has been
        captured yet."""
        if self._last_stable_objects is not None:
            return self._last_stable_objects
        return self._prev_objects or []

    def _check_divergence(self, new_objects) -> ObserveResult:
        predicted = self._current_predicted_fp
        actual_fp = _fingerprint(new_objects)
        if predicted is None:
            if self._last_stable_fp is None:
                self._last_stable_fp = actual_fp
                self._last_stable_objects = new_objects
                return ObserveResult(match=True)
            mismatch = (actual_fp != self._last_stable_fp)
        else:
            mismatch = (actual_fp != predicted)

        if not mismatch:
            self._debounce_remaining = 0
            self._consecutive_divergences = 0
            self._last_stable_fp = actual_fp
            self._last_stable_objects = new_objects
            self._record("object_match",
                         {"fingerprint_size": len(actual_fp[0])})
            return ObserveResult(match=True)

        if self._consecutive_divergences >= 1:
            self._consecutive_divergences += 1
            # R-F1: diff against the last STABLE objects.
            baseline = self._delta_baseline_objects()
            delta = _diff_objects(baseline, new_objects)
            self._last_patch = self._build_delta_packet(delta, None)
            self._record("premise_failure",
                         {"consecutive": self._consecutive_divergences})
            return ObserveResult(divergence=True, premise_failure=True)

        if self._debounce_remaining == 0:
            window = self._current_debounce_window()
            self._debounce_remaining = window
            self._record("debounce_started", {"window": window})
            return ObserveResult(debounce_started=True)

        self._debounce_remaining -= 1
        if self._debounce_remaining > 0:
            self._record("debounce_decremented",
                         {"remaining": self._debounce_remaining})
            return ObserveResult(debounce_decremented=True)

        # Survival.  R-F1: diff against the last STABLE objects.
        self._consecutive_divergences = 1
        baseline = self._delta_baseline_objects()
        delta = _diff_objects(baseline, new_objects)
        prediction = None
        if self._patch_hook is not None:
            provisional = self._build_delta_packet(delta, None)
            try:
                prediction = self._patch_hook(provisional)
            except Exception as exc:  # noqa: BLE001
                self._record("patch_hook_error", {"error": str(exc)})
                prediction = None
        self._last_patch = self._build_delta_packet(delta, prediction)
        if self._last_patch.predicted_post_state is not None:
            try:
                self._pending_prediction = _dict_to_fp(
                    self._last_patch.predicted_post_state)
            except (KeyError, TypeError, ValueError):
                self._pending_prediction = None
        elif prediction is not None:
            self._record("prediction_dropped_for_budget", {})
        self._post_patch_pending = True
        self._record("patch_triggered", {
            "char_count": self._last_patch.char_count,
            "has_prediction": self._last_patch.predicted_post_state
                              is not None,
            "delta_moved": len(self._last_patch.moved),
            "delta_new": len(self._last_patch.new),
            "delta_gone": len(self._last_patch.gone),
        })
        self._record("divergence_survived", {"consecutive": 1})
        return ObserveResult(divergence=True)

    def _current_debounce_window(self) -> int:
        if self._live.is_ready:
            med = self._live.median_revert_distance
            if med is not None:
                return int(med)
        return self._row.debounce_steps

    def _build_delta_packet(self, delta,
                            prediction: Optional[Dict[str, Any]]
                            ) -> DeltaPacket:
        packet = DeltaPacket(
            game_id=self.game_id,
            moved=list(delta.get("moved", [])),
            recolored=list(delta.get("recolored", [])),
            new=list(delta.get("new", [])),
            gone=list(delta.get("gone", [])),
            retry_history=list(self._retry_history),
            calibration_row=self._row.to_dict(),
            previous_checksums={
                "prev_settled": self._prev_checksum or ""},
            needs_full=False, char_count=0,
            predicted_post_state=prediction)
        js = json.dumps(packet.to_dict(), sort_keys=True)
        if len(js) > MAX_DELTA_PACKET_CHARS:
            packet.moved = []
            packet.recolored = []
            packet.new = []
            packet.gone = []
            packet.needs_full = True
            js = json.dumps(packet.to_dict(), sort_keys=True)
            if len(js) > MAX_DELTA_PACKET_CHARS:
                packet.predicted_post_state = None
                js = json.dumps(packet.to_dict(), sort_keys=True)
        packet.char_count = len(js)
        return packet

    def verify_observation(self, packet: DeltaPacket,
                           current_checksum: str) -> bool:
        prev = packet.previous_checksums.get("prev_settled")
        if prev and prev != current_checksum:
            raise StaleObservationError(
                f"packet referenced {prev!r}, current is "
                f"{current_checksum!r}")
        return True

    def _clear_queue(self) -> None:
        self._queue = []
        self._record("queue_cleared", {})

    def compare(self, prev_frame, curr_frame) -> Dict[str, Any]:
        prev_objs = _extract(prev_frame)
        curr_objs = _extract(curr_frame)
        prev_fp = _fingerprint(prev_objs)
        curr_fp = _fingerprint(curr_objs)
        return {"prev_fp": prev_fp, "curr_fp": curr_fp,
                "delta": _diff_objects(prev_objs, curr_objs),
                "match": prev_fp == curr_fp}


# ======================================================================
# Inline self-tests (v1.0.5, T-1..T-22)
# ======================================================================

def _eg(v=0, h=64, w=64):
    return [[v] * w for _ in range(h)]


def _pb(g, r0, c0, h, w, v):
    for r in range(r0, r0 + h):
        for c in range(c0, c0 + w):
            g[r][c] = v


def _fast_cal(game_id="ls20-9607627b", debounce=1):
    cal = dict(DEFAULT_CALIBRATION)
    cal[game_id] = CalibrationRow(
        game_id, 0.038, 0.591, debounce, 0.578, False, 2)
    return cal


def _t1():
    assert CLEAN_SET["games_count"] == 21
    assert CLEAN_SET["noop_pct"] == 3.78
    assert CLEAN_SET["exact_repeat_pct"] == 1.00
    assert CLEAN_SET["jitter_pct"] == 2.35
    assert CLEAN_SET["retry_success_pct"] == 59.1
    assert CLEAN_SET["switch_after_double_fail_pct"] == 3.9
    assert CLEAN_SET["pause_retry_s"] == 1.10
    assert CLEAN_SET["pause_fresh_s"] == 0.57
    assert CLEAN_SET["median_revert_distance"] == 5
    assert REJECT_LIST_IDS == ("lp85-305b61c3", "s5i5-18d95033",
                               "vc33-5430563c", "dc22-fdcac232")
    assert REJECT_LIST["noop_pct"] == 10.1
    assert REJECT_LIST["exact_repeat_pct"] == 45.3
    assert REJECT_LIST["retry_success_pct"] == 68.2
    assert REJECT_LIST["retry_budget"] == 3
    assert GLOBAL_DYNAMICS["oscillation_pct"] == 57.8
    assert GLOBAL_DYNAMICS["changeset_median"] == 54
    assert GLOBAL_DYNAMICS["changeset_p90"] == 200
    assert len(ALL_GAME_IDS) == 25
    assert len(DEFAULT_CALIBRATION) == 25


def _t2():
    p = ActionPipeline("ls20-9607627b")
    g = _eg(0)
    p.submit_plan(["right"]); p.step(); p.observe([g])
    p.submit_plan(["right"]); p.step()
    r = p.observe([g])
    assert r.no_op is True
    assert any(e.kind == "no_op_detected" for e in p.event_log())


def _t3():
    p = ActionPipeline("ls20-9607627b")
    g = _eg(0)
    p.submit_plan(["right"]); p.step(); p.observe([g])
    p.submit_plan(["right"]); p.step()
    r = p.observe([g]); assert r.no_op and r.retry_queued
    p.step(); r = p.observe([g]); assert r.no_op and r.retry_queued
    p.step(); r = p.observe([g])
    assert r.no_op and r.switch_flag_set
    assert p.switch_flag


def _t4():
    p = ActionPipeline("ls20-9607627b")
    g = _eg(0)
    p.submit_plan(["right"]); p.step(); p.observe([g])
    p.submit_plan(["right"]); p.step(); p.observe([g])
    assert p._retry_count >= 1
    g2 = _eg(0); _pb(g2, 10, 10, 2, 2, 5)
    p.submit_plan(["right"]); p.step(); p.observe([g2])
    assert p._retry_count == 0


def _t5():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0); _pb(a, 5, 5, 2, 2, 5)
    b = _eg(0); _pb(b, 20, 20, 2, 2, 5)
    cmp = p.compare(a, b)
    assert cmp["match"] is True
    assert len(cmp["delta"]["moved"]) >= 1


def _t6():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step(); r = p.observe([a])
    assert r.match is True
    assert p.last_patch() is None


def _t7():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); r = p.observe([b])
    assert r.debounce_started is True
    assert p.last_patch() is None
    assert any(e.kind == "debounce_started" for e in p.event_log())


def _t8():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    for i in range(LIVE_CALIB_N_TARGET):
        c = _eg(0); c[i % 60][i % 60] = 5
        p.submit_plan(["x"]); p.step()
        try:
            p.observe([c])
        except (DivergenceError, PremiseFailureError):
            pass
    assert p._live.is_ready is True
    assert p._live.median_revert_distance is not None


def _t9():
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b1 = _eg(0); _pb(b1, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step()
    r = p.observe([b1])
    assert r.debounce_started
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b1])
    except DivergenceError:
        pass
    else:
        raise AssertionError("expected DivergenceError on first survival")
    assert p._consecutive_divergences == 1
    b2 = _eg(0); _pb(b2, 10, 10, 2, 2, 5)
    p.submit_plan(["x", "y"]); p.step()
    try:
        p.observe([b2])
    except PremiseFailureError:
        pass
    else:
        raise AssertionError("expected PremiseFailureError")
    assert p._queue == []
    assert any(e.kind == "post_patch_reset"
               and e.data.get("kept_open") is True
               for e in p.event_log())


def _t10():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step(); r = p.observe([a])
    assert r.match is True
    assert p.last_patch() is None


def _t11():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0); _pb(a, 5, 5, 2, 2, 5)
    b = _eg(0); _pb(b, 20, 20, 2, 2, 5)
    delta = _diff_objects(_extract(a), _extract(b))
    assert len(delta["moved"]) >= 1
    entry = delta["moved"][0]
    assert isinstance(entry[1], str) and len(entry[1]) == 32


def _t12():
    p = ActionPipeline("ls20-9607627b")
    huge = {"moved": [[i, "h" * 40, [i] * 4, [i] * 4]
                      for i in range(500)],
            "recolored": [], "new": [], "gone": []}
    packet = p._build_delta_packet(huge, None)
    assert packet.char_count <= MAX_DELTA_PACKET_CHARS
    assert packet.needs_full is True
    assert packet.moved == []


def _t13():
    p = ActionPipeline("ls20-9607627b")
    packet = DeltaPacket(
        game_id="ls20-9607627b", moved=[], recolored=[], new=[], gone=[],
        retry_history=[], calibration_row={},
        previous_checksums={"prev_settled": "abc123"},
        needs_full=False, char_count=50)
    try:
        p.verify_observation(packet, "different")
    except StaleObservationError:
        return
    raise AssertionError("expected StaleObservationError")


def _t14():
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b])
    except DivergenceError:
        pass
    packet = p.last_patch()
    assert packet is not None
    assert packet.calibration_row["game_id"] == "ls20-9607627b"
    assert "noop_rate" in packet.calibration_row
    assert "debounce_steps" in packet.calibration_row
    assert isinstance(packet.retry_history, list)


def _t15():
    a = _eg(0); _pb(a, 5, 5, 2, 2, 5)
    b = _eg(0); _pb(b, 20, 20, 2, 2, 5)
    fp_b = _fingerprint(_extract(b))
    p = ActionPipeline("ls20-9607627b",
                       predict_hook=lambda s, act: fp_b)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    p.submit_plan(["x"]); p.step(); r = p.observe([b])
    assert r.match is True
    fp_empty = ((), 0)
    p2 = ActionPipeline("ls20-9607627b",
                        predict_hook=lambda s, act: fp_empty)
    p2.submit_plan(["x"]); p2.step(); p2.observe([a])
    p2.submit_plan(["x"]); p2.step(); r2 = p2.observe([b])
    assert r2.debounce_started is True


def _t16():
    p1 = ActionPipeline("ls20-9607627b")
    assert p1._row.retry_budget == 2
    assert p1._row.is_reject is False
    p2 = ActionPipeline("lp85-305b61c3")
    assert p2._row.retry_budget == 3
    assert p2._row.is_reject is True


def _t17():
    def run():
        p = ActionPipeline("ls20-9607627b")
        g = _eg(0)
        for _ in range(5):
            p.submit_plan(["x"]); p.step(); p.observe([g])
        return p.event_log_json()
    assert run() == run()


def _t18():
    import random as _r
    rng = _r.Random(2026)
    for _ in range(200):
        p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
        for i in range(20):
            g = [[rng.randrange(16) for _ in range(8)]
                 for _ in range(8)]
            p.submit_plan([f"a{i}"]); p.step()
            try:
                p.observe([g])
            except (DivergenceError, PremiseFailureError):
                pass
        assert p._retry_count <= p._row.retry_budget + 1
        for e in p.event_log():
            assert isinstance(e.kind, str)
            assert isinstance(e.tick, int)


def _t19():
    post = _eg(0); _pb(post, 30, 30, 2, 2, 5)
    pred_pass = _fp_to_dict(_fingerprint(_extract(post)))

    p1 = ActionPipeline(
        "ls20-9607627b", calibration=_fast_cal(),
        patch_hook=lambda packet: pred_pass)
    a = _eg(0)
    p1.submit_plan(["x"]); p1.step(); p1.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p1.submit_plan(["x"]); p1.step(); p1.observe([b])
    p1.submit_plan(["x"]); p1.step()
    try:
        p1.observe([b])
    except DivergenceError:
        pass
    assert p1.last_patch().predicted_post_state == pred_pass
    p1.submit_plan(["x"]); p1.step()
    r1 = p1.observe([post])
    assert r1.prediction_pass is True
    assert any(e.kind == "prediction_verified"
               and e.data.get("result") == "PASS"
               for e in p1.event_log())
    assert p1._consecutive_divergences == 0
    assert p1._post_patch_pending is False

    pred_fail = _fp_to_dict(_fingerprint(_extract(a)))
    p2 = ActionPipeline(
        "ls20-9607627b", calibration=_fast_cal(),
        patch_hook=lambda packet: pred_fail)
    p2.submit_plan(["x"]); p2.step(); p2.observe([a])
    b2 = _eg(0); _pb(b2, 5, 5, 2, 2, 5)
    p2.submit_plan(["x"]); p2.step(); p2.observe([b2])
    p2.submit_plan(["x"]); p2.step()
    try:
        p2.observe([b2])
    except DivergenceError:
        pass
    p2.submit_plan(["x"]); p2.step()
    escalated = False
    try:
        p2.observe([post])
    except PremiseFailureError:
        escalated = True
    assert escalated, "failed patch must escalate to PremiseFailureError"
    assert any(e.kind == "prediction_verified"
               and e.data.get("result") == "FAIL"
               for e in p2.event_log())
    assert any(e.kind == "post_patch_reset"
               and e.data.get("kept_open") is True
               for e in p2.event_log())


def _t20():
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step()
    r0 = p.observe([b])
    assert r0.debounce_started
    assert p._debounce_remaining == 1
    p.submit_plan(["x"]); p.step()
    raised = False
    try:
        p.observe([b])
    except DivergenceError:
        raised = True
    assert raised
    kinds = [e.kind for e in p.event_log()]
    assert "no_op_detected" in kinds
    assert "divergence_survived" in kinds


def _t21():
    post = _eg(0); _pb(post, 30, 30, 2, 2, 5)
    pred_pass = _fp_to_dict(_fingerprint(_extract(post)))
    p = ActionPipeline(
        "ls20-9607627b", calibration=_fast_cal(),
        patch_hook=lambda packet: pred_pass)
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b])
    except DivergenceError:
        pass
    assert p._consecutive_divergences == 1
    assert p._post_patch_pending is True
    p.submit_plan(["x"]); p.step()
    p.observe([post])
    assert p._consecutive_divergences == 0
    assert p._post_patch_pending is False
    assert p._last_stable_fp == _fingerprint(_extract(post))


def _t22():
    """R-F1: a persistent stable-jump divergence must produce a
    NON-empty delta packet.  The bug in v1.0.4 diffed against
    _prev_objects which, on a persistent divergence, is already the
    diverging frame."""
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    # Stable baseline: one object at (5,5).
    a = _eg(0); _pb(a, 5, 5, 3, 3, 5)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    # Jump to a stable-but-divergent state: one object at (40,40).
    b = _eg(0); _pb(b, 40, 40, 3, 3, 8)
    p.submit_plan(["x"]); p.step()
    r1 = p.observe([b])   # debounce_started
    assert r1.debounce_started
    # Hold the divergent state; survival fires on the next observe.
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b])   # held -> survival -> DivergenceError
    except DivergenceError as exc:
        packet = exc.packet
    else:
        raise AssertionError("expected DivergenceError")
    assert packet is not None
    # R-F1: the delta MUST describe the difference vs the stable
    # baseline (a), NOT vs b (which would be empty).
    total = len(packet.moved) + len(packet.recolored) + \
            len(packet.new) + len(packet.gone)
    assert total >= 1, (
        f"R-F1 regression: empty delta at survival. "
        f"moved={packet.moved} new={packet.new} "
        f"gone={packet.gone} recolored={packet.recolored}")


def _run_self_tests() -> int:
    tests = [
        ("T-1  frozen constants", _t1),
        ("T-2  no-op gate fires", _t2),
        ("T-3  retry policy max attempts", _t3),
        ("T-4  retry counter resets", _t4),
        ("T-5  moved object not divergence", _t5),
        ("T-6  oscillation tolerance", _t6),
        ("T-7  debounce first mismatch transient", _t7),
        ("T-8  debounce live estimate", _t8),
        ("T-9  divergence -> premise failure", _t9),
        ("T-10 transient self-corrects", _t10),
        ("T-11 delta packet structure", _t11),
        ("T-12 delta packet budget", _t12),
        ("T-13 stale observation", _t13),
        ("T-14 patch contains retry + calibration", _t14),
        ("T-15 falsifiable prediction (forward hook)", _t15),
        ("T-16 reject-list retry budget", _t16),
        ("T-17 determinism", _t17),
        ("T-18 fuzz", _t18),
        ("T-19 patch prediction PASS/FAIL logged", _t19),
        ("T-20 held frame advances debounce", _t20),
        ("T-21 post-patch reset (D-8)", _t21),
        ("T-22 stable-baseline delta (R-F1)", _t22),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            logging.info("PASS  %s", name)
        except Exception as e:
            failed += 1
            logging.info("FAIL  %s: %s: %s", name, type(e).__name__, e)
    logging.info("")
    logging.info("%d/%d passed", len(tests) - failed, len(tests))
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    import sys as _sys
    logging.basicConfig(level=logging.INFO,
                        format="%(message)s", stream=_sys.stdout)
    _sys.exit(_run_self_tests())

In [ ]:
%%writefile goa/agent/sandbox_v6.py
# sandbox_v6.py
"""sandbox_v6.py — 3-Layer Execution Defense. DIR-2026-076 §2. R11.

Fixes applied across rounds:
  A-P0-01 (CRITICAL)  no module-level non-stdlib import; lazy guarded
  A-P0-03 (HIGH)      safe __import__ in child builtins
  A-P0-04 (HIGH)      numpy I/O blocked across every alias
  A-P0-07 (MEDIUM)    canonical SAFE_BUILTIN_NAMES (drift-tested)
  A-P0-10 (LOW)       delattr retained as documented hardening
  A-P0-11 (LOW)       extended np I/O list documented as hardening
  A-P0-14 (CRITICAL)  full-AST numpy alias collector
  A-P0-20 (LOW)       from numpy import * -> disallowed_construct
  A-P0-26 (HIGH)      nested attribute + from-submodule bypass closed
  A-P0-28 (LOW)       module-level docstrings/constants allowed
  A-P0-32 (LOW)       collector stores root binding name only
  A-P0-33 (LOW)       dead `star_imported` field removed
  A-P0-34 (LOW)       _is_module_node_allowed narrowed to scalar constants
  A-P0-37..39         open_memmap / genfromtxt / fromregex / fromstring /
                      fromiter added to _NP_IO_ATTRS
  A-FN-01/02/03 (LOW) T_SUITE docstring traceability
  R11 (LABEL)         revision label bumped from R10 for traceability

FD-3: DIR §2.1 vs §2.3 self-contradiction satisfied by lazy loading.
`time` is stdlib, flagged for DIR correction.
"""
from __future__ import annotations
import ast
import json
import os
import subprocess
import sys
import time


class SandboxViolation(Exception):
    """Raised when candidate code fails Layer-1 analysis. .reason in
    {forbidden_import, dunder_access, dynamic_exec, global_stmt,
     io_access, disallowed_construct, namespace_violation}."""

    def __init__(self, reason: str, detail: str = ""):
        super().__init__(f"{reason}: {detail}" if detail else reason)
        self.reason = reason
        self.detail = detail


_ALLOWED_MODULE_NODES = (
    ast.FunctionDef, ast.AsyncFunctionDef, ast.Assign, ast.AnnAssign,
    ast.ClassDef, ast.Import, ast.ImportFrom,
)

_ALLOWED_IMPORTS = {"numpy", "math", "itertools", "collections"}

_BLACKLIST_NAMES = {
    "eval", "exec", "compile", "__import__",
    "getattr", "setattr", "delattr", "globals", "locals", "vars",
    "open", "input",
}

_NP_IO_ATTRS = frozenset({
    "save", "load", "fromfile", "tofile", "savetxt", "loadtxt",
    "savez", "savez_compressed", "memmap",
    "open_memmap", "genfromtxt", "fromregex", "fromstring", "fromiter",
})

_ALLOWED_CONSTANT_TYPES = (str, int, float, bool, type(None))


def _is_module_node_allowed(node: ast.AST) -> bool:
    if isinstance(node, _ALLOWED_MODULE_NODES):
        return True
    if isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant):
        return isinstance(node.value.value, _ALLOWED_CONSTANT_TYPES)
    return False


def _check_import(node: ast.AST) -> None:
    if isinstance(node, ast.Import):
        for alias in node.names:
            top = alias.name.split(".")[0]
            if top not in _ALLOWED_IMPORTS:
                raise SandboxViolation("forbidden_import", alias.name)
    elif isinstance(node, ast.ImportFrom):
        if node.level != 0 or node.module is None:
            raise SandboxViolation("forbidden_import", "relative import")
        top = node.module.split(".")[0]
        if top not in _ALLOWED_IMPORTS:
            raise SandboxViolation("forbidden_import", node.module)


class _NumpyAliasCollector(ast.NodeVisitor):
    def __init__(self) -> None:
        self.numpy_names: set = {"numpy"}

    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            top = alias.name.split(".")[0]
            if top == "numpy":
                self.numpy_names.add(alias.asname or "numpy")
        self.generic_visit(node)

    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        if node.module and node.module.split(".")[0] == "numpy":
            for alias in node.names:
                if alias.name == "*":
                    raise SandboxViolation(
                        "disallowed_construct",
                        "from numpy import * is disallowed")
                self.numpy_names.add(alias.asname or alias.name)
                if alias.name in _NP_IO_ATTRS:
                    raise SandboxViolation(
                        "io_access",
                        f"from numpy import {alias.name} is blocked")
        self.generic_visit(node)


def _attr_root_name(node: ast.AST):
    cur = node
    while isinstance(cur, ast.Attribute):
        cur = cur.value
    if isinstance(cur, ast.Name):
        return cur.id
    return None


def analyze(source: str) -> ast.Module:
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        raise SandboxViolation("disallowed_construct", f"syntax error: {e}")

    for node in tree.body:
        if not _is_module_node_allowed(node):
            raise SandboxViolation(
                "disallowed_construct",
                f"module-level {type(node).__name__} not allowed")
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            _check_import(node)

    collector = _NumpyAliasCollector()
    collector.visit(tree)

    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            _check_import(node)

        if isinstance(node, ast.Attribute):
            if node.attr.startswith("__") and node.attr.endswith("__"):
                raise SandboxViolation(
                    "dunder_access", f"attribute {node.attr!r}")
            if node.attr in _NP_IO_ATTRS:
                root = _attr_root_name(node)
                if root in collector.numpy_names:
                    raise SandboxViolation(
                        "io_access",
                        f"{root}.{node.attr} is blocked")

        if isinstance(node, ast.Name):
            if node.id in _BLACKLIST_NAMES:
                raise SandboxViolation(
                    "dynamic_exec", f"name {node.id!r} is blocked")
            if (node.id.startswith("__") and node.id.endswith("__")
                    and node.id != "__name__"):
                raise SandboxViolation(
                    "dunder_access", f"dunder name {node.id!r}")

        if isinstance(node, ast.Global):
            raise SandboxViolation("global_stmt", "global statement")
        if isinstance(node, ast.Nonlocal):
            raise SandboxViolation("global_stmt", "nonlocal statement")

    return tree


SAFE_BUILTIN_NAMES = (
    "abs", "all", "any", "bool", "dict", "divmod", "enumerate",
    "filter", "float", "frozenset", "int", "isinstance", "len", "list",
    "map", "max", "min", "next", "range", "reversed", "round", "set",
    "sorted", "str", "sum", "tuple", "zip", "print",
)


def _make_safe_builtins() -> dict:
    b = __builtins__
    return {n: (b[n] if isinstance(b, dict) else getattr(b, n))
            for n in SAFE_BUILTIN_NAMES}


def _lazy_import(name: str):
    try:
        return __import__(name)
    except Exception:
        return None


def build_namespace() -> dict:
    return {
        "__builtins__": _make_safe_builtins(),
        "np": _lazy_import("numpy"),
        "math": _lazy_import("math"),
        "itertools": _lazy_import("itertools"),
        "collections": _lazy_import("collections"),
    }


# IN-01 FIX (Indus, 2026-09-29): RLIMIT_NPROC=(1,1) blocks OpenBLAS
# pthread_create during numpy import in the child (verified: child dies at
# ~200ms with KeyboardInterrupt). Pinning thread counts to 1 via env vars
# fixes it with all rlimits intact (verified rc=0). Threads pinned to 1
# also keeps candidate execution deterministic.
_CHILD_THREAD_ENV = {
    "OPENBLAS_NUM_THREADS": "1",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
}


def _child_env() -> dict:
    env = dict(os.environ)
    env.update(_CHILD_THREAD_ENV)
    return env


def _set_limits(timeout_s: float, mem_mb: int) -> None:
    import resource
    try:
        hard = mem_mb * 1024 * 1024
        resource.setrlimit(resource.RLIMIT_AS, (hard, hard))
    except Exception:
        pass
    try:
        cpu = int(timeout_s) + 1
        resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu))
    except Exception:
        pass
    try:
        resource.setrlimit(resource.RLIMIT_NPROC, (1, 1))
    except Exception:
        pass
    try:
        resource.setrlimit(resource.RLIMIT_FSIZE, (0, 0))
    except Exception:
        pass


_CHILD_WRAPPER = r'''
import sys, json
sys.setrecursionlimit(200)

def _safe_default(o):
    try:
        import numpy as _np
        if isinstance(o, _np.ndarray):
            return o.tolist()
        if isinstance(o, _np.generic):
            return o.item()
    except Exception:
        pass
    try:
        return str(o)
    except Exception:
        return "<unserializable>"

def _emit(d):
    try:
        sys.stdout.write(json.dumps(d, default=_safe_default) + "\n")
        sys.stdout.flush()
    except Exception:
        sys.stdout.write(
            '{"ok": false, "result": null, "error": '
            '"serialization failure", "error_class": "ProtocolError"}\n')
        sys.stdout.flush()

_SAFE_NAMES = (
    "abs", "all", "any", "bool", "dict", "divmod", "enumerate",
    "filter", "float", "frozenset", "int", "isinstance", "len", "list",
    "map", "max", "min", "next", "range", "reversed", "round", "set",
    "sorted", "str", "sum", "tuple", "zip", "print",
)

_b = __builtins__
_SAFE = {n: (_b[n] if isinstance(_b, dict) else getattr(_b, n))
         for n in _SAFE_NAMES}

_ALLOWED_MODULES = {"numpy", "math", "itertools", "collections"}

def _safe_import(name, globals=None, locals=None, fromlist=(), level=0):
    top = (name or "").split(".")[0]
    if level != 0 or top not in _ALLOWED_MODULES:
        raise ImportError(f"module {name!r} not whitelisted in sandbox")
    real = _b["__import__"] if isinstance(_b, dict) else _b.__import__
    return real(name, globals, locals, fromlist, level)

_SAFE["__import__"] = _safe_import

try:
    raw = sys.stdin.read()
    payload = json.loads(raw)
    source = payload["source"]
    entry = payload["entry"]
    args = payload.get("args", {}) or {}
except Exception as e:
    _emit({"ok": False, "result": None,
           "error": f"payload error: {e}",
           "error_class": "ProtocolError"})
    sys.exit(0)

def _lazy(name):
    try:
        return _safe_import(name)
    except Exception:
        return None

try:
    ns = {
        "__builtins__": dict(_SAFE),
        "np": _lazy("numpy"),
        "math": _lazy("math"),
        "itertools": _lazy("itertools"),
        "collections": _lazy("collections"),
    }
    code = compile(source, "<candidate>", "exec")
    exec(code, ns)
    fn = ns.get(entry)
    if fn is None:
        raise NameError(f"entry {entry!r} not defined")
    result = fn(**args)
    _emit({"ok": True, "result": result, "error": None,
           "error_class": None})
except MemoryError as e:
    _emit({"ok": False, "result": None,
           "error": f"MemoryError: {e}",
           "error_class": "MemoryError"})
except RecursionError as e:
    _emit({"ok": False, "result": None,
           "error": f"RecursionError: {e}",
           "error_class": "RecursionError"})
except Exception as e:
    _emit({"ok": False, "result": None,
           "error": f"{type(e).__name__}: {e}",
           "error_class": type(e).__name__})
'''


def run_candidate(source: str, entry: str, args: dict,
                  timeout_s: float = 3.0, mem_mb: int = 512) -> dict:
    t0 = time.time()
    try:
        analyze(source)
    except SandboxViolation as e:
        return {
            "ok": False, "result": None, "error": str(e),
            "error_class": "SandboxViolation",
            "elapsed_ms": int((time.time() - t0) * 1000),
            "timeout": False, "mem_limit_hit": False,
            "stderr_tail": "",
        }
    payload = json.dumps({"source": source, "entry": entry, "args": args})
    popen_kwargs = dict(
        stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, text=True,
        env=_child_env(),
    )
    if os.name == "posix":
        popen_kwargs["preexec_fn"] = lambda: _set_limits(timeout_s, mem_mb)
    proc = subprocess.Popen(
        [sys.executable, "-I", "-c", _CHILD_WRAPPER], **popen_kwargs)
    timed_out = False
    out, err = "", ""
    try:
        out, err = proc.communicate(input=payload, timeout=timeout_s)
    except subprocess.TimeoutExpired:
        timed_out = True
        try: proc.kill()
        except Exception: pass
        try: out, err = proc.communicate(timeout=2.0)
        except Exception: pass
    elapsed_ms = int((time.time() - t0) * 1000)
    stderr_tail = (err or "")[-512:]
    if timed_out:
        return {
            "ok": False, "result": None,
            "error": f"timeout after {timeout_s}s",
            "error_class": "TimeoutError",
            "elapsed_ms": elapsed_ms,
            "timeout": True, "mem_limit_hit": False,
            "stderr_tail": stderr_tail,
        }
    if not out or not out.strip():
        return {
            "ok": False, "result": None,
            "error": "child produced no output",
            "error_class": "ProtocolError",
            "elapsed_ms": elapsed_ms,
            "timeout": False, "mem_limit_hit": False,
            "stderr_tail": stderr_tail,
        }
    last_line = out.strip().splitlines()[-1]
    try:
        res = json.loads(last_line)
    except Exception as e:
        return {
            "ok": False, "result": None,
            "error": f"bad child output: {e}; line={last_line[:120]!r}",
            "error_class": "ProtocolError",
            "elapsed_ms": elapsed_ms,
            "timeout": False, "mem_limit_hit": False,
            "stderr_tail": stderr_tail,
        }
    res.setdefault("ok", False)
    res.setdefault("result", None)
    res.setdefault("error", None)
    res.setdefault("error_class", None)
    res["elapsed_ms"] = elapsed_ms
    res["timeout"] = False
    res["stderr_tail"] = stderr_tail
    res["mem_limit_hit"] = (res.get("error_class") == "MemoryError")
    return res


In [ ]:
%%writefile goa/agent/counterexample_schema.json
{
  "$schema": "goa/coder-solver/counterexample/v1.0",
  "type": "object",
  "required": [
    "failure_location",
    "observed",
    "model_predicted",
    "diff_summary",
    "admissible_alternatives",
    "taxonomy_hint",
    "repair_scope"
  ],
  "properties": {
    "failure_location": {
      "type": "object",
      "required": [
        "function",
        "step",
        "action"
      ],
      "properties": {
        "function": {
          "type": "string",
          "enum": [
            "extract_objects",
            "transition",
            "reward",
            "next_action"
          ]
        },
        "step": {
          "type": "integer",
          "minimum": 0
        },
        "action": {
          "type": [
            "string",
            "null"
          ]
        },
        "object_ref": {
          "type": [
            "string",
            "null"
          ]
        }
      }
    },
    "observed": {
      "type": "object",
      "additionalProperties": {
        "type": "string"
      }
    },
    "model_predicted": {
      "type": "object",
      "additionalProperties": {
        "type": "string"
      }
    },
    "diff_summary": {
      "type": "object",
      "required": [
        "cells_differ",
        "first_mismatches",
        "shape_ok",
        "dtype_ok"
      ],
      "properties": {
        "cells_differ": {
          "type": "integer",
          "minimum": 0
        },
        "first_mismatches": {
          "type": "array",
          "items": {
            "type": "string"
          },
          "minItems": 0,
          "maxItems": 3
        },
        "shape_ok": {
          "type": "boolean"
        },
        "dtype_ok": {
          "type": "boolean"
        }
      }
    },
    "admissible_alternatives": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "minItems": 2,
      "maxItems": 4
    },
    "taxonomy_hint": {
      "type": "string",
      "enum": [
        "IndexError",
        "ValueError_shape",
        "timeout_rlimit",
        "replay_mismatch_few",
        "replay_mismatch_many"
      ]
    },
    "repair_scope": {
      "type": "string",
      "enum": [
        "change_only_rule_touching_counterexample",
        "replan_hypothesis",
        "bounds_guard",
        "reshape_guard",
        "complexity_reduction"
      ]
    }
  }
}


In [ ]:
%%writefile goa/agent/replay_verifier.py
# replay_verifier.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""Transition buffer + exact-replay admission (blueprint §C-1, §8).

Verification is free: the judge is deterministic Python replay, not an LLM.
Fail-closed: every helper degrades to a non-admitting verdict on any error.
"""
from __future__ import annotations
import json
from collections import deque

BUFFER_MAX = 64
MISMATCH_FEW_MAX = 3  # <=3 differing cells -> repair; >3 -> replan (frozen)


def _norm(grid) -> str:
    """Canonical serialization for exact comparison (astype-safe)."""
    if hasattr(grid, "tolist"):
        grid = grid.tolist()
    return json.dumps(grid, sort_keys=True)


class TransitionBuffer:
    """Bounded FIFO of (state, action, next_state)."""

    def __init__(self, maxlen: int = BUFFER_MAX):
        self._q = deque(maxlen=maxlen)

    def append(self, state, action, next_state) -> None:
        self._q.append((state, action, next_state))

    def as_list(self):
        return list(self._q)

    def __len__(self):
        return len(self._q)

    def clear(self) -> None:
        self._q.clear()


def replay_verify(source: str, run_candidate, buffer: "TransitionBuffer") -> dict:
    """Exact replay of `source`'s next_grid(state, action) over the buffer.

    Returns {"admitted": bool, "mismatches": int, "first_mismatches": [str],
             "error": str|None, "error_class": str|None}.
    """
    transitions = buffer.as_list()
    if not transitions:
        return {"admitted": False, "mismatches": 0, "first_mismatches": [],
                "error": "empty buffer", "error_class": "EmptyBuffer"}
    mismatches, first_mismatches = 0, []
    for state, action, next_state in transitions:
        r = run_candidate(source, "next_grid",
                          {"state": state, "action": action},
                          timeout_s=3.0)
        if not r.get("ok"):
            return {"admitted": False, "mismatches": 0, "first_mismatches": [],
                    "error": r.get("error"), "error_class": r.get("error_class")}
        try:
            if _norm(r.get("result")) != _norm(next_state):
                mismatches += 1
                if len(first_mismatches) < MISMATCH_FEW_MAX:
                    first_mismatches.append(f"t{mismatches}")
        except Exception as e:
            return {"admitted": False, "mismatches": 0, "first_mismatches": [],
                    "error": f"compare failed: {e}", "error_class": "CompareError"}
    return {"admitted": mismatches == 0, "mismatches": mismatches,
            "first_mismatches": first_mismatches, "error": None, "error_class": None}


def build_counterexample(function: str, step: int, action, observed: dict,
                         model_predicted: dict, cells_differ: int,
                         first_mismatches: list, shape_ok: bool = True,
                         dtype_ok: bool = True, taxonomy_hint: str = "replay_mismatch_few",
                         repair_scope: str = "change_only_rule_touching_counterexample",
                         alternatives: list = None) -> dict:
    """Schema-conformant counterexample (counterexample_schema.json, frozen)."""
    if alternatives is None:
        alternatives = ["player can push blue blocks",
                        "blue block is static + player moves diagonally",
                        "frame-diff encoding error"][:2]
    return {
        "failure_location": {"function": function, "step": step,
                             "action": action, "object_ref": None},
        "observed": {k: str(v) for k, v in observed.items()},
        "model_predicted": {k: str(v) for k, v in model_predicted.items()},
        "diff_summary": {"cells_differ": cells_differ,
                         "first_mismatches": [str(m) for m in first_mismatches[:3]],
                         "shape_ok": shape_ok, "dtype_ok": dtype_ok},
        "admissible_alternatives": [str(a) for a in alternatives[:4]],
        "taxonomy_hint": taxonomy_hint,
        "repair_scope": repair_scope,
    }


In [ ]:
%%writefile goa/agent/cegis_taxonomy.py
# cegis_taxonomy.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""Error taxonomy (blueprint §C-3): failure -> (taxonomy_hint, repair_scope)."""
from __future__ import annotations

TAXONOMY = ("IndexError", "ValueError_shape", "timeout_rlimit",
            "replay_mismatch_few", "replay_mismatch_many")

_SCOPE = {
    "IndexError": "bounds_guard",
    "ValueError_shape": "reshape_guard",
    "timeout_rlimit": "complexity_reduction",
    "replay_mismatch_few": "change_only_rule_touching_counterexample",
    "replay_mismatch_many": "replan_hypothesis",
}


def classify(error_class: str, mismatches: int = 0) -> tuple:
    """Map a runtime failure onto the frozen taxonomy.

    Rules (frozen):
      timeout / RLIMIT hits          -> timeout_rlimit
      MemoryError                    -> timeout_rlimit (resource exhaustion)
      IndexError                     -> IndexError
      ValueError w/ shape semantics   -> ValueError_shape
      replay mismatch count <= 3      -> replay_mismatch_few
      replay mismatch count >  3      -> replay_mismatch_many
    """
    ec = (error_class or "")
    if ec in ("TimeoutError", "MemoryError", "SandboxViolation"):
        hint = "timeout_rlimit" if ec != "SandboxViolation" else "IndexError"
        return (hint, _SCOPE[hint])
    if ec == "IndexError":
        return ("IndexError", _SCOPE["IndexError"])
    if ec == "ValueError":
        return ("ValueError_shape", _SCOPE["ValueError_shape"])
    if ec in ("ReplayMismatch", "CompareError", "EmptyBuffer"):
        hint = ("replay_mismatch_few" if mismatches <= 3
                else "replay_mismatch_many")
        return (hint, _SCOPE[hint])
    # unknown -> conservative default: replan
    return ("replay_mismatch_many", _SCOPE["replay_mismatch_many"])


In [ ]:
%%writefile goa/agent/cegis_router.py
# cegis_router.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""CEGIS routing (blueprint §C-3, frozen): repair vs replan decision."""
from __future__ import annotations

try:
    import cegis_taxonomy
except ImportError:
    from agent import cegis_taxonomy

REPAIR, REPLAN = "repair", "replan"


def route(error_class: str, mismatches: int = 0) -> str:
    """few-mismatch -> repair (touched rule only); everything else -> replan.

    A-CEGIS-13/16: replan forces a fresh reasoner hypothesis (Zone-2 reset
    + jolt). Repair only rewrites the rule touching the counterexample.
    """
    hint, _scope = cegis_taxonomy.classify(error_class, mismatches)
    if hint == "replay_mismatch_few":
        return REPAIR
    if hint == "replay_mismatch_many":
        return REPLAN
    # IndexError / ValueError_shape / timeout_rlimit: repair once, then
    # replan if the repair does not stick (guard-based scopes).
    return REPAIR


def max_repairs(taxonomy_hint: str) -> int:
    return 2 if taxonomy_hint in ("replay_mismatch_few", "IndexError",
                                  "ValueError_shape") else 0


In [ ]:
%%writefile goa/agent/adversarial_critic.py
# adversarial_critic.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""Structural spec critic (blueprint §1.5 VeriHarness pattern, zero-LLM).

Checks the spec the reasoner emits BEFORE any generation is spent:
signature-first, micro-example, assumptions listed, conservative
interpretation flag, alternatives 2-4. Structural only — fail-closed:
issues are advisory to the caller; an empty issue list never forces a
replan by itself.
"""
from __future__ import annotations

REQUIRED_MARKERS = ("def ",)          # signature-first
MAX_ALTERNATIVES = 4
MIN_ALTERNATIVES = 2


def critique_spec(spec: str, alternatives=None) -> list:
    issues = []
    if not spec or not spec.strip():
        return ["empty spec"]
    if not any(m in spec for m in REQUIRED_MARKERS):
        issues.append("spec does not lead with a function signature")
    if "return" not in spec:
        issues.append("spec does not state a return contract")
    if "assum" not in spec.lower():
        issues.append("spec does not list assumptions")
    if "conservative" not in spec.lower():
        issues.append("spec does not carry a conservative-interpretation directive")
    if alternatives is not None:
        n = len(alternatives)
        if not (MIN_ALTERNATIVES <= n <= MAX_ALTERNATIVES):
            issues.append(f"alternatives count {n} outside [2, 4]")
    return issues


In [ ]:
%%writefile goa/agent/inference_layer_v1_6.py
# inference_layer_v1_6.py — deployed-IL shim (DIR-078).
"""In the Run-21 notebook the deployed module IS v1.6
(goa/agent/inference_layer.py). This shim exists so that
`from inference_layer_v1_6 import ...` (diagnose extension, offline
tests) resolves against the deployed module without duplication."""
try:
    from inference_layer import *  # noqa: F401,F403
    from inference_layer import _log_call, call_log, build_spec, \
        CODER_CODE_CAP, CODER_CODE_CAP_EXCEEDED, _DEFAULT_PARAMS, \
        REASONER_ANSWER_CAP, DiagnoseResult
except ImportError:
    from agent.inference_layer import *  # noqa: F401,F403
    from agent.inference_layer import _log_call, call_log, build_spec, \
        CODER_CODE_CAP, CODER_CODE_CAP_EXCEEDED, _DEFAULT_PARAMS, \
        REASONER_ANSWER_CAP, DiagnoseResult


In [ ]:
%%writefile goa/agent/inference_layer_v1_6_additions.py
# inference_layer_v1_6_additions.py — GoA coder-solver. DIR-2026-076/078.
"""Thin re-export surface used by my_agent v1.0.19 (_il16_module).

Keeps my_agent decoupled from the full IL v1.6 module: only the
coder-facing constants and helpers cross this boundary. In the Run-21
notebook the deployed module IS v1.6 (goa/agent/inference_layer.py),
so we import from there first; the standalone inference_layer_v1_6
module name is the fallback for offline tests.
"""
from __future__ import annotations

try:
    from inference_layer import (CODER_CODE_CAP,
                                 CODER_CODE_CAP_EXCEEDED,
                                 build_spec, call_log, _log_call,
                                 _DEFAULT_PARAMS)
    _AVAILABLE = True
except ImportError:
    try:
        from agent.inference_layer import (CODER_CODE_CAP,
                                           CODER_CODE_CAP_EXCEEDED,
                                           build_spec, call_log, _log_call,
                                           _DEFAULT_PARAMS)
        _AVAILABLE = True
    except ImportError:
        from inference_layer_v1_6 import (CODER_CODE_CAP,
                                          CODER_CODE_CAP_EXCEEDED,
                                          build_spec, call_log, _log_call,
                                          _DEFAULT_PARAMS)
        _AVAILABLE = True
    except ImportError:
        _AVAILABLE = False
        CODER_CODE_CAP = 2048
        CODER_CODE_CAP_EXCEEDED = "coder_code_cap_exceeded"
        build_spec = None
        call_log = None
        _log_call = None
        _DEFAULT_PARAMS = {"temperature": 0.0, "top_p": 1.0, "top_k": -1}


In [ ]:
%%writefile goa/agent/coder_solver.py
# coder_solver.py — GoA coder-solver core. DIR-2026-076/078 (Indus).
"""Blueprint §8 core architecture, implemented fail-closed:

  TRIGGER (my_agent) -> reasoner emits TYPED SPEC -> CoderSolver:
    spec -> sandbox_v6 exec -> REPLAY VERIFY (pure Python, zero LLM)
      PASS -> admit + GuardedExecutor (1 action at a time)
      FAIL -> counterexample (frozen schema) -> repair (<=2/attempt)
              -> still FAIL -> replan signal -> my_agent reasoner path
  COUNTERS (frozen): 1 attempt = 1 synthesis + <=2 repairs;
                      <=3 attempts/level, <=6/game (18 generations worst).
  FALLBACK: any exception anywhere -> non-admitting result; the game
            NEVER aborts because of this module.
"""
from __future__ import annotations
import logging

def _dual(modname):
    """Flat import first (offline), then package import (notebook: goa/agent)."""
    try:
        import importlib
        return importlib.import_module(modname)
    except ImportError:
        return importlib.import_module("agent." + modname)

cegis_taxonomy = _dual("cegis_taxonomy")
cegis_router = _dual("cegis_router")
replay_verifier = _dual("replay_verifier")

logger = logging.getLogger(__name__)

try:
    _sb = _dual("sandbox_v6")
except ImportError:
    _sb = None

try:
    _build_spec = _dual("inference_layer_v1_6_additions").build_spec
except (ImportError, AttributeError):
    _build_spec = None

ATTEMPTS_PER_LEVEL_CAP = 3
ATTEMPTS_PER_GAME_CAP = 6
REPAIRS_PER_ATTEMPT_CAP = 2


class AttemptCounters:
    """Frozen attempt arithmetic (T-ATTEMPT-1 semantics)."""

    def __init__(self, max_apl=ATTEMPTS_PER_LEVEL_CAP,
                 max_apg=ATTEMPTS_PER_GAME_CAP,
                 max_rep=REPAIRS_PER_ATTEMPT_CAP):
        self.max_apl, self.max_apg, self.max_rep = max_apl, max_apg, max_rep
        self.apl = 0
        self.apg = 0
        self.repairs = 0
        self.generations = 0

    def begin_attempt(self):
        if self.apl >= self.max_apl or self.apg >= self.max_apg:
            return False
        self.apl += 1; self.apg += 1; self.repairs = 0
        self.generations += 1
        return True

    def begin_repair(self):
        if self.repairs >= self.max_rep:
            return False
        self.repairs += 1
        self.generations += 1
        return True

    def new_level(self):
        self.apl = 0

    def new_game(self):
        self.apl = 0; self.apg = 0; self.generations = 0


class SolveResult:
    """Fields consumed by my_agent v1.0.19 (see its mock tests T-64/T-65):
    admitted, replan_requested, mode, source (+ diagnostics)."""

    def __init__(self, admitted=False, replan_requested=False, mode="world",
                 source="", counterexample=None, error=None, error_class=None,
                 generations=0):
        self.admitted = admitted
        self.replan_requested = replan_requested
        self.mode = mode
        self.source = source
        self.counterexample = counterexample
        self.error = error
        self.error_class = error_class
        self.generations = generations


def _extract_code(raw: str) -> str:
    """Strip markdown fences from the coder's reply."""
    if not raw:
        return ""
    text = raw.strip()
    if "```" in text:
        parts = text.split("```")
        for p in parts:
            if "def " in p:
                body = p
                if body.startswith("python"):
                    body = body[6:]
                return body.strip()
    return text


def _spec_text(task_ctx: dict) -> str:
    if _build_spec is not None:
        try:
            return _build_spec(task_ctx)
        except Exception as e:
            logger.info("build_spec failed: %s", e)
    # minimal internal fallback (fail-closed, keeps chain alive)
    return (f"TASK: {task_ctx.get('task', '')}\n"
            f"HYPOTHESIS: {task_ctx.get('hypothesis', '')}\n"
            f"Return ONLY fenced Python defining "
            f"{task_ctx.get('entry_signature', 'next_action(frame, history)')}.")


class CoderSolver:
    """Spec -> sandbox exec -> replay verify -> admit / counterexample loop."""

    def __init__(self, call_llm, run_candidate=None):
        # call_llm(spec: str, mode: str) -> str  (my_agent._call_coder)
        self._call_llm = call_llm
        # LA-1 FIX (Indus, 2026-09-29 recheck audit): my_agent constructs
        # CoderSolver(call_llm=...) WITHOUT run_candidate; the old default
        # (NoSandbox lambda) silently killed every admit. Default to the
        # real sandbox when importable; explicit arg still wins; offline
        # without sandbox stays fail-closed NoSandbox.
        if run_candidate is not None:
            self._run = run_candidate
        elif _sb is not None:
            self._run = _sb.run_candidate
        else:
            self._run = (lambda *a, **k: {"ok": False,
                                         "error": "sandbox unavailable",
                                         "error_class": "NoSandbox"})
        self.counters = AttemptCounters()

    # ---------- mode 2: world model ----------
    def solve_world_model(self, task_ctx, buffer):
        return self._solve(task_ctx, buffer, "world", "next_grid")

    # ---------- mode 1: action program ----------
    def solve_action_program(self, task_ctx, buffer):
        return self._solve(task_ctx, buffer, "action", "next_action")

    # ---------- shared loop ----------
    def _solve(self, task_ctx, buffer, mode, entry):
        ctx = dict(task_ctx or {})
        ctx["mode"] = mode
        ctx.setdefault("entry_signature",
                        "next_grid(state, action)" if mode == "world"
                        else "next_action(frame, history)")
        # counters-before-cleverness: refuse before generating anything
        if not self.counters.begin_attempt():
            return SolveResult(mode=mode,
                               error="attempt cap reached",
                               error_class="CapExceeded")

        raw = ""
        try:
            raw = self._call_llm(_spec_text(ctx), mode)
        except Exception as e:
            logger.info("coder call failed: %s", e)
            return SolveResult(mode=mode, error=str(e), error_class="CallFailed")

        source = _extract_code(raw)
        if not source or "def " not in source:
            return SolveResult(mode=mode, error="no code in coder reply",
                               error_class="NoCode", generations=1)

        # static layer-1 first: never exec un-analyzed code
        if _sb is not None:
            try:
                _sb.analyze(source)
            except Exception as e:
                return SolveResult(mode=mode, error=f"L1: {e}",
                                   error_class="SandboxViolation", generations=1)

        # replay verify (mode "world") / dry exec (mode "action")
        if mode == "world":
            verdict = replay_verifier.replay_verify(source, self._run, buffer)
            if verdict.get("admitted"):
                return SolveResult(admitted=True, mode=mode, source=source,
                                   generations=1)
            return self._repair_loop(ctx, buffer, mode, entry, source, verdict)

        # action mode: single smoke exec, then admit (guarded per action)
        r = self._run(source, entry, {"frame": [[0]], "history": []},
                      timeout_s=3.0)
        if r.get("ok"):
            return SolveResult(admitted=True, mode=mode, source=source,
                               generations=1)
        return self._repair_loop(ctx, buffer, mode, entry, source, r)

    def _repair_loop(self, ctx, buffer, mode, entry, source, verdict):
        error_class = verdict.get("error_class") or "ReplayMismatch"
        mismatches = verdict.get("mismatches", 0)
        hint, scope = cegis_taxonomy.classify(error_class, mismatches)
        cx = replay_verifier.build_counterexample(
            function=entry, step=0, action=None,
            observed={"verdict": "recorded transitions"},
            model_predicted={"verdict": "candidate prediction"},
            cells_differ=mismatches,
            first_mismatches=verdict.get("first_mismatches") or [],
            taxonomy_hint=hint, repair_scope=scope,
            alternatives=ctx.get("alternatives"))
        gens = 1
        # repair only when routing says so, and only <= max_repairs
        if cegis_router.route(error_class, mismatches) == cegis_router.REPAIR:
            for _ in range(cegis_router.max_repairs(hint)):
                if not self.counters.begin_repair():
                    break
                gens += 1
                try:
                    repair_prompt = (_spec_text(ctx) +
                        f"\nREPAIR ONLY the rule touching this counterexample"
                        f" (scope: {scope}).\nCOUNTEREXAMPLE: {cx}"
                        f"\nPrevious code:\n{source}")
                    raw = self._call_llm(repair_prompt, mode)
                except Exception as e:
                    logger.info("repair call failed: %s", e)
                    break
                cand = _extract_code(raw)
                if not cand or "def " not in cand:
                    continue
                if _sb is not None:
                    try:
                        _sb.analyze(cand)
                    except Exception:
                        continue
                if mode == "world":
                    v2 = replay_verifier.replay_verify(cand, self._run, buffer)
                    if v2.get("admitted"):
                        return SolveResult(admitted=True, mode=mode,
                                           source=cand, generations=gens)
                else:
                    r2 = self._run(cand, entry,
                                   {"frame": [[0]], "history": []}, timeout_s=3.0)
                    if r2.get("ok"):
                        return SolveResult(admitted=True, mode=mode,
                                           source=cand, generations=gens)
        # exhausted: replan signal ONLY on many-mismatch (A-CEGIS-13/16)
        replan = hint in ("replay_mismatch_many", "timeout_rlimit")
        return SolveResult(admitted=False, replan_requested=replan,
                           mode=mode, counterexample=cx,
                           error=verdict.get("error") or f"{hint}",
                           error_class=error_class, generations=gens)


class GuardedExecutor:
    """Admitted program, one action at a time (Tycho guarded execution).

    my_agent records the transition and discards the route on mismatch;
    any exception here degrades to None -> reasoner path.
    """

    def __init__(self, source: str, mode: str = "action"):
        if _sb is None:
            raise RuntimeError("sandbox unavailable")
        self._source = source
        self._mode = mode
        self._entry = "next_action" if mode != "world" else "next_grid"
        _sb.analyze(source)  # raises SandboxViolation -> admit fails upstream

    def next_action(self, grid, history):
        try:
            r = _sb.run_candidate(self._source, self._entry,
                                   {"frame": grid, "history": list(history)},
                                   timeout_s=3.0)
            if r.get("ok"):
                return r.get("result")
            logger.info("guarded exec failed: %s", r.get("error"))
            return None
        except Exception as e:
            logger.info("guarded exec raised: %s", e)
            return None


In [ ]:
%%writefile goa/agent/my_agent.py
"""agent/my_agent.py — Kaggle entry point + single-file compiler, v1.0.21.

Changelog:
  v1.0.21 — Martial-law core audit (Chief order, 2026-09-30): Object Vision.
            extract_objects() ka structured output (id/color/size/bbox) ab
            cm.set_observation(objects=...) se observation block mein render
            hota hai — model ko RLE rows se column-arithmetic karke target
            nikalna nahi padta (core audit finding #1-#3). CERTIFIED_VERSIONS:
            context_manager 1.5 -> 1.6. + T-68/T-69 (render + palette
            cross-check).
  v1.0.20 — Run-23 postmortem (Chief order, 2026-09-30): proactive coder
            trigger. v1.0.19 mein coder SIRF reactive tha (jolt/divergence)
            — Run-23 mein reasoner ne alag actions try karte rahe, loop
            kabhi nahi pada, isliye 4 games mein ZERO coder calls. Naya
            config seam: coder_proactive_actions_at_level = N => N actions
            same level par bina advance ke par bhi coder attempt hota hai.
            Default None = v1.0.19 behavior (backward-compatible, T-suite).
            + T-66/T-67 tests.
  v1.0.19 — DIR/2026-076 (Operation Coder=Solver) + AGA fixes
            A-BH-01..A-BH-14 + A-CEGIS-13 + A-CEGIS-16 + A-CEGIS-17 +
            A-CEGIS-18 + A-CEGIS-19:
            (1)  Coder-solver wired via CoderSolver + ReplayVerifier.
            (2)  Persistent guarded execution (A-BH-04).
            (3)  _record_transition feeds the replay buffer.
            (4)  reset_level clears pending state (A-BH-03).
            (5)  AR-08: coder attempts vs deep-calls separate counters.
            (6)  Guarded-success clears stale jolt injections (A-BH-06).
            (7)  CERTIFIED_VERSIONS: inference_layer 1.5 -> 1.6.
            (8)  Lazy solver init — fail-closed additivity.
            (9)  coder_actions / coder_actions_level telemetry.
           (10)  Empty-buffer guard prevents budget drain (A-BH-10).
           (11)  Level-start trigger removed (A-BH-13).
           (12)  A-CEGIS-13: REPLAN signal consumed by agent loop.
           (13)  A-CEGIS-16: world-model REPLAN short-circuits BEFORE
                 action-program fallback; signal never overwritten.
           (14)  A-CEGIS-17: _handle_coder_replan sets _jolt_pending=True
                 to force a fresh deep reasoner call.
           (15)  A-CEGIS-18: defensive _cs_module / GuardedExecutor guard
                 at the top of _maybe_admit_coder — opaque AttributeError
                 avoided if the coder chain is partially importable.
           (16)  A-CEGIS-19: T-65 dead lines removed.
  v1.0.18 — DIR/2026-070 + DIR/2026-071 + DIR/2026-072 + INDUS-DIR-070.
"""
from __future__ import annotations
import argparse, hashlib, json, logging, re, statistics, sys, textwrap
from collections import Counter, deque
from dataclasses import dataclass
from pathlib import Path
from typing import (Any, Callable, Dict, FrozenSet, Iterable, List,
                    Mapping, Optional, Sequence, Tuple)

logger = logging.getLogger(__name__)

CERTIFIED_VERSIONS = {
    "context_manager": "1.6",
    "inference_layer": "1.6",
    "grid_text": "1.0.8",
    "action_pipeline": "1.0.5",
}
HARNESS_OWNED: FrozenSet[str] = frozenset({"arcengine", "agents.agent"})
DISPATCHED_LOG_CAP = 200
PROMPT_LOG_CAP = 50
JOLT_HISTORY_CAP = 50
CODER_ATTEMPTS_PER_GAME_CAP = 6
_WARNED_UNKNOWN_IDS: set = set()
_TACTICS_JOURNAL: List[Dict[str, Any]] = []


def _journal_brief(max_entries=6, max_chars=400):
    if not _TACTICS_JOURNAL: return ""
    parts = []
    for e in _TACTICS_JOURNAL[-max_entries:]:
        parts.append("[{g}:lv={lv}:{r}]".format(
            g=e.get("game", "?"), lv=e.get("levels", 0),
            r=str(e.get("rules", ""))[:60]))
    return " ".join(parts)[:max_chars]


try:
    from agent import context_manager as _cm_module
except ImportError:
    try: import context_manager as _cm_module
    except ImportError: _cm_module = None
try:
    from agent import inference_layer as _il_module
except ImportError:
    try: import inference_layer as _il_module
    except ImportError: _il_module = None
try:
    from agent import grid_text as _gt_module
except ImportError:
    try: import grid_text as _gt_module
    except ImportError: _gt_module = None
try:
    from agent import action_pipeline as _ap_module
except ImportError:
    try: import action_pipeline as _ap_module
    except ImportError: _ap_module = None

try:
    import coder_solver as _cs_module
except ImportError:
    try:
        from agent import coder_solver as _cs_module
    except ImportError:
        _cs_module = None
try:
    import replay_verifier as _rv_module
except ImportError:
    try:
        from agent import replay_verifier as _rv_module
    except ImportError:
        _rv_module = None
try:
    import inference_layer_v1_6_additions as _il16_module
except ImportError:
    try:
        from agent import inference_layer_v1_6_additions as _il16_module
    except ImportError:
        _il16_module = None

try:
    from arcengine import FrameData, GameAction, GameState  # noqa
    _HAS_ARCENGINE = True
except ImportError:
    FrameData = GameAction = GameState = None
    _HAS_ARCENGINE = False
try:
    from agents.agent import Agent as _FrameworkAgent  # noqa
    _HAS_FRAMEWORK_AGENT = True
except ImportError:
    class _FrameworkAgent:
        def __init__(self, *a, **k): pass
    _HAS_FRAMEWORK_AGENT = False


DEFAULT_CONFIG: Dict[str, Any] = {
    "reasoner_endpoint": "qwen3.8-27b",
    "coder_endpoint": "qwen3-coder-30b-a3b",
    "offline_fp8": False,
    "reasoner_mode": "fast",
    "connectivity": 4,
    "coder_solver_enabled": True,
}


class AgentError(RuntimeError): ...
class VersionMismatch(AgentError): ...
class VersionMissing(AgentError): ...
class ComponentUnavailable(AgentError): ...


def check_certified_versions(cm, il, gt, ap, expected=None,
                              version_manifest=None):
    expected = expected or CERTIFIED_VERSIONS
    manifest = version_manifest or {}
    verified = {}
    for name, mod in (("context_manager", cm), ("inference_layer", il),
                      ("grid_text", gt), ("action_pipeline", ap)):
        exp = expected[name]
        declared = getattr(mod, "__version__", None) or manifest.get(name)
        if declared is None:
            raise VersionMissing(
                f"certified module {name!r} has no __version__ or "
                f"manifest entry (expected {exp!r})")
        if declared != exp:
            raise VersionMismatch(f"{name}: {declared!r} != {exp!r}")
        verified[name] = True
    return verified


@dataclass(frozen=True)
class LocalCalibration:
    game_id: str
    is_reject: bool
    retry_budget: int
    debounce_steps: int
    pause_before_retry_s: float
    pause_before_fresh_s: float
    resolved_id: Optional[str] = None
    known: bool = True


def _prefix(game_id):
    s = str(game_id)
    return s.split("-", 1)[0] if "-" in s else s


def _require_attr(mod, name, *, reject_none=True):
    if not hasattr(mod, name):
        raise AgentError(f"action_pipeline missing {name!r}")
    v = getattr(mod, name)
    if reject_none and v is None:
        raise AgentError(f"action_pipeline.{name} is None")
    return v


def _require_clean_key(clean, key):
    try:
        if isinstance(clean, Mapping):
            if key not in clean:
                raise AgentError(f"CLEAN_SET missing {key!r}")
            return clean[key]
        if not hasattr(clean, key):
            raise AgentError(f"CLEAN_SET missing {key!r}")
        return getattr(clean, key)
    except AgentError: raise
    except Exception as e:
        raise AgentError(f"CLEAN_SET[{key!r}] unreadable: {e}") from e


def _row_get(row, key, default=None):
    if row is None: return default
    if isinstance(row, Mapping): return row.get(key, default)
    return getattr(row, key, default)


def _row_get_strict(row, key):
    if row is None:
        raise AgentError(f"calibration row is None (reading {key!r})")
    if isinstance(row, Mapping):
        if key not in row:
            raise AgentError(f"calibration row missing {key!r}")
        return row[key]
    if not hasattr(row, key):
        raise AgentError(f"calibration row missing {key!r}")
    return getattr(row, key)


def _aggregate_clean_defaults(rows):
    budgets, debounces = [], []
    for r in rows:
        if _row_get_strict(r, "is_reject"): continue
        b = _row_get(r, "retry_budget"); d = _row_get(r, "debounce_steps")
        if b is not None: budgets.append(int(b))
        if d is not None: debounces.append(int(d))
    if not budgets or not debounces:
        raise AgentError("DEFAULT_CALIBRATION has no clean rows to aggregate")
    return min(budgets), int(round(statistics.median(debounces)))


def lookup_calibration(game_id, ap_module):
    table = _require_attr(ap_module, "DEFAULT_CALIBRATION")
    clean = _require_attr(ap_module, "CLEAN_SET")
    reject_ids = tuple(_require_attr(ap_module, "REJECT_LIST_IDS"))

    resolved_id = None
    if game_id in table:
        resolved_id = game_id
    else:
        pref = _prefix(game_id)
        candidates = [k for k in table if _prefix(k) == pref]
        if len(candidates) == 1:
            resolved_id = candidates[0]

    effective_id = resolved_id if resolved_id is not None else game_id
    is_reject = effective_id in reject_ids

    if resolved_id is None:
        default_budget, default_debounce = _aggregate_clean_defaults(
            table.values())
        if game_id not in _WARNED_UNKNOWN_IDS:
            _WARNED_UNKNOWN_IDS.add(game_id)
            logger.warning(
                "unknown game_id %r; using aggregated clean defaults "
                "(retry_budget=%d, debounce=%d)", game_id,
                default_budget, default_debounce)
        return LocalCalibration(
            game_id=game_id, is_reject=False,
            retry_budget=default_budget, debounce_steps=default_debounce,
            pause_before_retry_s=float(_require_clean_key(clean, "pause_retry_s")),
            pause_before_fresh_s=float(_require_clean_key(clean, "pause_fresh_s")),
            resolved_id=None, known=False)

    row = table[resolved_id]
    b = _row_get(row, "retry_budget"); d = _row_get(row, "debounce_steps")
    if b is None or d is None:
        default_budget, default_debounce = _aggregate_clean_defaults(
            table.values())
        retry_budget = int(b) if b is not None else default_budget
        debounce = int(d) if d is not None else default_debounce
    else:
        retry_budget, debounce = int(b), int(d)

    def _pause(attr, key):
        v = _row_get(row, attr)
        if v is not None: return float(v)
        return float(_require_clean_key(clean, key))

    return LocalCalibration(
        game_id=game_id, is_reject=is_reject,
        retry_budget=retry_budget, debounce_steps=debounce,
        pause_before_retry_s=_pause("pause_before_retry_s", "pause_retry_s"),
        pause_before_fresh_s=_pause("pause_before_fresh_s", "pause_fresh_s"),
        resolved_id=resolved_id, known=True)


STUCK_CLEAN = 3
STUCK_MECHANIC = 4
JITTER_RADIUS = 2
DEADLOCK_WINDOWS = 2

_INSTRUCTION_RE = re.compile(
    r"^(?P<aid>[A-Za-z0-9_\-]+)"
    r"(?:\s+r(?P<r>\d+)\s+c(?P<c>\d+))?$")


@dataclass
class StuckSig:
    action_id: str
    target_xy: Optional[Tuple[int, int]]
    target_hash: Optional[str]
    target_bbox: Optional[Tuple[int, int, int, int]] = None


@dataclass
class JoltRequest:
    kind: str
    blacklist_sig: Optional[StuckSig] = None

    def to_dict(self):
        return {"kind": self.kind,
                "blacklist_sig": None if self.blacklist_sig is None else {
                    "action_id": self.blacklist_sig.action_id,
                    "target_xy": (None if self.blacklist_sig.target_xy is None
                                  else list(self.blacklist_sig.target_xy)),
                    "target_hash": self.blacklist_sig.target_hash,
                    "target_bbox": (None if self.blacklist_sig.target_bbox
                                    is None
                                    else list(self.blacklist_sig.target_bbox))}}


class LoopDetector:
    def __init__(self, game_id, is_reject, context_mgr, debounce_window,
                 allow_repetition_while=None, on_repetition_allowed=None):
        self.game_id = game_id
        self._threshold = STUCK_MECHANIC if is_reject else STUCK_CLEAN
        self._cm = context_mgr
        self._debounce_window = int(debounce_window)
        self._streak: List[StuckSig] = []
        self._frozen_count = 0
        self._last_objset_fp = None
        self._jolt_history = deque(maxlen=JOLT_HISTORY_CAP)
        self._allow_repetition = allow_repetition_while or (lambda: False)
        self._on_repetition_allowed = on_repetition_allowed or (lambda: None)

    def reset(self):
        self._streak.clear()
        self._frozen_count = 0
        self._last_objset_fp = None

    def tick(self, *, action_id, target_xy, target_hash, target_bbox,
             state_changed, objset_fp):
        if not state_changed:
            new_sig = StuckSig(action_id, target_xy, target_hash, target_bbox)
            if self._streak and self._matches(new_sig, self._streak[-1]):
                self._streak.append(new_sig)
            else:
                self._streak = [new_sig]
            if len(self._streak) > self._threshold:
                if self._allow_repetition():
                    self._on_repetition_allowed()
                else:
                    req = self._fire_stuck(new_sig)
                    self._jolt_history.append(req)
                    self._frozen_count = 0
                    self._last_objset_fp = None
                    return req
        else:
            self._streak.clear()

        if self._last_objset_fp is None:
            self._last_objset_fp = objset_fp
            self._frozen_count = 1
        elif objset_fp == self._last_objset_fp:
            self._frozen_count += 1
        else:
            self._last_objset_fp = objset_fp
            self._frozen_count = 1

        if self._frozen_count >= DEADLOCK_WINDOWS * self._debounce_window:
            self._frozen_count = 0
            self._last_objset_fp = None
            self._streak.clear()
            req = JoltRequest(kind="deadlock")
            self._jolt_history.append(req)
            return req
        return None

    def _matches(self, a, b):
        if a.action_id != b.action_id: return False
        if a.target_hash != b.target_hash: return False
        if a.target_xy is None and b.target_xy is None: return True
        if a.target_xy is None or b.target_xy is None: return False
        return (abs(a.target_xy[0] - b.target_xy[0]) <= JITTER_RADIUS
                and abs(a.target_xy[1] - b.target_xy[1]) <= JITTER_RADIUS)

    def _fire_stuck(self, sig):
        self._cm.register_dead_signature(self._format_blacklist(sig))
        self._streak.clear()
        return JoltRequest(kind="stuck", blacklist_sig=sig)

    @staticmethod
    def _format_blacklist(sig):
        if sig is None:
            return ("SYSTEM OVERRIDE: repeated action failed with no state "
                    "change.  DO NOT REPEAT.")
        sh = (sig.target_hash or "")[:6] or "<unknown>"
        if sig.target_bbox is not None:
            r0, r1, c0, c1 = sig.target_bbox
            bb = f"r{r0}-{r1} c{c0}-{c1}"
        elif sig.target_xy is not None:
            bb = f"r{sig.target_xy[0]} c{sig.target_xy[1]}"
        else:
            bb = "r? c?"
        return (f"SYSTEM OVERRIDE: action {sig.action_id} on object "
                f"<{sh}, {bb}> failed. DO NOT REPEAT.")

    def streak_length(self): return len(self._streak)
    def frozen_count(self): return self._frozen_count
    def jolt_history(self): return list(self._jolt_history)


@dataclass
class CRLFIntakeEntry:
    key: str
    cr_byte_count: int
    md5_original: str
    md5_normalized: str


class Agent:
    _DEEP_TRIGGER_MAX_PER_GAME = 8

    def __init__(self, game_id, config=None, *, cm=None, il=None,
                 gt=None, ap=None, model_config_hook=None,
                 version_manifest=None):
        self.game_id = game_id
        self.config = dict(DEFAULT_CONFIG)
        self._crlf_log = []
        if config:
            self.config.update(self._normalize_crlf(config))
        self._deep_cap()
        self._cm = cm if cm is not None else self._require(_cm_module, "context_manager")
        self._il = il if il is not None else self._require(_il_module, "inference_layer")
        self._gt = gt if gt is not None else self._require(_gt_module, "grid_text")
        self._ap_mod = ap if ap is not None else self._require(_ap_module, "action_pipeline")
        self.version_verified = check_certified_versions(
            self._cm, self._il, self._gt, self._ap_mod,
            version_manifest=version_manifest)
        self._pipeline = self._ap_mod.ActionPipeline(
            game_id=game_id, dispatch=self._on_dispatch, patch_hook=None)
        self._row = lookup_calibration(game_id, self._ap_mod)
        self._deep_calls_this_game = 0
        self._actions_at_level = 0
        self._jolt_pending = False
        self._divergence_pending = False
        self._switch_decision_pending = False
        self._p5_budget_remaining = 2
        self._loop = LoopDetector(
            game_id=game_id, is_reject=self._row.is_reject,
            context_mgr=self._cm, debounce_window=self._row.debounce_steps,
            allow_repetition_while=(lambda: self._p5_budget_remaining > 0),
            on_repetition_allowed=self._consume_p5_budget)
        self._model_config_hook = model_config_hook
        self._apply_model_config()
        self._prev_checksum = None
        self._prev_objset_fp = None
        self._last_anchor = None
        self._pending_action = None
        self._pending_injections = []
        self._last_reasoner_result = None
        self._dispatched_log = deque(maxlen=DISPATCHED_LOG_CAP)
        self._prompt_log = deque(maxlen=PROMPT_LOG_CAP)
        self._total_dispatches = 0
        self._force_anchor = False
        self._resync = False
        self._level_num = 0
        self._levels_completed = 0
        self._available_actions = []
        self._last_packet = None

        # ---------- v1.0.19: Coder-solver state ---------- #
        self._coder_attempts_this_game = 0
        self._coder_attempts_this_level = 0
        self._coder_actions_this_game = 0
        self._coder_actions_this_level = 0
        self._coder_solve_pending = False
        self._coder_solve_result = None
        self._coder_guarded = None
        self._transition_buffer = None
        self._solver = None
        self._prev_settled_grid = None
        # A-CEGIS-13: REPLAN consumption state.
        self._coder_replan_pending = False
        self._coder_replan_count = 0
        self._init_coder_solver()

    # ---------- v1.0.19: Coder-solver helpers ---------- #

    def _init_coder_solver(self) -> None:
        if not self.config.get("coder_solver_enabled", True):
            logger.info("coder_solver disabled by config")
            return
        if _cs_module is None or _rv_module is None:
            logger.info("coder_solver chain not importable — reasoner-only")
            return
        try:
            self._transition_buffer = _rv_module.TransitionBuffer()
            self._solver = _cs_module.CoderSolver(call_llm=self._call_coder)
        except Exception as e:
            logger.warning("coder_solver init failed: %s", e)
            self._solver = None
            self._transition_buffer = None

    def _call_coder(self, spec: str, mode: str) -> str:
        if _il16_module is None:
            return ""
        cap = getattr(_il16_module, "CODER_CODE_CAP", 2048)
        sentinel = getattr(_il16_module, "CODER_CODE_CAP_EXCEEDED",
                           "coder_code_cap_exceeded")
        # LA-4 FIX (Indus, 2026-09-29 final logic audit): _DEFAULT_PARAMS
        # is a MODULE global of inference_layer, never an instance attr —
        # getattr(il, ...) always returned {} and _call_transport raised
        # KeyError('temperature'), silently returning "" for every coder
        # call. Source params from the additions re-export with a literal
        # fallback.
        _dp = getattr(self._il, "_DEFAULT_PARAMS", None) \
            or getattr(_il16_module, "_DEFAULT_PARAMS", None) \
            or {"temperature": 0.0, "top_p": 1.0, "top_k": -1}
        try:
            rj = self._il._call_transport(
                model=self._il._coder_model, system=None,
                messages=[{"role": "user", "content": spec}],
                max_tokens=cap, mode=None,
                params=dict(_dp),
                chat_template_kwargs={})
        except Exception as e:
            logger.info("coder transport failed: %s", e)
            return ""
        try:
            fr = rj["choices"][0].get("finish_reason")
            content = rj["choices"][0]["message"]["content"]
        except Exception:
            return ""
        enf = getattr(_il16_module, "enforce_coder_cap", None)
        if callable(enf):
            if enf(fr, content):
                return sentinel
        elif fr == "length":
            return sentinel
        return content

    def _should_attempt_coder_solve(self) -> bool:
        if self._solver is None or self._transition_buffer is None:
            return False
        if self._coder_attempts_this_game >= CODER_ATTEMPTS_PER_GAME_CAP:
            return False
        # A-CEGIS-13: while a REPLAN is pending, skip coder attempts so
        # the next action goes through the reasoner for a fresh hypothesis.
        if self._coder_replan_pending:
            return False
        if len(self._transition_buffer) < 1:
            return False
        if self._jolt_pending or self._divergence_pending:
            return True
        # v1.0.20: proactive coder trigger (Run-23 postmortem). Reactive-only
        # path mein coder tabhi fire hota tha jab loopdetector ya AP failure
        # ho — blind exploration mein wo kabhi nahi hota. Ab harness config
        # se: N actions at-level bina level-advance ke => coder attempt.
        # Default None => v1.0.19 semantics (zero behavior change).
        _proactive = self.config.get("coder_proactive_actions_at_level", None)
        if (_proactive is not None
                and isinstance(_proactive, int)
                and not isinstance(_proactive, bool)
                and _proactive > 0
                and self._actions_at_level >= _proactive):
            return True
        return False

    def _maybe_admit_coder(self, task_ctx: Dict[str, Any]) -> bool:
        # A-CEGIS-18: defensive guard — the coder chain may be partially
        # importable in some deployments. Fail-closed with an explicit
        # return instead of an opaque AttributeError on GuardedExecutor.
        if _cs_module is None or getattr(_cs_module, "GuardedExecutor", None) is None:
            return False
        if self._coder_solve_pending:
            return False
        if self._transition_buffer is None or len(self._transition_buffer) < 1:
            return False
        self._coder_solve_pending = True
        try:
            self._coder_attempts_this_game += 1
            self._coder_attempts_this_level += 1

            # A-CEGIS-16: world-model REPLAN short-circuits BEFORE the
            # action-program fallback, so the signal is never overwritten.
            res = self._solver.solve_world_model(
                task_ctx, self._transition_buffer)
            if getattr(res, "replan_requested", False):
                self._coder_solve_result = res
                self._handle_coder_replan(res)
                return False

            if not res.admitted:
                res = self._solver.solve_action_program(
                    task_ctx, self._transition_buffer)

            self._coder_solve_result = res
            if getattr(res, "replan_requested", False):
                self._handle_coder_replan(res)

            if not res.admitted:
                return False
            self._coder_guarded = _cs_module.GuardedExecutor(
                res.source, res.mode)
            return True
        except Exception as e:
            logger.info("coder admit failed: %s", e)
            self._coder_guarded = None
            return False
        finally:
            self._coder_solve_pending = False

    # ---------- A-CEGIS-13 / A-CEGIS-17: REPLAN semantics ---------- #
    def _handle_coder_replan(self, res) -> None:
        """Blueprint §C-3: `replay_mismatch_many → REPLAN to reasoner;
        do not repair`.

        Consequences:
        1. Clear Zone 2 `hypotheses` and `world_model_status` so the
           reasoner is not re-anchored to the stale hypothesis.
        2. Set `_coder_replan_pending = True`. The next
           `_reason_and_plan` call skips coder attempts and goes
           directly to the reasoner for a fresh hypothesis.
        3. Reset the guarded executor — the previous admitted model is
           void.
        4. Bump `_coder_replan_count` for telemetry.
        5. A-CEGIS-17: set `_jolt_pending = True` so `_should_deep`
           returns True and the reasoner runs in deep mode on the next
           action. This makes the "force a fresh hypothesis" contract
           explicit rather than relying on incidental trigger state.
        """
        try:
            if hasattr(self._cm, "update_zone2"):
                self._cm.update_zone2("hypotheses", "", truncate=True)
                self._cm.update_zone2("world_model_status",
                                      "replan: prior hypothesis rejected",
                                      truncate=True)
        except Exception as e:
            logger.info("replan cm update failed: %s", type(e).__name__)
        self._coder_guarded = None
        self._coder_replan_pending = True
        self._coder_replan_count += 1
        # A-CEGIS-17: explicit deep-call trigger for the next action.
        self._jolt_pending = True
        logger.info("coder REPLAN requested; fresh hypothesis forced "
                    "(count=%d)", self._coder_replan_count)

    def _try_guarded_action(self) -> Optional[str]:
        if self._coder_guarded is None:
            return None
        if self._prev_settled_grid is None:
            return None
        try:
            history = (list(self._transition_buffer.as_list())
                       if self._transition_buffer else [])
            action = self._coder_guarded.next_action(
                self._prev_settled_grid, history)
        except Exception as e:
            logger.info("guarded next_action failed: %s", e)
            self._coder_guarded = None
            return None
        if action is None:
            return None
        self._coder_actions_this_game += 1
        self._coder_actions_this_level += 1
        self._actions_at_level += 1
        self._jolt_pending = False
        self._divergence_pending = False
        self._pending_injections.clear()
        self._force_anchor = False
        self._resync = False
        return action if isinstance(action, str) else str(action)

    def _record_transition(self, state, action, next_state) -> None:
        if self._transition_buffer is None:
            return
        try:
            self._transition_buffer.append(state, action, next_state)
        except Exception as e:
            logger.info("buffer append failed: %s", e)

    def _on_level_advance_coder(self) -> None:
        self._coder_attempts_this_level = 0
        self._coder_actions_this_level = 0
        self._coder_replan_pending = False
        if self._solver is not None:
            self._solver.counters.new_level()
        if _rv_module is not None:
            try:
                self._transition_buffer = _rv_module.TransitionBuffer()
            except Exception:
                self._transition_buffer = None
        self._coder_guarded = None
        self._coder_solve_result = None

    def _deep_cap(self):
        raw = self.config.get("deep_call_cap", self._DEEP_TRIGGER_MAX_PER_GAME)
        if isinstance(raw, bool) or not isinstance(raw, int):
            raise AgentError(f"deep_call_cap must be a positive int; got {raw!r}")
        if raw <= 0:
            raise AgentError(f"deep_call_cap must be > 0; got {raw}")
        return raw

    def _should_deep(self):
        if self._actions_at_level == 0: return True
        if self._jolt_pending: return True
        if self._divergence_pending: return True
        if self._switch_decision_pending: return True
        return False

    def _consume_p5_budget(self):
        if self._p5_budget_remaining > 0:
            self._p5_budget_remaining -= 1

    @property
    def pause_before_retry_s(self): return self._row.pause_before_retry_s
    @property
    def pause_before_fresh_s(self): return self._row.pause_before_fresh_s
    @property
    def is_reject_game(self): return self._row.is_reject

    def model_config(self):
        return {"reasoner_endpoint": self.config["reasoner_endpoint"],
                "coder_endpoint": self.config["coder_endpoint"],
                "offline_fp8": bool(self.config["offline_fp8"])}

    def crlf_intake_log(self): return list(self._crlf_log)
    def last_packet(self): return self._last_packet
    def total_dispatches(self): return self._total_dispatches
    def coder_attempts(self): return self._coder_attempts_this_game
    def coder_result(self): return self._coder_solve_result
    def coder_actions(self): return self._coder_actions_this_game
    def coder_actions_level(self): return self._coder_actions_this_level
    def coder_replan_count(self): return self._coder_replan_count
    def coder_replan_pending(self): return self._coder_replan_pending

    def _apply_model_config(self):
        mc = self.model_config()
        if self._model_config_hook is None:
            logger.warning("model_config_hook not supplied (payload=%r)", mc)
            return
        try:
            self._model_config_hook(mc)
        except Exception as e:
            raise AgentError(f"model_config_hook raised: {e}") from e

    @staticmethod
    def _require(mod, name):
        if mod is None:
            raise ComponentUnavailable(f"certified module {name!r} not importable")
        return mod

    def _normalize_crlf(self, d):
        out = dict(d)
        for k, v in d.items():
            if isinstance(v, str) and ("\r\n" in v or "\r" in v):
                cr = v.count("\r")
                orig = hashlib.md5(v.encode("utf-8")).hexdigest()
                fixed = v.replace("\r\n", "\n").replace("\r", "\n")
                norm = hashlib.md5(fixed.encode("utf-8")).hexdigest()
                self._crlf_log.append(CRLFIntakeEntry(k, cr, orig, norm))
                out[k] = fixed
        return out

    @staticmethod
    def _objset_fp(objects):
        return (tuple(sorted(Counter(o.ohash for o in objects).items())),
                len(objects))

    @staticmethod
    def _object_at(objects, r, c):
        for o in objects:
            for (rr, cc) in o.cells:
                if rr == r and cc == c: return o
        return None

    def _parse_instruction(self, instruction):
        s = (instruction or "").strip()
        m = _INSTRUCTION_RE.match(s)
        if not m: return s, None
        aid, r, c = m.group("aid"), m.group("r"), m.group("c")
        return (aid, None) if r is None or c is None else (aid, (int(r), int(c)))

    def _on_dispatch(self, action):
        self._dispatched_log.append(action)
        self._total_dispatches += 1
        self._cm.append_action(action, "dispatched", 0)

    def act(self, frame):
        if not isinstance(frame, (list, tuple)) or not frame:
            raise AgentError("frame must be a non-empty sequence")
        settled = frame[-1]

        if self._pending_action is not None and self._prev_settled_grid is not None:
            self._record_transition(self._prev_settled_grid,
                                    self._pending_action, settled)

        if self._pending_action is not None:
            try:
                self._pipeline.observe([settled])
            except self._ap_mod.DivergenceError as e:
                self._last_packet = e.packet
                self._divergence_pending = True
                self._coder_diagnose("divergence")
            except self._ap_mod.PremiseFailureError:
                self._divergence_pending = True
                self._coder_diagnose("premise_failure")
            self._pending_action = None

        anchor = self._gt.rle_anchor(settled)
        self._last_anchor = anchor
        checksum = anchor["checksum"]
        objects = self._gt.extract_objects(settled, background=0)
        objset_fp = self._objset_fp(objects)
        state_changed = (self._prev_checksum is not None
                         and checksum != self._prev_checksum)
        # v1.0.21 Object Vision: structured objects CM tak — prompt mein
        # "[id] color=.. rows=r0-r1 cols=c0-c1" render hote hain, model ko
        # RLE column-arithmetic nahi karna padta.
        obs_objects = [
            {"id": o.id, "color_char": o.color_char,
             "color_digit": int(o.color_digit), "size": int(o.size),
             "bbox": [int(v) for v in o.bbox]}
            for o in objects]
        self._cm.set_observation(
            encoding_dict=anchor,
            available_actions=list(self._available_actions),
            levels_completed=self._levels_completed, state="RUNNING",
            objects=obs_objects)
        if self._prev_objset_fp is not None and objset_fp != self._prev_objset_fp:
            self._cm.clear_dead_signatures()

        action_id, target_xy = (None, None)
        target_hash, target_bbox = (None, None)
        if self._dispatched_log:
            aid, txy = self._parse_instruction(str(self._dispatched_log[-1]))
            action_id, target_xy = aid, txy
            if txy is not None:
                o = self._object_at(objects, txy[0], txy[1])
                if o is not None:
                    target_hash, target_bbox = o.ohash, o.bbox
        if action_id is not None:
            jolt = self._loop.tick(
                action_id=action_id, target_xy=target_xy,
                target_hash=target_hash, target_bbox=target_bbox,
                state_changed=state_changed, objset_fp=objset_fp)
            if jolt is not None:
                self._handle_jolt(jolt)

        self._prev_checksum = checksum
        self._prev_objset_fp = objset_fp
        self._prev_settled_grid = settled

        next_action = self._reason_and_plan()
        self._pipeline.submit_plan([next_action])
        result = self._pipeline.step()
        if result is None: return None
        self._pending_action = result.action
        return result.action

    def reset_level(self, level_num=None, advanced=False):
        if level_num is not None: n = int(level_num)
        elif advanced: n = self._level_num + 1
        else: n = self._level_num
        self._cm.set_level(n)
        self._cm.clear_dead_signatures()
        self._loop.reset()
        self._level_num = n
        if advanced: self._levels_completed += 1
        if advanced: self._coder_world_model()
        self._pending_injections.clear()
        self._prev_objset_fp = None
        self._actions_at_level = 0
        self._p5_budget_remaining = 2
        self._jolt_pending = False
        self._divergence_pending = False
        self._pending_action = None
        self._prev_settled_grid = None
        self._on_level_advance_coder()

    def set_available_actions(self, actions):
        self._available_actions = list(actions)

    def _handle_jolt(self, jolt):
        self._jolt_pending = True
        if jolt.kind == "stuck":
            self._force_anchor = True
            self._pending_injections.append(
                "=== CHANGE OF PLAN ===\n"
                "The following action keeps failing with no state change.  "
                "Do not repeat it in this plan.\n"
                + LoopDetector._format_blacklist(jolt.blacklist_sig)
                + "\nConsider a completely different strategy.")
        elif jolt.kind == "deadlock":
            self._force_anchor = True
            self._resync = True
            self._pending_injections.append(
                "=== RESYNC ===\nNo object-level change over two debounce "
                "windows.  Fresh anchor attached below.  Re-derive the plan "
                "from scratch.")

    def _reason_and_plan(self):
        action = self._try_guarded_action()
        if action is not None:
            return action

        if self._should_attempt_coder_solve():
            task_ctx = {
                "task": ("Reproduce every buffered transition exactly. "
                         "Return the required functions."),
                "hypothesis": {
                    "game": self.game_id,
                    "level": self._level_num,
                    "actions_at_level": self._actions_at_level,
                },
                "replay_evidence": {
                    "transitions": len(self._transition_buffer or []),
                },
                "io_contract": {
                    "frame_format": "64x64 int8",
                    "must_replay": True,
                },
                "constraints": {"no_imports": True, "deterministic": True,
                                "max_lines": 80},
                "alternatives": [],
                "mode": "world",
            }
            if self._maybe_admit_coder(task_ctx):
                action = self._try_guarded_action()
                if action is not None:
                    return action
                self._coder_guarded = None

        pp = self._cm.build_prompt()
        text = getattr(pp, "text", str(pp))
        if self._force_anchor:
            text += ("\n\n[FRESH ANCHOR]\n"
                     + json.dumps(self._last_anchor or {}, sort_keys=True))
            self._force_anchor = False
        if self._resync:
            text = "[FULL RESYNC]\n" + text
            self._resync = False
        if self._pending_injections:
            text += "\n\n" + "\n\n".join(self._pending_injections)
            self._pending_injections.clear()
        self._prompt_log.append(text)

        cap = self._deep_cap()
        if self._deep_calls_this_game >= cap:
            mode = "fast"
        elif self._should_deep():
            mode = "deep"
        else:
            mode = self.config.get("reasoner_mode", "fast")
        self._jolt_pending = False
        self._divergence_pending = False

        result = self._il.ask_reasoner({"text": text}, mode=mode)
        if mode == "deep":
            self._deep_calls_this_game += 1
        self._last_reasoner_result = result
        self._cm.update_zone2("world_model_status",
                              str(result.observation_summary), truncate=True)
        if result.hypothesis_updates:
            self._cm.update_zone2("hypotheses",
                                  "\n".join(result.hypothesis_updates),
                                  truncate=True)
        self._cm.update_zone2("goal_notes", str(result.prediction),
                              truncate=True)
        self._actions_at_level += 1

        if self._coder_replan_pending:
            self._coder_replan_pending = False

        aid, txy = self._parse_instruction(result.instruction)
        return f"{aid} r{txy[0]} c{txy[1]}" if txy is not None else aid

    def _coder_world_model(self):
        try:
            r0 = self._last_reasoner_result
            hyp = "; ".join(getattr(r0, "hypothesis_updates", []) or [])
            brief = (
                "Write a short pure-python function world_model() returning a "
                "one-line summary of the current game model. Evidence: "
                f"game={self.game_id}; levels_completed={self._levels_completed}; "
                f"observation_summary={getattr(r0, 'observation_summary', '')}; "
                f"hypotheses={hyp}; prediction={getattr(r0, 'prediction', '')}. "
                "Only python code in one fenced block, no prose.")
            r = self._il.write_world_model(brief)
            digest = " ".join((getattr(r, "code", "") or "").split())[:120]
            self._cm.update_zone2("world_model_status", "wm-code: " + digest,
                                  truncate=True)
        except Exception as e:
            logger.info("coder world-model skipped: %s", type(e).__name__)

    def _coder_diagnose(self, kind, detail=""):
        try:
            r0 = self._last_reasoner_result
            trace = (f"{kind}: {detail}; last_prediction="
                     f"{getattr(r0, 'prediction', '')!r}")
            ce = None
            if self._last_packet is not None and hasattr(self._last_packet, "to_dict"):
                try:
                    ce = self._last_packet.to_dict()
                except Exception:
                    ce = None
            try:
                if ce is not None:
                    r = self._il.diagnose_failure(
                        function_name="plan", failure_trace=trace,
                        current_code="def plan(): pass",
                        counterexample=ce)
                else:
                    r = self._il.diagnose_failure(
                        function_name="plan", failure_trace=trace,
                        current_code="def plan(): pass")
            except TypeError:
                r = self._il.diagnose_failure(
                    function_name="plan", failure_trace=trace,
                    current_code="def plan(): pass")
            note = "{} (conf={})".format(
                getattr(r, "root_cause", ""), getattr(r, "confidence", "?"))
            self._cm.update_zone2("world_model_status", "diag: " + note,
                                  truncate=True)
        except Exception as e:
            logger.info("coder diagnose skipped: %s", type(e).__name__)

    def framework_choose(self, latest_frame, prev_state):
        state = _state_name(latest_frame)
        if state != prev_state:
            if state == "WIN": self.reset_level(advanced=True)
            elif state == "GAME_OVER": self.reset_level(advanced=False)
        if state in (None, "", "NOT_PLAYED", "GAME_OVER"):
            return {"name": "RESET", "x": None, "y": None,
                    "reasoning": f"reset:{state or 'init'}"}
        self.set_available_actions(_frame_available_actions(latest_frame))
        grid = _frame_grid(latest_frame)
        if grid is None:
            return {"name": "RESET", "x": None, "y": None,
                    "reasoning": "no-grid"}
        instr = self.act([grid])
        return _instruction_to_action_dict(instr)

    def prompt_log(self): return list(self._prompt_log)
    def dispatched_log(self): return list(self._dispatched_log)
    def jolt_history(self): return self._loop.jolt_history()


def _state_name(frame):
    s = getattr(frame, "state", None) if frame is not None else None
    if s is None: return None
    return str(s.name) if hasattr(s, "name") else str(s)


def _frame_grid(frame):
    f = getattr(frame, "frame", None) if frame is not None else None
    if f is None: return None
    if hasattr(f, "tolist"): f = f.tolist()
    if not isinstance(f, list) or not f: return None
    first = f[0]
    if isinstance(first, list) and first and isinstance(first[0], list):
        return [[int(cell[0]) for cell in row] for row in f]
    return [[int(c) for c in row] for row in f]


def _frame_available_actions(frame):
    aa = getattr(frame, "available_actions", None) if frame is not None else None
    return list(aa) if aa else []


def _instruction_to_action_dict(instruction):
    if not instruction:
        return {"name": "RESET", "x": None, "y": None,
                "reasoning": "no-instruction"}
    parts = str(instruction).split()
    aid = parts[0]
    x = None; y = None
    for p in parts[1:]:
        if p.startswith("r"):
            try: y = int(p[1:])
            except ValueError: pass
        elif p.startswith("c"):
            try: x = int(p[1:])
            except ValueError: pass
    return {"name": aid, "x": x, "y": y, "reasoning": str(instruction)}


class MyAgent(_FrameworkAgent):
    MAX_ACTIONS = 80

    def __init__(self, game_id, _orchestrator=None, **kwargs):
        super().__init__(**kwargs)
        self.game_id = game_id
        self._agent = _orchestrator or Agent(
            game_id=game_id, version_manifest=CERTIFIED_VERSIONS)
        self._prev_state = None
        _jb = _journal_brief()
        if _jb:
            try:
                _cm = getattr(self._agent, "_cm", None)
                if _cm is not None:
                    _cm.update_zone2("confirmed_rules",
                                     "cross-game: " + _jb, truncate=True)
            except Exception:
                pass

    def is_done(self, frames, latest_frame):
        return _state_name(latest_frame) == "WIN"

    def choose_action(self, frames, latest_frame):
        spec = self._agent.framework_choose(latest_frame, self._prev_state)
        self._prev_state = _state_name(latest_frame)
        return _make_framework_action(
            spec["name"], spec["x"], spec["y"], spec["reasoning"])

    @classmethod
    def _default_manifest(cls):
        return dict(CERTIFIED_VERSIONS)

    def cleanup(self, scorecard=None):
        try:
            lv = 0
            _fr = getattr(self, "frames", None)
            if _fr:
                lv = int(getattr(_fr[-1], "levels_completed", 0) or 0)
            _rules = ""
            _cm = getattr(self._agent, "_cm", None)
            if _cm is not None and hasattr(_cm, "snapshot"):
                try:
                    _snap = _cm.snapshot() or {}
                    _z2 = _snap.get("zone2", {}) if isinstance(_snap, dict) else {}
                    _rules = str(_z2.get("confirmed_rules", ""))[:200]
                except Exception:
                    pass
            _TACTICS_JOURNAL.append(
                {"game": self.game_id, "levels": lv, "rules": _rules})
        except Exception:
            pass
        _cleanup = getattr(super(), "cleanup", None)
        if callable(_cleanup):
            try: _cleanup(scorecard)
            except Exception: pass


_FRAMEWORK_ACTION_FACTORY = None


class _FallbackAction:
    def __init__(self, name, x=None, y=None, reasoning=""):
        self.name = name; self.x = x; self.y = y; self.reasoning = reasoning
        self.data = {"x": x, "y": y} if name == "ACTION6" else {}
    def __repr__(self):
        return f"_FallbackAction({self.name!r}, x={self.x}, y={self.y})"


def _make_framework_action(name, x=None, y=None, reasoning=""):
    if _FRAMEWORK_ACTION_FACTORY is not None:
        return _FRAMEWORK_ACTION_FACTORY(name, x, y, reasoning)
    if _HAS_ARCENGINE:
        try:
            act = getattr(GameAction, name)
        except AttributeError:
            act = GameAction.RESET
        if name == "ACTION6":
            try: act.set_data({"x": int(x) if x is not None else 0,
                               "y": int(y) if y is not None else 0})
            except Exception: pass
        try: act.reasoning = reasoning
        except Exception: pass
        return act
    return _FallbackAction(name, x=x, y=y, reasoning=reasoning)


MODULE_ORDER = ("grid_text", "context_manager", "inference_layer",
                "action_pipeline", "my_agent")
ARTIFACT_NAME = "my_agent.py"
DEFAULT_OUT_DIR = "submission"
_ARTIFACT_FIRST_LINE_PREFIX = '"""AUTO-GENERATED artifact.'

_ARTIFACT_HEADER = '''\
"""AUTO-GENERATED artifact.  Do not edit by hand.

Source of truth: the modular agent/ package.
Rebuild with: python agent/my_agent.py build

Public surface (what Kaggle imports):
    MyAgent, Agent, CERTIFIED_VERSIONS, check_certified_versions, ...
"""
'''


def read_modular_sources(root):
    sources = {}
    for name in MODULE_ORDER:
        path = root / "agent" / f"{name}.py"
        if not path.is_file():
            raise FileNotFoundError(f"missing modular source: {path}")
        text = path.read_text(encoding="utf-8")
        if name == "my_agent":
            first_line = text.split("\n", 1)[0].lstrip()
            if first_line.startswith(_ARTIFACT_FIRST_LINE_PREFIX):
                raise AgentError(
                    f"refusing to compile {path}: first line identifies "
                    f"a compiled artifact.")
        sources[name] = text
    return sources


def compile_sources(sources):
    parts = [_ARTIFACT_HEADER]
    parts.append("# ---- loader " + "-" * 52)
    parts.append("import types as _types, sys as _sys")
    parts.append("")
    parts.append("_agent_pkg = _sys.modules.get('agent')")
    parts.append("if _agent_pkg is None:")
    parts.append("    _agent_pkg = _types.ModuleType('agent')")
    parts.append("    _agent_pkg.__path__ = []")
    parts.append("    _sys.modules['agent'] = _agent_pkg")
    parts.append("")
    for name in ("grid_text", "context_manager",
                 "inference_layer", "action_pipeline"):
        if name not in sources: continue
        parts.append(f"# ---- {name} " + "-" * max(0, 52 - len(name)))
        parts.append(f"_SRC_{name} = {json.dumps(sources[name])}")
        parts.append(f"_mod_{name} = _types.ModuleType({name!r})")
        parts.append(f"_mod_{name}.__file__ = {name + '.py'!r}")
        parts.append(f"_sys.modules[{name!r}] = _mod_{name}")
        parts.append(f"_sys.modules['agent.{name}'] = _mod_{name}")
        parts.append(f"setattr(_agent_pkg, {name!r}, _mod_{name})")
        parts.append(f"exec(_SRC_{name}, _mod_{name}.__dict__)")
        parts.append("")
    parts.append("# ---- my_agent " + "-" * 45)
    parts.append(f"_SRC_my_agent = {json.dumps(sources['my_agent'])}")
    parts.append("_saved_name = __name__")
    parts.append("_self = _sys.modules[__name__]")
    parts.append("_sys.modules['my_agent'] = _self")
    parts.append("_sys.modules['agent.my_agent'] = _self")
    parts.append("try:")
    parts.append('    __name__ = "my_agent"')
    parts.append("    exec(_SRC_my_agent, globals())")
    parts.append("finally:")
    parts.append("    __name__ = _saved_name")
    parts.append("")
    parts.append("setattr(_agent_pkg, 'my_agent', _self)")
    parts.append("")
    return "\n".join(parts)


def md5_of(text): return hashlib.md5(text.encode("utf-8")).hexdigest()


def write_artifact(artifact, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(out_path.suffix + ".tmp")
    tmp.write_text(artifact, encoding="utf-8")
    tmp.replace(out_path)
    return md5_of(artifact)


def verify_artifact(out_path):
    import subprocess
    check = textwrap.dedent(f"""\
        import importlib.util, sys
        spec = importlib.util.spec_from_file_location(
            "_artifact_check", {str(out_path)!r})
        mod = importlib.util.module_from_spec(spec)
        sys.modules["_artifact_check"] = mod
        spec.loader.exec_module(mod)
        assert mod.__name__ == "_artifact_check"
        required = ("MyAgent", "Agent", "CERTIFIED_VERSIONS",
                    "check_certified_versions", "lookup_calibration",
                    "LoopDetector", "DEFAULT_CONFIG")
        for attr in required:
            if not hasattr(mod, attr):
                raise SystemExit(f"verify FAIL: missing {{attr!r}}")
        assert mod.CERTIFIED_VERSIONS == {{
            "context_manager": "1.6",
            "inference_layer": "1.6",
            "grid_text": "1.0.8",
            "action_pipeline": "1.0.5",
        }}, mod.CERTIFIED_VERSIONS
        print("verify OK")
    """)
    r = subprocess.run([sys.executable, "-c", check],
                       capture_output=True, text=True)
    if r.stdout: sys.stdout.write(r.stdout)
    if r.stderr: sys.stderr.write(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"artifact verification failed (exit {r.returncode})")


def _resolve_root(explicit):
    if explicit is not None: return explicit.resolve()
    here = Path(__file__).resolve()
    for cand in (here.parent, here.parent.parent, here.parent.parent.parent):
        if (cand / "agent" / "my_agent.py").is_file(): return cand
    return here.parent.parent


def _resolve_out(args, root):
    if args.in_place: return root / "agent" / ARTIFACT_NAME
    if args.out is not None:
        return args.out if args.out.is_absolute() else (root / args.out)
    return root / DEFAULT_OUT_DIR / ARTIFACT_NAME


def build_main(argv):
    p = argparse.ArgumentParser(prog="my_agent.py build")
    p.add_argument("--root", type=Path, default=None)
    p.add_argument("--out", type=Path, default=None)
    p.add_argument("--in-place", action="store_true")
    p.add_argument("--force", action="store_true")
    p.add_argument("--dry-run", action="store_true")
    p.add_argument("--no-verify", action="store_true")
    args = p.parse_args(argv)
    if args.in_place and not args.force:
        print("ERROR: --in-place requires --force", file=sys.stderr)
        return 2
    root = _resolve_root(args.root)
    out = _resolve_out(args, root)
    srcs = read_modular_sources(root)
    artifact = compile_sources(srcs)
    md5 = md5_of(artifact)
    if args.dry_run:
        print(f"dry-run md5: {md5}")
        return 0
    written_md5 = write_artifact(artifact, out)
    if written_md5 != md5:
        raise RuntimeError("write corrupted the artifact")
    print(f"wrote {out}")
    print(f"md5:   {md5}")
    if not args.no_verify:
        verify_artifact(out)
    return 0


class _FakeCM:
    __version__ = "1.6"
    def __init__(self):
        self._zone1 = ""
        self._zone2 = {"confirmed_rules": "", "hypotheses": "",
                       "world_model_status": "", "goal_notes": ""}
        self._zone3 = []; self._obs = None
        self.obs_objects = []
        self.dead_sigs = []; self._level = 0
        self.set_level_called_with = []
    def set_system_prompt(self, t): self._zone1 = t
    def set_observation(self, encoding_dict, available_actions=(),
                        levels_completed=0, state="unknown", objects=None):
        self._obs = (encoding_dict, list(available_actions),
                     levels_completed, state)
        self.obs_objects = list(objects or [])
    def update_zone2(self, f, t, truncate=False):
        if f in self._zone2: self._zone2[f] = t
    def append_action(self, a, r, d): self._zone3.append((a, r, d))
    def register_dead_signature(self, s):
        if s not in self.dead_sigs: self.dead_sigs.append(s)
    def clear_dead_signatures(self): self.dead_sigs.clear()
    def set_level(self, n):
        self._level = n; self.set_level_called_with.append(n)
    def build_prompt(self):
        return _PromptPacket(text=(
            f"[Z1]{self._zone1}\n[Z2]"
            f"{json.dumps(self._zone2, sort_keys=True)}\n[OBS]{self._obs}\n"
            f"do_not_retry={list(self.dead_sigs)}"))


@dataclass
class _PromptPacket: text: str


class _FakeIL:
    __version__ = "1.6"
    def __init__(self, instructions=None):
        self._instructions = list(instructions or []); self._idx = 0
        self.mode_calls = []
        self._coder_model = "qwen3-coder-30b-a3b"
        self._DEFAULT_PARAMS = {"temperature": 0.0}
    def ask_reasoner(self, prompt_packet, mode="fast"):
        self.mode_calls.append(mode)
        instr = ("right" if self._idx >= len(self._instructions)
                 else self._instructions[self._idx])
        self._idx += 1
        return _ReasonerResult("obs", ["h1"], instr, "p")
    def write_world_model(self, brief):
        return _WorldModelResult(code="def world_model(): return 'x'")
    def diagnose_failure(self, function_name, failure_trace, current_code=""):
        return _DiagnoseResult(root_cause="cause", confidence=0.5,
                               suggested_fix="fix")
    def _call_transport(self, **kw):
        return {"choices": [{"message": {"content": "```python\npass\n```"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}


@dataclass
class _WorldModelResult: code: str; notes: str = ""
@dataclass
class _DiagnoseResult:
    root_cause: str; confidence: float = 0.0; suggested_fix: str = ""


@dataclass
class _ReasonerResult:
    observation_summary: str; hypothesis_updates: list
    instruction: str; prediction: str


@dataclass
class _FakeObj:
    ohash: str; bbox: Tuple[int, int, int, int]
    cells: List[Tuple[int, int]]
    # v1.0.21 Object Vision: real grid_text.ObjectRec ke saath parity
    id: int = 0; color_digit: int = 0; color_char: str = "?"
    size: int = 0


class _FakeGT:
    __version__ = "1.0.8"
    def rle_anchor(self, grid):
        h = hashlib.md5()
        for row in grid: h.update(bytes(row))
        return {"type": "full", "dim": [len(grid), len(grid[0])],
                "checksum": h.hexdigest(), "rows": ["ok"]}
    def extract_objects(self, grid, background=0):
        by = {}
        for r, row in enumerate(grid):
            for c, v in enumerate(row):
                if v != background:
                    by.setdefault((v, "1x1"), []).append((r, c, v))
        objs = []
        for (v, shp), cells in by.items():
            rs = [r for r, _, _ in cells]; cs = [c for _, c, _ in cells]
            ohash = hashlib.md5(f"{v}|{shp}".encode()).hexdigest()
            objs.append(_FakeObj(ohash=ohash,
                bbox=(min(rs), max(rs), min(cs), max(cs)),
                cells=[(r, c) for r, c, _ in cells],
                id=len(objs) + 1, color_digit=v,
                color_char="BMPRgw"[v % 6], size=len(cells)))
        return objs


class _FakeAPModule:
    __version__ = "1.0.5"
    CLEAN_SET = {"games_count": 21, "noop_pct": 3.78, "exact_repeat_pct": 1.00,
                 "jitter_pct": 2.35, "retry_success_pct": 59.1,
                 "switch_after_double_fail_pct": 3.9,
                 "pause_retry_s": 1.10, "pause_fresh_s": 0.57}
    REJECT_LIST_IDS = ("lp85-305b61c3", "s5i5-18d95033",
                       "vc33-5430563c", "dc22-fdcac232")
    class CalibrationRow:
        def __init__(self, game_id, is_reject):
            self.game_id = game_id; self.is_reject = is_reject
            self.retry_budget = 3 if is_reject else 2
            self.debounce_steps = 5
    DEFAULT_CALIBRATION = None
    class DivergenceError(RuntimeError):
        def __init__(self, msg, packet):
            super().__init__(msg); self.packet = packet
    class PremiseFailureError(RuntimeError): pass
    class ActionResult:
        def __init__(self, action): self.action = action
    class ActionPipeline:
        def __init__(self, game_id, dispatch=None, patch_hook=None):
            self.game_id = game_id; self._dispatch = dispatch
            self._queue = []; self.step_calls = 0; self.observe_calls = 0
        def submit_plan(self, actions): self._queue = list(actions)
        def step(self):
            self.step_calls += 1
            if not self._queue: return None
            a = self._queue.pop(0)
            if self._dispatch is not None: self._dispatch(a)
            return _FakeAPModule.ActionResult(a)
        def observe(self, frame):
            self.observe_calls += 1; return None


_FakeAPModule.DEFAULT_CALIBRATION = {
    gid: _FakeAPModule.CalibrationRow(
        gid, gid in _FakeAPModule.REJECT_LIST_IDS)
    for gid in (
        "ar25-0c556536", "bp35-0a0ad940", "cd82-fb555c5d",
        "cn04-2fe56bfb", "dc22-fdcac232", "ft09-0d8bbf25",
        "g50t-5849a774", "ka59-38d34dbb", "lf52-271a04aa",
        "lp85-305b61c3", "ls20-9607627b", "m0r0-492f87ba",
        "r11l-495a7899", "re86-8af5384d", "s5i5-18d95033",
        "sb26-7fbdac44", "sc25-635fd71a", "sk48-d8078629",
        "sp80-589a99af", "su15-1944f8ab", "tn36-ef4dde99",
        "tr87-cd924810", "tu93-0768757b", "vc33-5430563c",
        "wa30-ee6fef47")}


def _frame(v=0, h=8, w=8, block=None):
    g = [[v] * w for _ in range(h)]
    if block is not None:
        r0, c0, rh, rw, vv = block
        for r in range(r0, r0 + rh):
            for c in range(c0, c0 + rw): g[r][c] = vv
    return [g]


def _agent(**kw):
    cfg = dict(kw.pop("config", None) or {})
    cfg.setdefault("coder_solver_enabled", False)
    return Agent(game_id=kw.pop("game_id", "ls20-9607627b"),
                 config=cfg,
                 cm=_FakeCM(), il=_FakeIL(instructions=kw.get("instructions")),
                 gt=_FakeGT(), ap=_FakeAPModule,
                 model_config_hook=kw.get("model_config_hook"),
                 version_manifest=kw.get("version_manifest"))


def _stuck_jolt():
    sig = StuckSig(action_id="right", target_xy=None,
                   target_hash=None, target_bbox=None)
    return JoltRequest(kind="stuck", blacklist_sig=sig)


def _t1():
    a = _agent()
    assert a.version_verified["context_manager"]
    class NoVer(_FakeCM): __version__ = None
    try: Agent(game_id="g", config={"coder_solver_enabled": False},
               cm=NoVer(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    except VersionMissing: pass
    else: raise AssertionError
    class Bad(_FakeCM): __version__ = "9.9"
    try: Agent(game_id="g", config={"coder_solver_enabled": False},
               cm=Bad(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    except VersionMismatch: pass
    else: raise AssertionError


def _t2():
    a = _agent(instructions=["right"]*4); f = _frame()
    a.act(f)
    for _ in range(3): a.act(f)
    assert a._row.retry_budget == 2 and a._row.is_reject is False


def _t3():
    class P: pass
    p = P(); err = _FakeAPModule.DivergenceError("d", p)
    assert err.packet is p


def _t4():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5)
    fp = ((), 0); fired = None
    for i in range(1, 5):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fired = i
    assert fired == 4


def _t5():
    ld = LoopDetector("lp85-305b61c3", True, _FakeCM(), 5)
    fp = ((), 0); fired = None
    for i in range(1, 6):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fired = i
    assert fired == 5


def _t6():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5)
    fp = ((), 0); th = "h"
    ld.tick(action_id="click", target_xy=(10, 10), target_hash=th,
            target_bbox=(10,10,10,10), state_changed=False, objset_fp=fp)
    ld.tick(action_id="click", target_xy=(12, 12), target_hash=th,
            target_bbox=(12,12,12,12), state_changed=False, objset_fp=fp)
    assert ld.streak_length() == 2
    ld.tick(action_id="click", target_xy=(15, 15), target_hash=th,
            target_bbox=(15,15,15,15), state_changed=False, objset_fp=fp)
    assert ld.streak_length() == 1


def _t7():
    a = _agent(instructions=["right"]*6); a.set_available_actions(["right"])
    f = _frame(); a.act(f)
    for _ in range(6): a.act(f)
    assert any(j.kind == "stuck" for j in a.jolt_history())


def _t8():
    cm = _FakeCM(); a = _agent(); a._cm = cm; a._loop._cm = cm
    cm.register_dead_signature("X"); assert len(cm.dead_sigs) == 1
    a.act(_frame(block=(1,1,1,1,5))); a.act(_frame(block=(2,2,1,1,5)))
    assert len(cm.dead_sigs) == 1
    a.act(_frame(h=8, w=8, block=(1,1,1,1,5)))
    f2 = _frame(h=8, w=8, block=(1,1,1,1,5)); f2[0][3][3] = 8
    a.act(f2); assert len(cm.dead_sigs) == 0


def _t9():
    a = _agent(instructions=["right"]*8); a.set_available_actions(["right"])
    f = _frame(); a.act(f)
    for _ in range(6): a.act(f)
    assert any(j.kind == "stuck" for j in a.jolt_history())
    assert a._pipeline.step_calls == a.total_dispatches()


def _t10():
    assert "enable_physical_jolt" not in DEFAULT_CONFIG
    assert "enable_physical_jolt" not in _agent().config


def _t11():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 2); fp = (("x",1),)
    fired = None
    for i in range(1, 6):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=True, objset_fp=fp)
        if r is not None: fired = i; break
    assert fired == 4


def _t12():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 2); fp = (("x",1),)
    fire = None
    for aid in ["a","b","c","d","e"]:
        r = ld.tick(action_id=aid, target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fire = r; break
    assert fire is not None and fire.kind == "deadlock"


def _t13():
    a = _agent()
    assert a.pause_before_retry_s == 1.10 and a.pause_before_fresh_s == 0.57


def _t14():
    a = _agent()
    assert a._row.retry_budget == 2
    a2 = Agent(game_id="lp85-305b61c3",
               config={"coder_solver_enabled": False},
               cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a2._row.retry_budget == 3 and a2._row.is_reject is True


def _t15():
    spy = []
    _agent(model_config_hook=lambda p: spy.append(dict(p)))
    assert len(spy) == 1


def _t16():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5)
    for b in ("dispatch", "step", "observe", "act"):
        assert not hasattr(ld, b)


def _t17():
    def run():
        a = _agent(instructions=["right"]*8); a.set_available_actions(["right"])
        return [a.act(_frame()) for _ in range(4)], a.prompt_log()
    assert run() == run()


def _t18():
    a = Agent(game_id="ls20-9607627b",
              config={"reasoner_endpoint": "q\r\nwen",
                      "coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert "\r" not in a.config["reasoner_endpoint"]


def _t19():
    a = _agent(instructions=["right"]*4)
    a._prev_objset_fp = (("x",1),)
    a._cm.register_dead_signature("SEED")
    a.reset_level(level_num=2, advanced=True)
    assert a._levels_completed == 1


def _t20():
    a = _agent(instructions=["right"]*4); a.set_available_actions(["right"])
    f = _frame(); a.act(f)
    a._handle_jolt(JoltRequest(kind="deadlock"))
    assert a._force_anchor and a._resync
    a.act(f)
    assert "[FULL RESYNC]" in "\n".join(a.prompt_log())


def _t21():
    a = _agent(instructions=["click r5 c7"]*4)
    f = _frame(block=(5,7,1,1,8)); a.act(f); a.act(f)
    aid, txy = a._parse_instruction(str(a._dispatched_log[0]))
    assert aid == "click" and txy == (5, 7)


def _t22():
    class NoCM(_FakeCM): __version__ = None
    try: Agent(game_id="x", config={"coder_solver_enabled": False},
               cm=NoCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    except VersionMissing: pass
    else: raise AssertionError


def _t23():
    a = Agent(game_id="ls20-9607627b",
              config={"reasoner_endpoint": "q\r\nwen\rmore",
                      "coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a.config["reasoner_endpoint"] == "q\nwen\nmore"


def _t24():
    a = _agent(instructions=["right"]); f = _frame()
    for _ in range(DISPATCHED_LOG_CAP + 50): a.act(f)
    assert len(a._dispatched_log) <= DISPATCHED_LOG_CAP


def _t25():
    c = {"reasoner_endpoint": "a\r\nb"}; _agent(config=c)
    assert c["reasoner_endpoint"] == "a\r\nb"


def _t26():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5); fp = (("x",1),)
    for _ in range(DEADLOCK_WINDOWS * 5 - 1):
        ld.tick(action_id="right", target_xy=None, target_hash=None,
                target_bbox=None, state_changed=True, objset_fp=fp)
    fired = None
    for _ in range(5):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None and r.kind == "stuck": fired = r; break
    assert fired is not None


def _t27():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 2); fp = (("x",1),)
    for _ in range(DEADLOCK_WINDOWS * 2 - 1):
        ld.tick(action_id="right", target_xy=None, target_hash=None,
                target_bbox=None, state_changed=True, objset_fp=fp)
    r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                target_bbox=None, state_changed=False, objset_fp=fp)
    assert r is not None and r.kind == "deadlock"


def _t28():
    a = Agent(game_id="ls20", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.resolved_id == "ls20-9607627b"


def _t29():
    _WARNED_UNKNOWN_IDS.discard("zzz-nope")
    a = Agent(game_id="zzz-nope", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.known is False


def _t30():
    a = Agent(game_id="dc22", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.is_reject is True


class _FakeFrame:
    def __init__(self, state="NOT_PLAYED", grid=None, avail=None):
        self.state = state
        self.frame = grid if grid is not None else [[0]*8 for _ in range(8)]
        self.available_actions = list(avail or [])


class _GameState:
    def __init__(self, name): self.name = name


def _with_real_state(frame, name):
    frame.state = _GameState(name); return frame


def _t31():
    a = _agent(instructions=["right"])
    w = MyAgent(game_id="ls20-9607627b", _orchestrator=a)
    f = _with_real_state(_FakeFrame("GAME_OVER"), "GAME_OVER")
    assert w.is_done([], f) is False
    f = _with_real_state(_FakeFrame("WIN"), "WIN")
    assert w.is_done([], f) is True


def _t32():
    spec = _instruction_to_action_dict("click r5 c7")
    assert spec["name"] == "click" and spec["x"] == 7


def _t33():
    sub = {"grid_text": "x", "context_manager": "x", "inference_layer": "x",
           "action_pipeline": "x", "my_agent": "x"}
    assert compile_sources(sub) == compile_sources(sub)


def _t34():
    assert MyAgent._default_manifest() == dict(CERTIFIED_VERSIONS)


def _t35():
    class OddAP(_FakeAPModule):
        DEFAULT_CALIBRATION = {
            "a-1": {"is_reject": False, "retry_budget": 2, "debounce_steps": 3},
            "b-2": {"is_reject": False, "retry_budget": 4, "debounce_steps": 4},
            "c-3": {"is_reject": False, "retry_budget": 5, "debounce_steps": 8}}
    budget, debounce = _aggregate_clean_defaults(OddAP.DEFAULT_CALIBRATION.values())
    assert budget == 2 and debounce == 4


def _t36():
    a = Agent(game_id="lp85", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.is_reject is True


def _t37():
    class DictAP(_FakeAPModule):
        DEFAULT_CALIBRATION = {"ls20-9607627b": {"is_reject": False,
                              "retry_budget": 7, "debounce_steps": 9}}
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=DictAP)
    assert a._row.retry_budget == 7


def _t38():
    class BadClean(_FakeAPModule): CLEAN_SET = {"games_count": 1}
    try: Agent(game_id="unknown-id",
               config={"coder_solver_enabled": False},
               cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=BadClean)
    except AgentError: pass
    else: raise AssertionError


def _t39():
    assert callable(build_main)
    try: build_main(["--help"])
    except SystemExit as e: assert e.code == 0


def _t40():
    import ast, inspect
    tree = ast.parse(inspect.getsource(sys.modules[__name__]))
    for n in tree.body:
        if isinstance(n, ast.Import):
            for a in n.names:
                assert a.name.split(".")[0] != "subprocess"


def _t41():
    import tempfile, shutil
    root = Path(tempfile.mkdtemp(prefix="t41_"))
    try:
        (root / "agent").mkdir()
        for name in ("grid_text", "context_manager",
                     "inference_layer", "action_pipeline"):
            (root / "agent" / f"{name}.py").write_text(f"# {name}\n", encoding="utf-8")
        (root / "agent" / "my_agent.py").write_text(
            '"""AUTO-GENERATED artifact.\n"""\nX = 1\n', encoding="utf-8")
        try: read_modular_sources(root)
        except AgentError: pass
        else: raise AssertionError
    finally:
        shutil.rmtree(root, ignore_errors=True)


def _t42():
    il = _FakeIL(instructions=["right"] * 8)
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    a.act(_frame()); a.act(_frame()); a.act(_frame())
    assert il.mode_calls[0] == "deep" and il.mode_calls[1] == "fast"


def _t43():
    il = _FakeIL(instructions=["right"] * 12)
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"]); a.act(_frame()); a.act(_frame())
    a._handle_jolt(_stuck_jolt()); a.act(_frame())
    assert il.mode_calls[-1] == "deep"


def _t44():
    cap = Agent._DEEP_TRIGGER_MAX_PER_GAME
    il = _FakeIL(instructions=["right"] * (cap + 8))
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    for _ in range(cap):
        a._handle_jolt(_stuck_jolt()); a.act(_frame())
    assert a._deep_calls_this_game == cap
    a._handle_jolt(_stuck_jolt()); a.act(_frame())
    assert il.mode_calls[-1] == "fast"


def _t45():
    cm = _FakeCM()
    ld = LoopDetector("ls20-9607627b", False, cm, 5,
                      allow_repetition_while=lambda: True,
                      on_repetition_allowed=lambda: None)
    fp = ((), 0); fired = None
    for i in range(1, 8):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fired = i; break
    assert fired is None


def _t46():
    cap = Agent._DEEP_TRIGGER_MAX_PER_GAME
    il = _FakeIL(instructions=["right"] * (cap + 8))
    a = Agent(game_id="ls20-9607627b",
              config={"reasoner_mode": "deep",
                      "coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    for _ in range(cap + 4): a.act(_frame())
    assert a._deep_calls_this_game == cap
    assert il.mode_calls[cap] == "fast"


def _t47():
    il = _FakeIL(instructions=["right"] * 16)
    a = Agent(game_id="ls20-9607627b",
              config={"deep_call_cap": 2, "coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    a.act(_frame())
    a._handle_jolt(_stuck_jolt()); a.act(_frame())
    a.act(_frame()); a.act(_frame()); a.act(_frame()); a.act(_frame())
    assert a._deep_calls_this_game == 2


def _t48():
    for bad in [-1, 0, True, False, 3.7, "abc", "8", None, [], {}]:
        try:
            Agent(game_id="ls20-9607627b",
                  config={"deep_call_cap": bad,
                          "coder_solver_enabled": False},
                  cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
        except AgentError: pass
        else: raise AssertionError(f"expected AgentError for {bad!r}")
    for good in (1, 2, 8, 100):
        a = Agent(game_id="ls20-9607627b",
                  config={"deep_call_cap": good,
                          "coder_solver_enabled": False},
                  cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
        assert a._deep_cap() == good


def _t49():
    out = LoopDetector._format_blacklist(None)
    assert isinstance(out, str) and "SYSTEM OVERRIDE" in out
    a = _agent(instructions=["right"] * 4)
    a.set_available_actions(["right"]); a.act(_frame())
    a._handle_jolt(JoltRequest(kind="stuck"))
    assert any("SYSTEM OVERRIDE" in inj for inj in a._pending_injections)


def _t50_coder_flag_default_on():
    a = Agent(game_id="ls20-9607627b",
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule,
              version_manifest={"inference_layer": "1.6"})
    assert a.config.get("coder_solver_enabled") is True


def _t51_coder_flag_off_reasoner_only():
    a = _agent()
    assert a._solver is None
    assert a._transition_buffer is None
    assert a._should_attempt_coder_solve() is False


def _t52_coder_attempt_cap():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _StubBuffer:
        def __len__(self): return 5
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _StubBuffer()
    a._coder_attempts_this_game = CODER_ATTEMPTS_PER_GAME_CAP - 1
    a._jolt_pending = True
    assert a._should_attempt_coder_solve() is True

    a._coder_attempts_this_game = CODER_ATTEMPTS_PER_GAME_CAP
    assert a._should_attempt_coder_solve() is False

    a._coder_attempts_this_game = 0
    a._jolt_pending = False
    a._divergence_pending = False
    assert a._should_attempt_coder_solve() is False


def _t53_il_16_certified():
    assert CERTIFIED_VERSIONS["inference_layer"] == "1.6"


def _t54_verify_artifact_expects_16():
    import inspect
    src = inspect.getsource(verify_artifact)
    assert '"inference_layer": "1.6"' in src


def _t55_coder_attempts_accessor():
    a = _agent()
    a._coder_attempts_this_game = 4
    assert a.coder_attempts() == 4


def _t56_coder_result_accessor():
    a = _agent()
    assert a.coder_result() is None


def _t57_reset_level_clears_pending():
    a = _agent()
    a._pending_action = "ACTION1"
    a._prev_settled_grid = [[1, 2], [3, 4]]
    a.reset_level(level_num=1, advanced=True)
    assert a._pending_action is None
    assert a._prev_settled_grid is None


def _t58_coder_actions_accessor():
    a = _agent()
    a._coder_actions_this_game = 3
    assert a.coder_actions() == 3


def _t59_empty_buffer_no_budget_drain():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
        def solve_world_model(self, *_a, **_kw):
            raise AssertionError("solve_world_model must not be called on empty buffer")
        def solve_action_program(self, *_a, **_kw):
            raise AssertionError("solve_action_program must not be called on empty buffer")

    class _EmptyBuffer:
        def __len__(self): return 0
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _EmptyBuffer()
    a._actions_at_level = 0
    assert a._should_attempt_coder_solve() is False

    for _ in range(6):
        a._maybe_admit_coder({"task": "t", "mode": "world"})

    assert a._coder_attempts_this_game == 0
    assert a._coder_attempts_this_level == 0


def _t66_proactive_default_off():
    """v1.0.20: default config (seam absent) => reactive-only, koi behavior
    change nahi — no jolt + no divergence = coder attempt nahi, chahe
    actions_at_level kuch bhi ho."""
    a = _agent()
    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []
    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._jolt_pending = False
    a._divergence_pending = False
    a._coder_replan_pending = False
    a._coder_attempts_this_game = 0
    a._actions_at_level = 99  # bohot zyada actions, par seam absent
    assert a._should_attempt_coder_solve() is False
    # bool/invalid values ko bhi ignore karna chahiye (fail-closed)
    a.config["coder_proactive_actions_at_level"] = True
    assert a._should_attempt_coder_solve() is False
    a.config["coder_proactive_actions_at_level"] = -5
    assert a._should_attempt_coder_solve() is False


def _t67_proactive_threshold_fires():
    """v1.0.20: coder_proactive_actions_at_level=N => N actions at-level par
    (bina jolt/divergence ke) coder attempt trigger hota hai."""
    a = _agent()
    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []
    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._jolt_pending = False
    a._divergence_pending = False
    a._coder_replan_pending = False
    a._coder_attempts_this_game = 0
    a.config["coder_proactive_actions_at_level"] = 12
    a._actions_at_level = 11
    assert a._should_attempt_coder_solve() is False
    a._actions_at_level = 12
    assert a._should_attempt_coder_solve() is True
    # attempts cap abhi bhi respected
    a._coder_attempts_this_game = CODER_ATTEMPTS_PER_GAME_CAP
    assert a._should_attempt_coder_solve() is False


def _t68_objects_passed_to_cm():
    """v1.0.21: extract_objects ka structured output cm.set_observation
    tak objects= kwarg ke through pahunchta hai (Object Vision)."""
    a = _agent(instructions=["right"] * 6)
    a.set_available_actions(["right"])
    a.act(_frame(block=(1, 1, 2, 3, 5)))
    objs = a._cm.obs_objects
    assert objs, "objects CM tak nahi pahunche"
    o = objs[0]
    for k in ("id", "color_char", "color_digit", "size", "bbox"):
        assert k in o, f"object payload missing key {k!r}"
    assert list(o["bbox"]) == [1, 2, 1, 3], o["bbox"]
    assert o["color_digit"] == 5 and o["color_char"] == "w", o
    assert o["size"] == 6, o["size"]


def _t69_palette_crosscheck():
    """v1.0.21 integration guard: CM _FALLBACK_PALETTE == grid_text
    CHAR_TO_DIGIT.  (Run-21 core audit: do certified modules alag format
    par test hue the — ye drift phir kabhi nahi hogi.)"""
    cm_pal = dict(_cm_module._FALLBACK_CHAR_TO_DIGIT)
    gt_pal = dict(_gt_module.CHAR_TO_DIGIT)
    assert cm_pal == gt_pal, (
        f"CM fallback palette drift: CM={cm_pal} GT={gt_pal}")


def _t60_level_start_no_coder_trigger():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _EmptyBuffer:
        def __len__(self): return 0
        def as_list(self): return []
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _EmptyBuffer()
    a._actions_at_level = 0
    assert a._should_attempt_coder_solve() is False

    a._jolt_pending = True
    assert a._should_attempt_coder_solve() is False

    a._transition_buffer = _FullBuffer()
    assert a._should_attempt_coder_solve() is True


def _t61_replan_clears_hypotheses():
    a = _agent()
    a._cm.update_zone2("hypotheses", "old stale hypothesis", truncate=True)
    a._cm.update_zone2("world_model_status", "old status", truncate=True)

    class _StubResult:
        admitted = False
        replan_requested = True
        mode = "world"
        source = ""

    a._handle_coder_replan(_StubResult())

    assert a._cm._zone2["hypotheses"] == "", \
        f"hypotheses not cleared: {a._cm._zone2['hypotheses']!r}"
    assert "replan" in a._cm._zone2["world_model_status"].lower()
    assert a._coder_replan_pending is True
    assert a._coder_replan_count == 1
    assert a._coder_guarded is None
    # A-CEGIS-17: explicit deep-call trigger.
    assert a._jolt_pending is True


def _t62_replan_blocks_coder_attempt():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._jolt_pending = True
    assert a._should_attempt_coder_solve() is True

    a._coder_replan_pending = True
    assert a._should_attempt_coder_solve() is False, \
        "coder attempt must be suppressed during REPLAN"


def _t63_replan_flag_cleared_after_reasoner():
    il = _FakeIL(instructions=["right"] * 4)
    a = _agent()
    a._il = il

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._coder_replan_pending = True

    a.act(_frame())

    assert a._coder_replan_pending is False, \
        "pending flag should be cleared after reasoner runs"


# ---------- A-CEGIS-16: world-model REPLAN short-circuits ---------- #

def _t64_world_model_replan_short_circuits():
    """A-CEGIS-16: when solve_world_model returns replan_requested=True,
    _handle_coder_replan is invoked and solve_action_program is NEVER
    called (REPLAN signal is not overwritten by the fallback)."""
    a = _agent()
    a._cm.update_zone2("hypotheses", "old", truncate=True)
    a._cm.update_zone2("world_model_status", "old", truncate=True)

    class _StubResult:
        def __init__(self, admitted=False, replan_requested=False,
                     mode="world", source=""):
            self.admitted = admitted
            self.replan_requested = replan_requested
            self.mode = mode
            self.source = source

    call_log = {"world": 0, "action": 0}

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
        def solve_world_model(self, *_a, **_kw):
            call_log["world"] += 1
            return _StubResult(admitted=False, replan_requested=True)
        def solve_action_program(self, *_a, **_kw):
            call_log["action"] += 1
            return _StubResult(admitted=True, replan_requested=False,
                               mode="action")

    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()

    admitted = a._maybe_admit_coder({"task": "t", "mode": "world"})

    assert admitted is False
    assert call_log["world"] == 1
    assert call_log["action"] == 0, \
        f"A-CEGIS-16: solve_action_program must NOT run after REPLAN; got {call_log['action']} calls"
    assert a._coder_replan_pending is True
    assert a._coder_replan_count == 1
    assert a._cm._zone2["hypotheses"] == ""
    assert a._coder_guarded is None
    assert a._jolt_pending is True


def _t65_world_model_not_admitted_falls_through_to_action():
    """A-CEGIS-16 complement: when world-model fails WITHOUT a REPLAN
    (e.g. few-mismatch repair exhausted), the action-program fallback
    still runs — the short-circuit only fires on REPLAN.

    A-CEGIS-19: dead lines removed from the previous revision.
    """
    a = _agent()

    class _StubResult:
        def __init__(self, admitted=False, replan_requested=False,
                     mode="world", source=""):
            self.admitted = admitted
            self.replan_requested = replan_requested
            self.mode = mode
            self.source = source

    call_log = {"world": 0, "action": 0}

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
        def solve_world_model(self, *_a, **_kw):
            call_log["world"] += 1
            return _StubResult(admitted=False, replan_requested=False)
        def solve_action_program(self, *_a, **_kw):
            call_log["action"] += 1
            return _StubResult(admitted=True, replan_requested=False,
                               mode="action",
                               source="def next_action(f, h): return 'right'")

    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()

    admitted = a._maybe_admit_coder({"task": "t", "mode": "world"})

    assert call_log["world"] == 1
    assert call_log["action"] == 1, \
        "action-program fallback must run when world-model fails WITHOUT REPLAN"
    assert admitted is True


def _run_self_tests():
    tests = [
        ("T-1", _t1), ("T-2", _t2), ("T-3", _t3), ("T-4", _t4), ("T-5", _t5),
        ("T-6", _t6), ("T-7", _t7), ("T-8", _t8), ("T-9", _t9), ("T-10", _t10),
        ("T-11", _t11), ("T-12", _t12), ("T-13", _t13), ("T-14", _t14),
        ("T-15", _t15), ("T-16", _t16), ("T-17", _t17), ("T-18", _t18),
        ("T-19", _t19), ("T-20", _t20), ("T-21", _t21), ("T-22", _t22),
        ("T-23", _t23), ("T-24", _t24), ("T-25", _t25), ("T-26", _t26),
        ("T-27", _t27), ("T-28", _t28), ("T-29", _t29), ("T-30", _t30),
        ("T-31", _t31), ("T-32", _t32), ("T-33", _t33), ("T-34", _t34),
        ("T-35", _t35), ("T-36", _t36), ("T-37", _t37), ("T-38", _t38),
        ("T-39", _t39), ("T-40", _t40), ("T-41", _t41), ("T-42", _t42),
        ("T-43", _t43), ("T-44", _t44), ("T-45", _t45), ("T-46", _t46),
        ("T-47", _t47), ("T-48", _t48), ("T-49", _t49),
        ("T-50 coder flag default ON", _t50_coder_flag_default_on),
        ("T-51 coder off reasoner-only", _t51_coder_flag_off_reasoner_only),
        ("T-52 coder attempt cap", _t52_coder_attempt_cap),
        ("T-53 IL 1.6 certified", _t53_il_16_certified),
        ("T-54 verify_artifact 1.6", _t54_verify_artifact_expects_16),
        ("T-55 coder_attempts accessor", _t55_coder_attempts_accessor),
        ("T-56 coder_result accessor", _t56_coder_result_accessor),
        ("T-57 reset_level clears pending", _t57_reset_level_clears_pending),
        ("T-58 coder_actions accessor", _t58_coder_actions_accessor),
        ("T-59 empty buffer no budget drain", _t59_empty_buffer_no_budget_drain),
        ("T-60 level-start no coder trigger", _t60_level_start_no_coder_trigger),
        ("T-61 replan clears hypotheses", _t61_replan_clears_hypotheses),
        ("T-62 replan blocks coder attempt", _t62_replan_blocks_coder_attempt),
        ("T-63 replan flag cleared after reasoner", _t63_replan_flag_cleared_after_reasoner),
        ("T-64 world-model REPLAN short-circuits (A-CEGIS-16)",
         _t64_world_model_replan_short_circuits),
        ("T-65 world-model non-REPLAN falls through (A-CEGIS-16)",
         _t65_world_model_not_admitted_falls_through_to_action),
        ("T-66 proactive coder default OFF (v1.0.20)",
         _t66_proactive_default_off),
        ("T-67 proactive coder threshold fires (v1.0.20)",
         _t67_proactive_threshold_fires),
        ("T-68 objects passed to CM (Object Vision v1.0.21)",
         _t68_objects_passed_to_cm),
        ("T-69 CM/grid_text palette cross-check (v1.0.21)",
         _t69_palette_crosscheck),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn(); logging.info("PASS  %s", name)
        except Exception as e:
            failed += 1
            logging.info("FAIL  %s: %s: %s", name, type(e).__name__, e)
    logging.info("")
    logging.info("%d/%d passed", len(tests) - failed, len(tests))
    return 0 if failed == 0 else 1


def _main(argv=None):
    if argv is None: argv = sys.argv[1:]
    if argv and argv[0] == "build": return build_main(argv[1:])
    if argv and argv[0] == "test": return _run_self_tests()
    if argv and argv[0] in ("-h", "--help", "help"):
        print("Usage: my_agent.py [command]\n\n"
              "Commands:\n  test\n  build [options]\n  help")
        return 0
    if argv:
        print(f"unknown command: {argv[0]!r}", file=sys.stderr)
        return 2
    return _run_self_tests()


if __name__ == "__main__":
    import sys as _sys
    logging.basicConfig(level=logging.INFO,
                        format="%(message)s", stream=_sys.stdout)
    _sys.exit(_main())


In [ ]:
%%writefile goa/agent/lora_datagen.py
#!/usr/bin/env python3
# lora_datagen.py — Task 8 Phase A dataset generator (Rev 6)
# CPU-only, deterministic (process-stable, per-game isolated), no network at runtime.
# Public API only from certified modules (DIR/060 §4.2).
#
# ============================================================
# DEVIATION REGISTER (S-8) — D-1 .. D-10
# ============================================================
# D-1  Zone 1/2 template loaded from external file
#      (--zone12-template / GOA_ZONE12_TEMPLATE). Certified reasoner
#      template not available to this author. OPEN — Chief supplies
#      file before Phase B.
# D-2  parse_replays import per DIR/060 §4.2 (certified). CLOSED.
# D-3  Engine-level seed not exposed by arcengine 0.9.3. Mitigated by
#      sha256-derived per-game seeding. Backend re-run is the arbiter.
# D-4  DG-5 stratified K=20 across episode (not first-K). CLOSED.
# D-5  Pre-flight zero-coverage check covers all 25 games per order
#      §3 STEP 1 literal text. Chief to rule if hold-out coverage may
#      be waived. OPEN.
# D-6  pyarrow required for inline fallback loader only. Not in the
#      §4.1 forbidden list. CLOSED.
# D-7  Manifest self-hash excluded (avoid self-reference). CLOSED.
# D-8  Split tolerance: warn if observed replay-guided fraction
#      deviates from 0.60 by >0.10. Order permits adjustment to replay
#      availability. CLOSED.
# D-9  Certified engine legend fixtures not supplied. Synthetic
#      fixtures prove code-path liveness, not production parity.
#      OPEN — backend to supply certified fixtures.
# D-10 Template validated at load time via dummy render (N-2 fix).
#      CLOSED.
#
# TEMPLATE CONTRACT: placeholders {before_grid}, {action}, {after_grid},
# {what_changed} are REQUIRED. Literal braces must be doubled ({{, }}).
# Any missing required placeholder fails at load time, before any
# rollout runs (N-6).
#
# REV 6 CHANGELOG (DIR/066):
#   FIX 1 (H-F1) — r11l-495a7899, sc25-635fd71a (two IDs corrected).
#   FIX 2 (H-F2) — local valid_grid helper; grid_text.valid_grid import
#                   removed (grid_text v1.0.8 __all__ does not export it).
#   FIX 3 (N-8)  — _load_cert_replays hard-aborts on ImportError unless
#                   --no-cert-replays explicitly passed.
#   FIX 4 (N-9)  — per-game global random module re-seed at game-loop
#                   entry, isolating any engine-side global RNG use.
#   N-11         — _cert_load_attempted guard; no double-log / double-raise.
# ============================================================

from __future__ import annotations

import argparse
import hashlib
import json
import logging
import os
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterator

# --- Certified module imports (public API only, §4.2) ---
# LA-2 FIX (Indus, 2026-09-29 final logic audit): bare flat import breaks
# in the notebook package context (only agent.* is importable there) —
# crashed FRAME PROBE's RGB branch. Dual import, fail-closed on neither.
try:
    from grid_text import extract_objects, object_delta
except ImportError:
    from agent.grid_text import extract_objects, object_delta

# parse_replays IS certified per DIR/060 §4.2 — default path.
_cert_iter_replays = None
_cert_load_attempted = False
_REPLAY_SOURCE = "inline_parquet"

def _load_cert_replays(prefer_cert: bool) -> None:
    """
    FIX 3 (N-8): hard abort on ImportError unless operator passed
    --no-cert-replays.  N-11: no retry / no double log on a second call.
    """
    global _cert_iter_replays, _REPLAY_SOURCE, _cert_load_attempted
    if _cert_iter_replays is not None:
        return
    if _cert_load_attempted:
        if prefer_cert:
            raise RuntimeError(
                "parse_replays previously failed to import; "
                "pass --no-cert-replays to use the inline parquet reader."
            )
        return
    _cert_load_attempted = True
    if not prefer_cert:
        _REPLAY_SOURCE = "inline_parquet"
        return
    try:
        from parse_replays import iter_replay_sessions as _fn  # noqa
        _cert_iter_replays = _fn
        _REPLAY_SOURCE = "parse_replays"
    except ImportError as e:
        raise RuntimeError(
            f"Certified parse_replays unavailable: {e}. "
            "Pass --no-cert-replays to use the inline parquet reader."
        ) from e

try:
    from arc_agi import Arcade
except ImportError:
    try:
        from arcengine import Arcade
    except ImportError:
        Arcade = None


# ============================================================
# Configuration
# ============================================================

HOLD_OUT_GAMES = {
    "ft09-0d8bbf25", "vc33-5430563c", "tu93-0768757b",
    "ka59-38d34dbb", "cd82-fb555c5d",
}

ALL_GAME_IDS = [
    "ar25-0c556536", "bp35-0a0ad940", "cd82-fb555c5d", "cn04-2fe56bfb",
    "dc22-fdcac232", "ft09-0d8bbf25", "g50t-5849a774", "ka59-38d34dbb",
    "lf52-271a04aa", "lp85-305b61c3", "ls20-9607627b", "m0r0-492f87ba",
    "r11l-495a7899", "re86-8af5384d", "s5i5-18d95033", "sb26-7fbdac44",
    "sc25-635fd71a", "sk48-d8078629", "sp80-589a99af", "su15-1944f8ab",
    "tn36-ef4dde99", "tr87-cd924810", "tu93-0768757b", "vc33-5430563c",
    "wa30-ee6fef47",
]

TRAIN_GAMES = [g for g in ALL_GAME_IDS if g not in HOLD_OUT_GAMES]
assert len(TRAIN_GAMES) == 20, f"expected 20 train games, got {len(TRAIN_GAMES)}"

ACTIONS_PER_GAME = 500
REPLAY_TARGET_FRACTION = 0.60
SPLIT_WARN_TOLERANCE = 0.10
DG5_SAMPLE_K = 20
SEED = 20260921
DATASET_VERSION = "1.0.0"
SCHEMA_VERSION = "1.0"

REPLAY_TOTAL_EXPECTED = 340
REPLAY_USABLE_EXPECTED = 336
REPLAY_RANGE_LO, REPLAY_RANGE_HI = 336, 342

UNUSABLE_REPLAYS = {"p0_44", "p1_0", "p1_9", "p2_21"}

ARC_PALETTE = [
    (0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64),
    (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),
    (127, 219, 255), (135, 12, 37), (255, 255, 255), (0, 255, 0),
    (0, 0, 255), (255, 0, 0), (255, 255, 0), (128, 128, 128),
]
_PALETTE_LOOKUP = {c: i for i, c in enumerate(ARC_PALETTE)}


# ============================================================
# Helpers — deterministic by construction
# ============================================================

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def _stable_hash(obj: Any) -> str:
    s = json.dumps(obj, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(s.encode()).hexdigest()

def _stable_seed_int(*parts: str) -> int:
    h = hashlib.sha256("|".join(parts).encode()).digest()
    return int.from_bytes(h[:8], "big")

def valid_grid(frame: Any) -> bool:
    """
    FIX 2 (H-F2): local engine-style frame validator, DIR/062 §4a shape.
    Frame is a non-empty list of rectangular lists of int in [0, 15].
    Certified grid_text v1.0.8 does NOT export valid_grid.
    """
    if not isinstance(frame, list) or not frame:
        return False
    if not isinstance(frame[0], list) or not frame[0]:
        return False
    w = len(frame[0])
    for row in frame:
        if not isinstance(row, list) or len(row) != w:
            return False
        for v in row:
            if not isinstance(v, int) or v < 0 or v > 15:
                return False
    return True

def log_warn(msg: str) -> None: logging.warning("[WARN] %s", msg)
def log_info(msg: str) -> None: logging.info("[INFO] %s", msg)


# ============================================================
# Replay loading
# ============================================================

def _inline_iter_replays(replay_dir: Path) -> Iterator[dict]:
    import pyarrow.parquet as pq
    for fp in sorted(replay_dir.glob("*.parquet")):
        pf = pq.ParquetFile(fp)
        for rg in range(pf.num_row_groups):
            for row in pf.read_row_group(rg).to_pylist():
                yield _normalize_replay_row(row)

def _normalize_replay_row(row: dict) -> dict:
    sid = row.get("id") or row.get("session_id") or row.get("name")
    gid = row.get("game_id") or row.get("game")
    actions = row.get("actions") or row.get("action_list") or []
    if isinstance(actions, str):
        actions = json.loads(actions)
    return {"id": sid, "game_id": gid, "actions": list(actions)}

def _iter_all_replays(replay_dir: Path, prefer_cert: bool) -> Iterator[dict]:
    if prefer_cert:
        _load_cert_replays(prefer_cert)
        if _cert_iter_replays is not None:
            yield from _cert_iter_replays(replay_dir)
            return
    yield from _inline_iter_replays(replay_dir)


# ============================================================
# STEP 1 — Replay pre-flight
# ============================================================

def verify_replays(replay_dir: Path, prefer_cert: bool) -> dict:
    if replay_dir is None or not replay_dir.exists():
        raise RuntimeError("Replay directory missing.")

    sessions = list(_iter_all_replays(replay_dir, prefer_cert))
    total = len(sessions)
    usable = [s for s in sessions if s.get("id") not in UNUSABLE_REPLAYS]
    usable_count = len(usable)

    if not (REPLAY_RANGE_LO <= total <= REPLAY_RANGE_HI):
        raise RuntimeError(
            f"Total sessions {total} outside {REPLAY_RANGE_LO}-{REPLAY_RANGE_HI}. "
            "STOP and report to Chief."
        )
    if not (REPLAY_RANGE_LO <= usable_count <= REPLAY_RANGE_HI):
        raise RuntimeError(
            f"Usable sessions {usable_count} outside "
            f"{REPLAY_RANGE_LO}-{REPLAY_RANGE_HI}. STOP and report to Chief."
        )
    if total != REPLAY_TOTAL_EXPECTED:
        log_warn(f"total sessions = {total}, Indus-verified = {REPLAY_TOTAL_EXPECTED}")
    if usable_count != REPLAY_USABLE_EXPECTED:
        log_warn(f"usable sessions = {usable_count}, "
                 f"Indus-verified = {REPLAY_USABLE_EXPECTED}")

    per_game = Counter()
    for s in usable:
        gid = s.get("game_id")
        if gid:
            per_game[gid] += 1

    missing = [g for g in ALL_GAME_IDS if per_game[g] == 0]
    if missing:
        raise RuntimeError(
            f"ZERO replay coverage for games: {missing}. STOP per §3 STEP 1."
        )

    return {
        "total_sessions": total,
        "usable_sessions": usable_count,
        "expected_total": REPLAY_TOTAL_EXPECTED,
        "expected_usable": REPLAY_USABLE_EXPECTED,
        "per_game_counts": dict(per_game),
        "unusable_skipped": sorted(UNUSABLE_REPLAYS),
        "replay_source": _REPLAY_SOURCE,
    }


# ============================================================
# STEP 2 — RGB -> grid adapter
# ============================================================

def _try_engine_legend(game_obj: Any, game_id: str) -> dict | None:
    legend = getattr(game_obj, "legend", None)
    if isinstance(legend, dict):
        return legend
    try:
        from arcengine import get_color_legend  # type: ignore
        lg = get_color_legend(game_id)
        if isinstance(lg, dict):
            return lg
    except Exception:
        pass
    return None

def detect_adapter_mode(game_obj: Any, game_id: str) -> tuple[str, dict | None]:
    legend = _try_engine_legend(game_obj, game_id)
    return ("legend", legend) if legend is not None else ("nearest-match", None)

def _nearest_palette_idx(pixel: tuple[int, int, int]) -> int:
    hit = _PALETTE_LOOKUP.get(pixel)
    if hit is not None:
        return hit
    r, g, b = pixel
    best_i, best_d = 0, 1 << 30
    for i, (pr, pg, pb) in enumerate(ARC_PALETTE):
        d = (r - pr) ** 2 + (g - pg) ** 2 + (b - pb) ** 2
        if d < best_d:
            best_d, best_i = d, i
    return best_i

def rgb_to_grid(frame: Any, legend: dict | None) -> list[list[int]]:
    if legend is not None:
        return [[legend.get(tuple(px), _nearest_palette_idx(tuple(px)))
                 for px in row] for row in frame]
    return [[_nearest_palette_idx(tuple(px)) for px in row] for row in frame]

def dg3_fixture_parity() -> dict:
    """Exercises both adapter paths. Synthetic legend (D-9)."""
    black, white = (0, 0, 0), (255, 255, 255)

    nm_cases = [
        ("nm_black_1x1", [[black]], [[0]]),
        ("nm_white_1x1", [[white]], [[10]]),
        ("nm_palette_id_2x2",
         [[ARC_PALETTE[0], ARC_PALETTE[1]],
          [ARC_PALETTE[2], ARC_PALETTE[3]]],
         [[0, 1], [2, 3]]),
    ]
    nm_p, nm_f, nm_d = 0, 0, []
    for name, rgb, exp in nm_cases:
        got = rgb_to_grid(rgb, None)
        ok = got == exp
        nm_p += int(ok); nm_f += int(not ok)
        nm_d.append({"name": name, "ok": ok})

    legend_identity = {c: i for i, c in enumerate(ARC_PALETTE)}
    legend_shifted = {ARC_PALETTE[i]: (i + 1) % 16 for i in range(16)}

    lg_cases = [
        ("lg_identity_black", legend_identity, [[black]], [[0]]),
        ("lg_identity_white", legend_identity, [[white]], [[10]]),
        ("lg_identity_palette_2x2", legend_identity,
         [[ARC_PALETTE[0], ARC_PALETTE[1]],
          [ARC_PALETTE[2], ARC_PALETTE[3]]],
         [[0, 1], [2, 3]]),
        ("lg_shifted_black", legend_shifted, [[black]], [[1]]),
        ("lg_shifted_white", legend_shifted, [[white]], [[11]]),
    ]
    lg_p, lg_f, lg_d = 0, 0, []
    for name, lg, rgb, exp in lg_cases:
        got = rgb_to_grid(rgb, lg)
        ok = got == exp
        lg_p += int(ok); lg_f += int(not ok)
        lg_d.append({"name": name, "ok": ok})

    return {
        "nearest_match": {"passed": nm_p, "failed": nm_f, "details": nm_d},
        "legend_path":   {"passed": lg_p, "failed": lg_f, "details": lg_d},
        "certified_engine_fixtures": "NOT SUPPLIED (D-9)",
    }


# ============================================================
# STEP 3 — Rollout core
# ============================================================

def _normalize_action(payload: Any) -> dict:
    if isinstance(payload, dict):
        a = int(payload.get("action", -1))
        if a == 6:
            x, y = payload.get("x"), payload.get("y")
            if not (isinstance(x, int) and isinstance(y, int)
                    and 0 <= x <= 63 and 0 <= y <= 63):
                raise ValueError(f"ACTION6 missing/invalid x,y: {payload!r}")
            return {"action": 6, "x": x, "y": y}
        return {"action": a}
    a = int(payload)
    if a == 6:
        raise ValueError("bare ACTION6 without x,y is illegal")
    return {"action": a}

def _step_env(env: Any, payload: dict) -> Any:
    if payload["action"] == 6:
        return env.step(6, payload["x"], payload["y"])
    return env.step(payload["action"])

def _sorted_available(env: Any) -> list[int]:
    return sorted(int(a) for a in env.available_actions())

def _validate_transition_schema(t: dict) -> None:
    required = ("game_id", "step_index", "before_grid", "action",
                "after_grid", "what_changed", "source", "level")
    for k in required:
        if k not in t or t[k] is None:
            raise ValueError(f"transition missing/None: {k} in {t.get('game_id')}")
    if not isinstance(t["before_grid"], list) or not isinstance(t["after_grid"], list):
        raise ValueError("grids must be lists")
    if not isinstance(t["step_index"], int) or t["step_index"] < 0:
        raise ValueError("step_index must be non-negative int")
    if t["source"] not in ("random", "replay-guided"):
        raise ValueError(f"bad source: {t['source']}")
    if not isinstance(t["what_changed"], dict):
        raise ValueError(f"what_changed must be dict, got {type(t['what_changed'])}")
    a = t["action"]
    if not isinstance(a, dict) or "action" not in a:
        raise ValueError(f"bad action payload: {a!r}")
    if a["action"] == 6:
        if not (isinstance(a.get("x"), int) and isinstance(a.get("y"), int)
                and 0 <= a["x"] <= 63 and 0 <= a["y"] <= 63):
            raise ValueError(f"ACTION6 payload invalid: {a!r}")

def _make_transition(game_id, step_index, before_grid, action_payload,
                     after_grid, what_changed, source, level) -> dict:
    return {
        "game_id": game_id,
        "step_index": step_index,
        "before_grid": before_grid,
        "action": action_payload,
        "after_grid": after_grid,
        "what_changed": what_changed,
        "source": source,
        "level": level,
    }

def random_branch_from(env, game_id, legend, before_grid, before_objs,
                       level, num_actions, rng, step_offset) -> list[dict]:
    out: list[dict] = []
    if num_actions <= 0:
        return out
    for i in range(num_actions):
        avail = _sorted_available(env)
        if not avail:
            break
        a = rng.choice(avail)
        if a == 6:
            payload = {"action": 6, "x": rng.randint(0, 63),
                       "y": rng.randint(0, 63)}
        else:
            payload = {"action": a}
        try:
            obs, _r, done, info = _step_env(env, payload)
        except Exception as e:
            log_warn(f"{game_id}: step failed at {step_offset+i}: {e}")
            break
        after_grid = rgb_to_grid(obs, legend)
        after_objs = extract_objects(after_grid)
        delta = object_delta(before_objs, after_objs)
        lvl = info.get("level", level) if isinstance(info, dict) else level
        out.append(_make_transition(game_id, step_offset + i, before_grid,
                                    payload, after_grid, delta,
                                    "random", lvl))
        if done:
            obs = env.reset()
            before_grid = rgb_to_grid(obs, legend)
            before_objs = extract_objects(before_grid)
            level = 0
        else:
            before_grid, before_objs, level = after_grid, after_objs, lvl
    return out

def replay_guided_rollout(env, game_id, legend, replay_actions,
                          target_total, rng) -> list[dict]:
    out: list[dict] = []
    obs = env.reset()
    before_grid = rgb_to_grid(obs, legend)
    before_objs = extract_objects(before_grid)
    level = 0

    for step, raw in enumerate(replay_actions):
        if len(out) >= target_total:
            break
        try:
            payload = _normalize_action(raw)
        except ValueError:
            continue
        try:
            obs, _r, done, info = _step_env(env, payload)
        except Exception as e:
            log_warn(f"{game_id}: replay step {step} failed: {e}")
            break
        after_grid = rgb_to_grid(obs, legend)
        after_objs = extract_objects(after_grid)
        delta = object_delta(before_objs, after_objs)
        lvl = info.get("level", level) if isinstance(info, dict) else level
        out.append(_make_transition(game_id, step, before_grid, payload,
                                    after_grid, delta, "replay-guided", lvl))
        if done:
            obs = env.reset()
            before_grid = rgb_to_grid(obs, legend)
            before_objs = extract_objects(before_grid)
            level = 0
        else:
            before_grid, before_objs, level = after_grid, after_objs, lvl

    remaining = target_total - len(out)
    if remaining > 0:
        more = random_branch_from(
            env, game_id, legend, before_grid, before_objs, level,
            remaining, rng, step_offset=len(out),
        )
        out.extend(more)
    return out

def generate_game_transitions(game_id, replay_sessions) -> dict:
    if Arcade is None:
        raise RuntimeError("arcengine/arc_agi not importable — CPU-only firewall broken.")

    # FIX 4 (N-9): re-seed the GLOBAL random module per game, so any
    # engine-side consumption of global random is re-isolated per game.
    random.seed(_stable_seed_int(str(SEED), game_id, "global"))

    game_rng = random.Random(_stable_seed_int(str(SEED), game_id))

    env = Arcade.make(game_id)
    try:
        mode, legend = detect_adapter_mode(env, game_id)

        game_replays = [s for s in replay_sessions if s.get("game_id") == game_id]
        game_replays.sort(key=lambda s: str(s.get("id", "")))
        replay_actions: list = []
        for s in game_replays:
            replay_actions.extend(s.get("actions", []))

        target_replay = int(ACTIONS_PER_GAME * REPLAY_TARGET_FRACTION)
        transitions: list[dict] = []

        if replay_actions:
            transitions = replay_guided_rollout(
                env, game_id, legend, replay_actions, target_replay, game_rng
            )
        remaining = ACTIONS_PER_GAME - len(transitions)
        if remaining > 0:
            obs = env.reset()
            bg = rgb_to_grid(obs, legend)
            bo = extract_objects(bg)
            more = random_branch_from(
                env, game_id, legend, bg, bo, 0, remaining, game_rng,
                step_offset=len(transitions),
            )
            transitions.extend(more)
        transitions = transitions[:ACTIONS_PER_GAME]

        for t in transitions:
            _validate_transition_schema(t)

        guided = sum(1 for t in transitions if t["source"] == "replay-guided")
        frac = guided / max(1, len(transitions))
        if abs(frac - REPLAY_TARGET_FRACTION) > SPLIT_WARN_TOLERANCE:
            log_warn(f"{game_id}: replay-guided fraction {frac:.3f} deviates "
                     f"from target {REPLAY_TARGET_FRACTION} by >{SPLIT_WARN_TOLERANCE}")

        dg5 = dg5_sample_replay(game_id, legend, transitions)

        return {"transitions": transitions,
                "adapter_mode": mode,
                "dg5": dg5,
                "replay_fraction": frac}
    finally:
        try:
            env.close()
        except Exception:
            pass

def dg5_sample_replay(game_id, legend, transitions) -> dict:
    if Arcade is None or not transitions:
        return {"checked": 0, "matched": 0, "mismatched": 0, "indices": []}
    N = len(transitions)
    k = min(DG5_SAMPLE_K, N)
    if k == 0:
        return {"checked": 0, "matched": 0, "mismatched": 0, "indices": []}
    idxs = sorted({int(round(i * (N - 1) / (k - 1))) for i in range(k)}) if k > 1 else [0]
    env = Arcade.make(game_id)
    checked = matched = 0
    try:
        env.reset()
        last = -1
        for i in idxs:
            while last + 1 <= i:
                last += 1
                try:
                    payload = _normalize_action(transitions[last]["action"])
                    obs, _r, done, _info = _step_env(env, payload)
                except Exception:
                    return {"checked": checked, "matched": matched,
                            "mismatched": checked - matched, "indices": idxs}
                if done:
                    env.reset()
            got = rgb_to_grid(obs, legend)
            checked += 1
            if _stable_hash(got) == _stable_hash(transitions[i]["after_grid"]):
                matched += 1
    finally:
        try:
            env.close()
        except Exception:
            pass
    return {"checked": checked, "matched": matched,
            "mismatched": checked - matched, "indices": idxs}


# ============================================================
# STEP 4 — Zone 1/2 encoding
# ============================================================

_REQUIRED_PLACEHOLDERS = ("before_grid", "action", "after_grid", "what_changed")

def _load_zone12_template(template_path: Path | None) -> str:
    if template_path is None:
        env_path = os.environ.get("GOA_ZONE12_TEMPLATE")
        if env_path:
            template_path = Path(env_path)
    if template_path is None or not template_path.exists():
        raise RuntimeError(
            "Zone 1/2 template not supplied. Provide --zone12-template <file> "
            "or GOA_ZONE12_TEMPLATE=<file>. Phase B blocked until then (D-1)."
        )
    template = template_path.read_text()

    # N-6 fix: missing required placeholder is a HARD FAIL, not a warning.
    missing = [ph for ph in _REQUIRED_PLACEHOLDERS if "{" + ph + "}" not in template]
    if missing:
        raise RuntimeError(
            f"Zone 1/2 template missing required placeholder(s): {missing}. "
            "On-distribution mandate (D6) cannot be satisfied. Reject before rollout."
        )

    # N-2 fix: dummy render validates brace escaping / formatter syntax.
    dummy = {
        "before_grid": [[0]], "action": {"action": 1},
        "after_grid": [[0]], "what_changed": {},
    }
    try:
        template.format(**{k: json.dumps(dummy[k], separators=(",", ":"))
                           for k in _REQUIRED_PLACEHOLDERS})
    except (KeyError, IndexError, ValueError) as e:
        raise RuntimeError(
            f"Zone 1/2 template invalid (brace/formatter error): {e}. "
            "Literal braces must be doubled ({{, }})."
        )
    return template

def encode_zone12(transition: dict, template: str) -> str:
    return template.format(
        before_grid=json.dumps(transition["before_grid"], separators=(",", ":")),
        action=json.dumps(transition["action"], separators=(",", ":")),
        after_grid=json.dumps(transition["after_grid"], separators=(",", ":")),
        what_changed=json.dumps(transition["what_changed"], separators=(",", ":")),
    )


# ============================================================
# STEP 5 — Balance report + manifest
# ============================================================

def build_balance_report(transitions: list[dict]) -> dict:
    per_game = Counter(t["game_id"] for t in transitions)
    source_split = Counter(t["source"] for t in transitions)
    level_hist = defaultdict(Counter)
    for t in transitions:
        level_hist[t["game_id"]][t["level"]] += 1
    return {
        "per_game_counts": dict(per_game),
        "source_split": dict(source_split),
        "level_histogram": {g: dict(h) for g, h in level_hist.items()},
    }

def build_manifest(dataset_dir, seed, code_hash, adapter_modes, balance,
                   dg3, dg5, replay_meta) -> dict:
    files = {}
    for p in sorted(dataset_dir.rglob("*")):
        if p.is_file() and p.name != "manifest.json":
            files[str(p.relative_to(dataset_dir))] = _sha256_file(p)
    return {
        "seed": seed,
        "code_hash": code_hash,
        "adapter_modes": adapter_modes,
        "schema_version": SCHEMA_VERSION,
        "dataset_version": DATASET_VERSION,
        "files": files,
        "balance_report": balance,
        "dg3_fixture_parity": dg3,
        "dg5_sample_replay": dg5,
        "hold_out_games": sorted(HOLD_OUT_GAMES),
        "train_games": TRAIN_GAMES,
        "replay_meta": replay_meta,
        "total_transitions": sum(balance["per_game_counts"].values()),
    }


# ============================================================
# Main
# ============================================================

def main() -> int:
    parser = argparse.ArgumentParser(description="Task 8 Phase A dataset generator")
    parser.add_argument("--replay-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--zone12-template", type=Path, default=None)
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--no-cert-replays", action="store_true",
                        help="Force inline parquet loader (default: certified parse_replays)")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    random.seed(args.seed)  # informational only; per-game RNG is isolated

    prefer_cert = not args.no_cert_replays

    # STEP 1 — pre-flight
    replay_meta = verify_replays(args.replay_dir, prefer_cert=prefer_cert)
    log_info(f"replays OK: total={replay_meta['total_sessions']} "
             f"usable={replay_meta['usable_sessions']} "
             f"source={replay_meta['replay_source']}")

    all_sessions = [s for s in _iter_all_replays(args.replay_dir, prefer_cert)
                    if s.get("id") not in UNUSABLE_REPLAYS]

    # STEP 2 — DG-3 fixtures
    dg3 = dg3_fixture_parity()
    if dg3["nearest_match"]["failed"] > 0 or dg3["legend_path"]["failed"] > 0:
        raise RuntimeError(f"DG-3 fixture parity FAILED: {dg3}")

    # STEP 4 pre-flight — template must validate before rollouts
    template = _load_zone12_template(args.zone12_template)

    # STEP 3 — per-game generation (isolated RNG per game)
    all_transitions: list[dict] = []
    adapter_modes: dict[str, str] = {}
    dg5_report: dict[str, dict] = {}
    for game_id in TRAIN_GAMES:
        log_info(f"rollout {game_id}")
        res = generate_game_transitions(game_id, all_sessions)
        adapter_modes[game_id] = res["adapter_mode"]
        dg5_report[game_id] = res["dg5"]
        all_transitions.extend(res["transitions"])

    # DG-2 firewall
    for t in all_transitions:
        assert t["game_id"] not in HOLD_OUT_GAMES, f"hold-out leak: {t['game_id']}"
    assert len(all_transitions) == len(TRAIN_GAMES) * ACTIONS_PER_GAME, \
        f"expected {len(TRAIN_GAMES)*ACTIONS_PER_GAME}, got {len(all_transitions)}"

    # STEP 4 — encode
    for t in all_transitions:
        t["text"] = encode_zone12(t, template)

    # STEP 5 — write
    out = args.output_dir
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "transitions.jsonl", "w") as f:
        for t in all_transitions:
            f.write(json.dumps(t, separators=(",", ":")) + "\n")

    balance = build_balance_report(all_transitions)
    with open(out / "balance_report.json", "w") as f:
        json.dump(balance, f, indent=2)

    code_hash = _sha256_file(Path(__file__))
    manifest = build_manifest(out, args.seed, code_hash, adapter_modes, balance,
                              dg3, dg5_report, replay_meta)
    with open(out / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    log_info(f"dataset written: {out} ({len(all_transitions)} transitions)")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /kaggle/working/goa/vllm_transport.py
"""GoA test-harness vLLM transport adapter v26.3.4 — Deep Mode."""
from __future__ import annotations
import hashlib, json, os
from typing import Any, Dict, Optional, Tuple

REASONER_DEEP_THINKING_BUDGET = 2000
REASONER_DEEP_MAX_TOKENS      = 6144
REASONER_DEEP_EFFORT          = "medium"
REASONER_MIN_FLOOR            = 1024
ENABLE_THINKING_FLOOR         = 6144
RETRY_DOUBLED_CAP             = 8192

STALE_CHAT_TEMPLATE_KEYS: Tuple[str, ...] = (
    "thinking_budget", "reasoning_effort", "enable_thinking",
)
DEFAULT_CODER_MODEL_NAME = "qwen3-coder-30b-a3b"
V25_1_A_ENV_VAR = "V25_1_A_ENABLE_EXPLICIT_THINKING"


class VLLMTransport:
    def __init__(self, *args, client=None, base_url=None,
                 coder_base_url=None, coder_model_name=None, **kwargs):
        if len(args) >= 2 and isinstance(args[0], str) and isinstance(args[1], str):
            base_url, coder_base_url = args[0], args[1]
        elif len(args) == 1 and isinstance(args[0], str):
            base_url = args[0]
        self._client = client
        self._base_url = base_url or os.environ.get(
            "VLLM_BASE_URL", "http://localhost:8000")
        self._coder_base_url = coder_base_url or os.environ.get(
            "VLLM_CODER_URL", "http://localhost:8001")
        self._coder_model_name = (coder_model_name
                                  or os.environ.get("VLLM_CODER_NAME")
                                  or DEFAULT_CODER_MODEL_NAME)
        self._v25_1_a_enabled = os.environ.get(V25_1_A_ENV_VAR) == "1"

    def set_v25_1_a(self, enabled):
        self._v25_1_a_enabled = bool(enabled)

    def get_v25_1_a(self):
        return self._v25_1_a_enabled

    def _is_coder(self, model):
        return model == self._coder_model_name

    def _target_url(self, model):
        return self._coder_base_url if self._is_coder(model) else self._base_url

    def __call__(self, *, model, system=None, messages, max_tokens,
                 temperature=0.0, top_p=1.0, top_k=-1, stop_sequences=None,
                 chat_template_kwargs=None, presence_penalty=0.0,
                 repetition_penalty=1.0, reasoning_mode=None, **_unexpected):
        if _unexpected:
            print(f"WARN vllm_transport: ignored kwargs: {sorted(_unexpected)}")

        if reasoning_mode not in ("deep", "fast"):
            if reasoning_mode is not None:
                print(f"WARN vllm_transport: unknown reasoning_mode {reasoning_mode!r}")
            print("WARN vllm_transport: reasoning_mode absent/invalid — FAILING CLOSED TO FAST")
            reasoning_mode = "fast"

        _ctk = dict(chat_template_kwargs or {})
        for _stale in STALE_CHAT_TEMPLATE_KEYS:
            _ctk.pop(_stale, None)

        diag = os.environ.get("TRANSPORT_FLOOR_OVERRIDE") == "diag"
        if not diag:
            if not self._is_coder(model) and max_tokens < REASONER_MIN_FLOOR:
                max_tokens = REASONER_MIN_FLOOR
            if reasoning_mode == "deep" and max_tokens < ENABLE_THINKING_FLOOR:
                max_tokens = ENABLE_THINKING_FLOOR

        body: Dict[str, Any] = {
            "model": model, "messages": messages, "max_tokens": max_tokens,
            "temperature": temperature, "top_p": top_p, "top_k": top_k,
            "stop_sequences": list(stop_sequences or []),
            "presence_penalty": presence_penalty,
            "repetition_penalty": repetition_penalty,
        }
        if system is not None:
            body["system"] = system

        if reasoning_mode == "deep":
            body["reasoning_effort"] = REASONER_DEEP_EFFORT
            body["thinking_token_budget"] = REASONER_DEEP_THINKING_BUDGET
            # LA-7 FIX (Indus, 2026-09-29, Run-15 attempt-3 postmortem): the
            # Qwen3 template reads ONLY chat_template_kwargs. Top-level
            # enable_thinking/thinking_token_budget are silently ignored by
            # vLLM 0.30 (wire evidence: rt=0, ct=6144, finish=length on
            # attempt-3 STEP[1]). Nested form = the documented mechanism,
            # same key the server launch flag uses.
            _ctk["enable_thinking"] = True
            _ctk["thinking_budget"] = REASONER_DEEP_THINKING_BUDGET
            if self._v25_1_a_enabled:
                body["enable_thinking"] = True  # legacy A/B evidence only
                print("WARN vllm_transport: v25.1-A — TOP-LEVEL enable_thinking=True")
        else:
            _ctk["enable_thinking"] = False

        if _ctk:
            body["chat_template_kwargs"] = _ctk

        target = self._target_url(model)
        retry_on = os.environ.get("DISABLE_LENGTH_RETRY") != "1"
        _attempt_mt = max_tokens
        while True:
            body["max_tokens"] = _attempt_mt
            resp = self._client.post(f"{target}/v1/chat/completions", json=body)
            if not retry_on:
                break
            fr = self._extract_finish_reason(resp)
            if fr != "length" or _attempt_mt >= RETRY_DOUBLED_CAP:
                break
            _attempt_mt = min(_attempt_mt * 2, RETRY_DOUBLED_CAP)
            print(f"WARN vllm_transport: finish_reason=length; retrying mt={_attempt_mt}")
        return resp

    def post_wire_capture(self, body):
        wire_body_bytes = json.dumps(
            body, separators=(",", ":"), sort_keys=False).encode("utf-8")
        wire_sha256 = hashlib.sha256(wire_body_bytes).hexdigest()
        rec = {"wire_sha256": wire_sha256,
               "body_length_bytes": len(wire_body_bytes), "body": body}
        target = self._target_url(body.get("model", ""))
        resp = self._client.post(f"{target}/v1/chat/completions",
                                 data=wire_body_bytes,
                                 headers={"Content-Type": "application/json"})
        return {"record": rec, "response": resp}

    @staticmethod
    def _extract_finish_reason(resp):
        try:
            data = resp.json() if hasattr(resp, "json") else resp
            return data["choices"][0].get("finish_reason")
        except Exception:
            return None


def _self_check():
    captured = {}
    class _FakeClient:
        def post(self, url, json=None, data=None, headers=None):
            captured["body"] = json if json is not None else data
            captured["url"] = url
            class _R:
                def json(self_inner):
                    return {"choices": [{"finish_reason": "stop"}],
                            "usage": {"completion_tokens": 1}}
            return _R()

    t = VLLMTransport(client=_FakeClient(), base_url="http://x",
                      coder_base_url="http://y",
                      coder_model_name="qwen3-coder-30b-a3b")
    t(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
      max_tokens=1024, chat_template_kwargs={"thinking_budget": 999},
      reasoning_mode="fast")
    wf = captured["body"]
    assert wf.get("chat_template_kwargs") == {"enable_thinking": False}
    assert "enable_thinking" not in wf
    assert "thinking_token_budget" not in wf
    assert "reasoning_effort" not in wf
    assert "http://x" in captured["url"]

    t(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
      max_tokens=1024, chat_template_kwargs={"enable_thinking": True},
      reasoning_mode="deep")
    wd = captured["body"]
    assert wd["reasoning_effort"] == "medium"
    assert wd["thinking_token_budget"] == 2000
    assert wd["max_tokens"] >= 6144
    assert wd["chat_template_kwargs"].get("enable_thinking") is True   # LA-7
    assert wd["chat_template_kwargs"].get("thinking_budget") == 2000   # LA-7
    assert "enable_thinking" not in wd   # top-level only under v25.1-A

    t(model="qwen3-coder-30b-a3b", messages=[{"role": "user", "content": "h"}],
      max_tokens=256, reasoning_mode="fast")
    assert "http://y" in captured["url"]

    t2 = VLLMTransport(client=_FakeClient(), base_url="http://x",
                       coder_base_url="http://y",
                       coder_model_name="my-custom-coder")
    t2(model="my-custom-coder", messages=[{"role": "user", "content": "h"}],
       max_tokens=256, reasoning_mode="fast")
    assert "http://y" in captured["url"]

    os.environ[V25_1_A_ENV_VAR] = "1"
    t3 = VLLMTransport(client=_FakeClient(), base_url="http://x",
                       coder_base_url="http://y")
    t3(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
       max_tokens=1024, reasoning_mode="deep")
    assert captured["body"].get("enable_thinking") is True
    os.environ.pop(V25_1_A_ENV_VAR, None)

    t(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
      max_tokens=1024, reasoning_mode=None)
    assert captured["body"]["chat_template_kwargs"]["enable_thinking"] is False

    t4 = VLLMTransport("http://r:8000", "http://c:8001")
    assert t4._base_url == "http://r:8000"
    assert t4._coder_base_url == "http://c:8001"
    print("vllm_transport v26.3.4 self-check: 7/7 PASS")


if __name__ == "__main__":
    _self_check()


In [ ]:
# CERTIFIED FILE REPAIR v15 (Run-21 v18: CM 1.5->1.6 + my_agent 1.0.20->1.0.21 — Object Vision, martial-law core audit)
import hashlib, os, itertools

_expected = {
    "grid_text":        "bdbc329a990d82c19bf4b4e22b838bdd",
    "context_manager":  "18b8dfa5be82c8224671e0fa92ee8135",
    "inference_layer":  "8ec5288b371b0e15f3f4d8ed49b0fa46",
    "action_pipeline":  "826cf5f96000f615e57f1a04233102e9",
    "my_agent":         "1c1151af8c0dcf5931af02138c2bb590",
}

for _fn, _emd5 in _expected.items():
    if _emd5.startswith("$") and _emd5.endswith("$"):
        raise SystemExit(
            f"cell11 placeholder not replaced for {_fn}: {_emd5!r}. "
            f"Run assembly to inject, or paste the real md5 manually.")


def _m(t):
    return hashlib.md5(t.encode("utf-8")).hexdigest()


_transforms = [
    ("strip_trailing_lf",   lambda t: t[:-1] if (t.endswith("\n") and not t.endswith("\r\n")) else t),
    ("strip_trailing_crlf", lambda t: t[:-2] if t.endswith("\r\n") else t),
    ("strip_bom",           lambda t: t[1:] if t.startswith("\ufeff") else t),
    ("normalize_crlf",      lambda t: t.replace("\r\n", "\n").replace("\r", "\n")),
]

_already, _repaired, _total = 0, 0, len(_expected)

for _fn, _emd5 in _expected.items():
    _p = f"/kaggle/working/goa/agent/{_fn}.py"
    if not os.path.exists(_p):
        raise SystemExit(f"{_fn}: file missing: {_p}")
    try:
        with open(_p, encoding="utf-8-sig") as _f:
            _t = _f.read()
    except Exception as e:
        raise SystemExit(f"{_fn}: read failed: {e}")

    if _m(_t) == _emd5:
        print(f"{_fn:18} already certified bytes")
        _already += 1
        continue

    _fixed, _applied = None, None
    for _r in (1, 2, 3):
        for _combo in itertools.combinations(_transforms, _r):
            _cand = _t
            for _cname, _cfunc in _combo:
                _cand = _cfunc(_cand)
            if _m(_cand) == _emd5:
                _fixed = _cand
                _applied = [_cname for _cname, _ in _combo]
                break
        if _fixed is not None:
            break

    if _fixed is None:
        raise SystemExit(f"{_fn}: md5 mismatch, no repair combo matched")

    _tmp = _p + ".tmp"
    with open(_tmp, "w", encoding="utf-8", newline="") as _f:
        _f.write(_fixed)
    os.replace(_tmp, _p)
    print(f"{_fn:18} repaired via {_applied}")
    _repaired += 1

for _fn, _emd5 in _expected.items():
    _p = f"/kaggle/working/goa/agent/{_fn}.py"
    with open(_p, encoding="utf-8-sig") as _f:
        assert _m(_f.read()) == _emd5, f"{_fn}: post-repair md5 mismatch"

print(f"ALL {_total} CERTIFIED FILES VERIFIED ({_already} already, {_repaired} repaired)")


In [ ]:
# R-13 build check v7 (Run-21 v18 artifact; v1.0.20 hash 1e4f0d0d retired.
# Build now: CM v1.6 + my_agent v1.0.21 + 9 coder-solver files.
# Reproduced locally 2026-09-30 — Object Vision certified upgrade.)
import subprocess, hashlib, os, sys

GOA_DIR  = "/kaggle/working/goa"
OUT_PATH = f"{GOA_DIR}/submission/my_agent.py"
EXPECTED = "adb470eae95e94d557fd110420676096"

if EXPECTED.startswith("$") and EXPECTED.endswith("$"):
    raise SystemExit(
        f"cell12 placeholder not replaced: {EXPECTED!r}. "
        f"Run assembly to inject, or paste real artifact md5.")

os.makedirs(f"{GOA_DIR}/submission", exist_ok=True)

if os.path.exists(OUT_PATH):
    os.remove(OUT_PATH)
    print(f"INFO cleaned stale output: {OUT_PATH}")

r = subprocess.run(
    [sys.executable, "agent/my_agent.py", "build",
     "--out", "submission/my_agent.py"],
    cwd=GOA_DIR, capture_output=True, text=True, check=False)

if r.returncode != 0:
    print(f"FAIL build (exit={r.returncode})")
    print("STDOUT:", r.stdout[-2000:] or "(empty)")
    print("STDERR:", r.stderr[-2000:] or "(empty)")
    raise SystemExit("my_agent build fail — R-13 abort")
print("OK   build complete")

if not os.path.exists(OUT_PATH):
    raise SystemExit(f"build returned 0 but output missing: {OUT_PATH}")

with open(OUT_PATH, "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print(f"artifact md5: {md5}")
if md5 != EXPECTED:
    raise SystemExit(
        f"R-13 MISMATCH:\n  expected {EXPECTED}\n  got      {md5}")
print("R-13 PASS: byte-identical artifact")


In [ ]:
"""Cell 13 v12.4 — vLLM launch + runbook globals."""
import atexit, json, os, socket, subprocess, sys, time
import glob, sysconfig, site

GOA_DIR = "/kaggle/working/goa"
if GOA_DIR not in sys.path:
    sys.path.insert(0, GOA_DIR)
    print(f"OK   sys.path += {GOA_DIR}")

ENV_RECORD = {
    "server_version":               os.environ.get("VLLM_VERSION", "unknown"),
    "vllm_commit":                  os.environ.get("VLLM_COMMIT", "unknown"),
    "reasoning_parser":             "qwen3",
    "default_chat_template_kwargs": {"enable_thinking": False},
    "transport_floor_override":     os.environ.get("TRANSPORT_FLOOR_OVERRIDE", "unset"),
    "model_id":                     "qwen3.8-27b",
    "wire_capture_available":       os.environ.get("WIRE_CAPTURE_DISABLED") != "1",
    "transport_version":            "v26.3.4",
}
print("ENV_RECORD " + json.dumps(ENV_RECORD, sort_keys=True))
if not ENV_RECORD["wire_capture_available"]:
    print("WARN: wire capture unavailable — GATE-A will be BLOCKED (D-5)")

REASONER_PATH = os.environ.get(
    "REASONER_PATH", "/kaggle/input/datasets/gray2488/qwen38-27b-fp8")
CODER_PATH = os.environ.get(
    "CODER_PATH", "/kaggle/input/datasets/gray2488/qwen3-coder-30b-a3b-instruct")
_REASONER_NAME = os.environ.get("VLLM_REASONER_NAME", "qwen3.8-27b")
_CODER_NAME    = os.environ.get("VLLM_CODER_NAME", "qwen3-coder-30b-a3b")
LOG_DIR = "/kaggle/working/logs"
os.makedirs(LOG_DIR, exist_ok=True)

REASONER_UTIL, CODER_UTIL = 0.40, 0.42
MAX_MODEL_LEN = 16384

_site_dirs = set()
for _p in ("purelib", "platlib"):
    _d = sysconfig.get_paths().get(_p)
    if _d and os.path.isdir(_d): _site_dirs.add(_d)
try:
    for _d in site.getsitepackages():
        if _d and os.path.isdir(_d): _site_dirs.add(_d)
except Exception:
    pass

_LIBS = []
for base in _site_dirs:
    _LIBS += glob.glob(f"{base}/nvidia/*/lib")
    _LIBS += glob.glob(f"{base}/cuda/*/lib")
_LIBS += glob.glob("/usr/local/cuda*/lib64")
_LIBS += glob.glob("/usr/local/cuda*/lib")
_LIBS = list(dict.fromkeys(_LIBS))

VLLM_ENV = dict(os.environ)
_existing_ld = VLLM_ENV.get("LD_LIBRARY_PATH", "")
VLLM_ENV["LD_LIBRARY_PATH"] = ":".join(
    list(dict.fromkeys(_LIBS + [d for d in _existing_ld.split(":") if d])))
VLLM_ENV["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
VLLM_ENV["VLLM_USE_DEEP_GEMM"] = "0"
VLLM_ENV["VLLM_MOE_USE_DEEP_GEMM"] = "0"
VLLM_ENV["VLLM_REASONER_NAME"] = _REASONER_NAME
VLLM_ENV["VLLM_CODER_NAME"] = _CODER_NAME


def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1.0)
        return s.connect_ex(("127.0.0.1", port)) == 0

for _port in (8000, 8001):
    if _port_in_use(_port):
        raise SystemExit(f"Port {_port} already in use")

_procs = []
_log_handles = []

_REASONER_FLAGS = [
    "--default-chat-template-kwargs", '{"enable_thinking": false}',
    "--enable-prefix-caching",
    "--mamba-cache-mode", "align",
    "--enable-prompt-tokens-details",
]
_FALLBACK_DROP = ("--mamba-cache-mode", "align", "--enable-prompt-tokens-details")
_MAMBA_SIGNATURES = ("mamba-cache-mode", "unrecognized arguments",
                     "unrecognized argument")


def _launch(port, model_path, served_name, util, log_path,
            *, is_reasoner, flags=None, attempt=1):
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--served-model-name", served_name,
        "--port", str(port),
        "--gpu-memory-utilization", str(util),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--no-enable-log-requests",
        "--max-num-seqs", "64",
    ]
    if is_reasoner:
        cmd += ["--reasoning-parser", "qwen3"]
        cmd += (flags if flags is not None else _REASONER_FLAGS)
    print(f"LAUNCH[port={port} is_reasoner={is_reasoner} attempt={attempt}]: "
          f"{' '.join(cmd)}", flush=True)
    _mode = "w" if attempt == 1 else "a"
    lf = open(log_path, _mode)
    if attempt >= 2:
        lf.write(f"\n\n===== RELAUNCH ATTEMPT {attempt} =====\n")
        lf.flush()
    _log_handles.append(lf)
    p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT,
                         env=VLLM_ENV, start_new_session=True)
    _procs.append(p)
    return p


def _tail_log(log_path, n=2000):
    try:
        with open(log_path, encoding="utf-8", errors="replace") as f:
            return f.read()[-n:]
    except Exception:
        return ""


def _wait_healthy(port, proc, log_path, *, is_reasoner,
                  timeout_s=1800, allow_fallback=True):
    t0 = time.time()
    relaunched = False
    attempt = 1
    current_proc = proc
    while time.time() - t0 < timeout_s:
        if current_proc.poll() is not None:
            tail = _tail_log(log_path)
            hit = any(sig in tail for sig in _MAMBA_SIGNATURES)
            if is_reasoner and allow_fallback and not relaunched and hit:
                print("WARN: reasoner rejected --mamba-cache-mode — relaunching "
                      "(append log, first-attempt trace preserved)")
                _new_flags = [a for a in _REASONER_FLAGS
                              if a not in _FALLBACK_DROP]
                attempt += 1
                current_proc = _launch(
                    port, REASONER_PATH, _REASONER_NAME, REASONER_UTIL,
                    log_path, is_reasoner=True, flags=_new_flags,
                    attempt=attempt)
                relaunched = True
                continue
            raise RuntimeError(
                f"vLLM on port {port} died (exit={current_proc.returncode})\n{tail}")
        try:
            s = socket.create_connection(("127.0.0.1", port), timeout=2.0)
            s.close()
            print(f"OK   port {port} healthy in {time.time()-t0:.0f}s "
                  f"(relaunched={relaunched} attempt={attempt})", flush=True)
            return current_proc
        except OSError:
            time.sleep(5)
    raise RuntimeError(f"vLLM port {port} not ready in {timeout_s}s")


print(f"REASONER launch — {REASONER_PATH}", flush=True)
p_reasoner = _launch(8000, REASONER_PATH, _REASONER_NAME, REASONER_UTIL,
                     f"{LOG_DIR}/vllm_reasoner.log", is_reasoner=True,
                     attempt=1)
p_reasoner = _wait_healthy(8000, p_reasoner,
                           f"{LOG_DIR}/vllm_reasoner.log", is_reasoner=True)

print(f"CODER launch — {CODER_PATH}", flush=True)
p_coder = _launch(8001, CODER_PATH, _CODER_NAME, CODER_UTIL,
                  f"{LOG_DIR}/vllm_coder.log", is_reasoner=False, attempt=1)
p_coder = _wait_healthy(8001, p_coder,
                        f"{LOG_DIR}/vllm_coder.log", is_reasoner=False)

print("OK   both vLLM servers live (reasoner :8000 + coder :8001)")

import httpx as _httpx
from vllm_transport import VLLMTransport as _VTClass
# LA-5 FIX (Indus, 2026-09-29, Run-15 attempt-2 postmortem): kernel-side
# `from transformers import AutoTokenizer` dies with ImportError
# (torch/transformers version mismatch on the Kaggle image: transformers
# wants torch.distributed._functional_collectives._maybe_view_chunk_cat,
# absent in torch 2.10). vLLM servers are unaffected (subprocess). Fail-open:
# guarded import + vLLM-server /tokenize shim for exact token counts.
try:
    from transformers import AutoTokenizer as _AT
except Exception as _ate:
    print(f"WARN: kernel transformers unavailable ({type(_ate).__name__}: "
          f"{str(_ate)[:120]}) — /tokenize shim will be used")
    _AT = None

_httpx_client = _httpx.Client(
    timeout=_httpx.Timeout(connect=10.0, read=300.0, write=30.0, pool=5.0))
atexit.register(_httpx_client.close)

TRANSPORT = _VTClass(client=_httpx_client,
                     base_url="http://127.0.0.1:8000",
                     coder_base_url="http://127.0.0.1:8001",
                     coder_model_name=_CODER_NAME)
tr = TRANSPORT

BASE_URL = TRANSPORT._base_url
print(f"OK   BASE_URL derived from TRANSPORT: {BASE_URL}")

_TOKENIZER_ID = os.environ.get("TOKENIZER_ID", REASONER_PATH)

class _ServerTokenizerShim:
    """LA-5: exact token counts via the running vLLM server's /tokenize."""
    def __init__(self, client, base_url, model):
        self._c, self._u, self._m = client, base_url.rstrip("/"), model
    def encode(self, text):
        try:
            r = self._c.post(self._u + "/tokenize",
                             json={"model": self._m, "prompt": text})
            return r.json().get("tokens", [])
        except Exception:
            return []

TOKENIZER = None
if _AT is not None:
    try:
        TOKENIZER = _AT.from_pretrained(_TOKENIZER_ID, trust_remote_code=True)
        print(f"OK   TOKENIZER loaded from {_TOKENIZER_ID}")
    except Exception as e:
        print(f"WARN: TOKENIZER load from {_TOKENIZER_ID} failed: {e}")
        try:
            TOKENIZER = _AT.from_pretrained("Qwen/Qwen3.8-27B",
                                           trust_remote_code=True,
                                           local_files_only=True)
            print("OK   TOKENIZER loaded from HF hub cache")
        except Exception as e2:
            print(f"WARN: hub fallback failed too: {e2}")
if TOKENIZER is None:
    TOKENIZER = _ServerTokenizerShim(_httpx_client, "http://127.0.0.1:8000",
                                     _REASONER_NAME)
    print("OK   TOKENIZER = vLLM /tokenize server shim (exact counts)")

print(f"OK   TRANSPORT bound (reasoner {TRANSPORT._base_url}, "
      f"coder {TRANSPORT._coder_base_url}, coder_name={_CODER_NAME})")
print(f"OK   TOKENIZER ready ({type(TOKENIZER).__name__})")


In [ ]:
%%writefile Indus_DeepMode_IMP2P_Prompt.txt
You are solving a constraint-elimination puzzle. Reason through it completely and carefully before giving any answer.

SIX RESEARCHERS work at the Meridian Institute: Anaya, Bijoy, Chitra, Dev, Esha, and Farhan. Each is assigned to exactly one of six labs (Alpha, Bravo, Charlie, Delta, Echo, Foxtrot), holds exactly one of six specialties (biophysics, cryptography, epidemiology, geology, hydrology, immunology), joined in exactly one of six years (2016, 2017, 2018, 2019, 2020, 2021), and drinks exactly one of six coffee orders (espresso, americano, latte, cappuccino, mocha, black). Each lab, specialty, year, and coffee order is used by exactly one researcher. No two researchers share any attribute.

The 16 clues:

1. The researcher who joined in 2016 is in Bravo.
2. Chitra joined two years after the epidemiologist.
3. The cryptographer drinks mocha.
4. Dev is in Foxtrot but is not the hydrologist.
5. The researcher who joined in 2018 drinks espresso.
6. Anaya joined before both Esha and Farhan.
7. The geologist is in Charlie and joined in 2020.
8. The researcher who drinks cappuccino joined in 2017.
9. Bijoy is the biophysicist.
10. The researcher in Alpha drinks black coffee and is not the immunologist.
11. Esha joined exactly one year after the immunologist.
12. The hydrologist drinks americano.
13. Farhan is not in Echo and did not join in 2021.
14. The researcher in Delta joined before the researcher in Echo.
15. The immunologist is not in Bravo and does not drink espresso, cappuccino, or black coffee.
16. Bijoy joined in a year that is exactly two years after the researcher in Delta and exactly one year before Esha.

Tasks:
A. Determine the complete assignment: for each researcher, the lab, specialty, join year, and coffee order.
B. Prove the assignment is unique: explicitly consider at least three alternative hypotheses during your reasoning (for example, alternative placements of the immunologist or alternative year assignments) and show why each leads to a contradiction.
C. State, for each researcher, the single clue that most directly pins down their join year, and explain the chain of deductions that forced it.

Do not skip deduction steps. Show the full elimination reasoning.


In [ ]:
%%writefile Indus_DeepMode_IMP2B_Prompt.txt
You are solving an extended constraint-elimination puzzle with counterfactual analysis. Reason through it completely, exhaustively, and carefully before giving any answer. Enumerate every partial assignment you consider and explicitly reject, including the ones that fail only after several steps of deduction.

EIGHT RESEARCHERS work at the Meridian Institute: Anaya, Bijoy, Chitra, Dev, Esha, Farhan, Gauri, and Hemant. Each is assigned to exactly one of eight labs (Alpha, Bravo, Charlie, Delta, Echo, Foxtrot, Golf, Hotel), holds exactly one of eight specialties (biophysics, cryptography, epidemiology, geology, hydrology, immunology, mechanics, optics), joined in exactly one of eight years (2014 through 2021, one each), drinks exactly one of eight coffee orders (espresso, americano, latte, cappuccino, mocha, black, macchiato, flat white), and speaks exactly one of eight languages (Assamese, Bengali, Kannada, Marathi, Odia, Punjabi, Tamil, Telugu). Each value in every dimension is used exactly once. No two researchers share any attribute.

The 24 clues:

1. The researcher who joined in 2014 is in Bravo.
2. Chitra joined two years after the epidemiologist.
3. The cryptographer drinks mocha.
4. Dev is in Foxtrot but is neither the hydrologist nor the mechanic.
5. The researcher who joined in 2018 drinks espresso.
6. Anaya joined before both Esha and Farhan.
7. The geologist is in Charlie and joined in 2020.
8. The researcher who drinks cappuccino joined in 2017.
9. Bijoy is the biophysicist.
10. The researcher in Alpha drinks black coffee and is neither the immunologist nor the optical scientist.
11. Esha joined exactly one year after the immunologist.
12. The hydrologist drinks americano.
13. Farhan is not in Echo, is not in Golf, and did not join in 2021.
14. The researcher in Delta joined before the researcher in Echo, who joined before the researcher in Golf.
15. The immunologist is not in Bravo and does not drink espresso, cappuccino, black coffee, macchiato, or flat white.
16. Bijoy joined in a year exactly two years after the researcher in Delta and exactly one year before Esha.
17. The researcher who speaks Tamil is the opticist and is in Hotel.
18. Gauri speaks Kannada and joined before the researcher who speaks Marathi.
19. The researcher who joined in 2015 speaks Punjabi.
20. Hemant is in Delta and speaks neither Bengali nor Odia.
21. The mechanic drinks flat white and joined after both the geologist and the cryptographer.
22. The researcher who speaks Telugu joined exactly three years after the researcher in Alpha.
23. Esha does not speak Odia and is not the epidemiologist.
24. The researcher who speaks Bengali joined before the researcher who speaks Assamese, who joined before Chitra.

Tasks:
A. Determine the complete assignment: for each researcher, the lab, specialty, join year, coffee order, and language.
B. Prove uniqueness by exhaustive enumeration: for each of the eight researchers, list the alternative labs or specialties you considered and eliminated, and the clue chain that eliminated each.
C. COUNTERFACTUAL ONE: Suppose instead that Anaya's lab were Golf (rather than her actual lab) and that Dev's join year were 2015 (rather than his actual year), with all clues otherwise unchanged. Determine which assignments would have to change, whether the resulting configuration remains consistent with every clue, and if not, identify the first clue that fails and the full deduction chain that exposes the failure.
D. COUNTERFACTUAL TWO: Suppose instead that Esha's specialty were swapped with Chitra's specialty (all else as actually determined). Re-derive which coffee orders and join years remain forced, which become underdetermined, and whether clue 11 and clue 16 still hold; explain the interaction between these two clues under the swap.
E. For each of clues 1 through 24, state whether it was used in its direct form, its contrapositive form, or both, and give one example of a deduction where a contrapositive use was essential.

Do not skip any deduction step. Enumerate rejections explicitly. Show the complete reasoning.


## Template-conformance cell (report §12) — appended

In [ ]:
"""Template-conformance cell (report §12) — sets CONFORMANCE_PASSED flag."""
# LA-5c FIX: kernel transformers import is guarded (torch/transformers
# mismatch on the Kaggle image — see LA-5 note in the launch cell).
try:
    from transformers import AutoTokenizer
except Exception as _ate31:
    print(f"WARN: kernel transformers unavailable ({type(_ate31).__name__}) — "
          "template-conformance checks will be SKIPPED; kwarg safety is "
          "enforced live by the runbook GATE-A (C0/C1/C2, D-10 A/B).")
    AutoTokenizer = None
import os

CONFORMANCE_PASSED = False
CONFORMANCE_MODE = "full"

if AutoTokenizer is None:
    # Honest degradation: no local tokenizer -> record SKIP, arm the gate so
    # Run-All is not blocked; real server-side kwarg evidence = GATE-A.
    CONFORMANCE_PASSED = True
    CONFORMANCE_MODE = "SKIPPED_NO_KERNEL_TOKENIZER"
    print("TEMPLATE-CONFORMANCE: SKIPPED (no kernel tokenizer) — "
          "server-side equivalents enforced by runbook GATE-A")
    print("OK   CONFORMANCE_PASSED = True (SKIPPED mode, recorded)")
elif 'TOKENIZER' in globals() and TOKENIZER is not None:
    tok = TOKENIZER
    print("OK   reusing TOKENIZER from cell 13")
else:
    _REASONER_PATH = os.environ.get(
        "REASONER_PATH", "/kaggle/input/datasets/gray2488/qwen38-27b-fp8")
    _TID = os.environ.get("TOKENIZER_ID", _REASONER_PATH)
    if AutoTokenizer is None:
        tok = None
    else:
        try:
            tok = AutoTokenizer.from_pretrained(_TID, trust_remote_code=True)
            print(f"OK   TOKENIZER loaded from {_TID} (standalone fallback)")
        except Exception:
            tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B",
                                                trust_remote_code=True,
                                                local_files_only=True)
            print("OK   TOKENIZER loaded from HF hub cache (standalone)")

# LA-5c cont.: run the render checks only when a local tokenizer is live;
# never SystemExit on the skip path (papermill fails the notebook on it).
if 'tok' in globals() and tok is not None and hasattr(tok, "apply_chat_template"):
    def _tconf_main():
        MSGS = [{"role": "user", "content": "Reply with JSON {}."}]


        def _render(**kw):
            try:
                out = tok.apply_chat_template(MSGS, tokenize=False,
                                              add_generation_prompt=True, **kw)
                return True, out
            except Exception as e:
                return False, e


        def _check(name, condition, detail=""):
            print(f"[{'PASS' if condition else 'FAIL'}] {name} {detail}")
            return condition


        results = []
        ok1, r1 = _render(reasoning_effort="medium")
        results.append(_check("CHECK-1 reasoning_effort=medium accept", ok1))

        ok2, r2 = _render(reasoning_effort="high")
        results.append(_check("CHECK-2 reasoning_effort=high reject", not ok2))

        ok3a, ra = _render()
        ok3b, rb = _render(chat_template_kwargs={"thinking_budget": 1000})
        ok3_pre = ok3a and ok3b and ra == rb
        results.append(_check("CHECK-3a thinking_budget ignore (offline text-eq)", ok3_pre))
        print("  NOTE: D-10 full rt+answer equality in Cell 14 step [7.5]")

        ok4, r4 = _render(thinking_token_budget=2048)
        results.append(_check("CHECK-4 thinking_token_budget accept", ok4))

        ok5, r5 = _render(enable_thinking=False)
        results.append(_check("CHECK-5 enable_thinking=false accept", ok5))

        n_pass = sum(1 for x in results if x)
        print(f"CONFORMANCE {'PASS' if n_pass == 5 else 'FAIL'} ({n_pass}/5)")
        if n_pass != 5:
            raise SystemExit("TEMPLATE-CONFORMANCE FAIL — GATE-A must not proceed")

    _tconf_main()
    # LA-9a (external-audit finding, VERIFIED TRUE): the assignment used to
    # live inside _tconf_main — Python scoping made it a LOCAL and the
    # module-level flag never armed when the full path ran (masked today
    # only because the broken transformers import forces the SKIP arm).
    # Arming now happens at module level, auditable at cell scope.
    CONFORMANCE_PASSED = True
    print("OK   CONFORMANCE_PASSED = True (cell 14 gate armed)")
elif 'tok' in globals() and tok is not None:
    # LA-9b (Indus addendum, beyond the audit): a shim-only tokenizer
    # (encode-only, no apply_chat_template) must degrade to an honest
    # SKIP, not an AttributeError-driven false FAIL inside the renders.
    CONFORMANCE_PASSED = True
    CONFORMANCE_MODE = "SKIPPED_NO_TEMPLATE_CAPABLE_TOKENIZER"
    print("WARN: tokenizer = /tokenize shim (no apply_chat_template) — SKIP mode")
    print("OK   CONFORMANCE_PASSED = True (SKIP mode, recorded)")
else:
    print("TEMPLATE-CONFORMANCE: SKIP mode (armed above) — checks not run")


In [ ]:
"""Cell 14 v3.6 — frozen pre-flight runbook (report §18-§19).

F-3 (P2, INDUS-DIR-070 closure): C1/C2 branch split.  Previously any
in-tolerance failure routed to a single FAIL.  Per D-6, rt > budget + d
is FAIL-breach; rt < budget - d with FT present is INCONCLUSIVE
(measurement/parser error) and routes to GATE-AC record.  Both directions
are conservative (never a false PASS); the split is semantic precision.
"""
import hashlib, json, os, re
from pathlib import Path
# LA-8 FIX (Indus, 2026-09-30, attempt-4 postmortem): _step_4_imp3 uses
# InferenceLayer directly but the runbook cell never imported it — first
# live execution died with NameError at 751s (steps 4+ had never run
# before; every earlier attempt stopped at STEP[1]).
try:
    from agent.inference_layer import InferenceLayer
except ImportError:
    from inference_layer import InferenceLayer

IMP2P_SHA  = "940c73a0e386d3ccf9ed3b9956c367833990068c1c8fd459550748fa32707bf1"
IMP2B_SHA  = "377a7c8134fe2eacc5806a79183a26d090e9532f23f33efa75d1f89c99982d47"
IMP2P_PATH = "Indus_DeepMode_IMP2P_Prompt.txt"
IMP2B_PATH = "Indus_DeepMode_IMP2B_Prompt.txt"
assert len(IMP2P_SHA) == 64 and len(IMP2B_SHA) == 64


def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def _verify(path, expected):
    a = _sha(path)
    if a != expected:
        raise SystemExit(f"STOP: {path} sha256 mismatch\n"
                         f"  expected {expected}\n  actual   {a}")
    return a


IMP2P_PROMPT = Path(IMP2P_PATH).read_text(encoding="utf-8")
_verify(IMP2P_PATH, IMP2P_SHA)

D = None
WIRE_CAPTURE_AVAILABLE = os.environ.get("WIRE_CAPTURE_DISABLED") != "1"

FT_LOG_REGEX = re.compile(
    r"(?i)(thinking_budget|reasoning_budget).*(force|terminat|exhaust|stop)")
FT_METRIC_NAME = "vllm:reasoning_budget_forced_termination_total"
FT_LOG_PATH    = os.environ.get("VLLM_LOG_PATH", "")
FT_METRICS_URL = os.environ.get("VLLM_METRICS_URL", "")

PREFLIGHT = {
    "env": {
        "server_version":               os.environ.get("VLLM_VERSION", "unknown"),
        "vllm_commit":                  os.environ.get("VLLM_COMMIT", "unknown"),
        "reasoning_parser":             "qwen3",
        "default_chat_template_kwargs": {"enable_thinking": False},
        "transport_floor_override":     os.environ.get("TRANSPORT_FLOOR_OVERRIDE", "unset"),
        "wire_capture_available":       WIRE_CAPTURE_AVAILABLE,
    },
    "runbook_note": (
        "AGA-20: step [7.5] D-10 A/B is pre-GATE-A per report §12. "
        "AGA-26: hand-built probes carry DEEP wire inline. "
        "NEW-D1: conformance cell must be run before _run() (flag check). "
        "F-3: C1/C2 split — rt < budget-d with FT present -> INCONCLUSIVE."),
    "steps": {},
}


class _SpyClient:
    def __init__(self, real_client):
        self._real = real_client
        self.first_body = None
        self.last_body = None
        self.last_response = None
        self._post_count = 0

    def post(self, url, json=None, data=None, headers=None, **kw):
        self._post_count += 1
        self.last_body = json
        if self.first_body is None:
            self.first_body = json
        if self._real is not None:
            resp = self._real.post(url, json=json, data=data,
                                   headers=headers, **kw)
        else:
            class _R:
                def json(self_inner):
                    return {"choices": [{"finish_reason": "stop",
                                         "message": {"content": "{}"}}],
                            "usage": {"completion_tokens": 1}}
            resp = _R()
        self.last_response = resp
        return resp


def _call_via_transport(*, model, messages, max_tokens, reasoning_mode,
                        temperature=0.0, chat_template_kwargs=None,
                        v25_1_a=None):
    real_client = getattr(TRANSPORT, "_client", None)
    spy = _SpyClient(real_client)
    t = TRANSPORT.__class__(client=spy, base_url=TRANSPORT._base_url,
                            coder_base_url=TRANSPORT._coder_base_url,
                            coder_model_name=TRANSPORT._coder_model_name)
    if v25_1_a is not None:
        t.set_v25_1_a(bool(v25_1_a))
    else:
        t.set_v25_1_a(TRANSPORT.get_v25_1_a())
    resp = t(model=model, messages=messages, max_tokens=max_tokens,
             reasoning_mode=reasoning_mode, temperature=temperature,
             chat_template_kwargs=chat_template_kwargs)
    return spy, resp


def _post(body, capture=True):
    if capture and WIRE_CAPTURE_AVAILABLE:
        r = TRANSPORT.post_wire_capture(body)
        return r["response"], r["record"]
    resp = TRANSPORT._client.post(f"{BASE_URL}/v1/chat/completions", json=body)
    return resp, None


def _rt(rj):
    try:
        return rj["usage"]["completion_tokens_details"]["reasoning_tokens"]
    except Exception:
        return None


def _ct(rj): return rj["usage"]["completion_tokens"]
def _fr(rj): return rj["choices"][0]["finish_reason"]


def _ft_evidence_via_log(since_byte=0):
    if not FT_LOG_PATH or not Path(FT_LOG_PATH).is_file():
        return None
    try:
        with open(FT_LOG_PATH, "rb") as f:
            f.seek(since_byte)
            for raw in f:
                line = raw.decode("utf-8", errors="replace")
                if FT_LOG_REGEX.search(line):
                    return line.strip()
    except Exception:
        return None
    return None


def _ft_metric_delta(before_value):
    if not FT_METRICS_URL: return None
    try:
        import urllib.request
        with urllib.request.urlopen(FT_METRICS_URL, timeout=2.0) as r:
            text = r.read().decode("utf-8", errors="replace")
    except Exception:
        return None
    for line in text.splitlines():
        if line.startswith(FT_METRIC_NAME):
            try:
                val = float(line.rsplit(" ", 1)[1])
                if before_value is not None and val > before_value:
                    return val - before_value
                return None
            except Exception:
                return None
    return None


def _ft_metric_current():
    if not FT_METRICS_URL: return None
    try:
        import urllib.request
        with urllib.request.urlopen(FT_METRICS_URL, timeout=2.0) as r:
            text = r.read().decode("utf-8", errors="replace")
        for line in text.splitlines():
            if line.startswith(FT_METRIC_NAME):
                return float(line.rsplit(" ", 1)[1])
    except Exception:
        return None
    return None


def _ft_evidence(log_since_byte=0, metric_before=None):
    log_hit = _ft_evidence_via_log(log_since_byte)
    if log_hit is not None:
        return {"type": "log_line", "value": log_hit}
    metric_hit = _ft_metric_delta(metric_before)
    if metric_hit is not None:
        return {"type": "metric_delta", "value": metric_hit}
    return None


def _step_1_imp2p():
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": IMP2P_PROMPT}],
            "max_tokens": 6144, "thinking_token_budget": 4096,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": 4096},
            "temperature": 0.0}
    resp, wire = _post(body)
    rj = resp.json(); rt = _rt(rj)
    rec = {"prompt_source": IMP2P_PATH, "prompt_sha256": IMP2P_SHA,
           "reasoning_tokens": rt, "completion_tokens": _ct(rj),
           "finish_reason": _fr(rj),
           "wire_sha256": wire["wire_sha256"] if wire else None}
    rec["verdict"] = ("BLOCKED" if rt is None
                      else "PASS" if rt > 2000 else "PROMPT-FIX")
    PREFLIGHT["steps"]["1_imp2p"] = rec
    print(f"STEP[1] IMP-2P: {rec['verdict']} (rt={rt})")
    return rec["verdict"]


def _step_2_imp2():
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": IMP2P_PROMPT}],
            "max_tokens": 1000, "thinking_token_budget": 2000,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": 2000},
            "temperature": 0.0}
    resp, wire = _post(body)
    rj = resp.json(); rt = _rt(rj); ct = _ct(rj)
    rec = {"reasoning_tokens": rt, "completion_tokens": ct,
           "finish_reason": _fr(rj)}
    if rt is not None and rt >= 1500:
        rec["outcome"] = "EXCLUDES"
    elif rt is not None and rt <= 1000 and ct <= 1000:
        rec["outcome"] = "INCONCLUSIVE-low-rt"
    else:
        rec["outcome"] = "INCONCLUSIVE"
    PREFLIGHT["steps"]["2_imp2"] = rec
    print(f"STEP[2] IMP-2: {rec['outcome']} (rt={rt}, ct={ct})")
    return rec["outcome"]


def _step_3_par_cal():
    MILDLY = "Reason briefly then answer: 2+2=?"
    ds = []
    for _ in range(2):
        body = {"model": "qwen3.8-27b",
                "messages": [{"role": "user", "content": MILDLY}],
                "max_tokens": 2048, "thinking_token_budget": 2048,
                "chat_template_kwargs": {"enable_thinking": True,
                                         "thinking_budget": 2048},
                "temperature": 0.0, "seed": 42}
        resp, _ = _post(body)
        rj = resp.json()
        rt_server = _rt(rj) or 0
        msg = rj["choices"][0]["message"]
        content = msg.get("reasoning_content") or msg.get("reasoning") or ""
        # LA-5b: TOKENIZER may be the /tokenize shim or None — never crash
        # the runbook; an unreachable tokenizer skips the delta instead of
        # distorting it.
        rt_local = None
        if content and TOKENIZER is not None:
            try:
                rt_local = len(TOKENIZER.encode(content))
            except Exception:
                rt_local = None
        if rt_local is not None:
            ds.append(abs(rt_local - rt_server))
    max_d = max(ds) if ds else 0
    # LA-9c (external-audit OBS-2, accepted): empty ds previously produced
    # a false-positive PASS (fail-open). The distinction is now explicit;
    # return semantics unchanged (PAR-CAL is evidence, GATE-A is the gate).
    rec = {"run_ds": ds, "max_d": max_d,
           "verdict": ("PASS" if ds and max_d <= 5
                       else "INCONCLUSIVE-no-tokenizer" if not ds
                       else "BLOCKED")}
    PREFLIGHT["steps"]["3_par_cal"] = rec
    print(f"STEP[3] IMP-7 PAR-CAL: {rec['verdict']} (max_d={max_d})")
    return max_d if max_d <= 5 else None


def _step_4_imp3():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    _il = InferenceLayer(transport=_fake)
    _il.ask_reasoner({"text": "x"}, mode="deep")
    ok = bool(spy) and spy[-1] == "deep"
    PREFLIGHT["steps"]["4_imp3"] = {"observed": spy,
                                    "verdict": "PASS" if ok else "FAIL"}
    print(f"STEP[4] IMP-3: {'PASS' if ok else 'FAIL'}")
    return ok


def _step_5_imp4():
    import ast, inspect
    from agent import my_agent as _ma
    tree = ast.parse(inspect.getsource(_ma))
    sites, with_mode = 0, 0
    for n in ast.walk(tree):
        if isinstance(n, ast.Call) and \
           getattr(n.func, "attr", None) == "ask_reasoner":
            sites += 1
            if any(k.arg == "mode" for k in n.keywords):
                with_mode += 1
    ok = sites > 0 and sites == with_mode
    PREFLIGHT["steps"]["5_imp4"] = {"call_sites": sites,
                                    "with_mode": with_mode,
                                    "verdict": "PASS" if ok else "FAIL"}
    print(f"STEP[5] IMP-4: {'PASS' if ok else 'FAIL'} ({with_mode}/{sites})")
    return ok


def _step_6_imp5():
    prior_disable = os.environ.get("DISABLE_LENGTH_RETRY")
    os.environ["DISABLE_LENGTH_RETRY"] = "1"
    try:
        spy, _ = _call_via_transport(
            model="qwen3.8-27b",
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=6144, reasoning_mode="deep", temperature=0.0)
    finally:
        if prior_disable is None:
            os.environ.pop("DISABLE_LENGTH_RETRY", None)
        else:
            os.environ["DISABLE_LENGTH_RETRY"] = prior_disable

    body = spy.first_body or {}
    wire_sha = hashlib.sha256(
        json.dumps(body, separators=(",", ":"), sort_keys=False).encode()
    ).hexdigest()
    ok = body.get("reasoning_effort") == "medium"
    PREFLIGHT["steps"]["6_imp5"] = {
        "wire_reasoning_effort": body.get("reasoning_effort"),
        "wire_sha256": wire_sha, "post_count": spy._post_count,
        "first_body_snapshot": True,
        "verdict": "PASS" if ok else "FAIL"}
    print(f"STEP[6] IMP-5: {'PASS' if ok else 'FAIL'}")
    return ok


def _step_7_imp6():
    def _probe(explicit):
        body = {"model": "qwen3.8-27b",
                "messages": [{"role": "user", "content": "2+2=?"}],
                "max_tokens": 6144, "thinking_token_budget": 2000,
                "reasoning_effort": "medium", "temperature": 0.0}
        if explicit:
            # LA-7: nested form (top-level ignored by vLLM 0.30)
            body["chat_template_kwargs"] = {"enable_thinking": True}
        resp, wire = _post(body)
        return _rt(resp.json()), wire

    rt7a, w7a = _probe(explicit=False)
    rt7b, w7b = _probe(explicit=True)
    if rt7a is not None and rt7a > 0:
        resolution, by = "v25.1-B", "auto"
    elif rt7b is not None and rt7b > 0:
        resolution, by = "v25.1-A", "auto"
    else:
        resolution, by = "BLOCKED_ESCALATED", "auto"

    rec = {"resolution": resolution, "determined_by": by,
           "step_7a_rt": rt7a, "step_7b_rt": rt7b,
           "evidence": {"wire_7a": w7a, "wire_7b": w7b}}

    if resolution == "v25.1-A":
        try:
            TRANSPORT.set_v25_1_a(True)
            rec["v25_1_a_gate_enabled"] = True
        except Exception:
            rec["v25_1_a_gate_enabled"] = False
    else:
        try: TRANSPORT.set_v25_1_a(False)
        except Exception: pass
        rec["v25_1_a_gate_enabled"] = False

    Path("IMP-6-resolution.json").write_text(
        json.dumps(rec, indent=2), encoding="utf-8")
    PREFLIGHT["steps"]["7_imp6"] = rec
    print(f"STEP[7] IMP-6: {resolution} (gate={rec['v25_1_a_gate_enabled']})")
    return resolution


def _step_7_5_d10_ab():
    def _one(include_budget):
        body = {"model": "qwen3.8-27b",
                "messages": [{"role": "user",
                              "content": "Reply with JSON {}."}],
                "max_tokens": 6144, "thinking_token_budget": 2000,
                "reasoning_effort": "medium", "temperature": 0.0,
                "top_p": 1.0, "seed": 42,
                "chat_template_kwargs": {"enable_thinking": True}}
        if include_budget:
            body["chat_template_kwargs"] = {"enable_thinking": True,
                                            "thinking_budget": 1000}
        resp, wire = _post(body)
        rj = resp.json()
        return {"reasoning_tokens": _rt(rj),
                "final_answer": rj["choices"][0]["message"].get("content", ""),
                "wire_sha256": wire["wire_sha256"] if wire else None}

    present = _one(True); absent = _one(False)
    rt_eq = present["reasoning_tokens"] == absent["reasoning_tokens"]
    ans_eq = present["final_answer"] == absent["final_answer"]
    verdict = "PASS" if (rt_eq and ans_eq) else "FAIL"
    rec = {"present": present, "absent": absent,
           "rt_equal": rt_eq, "answer_equal": ans_eq, "verdict": verdict,
           "note": "AGA-26: hand-built probe"}
    PREFLIGHT["steps"]["7_5_d10_ab"] = rec
    print(f"STEP[7.5] D-10 A/B: {verdict} (rt_eq={rt_eq}, ans_eq={ans_eq})")
    return verdict == "PASS"


def _gate_a_c0_attempt(budget=1024):
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": "count to 5"}],
            "max_tokens": 4096, "thinking_token_budget": budget,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": budget},
            "temperature": 0.0, "seed": 42}
    resp, wire = _post(body)
    rj = resp.json()
    return rj, wire, _rt(rj)


def _gate_a_c0():
    d = D or 0
    rj, wire, rt = _gate_a_c0_attempt(1024)
    wire_confirms_1 = (wire is not None
                     and wire["body"].get("thinking_token_budget") == 1024)
    rec = {"configured_budget": 1024, "reasoning_tokens": rt,
           "finish_reason": _fr(rj),
           "wire_sha256": wire["wire_sha256"] if wire else None,
           "wire_confirms_budget_1024_attempt1": wire_confirms_1,
           "note": "AGA-26: hand-built probe"}

    if rt is not None and 0 < rt < 192:
        print(f"STEP[8] C0: 0<rt<192 (rt={rt}); retry-once per D-13")
        rj2, wire2, rt2 = _gate_a_c0_attempt(1024)
        wire_confirms_2 = (wire2 is not None
                     and wire2["body"].get("thinking_token_budget") == 1024)
        rec["retry_reasoning_tokens"] = rt2
        rec["retry_wire_sha256"] = (wire2["wire_sha256"] if wire2 else None)
        rec["wire_confirms_budget_1024_attempt2"] = wire_confirms_2
        if rt2 is not None and 192 <= rt2 <= 1024 + d:
            rec["verdict"] = "PASS-after-retry"
        elif rt2 is not None and rt2 > 1024 + d and wire_confirms_2:
            rec["verdict"] = "FAIL-breach"
        elif rt2 is not None and rt2 > 1024 + d:
            rec["verdict"] = "INCONCLUSIVE-no-wire-confirm"
        elif rt2 == 0:
            rec["verdict"] = "BLOCKED-retry-rt0"
        else:
            rec["verdict"] = "BLOCKED-retry-exhausted"
        PREFLIGHT["steps"]["8_c0"] = rec
        print(f"STEP[8] C0: {rec['verdict']} (rt={rt}, rt2={rt2})")
        return rec

    if rt is None:
        rec["verdict"] = "GATE-B-trigger"
    elif rt == 0:
        rec["verdict"] = "GATE-B-trigger"
    elif 192 <= rt <= 1024 + d:
        rec["verdict"] = "PASS"
    elif rt > 1024 + d and wire_confirms_1:
        rec["verdict"] = "FAIL-breach"
    elif rt > 1024 + d:
        rec["verdict"] = "INCONCLUSIVE-no-wire-confirm"
    else:
        rec["verdict"] = "UNCLASSIFIED"
    PREFLIGHT["steps"]["8_c0"] = rec
    print(f"STEP[8] C0: {rec['verdict']} (rt={rt})")
    return rec


def _gate_a_cn(rt_expected, budget):
    """F-3 (P2, INDUS-DIR-070 closure): D-6 branch split.

    Per D-6: measurement-error-aware decision.
      rt > budget + d  → FAIL-breach (only when FT confirms enforcement)
      abs(rt - budget) <= d with FT present → PASS
      rt < budget - d with FT present → INCONCLUSIVE (measurement/parser
                                     error) → GATE-AC record (manual review)
      FT absent → INCONCLUSIVE-no-FT
    """
    d = D or 0
    log_since = 0
    if FT_LOG_PATH and Path(FT_LOG_PATH).is_file():
        try: log_since = Path(FT_LOG_PATH).stat().st_size
        except Exception: log_since = 0
    metric_before = _ft_metric_current()

    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": "list 3 colors"}],
            "max_tokens": 4096, "thinking_token_budget": budget,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": budget},
            "temperature": 0.0, "seed": 42}
    resp, wire = _post(body)
    rj = resp.json(); rt = _rt(rj); ft = _ft_evidence(log_since, metric_before)
    rec = {"configured_budget": budget, "reasoning_tokens": rt,
           "ft_evidence": ft, "d_used": d,
           "wire_sha256": wire["wire_sha256"] if wire else None,
           "note": "AGA-26: hand-built probe; F-3 split applied"}

    if rt is None:
        rec["verdict"] = "GATE-B-trigger"
        rec["branch"] = "rt_absent"
    elif ft is None:
        rec["verdict"] = "INCONCLUSIVE-no-FT"
        rec["branch"] = "ft_absent"
    elif abs(rt - budget) <= d:
        rec["verdict"] = "PASS"
        rec["branch"] = "in_tolerance_with_FT"
    elif rt > budget + d:
        # breach direction — server emitted more reasoning tokens than the
        # configured budget permits
        rec["verdict"] = "FAIL-breach"
        rec["branch"] = "rt_above_budget_plus_d"
    elif rt < budget - d:
        # F-3: measurement / parser error — NOT a FAIL.  Routes to GATE-AC.
        rec["verdict"] = "INCONCLUSIVE-measurement-error"
        rec["branch"] = "rt_below_budget_minus_d_with_FT"
        # Also record into GATE-AC for traceability
        PREFLIGHT.setdefault("steps", {}).setdefault(
            "8_gate_ac_extra", []).append({
                "case": f"8_c{rt_expected}",
                "note": "rt < budget - d with FT present (D-6)",
                "reasoning_tokens": rt, "budget": budget, "d": d})
    else:
        rec["verdict"] = "UNCLASSIFIED"
        rec["branch"] = "unreachable"

    PREFLIGHT["steps"][f"8_c{rt_expected}"] = rec
    print(f"STEP[8] C{rt_expected}: {rec['verdict']} "
          f"(rt={rt}, branch={rec['branch']})")
    return rec


def _step_9_e2e(imp2_outcome, imp6_resolution):
    # FIX-α (audit round-5 §2, VERIFIED TRUE): diagnostic env vars pehle
    # bina restore ke set hote the — diag mode (token floors off, length-
    # retry off) production THE RUN tak leak karta tha. Ab: save -> set ->
    # try/finally restore. Production semantics THE RUN mein wapas normal.
    _saved_diag_env = {k: os.environ.get(k)
                       for k in ("DISABLE_LENGTH_RETRY",
                                 "TRANSPORT_FLOOR_OVERRIDE")}
    os.environ["DISABLE_LENGTH_RETRY"] = "1"
    os.environ["TRANSPORT_FLOOR_OVERRIDE"] = "diag"
    try:
        return _step_9_e2e_diag(imp2_outcome, imp6_resolution)
    finally:
        for _k, _v in _saved_diag_env.items():
            if _v is None:
                os.environ.pop(_k, None)
            else:
                os.environ[_k] = _v


def _step_9_e2e_diag(imp2_outcome, imp6_resolution):

    spy_d, resp_d = _call_via_transport(
        model="qwen3.8-27b",
        messages=[{"role": "user", "content": "give JSON"}],
        max_tokens=6144, reasoning_mode="deep")
    body_d = spy_d.first_body or {}
    wire_d_sha = hashlib.sha256(
        json.dumps(body_d, separators=(",", ":"), sort_keys=False).encode()
    ).hexdigest()
    try: rj_d = resp_d.json()
    except Exception:
        rj_d = {"choices": [{"message": {"content": ""}}],
                "usage": {"completion_tokens": 0}}

    a0_checks = {
        "transport_emitted_effort_medium":
            body_d.get("reasoning_effort") == "medium",
        "transport_emitted_budget_2000":
            body_d.get("thinking_token_budget") == 2000,
        "transport_emitted_max_tokens_6144":
            body_d.get("max_tokens") == 6144,
        "transport_no_stale_top_level": "thinking_budget" not in body_d,
        "transport_ctk_enable_thinking_true":
            (body_d.get("chat_template_kwargs") or {}).get("enable_thinking") is True,
        "transport_ctk_thinking_budget_2000":
            (body_d.get("chat_template_kwargs") or {}).get("thinking_budget") == 2000,
        "transport_no_reasoning_mode_on_wire":
            "reasoning_mode" not in body_d,
        "post_reasoning_tokens_gt_0": (_rt(rj_d) or 0) > 0,
    }
    if imp6_resolution == "v25.1-A":
        a0_checks["transport_v251a_toplevel_flag"] = (
            body_d.get("enable_thinking") is True)
        # LA-7: primary assertion is the nested form (always present now)
    PREFLIGHT["steps"]["9_a0"] = {"checks": a0_checks,
        "transport_wire_sha256": wire_d_sha,
        "verdict": "PASS" if all(a0_checks.values()) else "FAIL"}

    spy_f, resp_f = _call_via_transport(
        model="qwen3.8-27b",
        messages=[{"role": "user", "content": "give JSON"}],
        max_tokens=1024, reasoning_mode="fast")
    body_f = spy_f.first_body or {}
    a0f_checks = {
        "transport_enable_thinking_false":
            body_f.get("chat_template_kwargs", {}).get("enable_thinking") is False,
        "transport_no_budget": "thinking_token_budget" not in body_f,
        "transport_no_effort": "reasoning_effort" not in body_f,
        "transport_no_stale":
            "thinking_budget" not in (body_f.get("chat_template_kwargs") or {}),
    }
    PREFLIGHT["steps"]["9_a0_fast"] = {"checks": a0f_checks,
        "verdict": "PASS" if all(a0f_checks.values()) else "FAIL"}

    ct = _ct(rj_d); rt = _rt(rj_d) or 0
    # FIX-ι3 (audit round-6 §5, VERIFIED TRUE): +2000 allowance nonstandard
    # accounting assume karta tha. LIVE EVIDENCE (Run-22 STEP[2]): max_tokens
    # =1000 par rt=1000, ct=1000 => completion_tokens REASONING INCLUDE
    # karta hai AUR max_tokens se capped hai => ct kabhi 6144 se zyada ho hi
    # nahi sakta. Dono branches ab conservative 6144 par.
    if imp2_outcome == "EXCLUDES":
        a5_ok = ct <= 6144; bound = "EXCLUDES-conservative-fixed"
    else:
        a5_ok = ct <= 6144; bound = "INCLUDES-conservative"
    a5 = {"bound": bound, "completion_tokens": ct, "max_tokens": 6144,
          "budget": 2000, "check": a5_ok,
          "reasoning_tokens_le_budget": rt <= 2000}
    a5["verdict"] = "PASS" if a5["check"] and a5["reasoning_tokens_le_budget"] else "FAIL"
    PREFLIGHT["steps"]["9_a5"] = a5

    try:
        json.loads(rj_d["choices"][0]["message"]["content"]); a6 = True
    except Exception: a6 = False
    PREFLIGHT["steps"]["9_a6"] = {"verdict": "PASS" if a6 else "FAIL"}

    if (a5["verdict"] == "FAIL" and bound == "INCLUDES-conservative"
            and ct > 6144 and imp2_outcome == "INCONCLUSIVE"):
        return _d15_escalation(IMP2B_PATH)

    return {"a0": PREFLIGHT["steps"]["9_a0"]["verdict"],
            "a0_fast": PREFLIGHT["steps"]["9_a0_fast"]["verdict"],
            "a5": a5["verdict"], "a6": PREFLIGHT["steps"]["9_a6"]["verdict"]}


def _d15_escalation(imp2b_path):
    _verify(imp2b_path, IMP2B_SHA)
    imp2b_prompt = Path(imp2b_path).read_text(encoding="utf-8")
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": imp2b_prompt}],
            "max_tokens": 500, "thinking_token_budget": 2000,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": 2000},
            "temperature": 0.0}
    resp, _ = _post(body); rt = _rt(resp.json())
    if rt is not None and rt >= 1000:
        PREFLIGHT["steps"]["9_d15"] = {"imp2b_rt": rt,
                                       "verdict": "EXCLUDES-proven"}
        return {"a0": "PASS", "a0_fast": "PASS", "a5": "RE-EVAL", "a6": "PASS"}
    PREFLIGHT["steps"]["9_d15"] = {"imp2b_rt": rt,
                                   "verdict": "INCONCLUSIVE-manual-review"}
    return {"a0": "PASS", "a0_fast": "PASS", "a5": "MANUAL-REVIEW", "a6": "PASS"}


def _gate_ac_record():
    ac = []
    for k in ("8_c0", "8_c48", "8_c96"):
        s = PREFLIGHT["steps"].get(k)
        if s and s.get("reasoning_tokens") is not None:
            rt = s["reasoning_tokens"]; budget = s["configured_budget"]
            if abs(rt - budget) <= (D or 0) and s.get("ft_evidence") is None:
                ac.append({"case": k, "note": "abs(rt-budget)<=d without FT"})
            if rt < budget - (D or 0) and s.get("ft_evidence") is not None:
                ac.append({"case": k, "note": "rt<budget-d with FT present"})
    PREFLIGHT["steps"]["10_gate_ac"] = {"records": ac,
        "note": "diagnostic only (D-14)"}


def _step_11_gate_b():
    # FIX-ι1 (audit round-6 BLOCKER-2, VERIFIED TRUE): `is not None` predicate
    # rt=0 ko bhi "parser evidence" maan leta tha — jabki C0 ka apna logic
    # rt==0 ko suspicious (GATE-B-trigger) treat karta hai. Ab parser
    # evidence = POSITIVE rt (rt>0). Live Run-22 evidence: rt=50/23/4095 —
    # naye predicate ke saath bhi PASS; sirf rt=0/None ab BLOCKED.
    any_rt = any(
        (lambda v: v is not None and v > 0)(
            PREFLIGHT["steps"].get(k, {}).get("reasoning_tokens"))
        for k in PREFLIGHT["steps"]
        if k.startswith("8_") or k.startswith("9_"))
    rec = {"parser_derived_rt_positive": any_rt,
           "verdict": "PASS" if any_rt else "BLOCKED"}
    PREFLIGHT["steps"]["11_gate_b"] = rec
    print(f"STEP[11] GATE-B: {rec['verdict']}")
    return any_rt


def _step_12_gate_c():
    rec = {"parser_flag": "qwen3",
           "field_path": "usage.completion_tokens_details.reasoning_tokens",
           "server_version": os.environ.get("VLLM_VERSION", "unknown"),
           "vllm_commit":    os.environ.get("VLLM_COMMIT", "unknown"),
           "gate_b_precheck": PREFLIGHT["steps"].get("11_gate_b", {}).get("verdict")}
    rec["verdict"] = "PASS" if rec["gate_b_precheck"] == "PASS" else "BLOCKED"
    PREFLIGHT["steps"]["12_gate_c"] = rec
    print(f"STEP[12] GATE-C: {rec['verdict']}")
    return rec["verdict"] == "PASS"


def _step_13_gate_e(traces=None, manual_review=None):
    """F-5 closure (v3.6): D-7 multi-class semantics.
    PASS iff off_game == 0 AND (unknown == 0 OR all unknowns cleared).
    90% coverage = pre-review automation gate; manual review can rescue.
    cannot-cover = manual review unavailable OR total_lines == 0."""
    traces = traces or []
    total = len(traces)
    if total == 0:
        rec = {"on_game": 0, "off_game": 0, "unknown": 0, "total_lines": 0,
               "verdict": "CANNOT-COVER", "reason": "total_lines == 0 (D-7)"}
    else:
        def _classify(t):
            tl = t.lower()
            if any(k in tl for k in ("harness", "kaggle", "vllm server", "notebook")):
                return "off"
            if any(k in tl for k in ("game", "level", "action", "grid", "move")):
                return "on"
            return "unknown"
        classes = [_classify(t) for t in traces]
        on_game, off_game, unknown = (classes.count("on"), classes.count("off"),
                                      classes.count("unknown"))
        cleared_unknown, rescued_off = 0, 0
        if manual_review is not None:
            for i, c in enumerate(classes):
                r = manual_review(i, traces[i], c)
                if c == "unknown" and r == "clear": cleared_unknown += 1
                elif c == "off" and r == "on": rescued_off += 1
        off_game -= rescued_off
        uncleared = unknown - cleared_unknown
        coverage = (on_game + off_game + rescued_off) / total
        if off_game == 0 and uncleared == 0:
            verdict, reason = "PASS", (f"off_game=0; unknown fully cleared "
                                      f"(coverage {coverage:.0%})")
        elif uncleared > 0 and manual_review is None:
            verdict, reason = "CANNOT-COVER", ("manual review unavailable with "
                                               "unresolved unknowns (D-7)")
        elif coverage >= 0.90:
            verdict, reason = "PRE-REVIEW", (f"coverage {coverage:.0%} >= 90% automation "
                                             f"gate; manual review required: {off_game} "
                                             f"off-game, {uncleared} unknown unresolved")
        else:
            verdict, reason = "FAIL", (f"coverage {coverage:.0%} < 90% automation gate; "
                                        f"{off_game} off-game, {uncleared} unresolved unknown")
        rec = {"on_game": on_game, "off_game": off_game, "unknown": unknown,
               "cleared_unknown": cleared_unknown, "rescued_off": rescued_off,
               "total_lines": total, "coverage_pct": round(coverage * 100, 1),
               "verdict": verdict, "reason": reason}
    PREFLIGHT["steps"]["13_gate_e"] = rec
    print(f"STEP[13] GATE-E: {rec['verdict']}")
    return rec["verdict"] == "PASS"

def _step_14_certify(imp2p, e2e, gate_b, gate_c, gate_e):
    gate_a = (e2e["a0"] == "PASS" and e2e["a0_fast"] == "PASS"
              and e2e["a5"] in ("PASS", "RE-EVAL") and e2e["a6"] == "PASS")
    phase1 = (imp2p == "PASS" and gate_a and gate_b and gate_c)
    chain  = phase1 and gate_e
    # FIX-ι2 (audit round-6 BLOCKER-1, VERIFIED TRUE): C0/C48/C96 verdicts
    # phase1/chain mein nahi hain — auditor ne "mixed semantics" pakda.
    # DECISION (Indus, adjudicated): DIAGNOSTIC-ONLY, explicitly. Kyunki:
    # (a) unka EVIDENCE (reasoning_tokens) GATE-B mein already jaata hai;
    # (b) C0 ko hard-gate banana is environment mein hamesha-FAIL deta —
    # live Run-22: C0 BLOCKED-retry-exhausted (rt=50, small-budget thinking
    # characteristic, defect nahi) — waisa gate certification ko meaningless
    # kar deta. Ab record mein policy explicit hai — koi ambiguity nahi.
    _cert = {"c_probes_policy": "DIAGNOSTIC-ONLY: C0/C48/C96 evidence feeds "
             "GATE-B; verdicts do NOT gate phase1/chain (D-15 ruling)",
             "gate_a": "PASS" if gate_a else "FAIL",
             "gate_b": "PASS" if gate_b else "FAIL",
             "gate_c": "PASS" if gate_c else "FAIL",
             "gate_e": "PASS" if gate_e else "FAIL",
             "phase1": "PASS" if phase1 else "FAIL",
             "chain":  "PASS" if chain  else "FAIL"}
    PREFLIGHT["certification"] = _cert
    assert "10_gate_ac" not in PREFLIGHT.get("certification", {}), "D-14 violation"
    for _k in PREFLIGHT["certification"]:
        assert "gate_ac" not in _k.lower() and _k.lower() != "ac", \
            f"D-14 violation: key {_k!r}"
    return chain


def _run():
    global D
    print("=== FROZEN PRE-FLIGHT RUNBOOK ===")

    if not globals().get("CONFORMANCE_PASSED", False):
        # Indus forward-fix (audit round-3, FIX-Y follow-through): ye _run()
        # ke andar ka EKMAATRA SystemExit tha — advisory wrapper ko bypass
        # karke notebook abort kar deta. Stop semantics consistent with baaki
        # sab paths: record + return 1 (Run-All mein order fix hai — cell 26
        # pehle chalta hai — isliye ye path normally unreachable hai).
        Path("PRE-FLIGHT-RECORD.json").write_text(json.dumps({
            "verdict": "STOP", "reason": "conformance-not-run",
            "note": "template-conformance must arm before GATE-A (report §12)"}))
        print("RUNBOOK STOP: conformance cell has not been run (recorded)")
        return 1

    print("OK   CONFORMANCE_PASSED=True (report §12 gate satisfied)")

    imp2p = _step_1_imp2p()
    if imp2p != "PASS":
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    imp2 = _step_2_imp2()
    D = _step_3_par_cal()
    if D is None:
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    if not _step_4_imp3(): return 1
    if not _step_5_imp4(): return 1
    if not _step_6_imp5(): return 1
    imp6 = _step_7_imp6()
    if imp6 == "BLOCKED_ESCALATED":
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    if not _step_7_5_d10_ab():
        print("RUNBOOK STOP: D-10 A/B FAIL")
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    _gate_a_c0()
    _gate_a_cn(48, 48)
    _gate_a_cn(96, 96)
    _gate_ac_record()
    e2e = _step_9_e2e(imp2, imp6)
    gb  = _step_11_gate_b()
    gc  = _step_12_gate_c()
    ge  = _step_13_gate_e(traces=[])  # stub by design: GATE-E certifies the CELL-25 run, not this pre-flight runbook; real traces come from the game loop
    chain = _step_14_certify(imp2p, e2e, gb, gc, ge)
    Path("PRE-FLIGHT-RECORD.json").write_text(
        json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
    print(f"=== CERTIFICATION: phase1={PREFLIGHT['certification']['phase1']} "
          f"chain={PREFLIGHT['certification']['chain']} ===")
    return 0 if chain else 1


if __name__ == "__main__":
    # LA-10 / FIX-U (Indus, 2026-09-30, Run-22 postmortem): FIX-Y ka
    # "exit(0) never aborts" JUPYTER MEIN GALAT THA — IPython har SystemExit
    # (code 0 bhi) catch karke cell ko error-mark karta hai aur Run-All
    # wahin ruk jaata hai. Evidence: log 865.4s par advisory-warn ke turant
    # baad IPython exit-UserWarning, phir nbconvert finalization — coder-
    # conf/frame-probe/THE RUN/koi game cell NAHI chala. Fix: exit call
    # POORA hata do — cell normally complete ho, Run-All aage badhe.
    # (Audit round-3 ka proposed fix bhi yahi galti karta tha — meri miss:
    # maine Python exit-semantics test kiya tha, Jupyter-semantics nahi.)
    _rc = _run()
    if _rc:
        print("WARN: runbook preflight rc=%d — ADVISORY MODE: games proceed, "
              "preflight evidence recorded for post-hoc audit" % _rc, flush=True)
    # NO exit — Jupyter cell normal completion = Run-All continues.


In [ ]:
# CODER-SOLVER CONFORMANCE (Run-21) — 8 checks, arms CODER_CONFORMANCE_PASSED
# FIX-V (external-audit round-4, adjudicated PARTIALLY TRUE — coder-conf
# THE RUN ke baad tha: fail hone par 6h games ke BAAD papermill abort =
# slot waste. Ab runbook ke turant baad, THE RUN se PEHLE chalta hai —
# fail-closed at the RIGHT time (code regression = pre-game abort sahi,
# environment-certification = advisory, runbook FIX-Y). Dependencies: sirf
# agent modules (cells 4-17) — live-exec verified position-independent.
import hashlib, sys
sys.path.insert(0, "/kaggle/working/goa")
from agent import sandbox_v6 as _sb
from agent import coder_solver as _cs
from agent import inference_layer as _il

_ok = 0
def _chk(name, cond):
    global _ok
    print(("PASS " if cond else "FAIL ") + name)
    if cond: _ok += 1

raw = open("/kaggle/working/goa/agent/counterexample_schema.json", "rb").read()
_chk("CS-1 schema md5 frozen",
     hashlib.md5(raw).hexdigest() == "97073721699cf92d53bb4909322ad212")
r1 = _sb.run_candidate("def f():\n    import os\n    return os", "f", {})
_chk("CS-2 sandbox blocks import", (not r1["ok"]) and r1["error_class"] == "SandboxViolation")
r2 = _sb.run_candidate("def f():\n    return __import__('os')", "f", {})
_chk("CS-3 sandbox blocks __import__", (not r2["ok"]) and r2["error_class"] == "SandboxViolation")
r3 = _sb.run_candidate("def f(frame, history):\n    return 'right'", "f",
                       {"frame": [[0, 0]], "history": []})
_chk("CS-4 benign round-trip", r3["ok"] and r3["result"] == "right")
_chk("CS-5 IL v1.6 call_log + CAP",
     _il.__version__ == "1.6" and _il.CODER_CODE_CAP == 2048
     and callable(_il.call_log))
c = _cs.AttemptCounters()
for _ in range(6):
    c.new_level(); c.begin_attempt(); c.begin_repair(); c.begin_repair()
_chk("CS-6 attempt arithmetic worst-case 18", c.generations == 18)
s7 = _cs.CoderSolver(call_llm=lambda spec, mode: "")
_chk("CS-7 solver wires real sandbox (LA-1)", s7._run is _sb.run_candidate)
chk8 = _il.InferenceLayer(transport=None)
_chk("CS-8 IL instance call_log + coder params seam (LA-3/LA-4)",
     callable(getattr(chk8, "call_log", None))
     and hasattr(_il, "_DEFAULT_PARAMS"))

# FIX-X (Indus, 2026-09-30, external-audit BUG-X, VERIFIED TRUE): the
# threshold was left at 6 when CS-7/CS-8 were added — the green path
# (8/8) raised SystemExit and aborted Run-All after THE RUN. My v6 miss:
# message strings were bumped, the condition was not.
if _ok != 8:
    raise SystemExit(f"CODER-SOLVER CONFORMANCE FAILED ({_ok}/8)")
CODER_CONFORMANCE_PASSED = True
print("CODER-SOLVER CONFORMANCE: 8/8 PASSED")


In [ ]:
# FRAME PROBE v14: Run#9 fix — framework Agent direct-load shim (agents/__init__ zoo bypass). v13: OperationMode.OFFLINE. v12: H-4 fix (arc_probe bhi close — env ke saath). v11: L-1 fix (print _ProbeProxy). v10: live FrameDataRaw.frame = List[ndarray] — har action ke beech engine
# MULTIPLE 64x64 int8 renders deta hai (base_game.perform_action loop). Certified
# _frame_grid corpus shape (flat 2D grid) expect karta hai — adapter LAST grid flatten
# karta hai (int cells) ya flatten+rgb_to_grid (RGB cells). Mini-e2e REAL engine frame
# par certified path end-to-end verify karta hai.
import sys, os

# ---------- Path setup (dedup + FRAMEWORK_DIR guard) ----------

if 'FRAMEWORK_DIR' not in globals() or not FRAMEWORK_DIR:
    raise SystemExit("FRAMEWORK_DIR undefined — Cell 1 preflight pehle chalao")

for _p in ["/kaggle/working/goa", "/kaggle/working/goa/agent", FRAMEWORK_DIR]:
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)
        print(f"OK   sys.path += {_p}")

# ---------- ARC API import (error context) ----------

try:
    import arc_agi
    from arc_agi import OperationMode
except Exception as e:
    raise SystemExit(f"arc_agi import fail: {type(e).__name__}: {e} "
                     f"— Cell 2 pip install + Cell 3 bootstrap check karo")

# ---------- Arcade + env + reset (each step wrapped) ----------

try:
    arc_probe = arc_agi.Arcade(operation_mode=OperationMode.OFFLINE)   # v13: Run#8 - NORMAL API try karta tha (three.arcprize.org), internet OFF hai
    print("OK   Arcade created (OperationMode.OFFLINE)")
except Exception as e:
    raise SystemExit(f"Arcade() fail: {type(e).__name__}: {e} "
                     f"— OperationMode.OFFLINE bhi fail - local environment_files check karo (Cell 2 cache)")

try:
    env_probe = arc_probe.make("ls20")
except Exception as e:
    raise SystemExit(f"make('ls20') fail: {type(e).__name__}: {e} "
                     f"— game ID ya cache issue")

if env_probe is None:
    raise SystemExit("make('ls20') returned None — games cache check karo")

try:
    r = env_probe.reset()
except Exception as e:
    raise SystemExit(f"reset() fail: {type(e).__name__}: {e}")

if r is None or not getattr(r, "frame", None):
    raise SystemExit("reset() frame missing — arc-agi API check karo")

raw_frames = r.frame
print(f"OK   reset(): {len(raw_frames)} frame(s), "
      f"shape={raw_frames[0].shape}, dtype={raw_frames[0].dtype}")

# ---------- Robust _norm_grid (ndarray-safe) ----------

def _norm_grid(frame_attr):
    """Live List[grid] -> LAST grid (2D, tolist). Single ndarray/list passthrough."""
    if frame_attr is None:
        raise ValueError("empty frame (None)")
    if isinstance(frame_attr, list):
        if not frame_attr:
            raise ValueError("empty frame list")
        last = frame_attr[-1]
    else:
        last = frame_attr
    if hasattr(last, "tolist"):
        last = last.tolist()
    if not last:
        raise ValueError("frame empty after conversion")
    return last

grid = _norm_grid(raw_frames)
cell = grid[0][0]
print(f"OK   grid: {len(grid)} x {len(grid[0])} | sample cell: {cell!r}")

# ---------- Format detection ----------

if isinstance(cell, (int, float)) and not isinstance(cell, bool):
    FRAME_ADAPTER = _norm_grid
    print("GREEN: int cells — adapter = last-grid flatten")
elif isinstance(cell, (list, tuple)) and len(cell) >= 3:
    import agent.lora_datagen as ld
    _smoke = ld.rgb_to_grid(grid, None)
    print(f"OK   adapter smoke: {grid[0][0]!r} -> {_smoke[0][0]!r}")
    FRAME_ADAPTER = lambda fa: ld.rgb_to_grid(_norm_grid(fa), None)
    print("GREEN: RGB cells — last-grid flatten + rgb_to_grid (lora_datagen D5)")
else:
    raise SystemExit(f"UNKNOWN cell format: {cell!r} — run yahin ruka (safety gate)")

# ---------- Mini-e2e: adapter -> _ProbeProxy -> certified _frame_grid ----------

# ---------- Run#9 fix: framework Agent DIRECT-LOAD shim (v14) ----------
# Root cause: my_agent.py ka 'from agents.agent import Agent' pehle agents/__init__.py
# chalata hai — wo langgraph/smolagents/langsmith template zoo import karta hai (offline
# session mein absent) -> ImportError -> my_agent ka _FrameworkAgent STUB fallback ->
# main() missing -> Run#9 mein 25/25 games instant-fail ('GoAMyAgent' has no 'main').
# Fix: package stub ke through agents/agent.py DIRECT load (uski deps — arcengine,
# pydantic, arc_agi, .recorder, .tracing(agentops-guarded) — sab installed hain).
import types as _types15, importlib as _il15

if "agents.agent" not in sys.modules:
    _agents_stub = _types15.ModuleType("agents")
    _agents_stub.__path__ = [os.path.join(FRAMEWORK_DIR, "agents")]
    sys.modules["agents"] = _agents_stub
    sys.modules["agents.agent"] = _il15.import_module("agents.agent")

_FW_AGENT = sys.modules["agents.agent"].Agent
if not hasattr(_FW_AGENT, "main"):
    raise SystemExit("framework Agent.main missing — direct-load fail, ARC repo attach check karo")
print(f"OK   framework Agent direct-loaded (MAX_ACTIONS={getattr(_FW_AGENT, 'MAX_ACTIONS', '?')}, main loop live)")
print("OK   my_agent ab REAL framework Agent inherit karega — main() available hoga")

import agent.my_agent as _ma

class _ProbeProxy:
    """Mimics FrameData surface for _frame_grid, but swaps .frame for adapted grid."""
    def __init__(self, fd, g):
        try:
            self.__dict__.update({k: v for k, v in vars(fd).items() if k != "frame"})
        except TypeError:
            for attr in ("state", "available_actions", "levels_completed",
                         "win_levels", "action_count", "reward", "score", "guid",
                         "full_reset", "game_id"):
                v = getattr(fd, attr, None)
                if v is not None:
                    setattr(self, attr, v)
        self.frame = g

_pg = FRAME_ADAPTER(r.frame)
_g = _ma._frame_grid(_ProbeProxy(r, _pg))

# ---------- Shape check ----------

if not _g:
    raise SystemExit("_frame_grid returned empty")
if len(_g) != len(_pg) or len(_g[0]) != len(_pg[0]):
    raise SystemExit(
        f"mini-e2e shape mismatch: got {len(_g)}x{len(_g[0])}, "
        f"want {len(_pg)}x{len(_pg[0])}"
    )

# ---------- Full-cell value check (not just [1][1]) ----------

_mismatches = [
    (i, j, _g[i][j], _pg[i][j])
    for i in range(len(_g))
    for j in range(len(_g[0]))
    if _g[i][j] != _pg[i][j]
]
if _mismatches:
    _first = _mismatches[0]
    raise SystemExit(
        f"_frame_grid != adapter output — {len(_mismatches)} mismatches. "
        f"First at [{_first[0]}][{_first[1]}]: got {_first[2]!r} want {_first[3]!r}"
    )

print(f"MINI-E2E PASS: adapter + _ProbeProxy + _frame_grid -> "
      f"{len(_g)}x{len(_g[0])} grid, certified path verified "
      f"(all {len(_g)*len(_g[0])} cells matched)")

# ---------- Cleanup ----------

try:
    if hasattr(env_probe, "close"):
        env_probe.close()
        print("OK   env_probe closed")
except Exception:
    pass
try:
    if hasattr(arc_probe, "close"):
        arc_probe.close()
        print("OK   arc_probe closed")
except Exception:
    pass


In [ ]:
# THE RUN v9 (DIR/2026-068 — Memory & Coder Awakening): CM v1.5 + my_agent v1.0.17 (coder wiring + tactics journal) + time caps 20min/7.5h + total_hard_cap 14000. v8: context_window=16384. v7: fw base init. v6: Agent gate. v5: OFFLINE.
# MyAgent artifact class (v1.0.16) + injected real-model orchestrator (_orchestrator).
# FIXES v4 (external audit): H-1 health-check route keys hardcoded (env-proof), M-2 ETA append env-None.
# FIXES v3: hard 25-game gate, 9h Kaggle cap reference, env-None JSON save.
# FIXES v2: 12 min/game + 5.5h global budget, FRAME_ADAPTER guard, a.frames safety,
# API checks, TRANSPORT reuse, vLLM health, defensive attrs.
import sys, time, json, os, threading, traceback
sys.path.insert(0, "/kaggle/working/goa")
if 'FRAMEWORK_DIR' not in globals() or not FRAMEWORK_DIR:
    raise SystemExit("FRAMEWORK_DIR undefined — Cell 1 preflight pehle chalao")
sys.path.insert(0, FRAMEWORK_DIR)
os.chdir("/kaggle/working")   # Arcade ./environment_files cache yahin hai
                              # NOTE: cells 17-18 ko absolute paths use karni hongi

import arc_agi
from arc_agi import OperationMode
import agent.context_manager as cm_mod
import agent.inference_layer as il_mod
import agent.grid_text as gt_mod
import agent.action_pipeline as ap_mod
import agent.my_agent as ma

# ---------- Defensive: MyAgent API surface check (Cell 8 version pe depend karta hai) ----------

_REQUIRED_MA_ATTRS = ["Agent", "CERTIFIED_VERSIONS", "_state_name", "_make_framework_action"]
_missing_ma = [a for a in _REQUIRED_MA_ATTRS if not hasattr(ma, a)]
if _missing_ma:
    raise SystemExit(f"my_agent missing attrs: {_missing_ma} — Cell 8 version mismatch")

_VERSIONS = getattr(ma, "CERTIFIED_VERSIONS", None) or {}
if not _VERSIONS:
    print("WARN ma.CERTIFIED_VERSIONS missing/empty — empty manifest use hoga")

# ---------- Run#9 fix: framework Agent live hona CHAHIYE (main() = game loop) ----------
# Run#9: har game 'GoAMyAgent' object has no attribute 'main' se 0.0 min mein fail hua.
# Shim (Cell 15 v14 twin — idempotent): agents package stub -> agents/agent.py direct load.
# Agar my_agent pehle stub ke saath cached tha to purge + fresh import (self-heal).
import sys as _sys16, types as _types16, importlib as _il16

if "FRAMEWORK_DIR" not in globals() or not FRAMEWORK_DIR:
    raise SystemExit("FRAMEWORK_DIR undefined — Cell 1 preflight pehle chalao")

if "agents.agent" not in _sys16.modules:
    _stub16 = _types16.ModuleType("agents")
    _stub16.__path__ = [os.path.join(FRAMEWORK_DIR, "agents")]
    _sys16.modules["agents"] = _stub16
    _sys16.modules["agents.agent"] = _il16.import_module("agents.agent")

if not getattr(ma, "_HAS_FRAMEWORK_AGENT", False):
    print("WARN my_agent stub-ke-saath cached tha — shim ke baad fresh import (Run#9 self-heal)")
    for _m16 in [m for m in list(_sys16.modules) if m in ("agent.my_agent", "my_agent")]:
        _sys16.modules.pop(_m16, None)
    ma = _il16.import_module("agent.my_agent")

if not getattr(ma, "_HAS_FRAMEWORK_AGENT", False):
    raise SystemExit("my_agent._HAS_FRAMEWORK_AGENT False — framework Agent inherit fail (Run#9 bug) — ARC repo attach check karo")
if not hasattr(ma.MyAgent, "main"):
    raise SystemExit("MyAgent.main missing — framework main() loop nahi mila (Run#9 bug)")
print(f"OK   framework Agent live — MyAgent.main() available (MAX_ACTIONS={getattr(ma.MyAgent, 'MAX_ACTIONS', '?')})")

# ---------- Transport (reuse Cell 13 if available) ----------

if 'tr' in globals() and tr is not None:
    TRANSPORT = tr
    print("OK   reusing Cell 13 transport")
else:
    from vllm_transport import VLLMTransport
    TRANSPORT = VLLMTransport("http://127.0.0.1:8000", "http://127.0.0.1:8001")
    print("INFO new transport created (Cell 13 tr not found)")

# ---------- vLLM health check (fail fast before 25-game loop) ----------

# P-1/P-2 FIX (DIR-078 addendum): VLLMTransport v26.3.4 has no wait_until_healthy()
# or routes API (older transport revision's API — latent crash since Run-20, never
# hit because no Run was launched after it was introduced).
# V4 integration contract: every TRANSPORT method this cell calls must exist.
for _m25 in ("__call__", "post_wire_capture", "set_v25_1_a", "get_v25_1_a"):
    if not hasattr(TRANSPORT, _m25):
        raise SystemExit(f"TRANSPORT.{_m25} missing — transport/cell API mismatch")
# Health check via raw sockets (Cell 13's _wait_healthy already verified launch;
# this is the pre-loop re-check, transport-independent):
import socket as _sock25
for _port25 in (8000, 8001):
    try:
        _s25 = _sock25.create_connection(("127.0.0.1", _port25), timeout=3.0)
        _s25.close()
    except OSError as _e25:
        raise SystemExit(f"vLLM server on port {_port25} not reachable: {_e25} — Cell 13 dobara chalao")
print("OK   both vLLM servers healthy (ports 8000, 8001 reachable; transport API contract 4/4)")

# ---------- FRAME_ADAPTER guard (Cell 15 set karta hai) ----------

if 'FRAME_ADAPTER' not in globals() or FRAME_ADAPTER is None:
    raise SystemExit("FRAME_ADAPTER undefined — Cell 15 (frame probe) pehle chalao")
print(f"OK   FRAME_ADAPTER set ({type(FRAME_ADAPTER).__name__})")

# ---------- Time budget (9h Kaggle session cap ke andar rehna hai) ----------

MAX_MINUTES_PER_GAME = 17.0    # FIX-θ (Chief order: "kam se kam har game par 15 min do"):
                                # 15->17. FIX-η ke saath worst-case math: 8 deep × ~90s = 12 min
                                # + 72 fast × ~5s = 6 min => ~18 min worst, ~14 typical.
                                # 25×17=7.08h + infra 0.2h = 7.3h < 7.5h global, Kaggle 9h OK.
                                # Win hone par game early khatam — realistic mix isse kam.
GLOBAL_TIME_BUDGET_H = 7.5     # v9: 5.5->7.5 — 25 × ~15 min + infra < 9h Kaggle cap

class _GridFrameProxy:
    """FrameData proxy: saare attributes same, sirf .frame int-grid (W-2) banaya gaya."""
    def __init__(self, fd, grid):
        try:
            self.__dict__.update({k: v for k, v in vars(fd).items() if k != "frame"})
        except TypeError:
            for attr in ("state", "available_actions", "levels_completed",
                         "win_levels", "action_count", "reward", "score"):
                v = getattr(fd, attr, None)
                if v is not None:
                    setattr(self, attr, v)
        self.frame = grid

class GoAMyAgent(ma.MyAgent):
    """Test-harness wrapper: MyAgent v1.0.16 as-is, documented _orchestrator
    constructor parameter (F-1 workaround) ke through real-model orchestrator inject.
    MUST-FIX 1: is_done() mein per-game wall-clock cap (MAX_MINUTES_PER_GAME) —
    framework actions ke beech is_done check karta hai: cooperative stop, core untouched."""
    def __init__(self, **kw):
        orch = ma.Agent(
            game_id=kw["game_id"],
            # v9 (DIR/2026-068): CM v1.5 field caps (350/200/100/200) ke saath
            # total_hard_cap bhi 7800 -> 14000 bytes (14000+system+1024 < 16384
            # IL gate — PromptTooLong structurally impossible).
            cm=cm_mod.ContextManager(total_hard_cap=14000),
            # v8 (Run#12): IL ka certified default CONTEXT_WINDOW=8192 (byte-counter
            # conservative hai) 7/25 games ko pehle hi reasoner call pe PromptTooLongError
            # de gaya (prompt+max_tokens 8214-9356). Actual vLLM reasoner max_model_len =
            # 16384 (Cell 13 launch flag) — designed constructor injection se TRUE value.
            il=il_mod.InferenceLayer(transport=TRANSPORT, context_window=16384),
            gt=gt_mod, ap=ap_mod,
            version_manifest=_VERSIONS,
        )
        # v7 (Run#10 fix): certified MyAgent.__init__ (v1.0.16, stub-era L-3 design)
        # game_id consume karke framework Agent.__init__ ko forward NAHI karta — real
        # base ke saath 'TypeError: missing game_id' (Run#10: 25/25 instant-fail).
        # Certified file S-8 untouchable: framework base __init__ DIRECT call
        # (game_id samet — fw khud self.game_id set karta hai) + MyAgent ka state
        # replicate (sirf _agent + _prev_state — game_id fw se milta hai).
        ma.MyAgent.__mro__[1].__init__(self, **kw)   # REAL framework Agent.__init__
        self._agent = orch                           # MyAgent F-1 orchestrator inject
        self._prev_state = None                      # MyAgent state replicate
        self._goa_t0 = time.time()
        # FIX-Z (external-audit round-3, adjudicated ACCEPTED): CM v1.5 ka
        # set_system_prompt kabhi call nahi hota tha — Zone1 hamesha empty
        # render hota tha, reasoner ko response contract ka directive nahi
        # milta. Test-harness layer: certified my_agent.py untouched
        # (artifact 001ce125 stable). Wording = actual contract: 4 JSON keys
        # (dataclass ReasonerResult) + _INSTRUCTION_RE ("aid" / "aid r# c#").
        if hasattr(orch, "_cm") and hasattr(orch._cm, "set_system_prompt"):
            orch._cm.set_system_prompt(
                # 185 chars < 200 zone1 cap (CM _tc = ~1 token/char, byte-
                # conservative). Audit ka original wording 204 chars tha —
                # cap overflow, trimmed bina contract change ke.
                "ARC-AGI-3 agent. Reply ONLY JSON: "
                "{observation_summary, hypothesis_updates, instruction, prediction}. "
                "instruction=ACTION_ID or ACTION_ID r<row> c<col>, "
                "e.g. ACTION6 r5 c7. No prose."
            )
            print("  [zone1] system prompt armed (%d chars)"
                  % len(orch._cm._zone1))
        # FIX-η (Chief order, Run-23 stop @ game 4): "3-4 deep chances mein
        # jeet nahi" — my_agent ka default trigger-based deep (sirf level-
        # start/jolt/divergence par) 80 actions mein sirf 2-4 soch deta tha,
        # baaki blind fast. Ab: reasoner_mode="deep" => HAR action deep jab
        # tak cap (hit-and-trial WITH thinking), deep_call_cap=8 (2000-token
        # thinks, ~65-90s each => ~9-12 min thinking + fast exploration).
        # Certified my_agent.py UNTOUCHED — documented config seam
        # (_deep_cap/Agent.config dono already exist, v1.0.19 design).
        # FIX-η2 (Run-23 postmortem cont.): deep 8->6 (time-balance) +
        # coder PROACTIVE (v1.0.20 certified upgrade): action 10 se coder
        # khud solve try karega (transition-buffer replay se). Budget math:
        # 6 deep × ~80s = 8 min + coder attempts (~1-2 min each, cap 6,
        # realistically 2-4) + fast actions => ~15 min of 17.
        orch.config["reasoner_mode"] = "deep"
        orch.config["deep_call_cap"] = 6
        orch.config["coder_proactive_actions_at_level"] = 10
        print("  [deep] reasoner_mode=deep, deep_call_cap=6 | "
              "[coder] proactive at 10 actions/level")

    def is_done(self, frames, latest_frame):
        if ma._state_name(latest_frame) == "WIN":
            return True
        _HB["acts"] = getattr(self, "action_counter", _HB["acts"])
        if time.time() - self._goa_t0 > MAX_MINUTES_PER_GAME * 60.0:
            print(f"  [time-cap] {self.game_id}: {MAX_MINUTES_PER_GAME} min limit — aage badh raha hoon",
                  flush=True)
            return True
        return False

    def choose_action(self, frames, latest_frame):
        # W-2: RGB frame par adapter conversion (probe cell FRAME_ADAPTER set karta hai).
        # MyAgent.choose_action test-harness replica + conversion insert — core untouched.
        if FRAME_ADAPTER is not None:
            conv = FRAME_ADAPTER(latest_frame.frame)
            proxy = _GridFrameProxy(latest_frame, conv)
            latest_frame = proxy
        spec = self._agent.framework_choose(latest_frame, self._prev_state)
        self._prev_state = ma._state_name(latest_frame)
        return ma._make_framework_action(spec["name"], spec["x"], spec["y"], spec["reasoning"])

assert hasattr(GoAMyAgent, "main"), "GoAMyAgent.main missing — Run#9 regression (inheritance)"

# ---------- Heartbeat thread: har 60s print (game, actions, time, ETA) ----------

_WALL0 = time.time()
_HB = {"game": "-", "acts": 0, "done": 0, "total": 25, "stop": False, "t0": time.time()}
_HB_DONE_MIN = []

def _heartbeat():
    while not _HB["stop"]:
        time.sleep(60)
        cur = (time.time() - _HB["t0"]) / 60
        wall = (time.time() - _WALL0) / 3600
        dm = _HB_DONE_MIN
        if dm:
            eta = (sum(dm) / len(dm)) * (_HB["total"] - _HB["done"])
            print(f"  [hb {wall:.1f}h] {_HB['game']}: {_HB['acts']} acts, "
                  f"{cur:.1f} min chal raha hai | done {_HB['done']}/{_HB['total']} "
                  f"| ETA ~{eta/60:.1f}h", flush=True)
        else:
            print(f"  [hb {wall:.1f}h] {_HB['game']}: {_HB['acts']} acts, "
                  f"{cur:.1f} min chal raha hai | done {_HB['done']}/{_HB['total']}",
                  flush=True)

threading.Thread(target=_heartbeat, daemon=True).start()

# ---------- Arcade + environment list (API guards) ----------

arc = arc_agi.Arcade(operation_mode=OperationMode.OFFLINE)   # v5: Run#8 - offline run, NORMAL API pe jaata tha; OFFLINE sirf local 25 cached envs

if not hasattr(arc, "get_environments"):
    raise SystemExit("Arcade.get_environments() nahi hai — ARC API check karo")
try:
    all_envs = arc.get_environments()
except Exception as e:
    raise SystemExit(f"get_environments() fail: {type(e).__name__}: {e}")

print(f"OK   {len(all_envs)} envs retrieved")
for _e in all_envs[:3]:
    print(f"     sample: game_id={getattr(_e, 'game_id', '?')!r}")

game_ids = sorted({e.game_id.split("-")[0] for e in all_envs})
print(f"{len(game_ids)} games: {game_ids}")

# ---------- Game count: hard gate (BUG-A fix) ----------

EXPECTED_GAMES = 25
if len(game_ids) != EXPECTED_GAMES:
    raise SystemExit(
        f"game count = {len(game_ids)} (expected {EXPECTED_GAMES}) "
        f"— cache/attachment mismatch"
    )
_HB["total"] = EXPECTED_GAMES

# ---------- Global budget helper ----------

def _global_budget_exceeded():
    return (time.time() - _WALL0) / 3600 > GLOBAL_TIME_BUDGET_H

# ---------- Run#10 fix gate: GoAMyAgent instantiation smoke (0 actions, discard) ----------
# Run#10: MyAgent.__init__ game_id forward nahi karta tha — real base ke saath TypeError.
# Ab single instantiate-then-discard se verify: 25x silent-fail possibility khatam.
_env_smoke = arc.make(game_ids[0])
_a_smoke = GoAMyAgent(card_id="goa-rtx-test", game_id=game_ids[0],
                      agent_name="GoA.smoke", ROOT_URL="http://localhost",
                      record=False, arc_env=_env_smoke, tags=["goa-smoke"])
assert hasattr(_a_smoke, "main"), "smoke agent: main missing (shim fail)"
assert _a_smoke._agent is not None, "smoke agent: orchestrator not wired"
assert _a_smoke.arc_env is _env_smoke, "smoke agent: arc_env not wired"
assert _a_smoke.game_id == game_ids[0], "smoke agent: game_id not set"
del _a_smoke, _env_smoke
print("OK   GoAMyAgent instantiation smoke — framework base + orchestrator wired, main() live")

# ---------- Main loop ----------

results = []
for i, gid in enumerate(game_ids, 1):
    if _global_budget_exceeded():
        print(f"GLOBAL BUDGET EXCEEDED ({GLOBAL_TIME_BUDGET_H}h) — abort at game {i}", flush=True)
        break

    t0 = time.time()
    print(f"[{i}/{len(game_ids)}] {gid} START", flush=True)
    _HB.update(game=gid, t0=t0, acts=0, done=i - 1)
    try:
        env = arc.make(gid)
        if env is None:
            rec = dict(game=gid, error="env is None",
                       minutes=round((time.time() - t0) / 60, 1),
                       time_capped=False)
            results.append(rec)
            _HB_DONE_MIN.append(rec.get("minutes", 0))  # M-2: ETA mein count
            _HB["done"] = i
            json.dump(results, open("/kaggle/working/run_results.json", "w"), indent=1)
            print(f"[{i}/{len(game_ids)}] {gid} env None — recorded, skipping", flush=True)
            continue

        a = GoAMyAgent(
            card_id="goa-rtx-test", game_id=gid, agent_name=f"GoA.{gid}",
            ROOT_URL="http://localhost", record=False, arc_env=env, tags=["goa-test"],
        )
        a.main()

        # ---------- Frames safety ----------
        final = a.frames[-1] if getattr(a, "frames", None) else None
        if final is None:
            raise RuntimeError("a.main() returned without any frames")

        # ---------- Log / jolts safety ----------
        _il = getattr(a._agent, "_il", None)
        log = _il.call_log() if _il and hasattr(_il, "call_log") else []
        jolts = a._agent.jolt_history() if hasattr(a._agent, "jolt_history") else []

        rec = dict(
            game=gid,
            state=str(getattr(final, "state", "?")),
            levels=getattr(final, "levels_completed", 0),
            actions=getattr(a, "action_counter", 0),
            minutes=round((time.time() - t0) / 60, 1),
            il_calls=len(log),
            # FIX-β (audit round-5 §6, VERIFIED TRUE): IL call_log entries
            # DICTS hain (ts/model/mode/prompt_tokens/completion_tokens/
            # latency_ms/finish_reason) — getattr hamesha default de raha
            # tha => reasoner/coder counts aur tokens FALSE ZEROS the.
            # Ab: r.get() + sahi keys. Coder detection model-name se (IL
            # coder calls mode=None log karta hai — audit sahi tha).
            # Raw log bhi dump hota hai: post-hoc recovery possible.
            reasoner_fast=sum(1 for r in log if r.get("mode") == "fast"),
            reasoner_deep=sum(1 for r in log if r.get("mode") == "deep"),
            coder_calls=sum(1 for r in log
                            if "coder" in str(r.get("model", ""))),
            finish_length=sum(1 for r in log
                              if r.get("finish_reason") == "length"),
            finish_other=sum(1 for r in log
                            if r.get("finish_reason") not in ("stop", None)),
            tokens_in=sum(r.get("prompt_tokens", 0) for r in log),
            tokens_out=sum(r.get("completion_tokens", 0) for r in log),
            il_log_raw=log,
            stuck=sum(1 for j in jolts if getattr(j, "kind", None) == "stuck"),
            deadlock=sum(1 for j in jolts if getattr(j, "kind", None) == "deadlock"),
        )
    except Exception as e:
        rec = dict(
            game=gid,
            error=f"{type(e).__name__}: {str(e)[:1000]}",
            trace=traceback.format_exc()[-800:],
            minutes=round((time.time() - t0) / 60, 1),
            time_capped=(time.time() - t0) > MAX_MINUTES_PER_GAME * 60.0,
        )

    results.append(rec)
    _HB_DONE_MIN.append(rec.get("minutes", 0))
    _HB["done"] = i
    json.dump(results, open("/kaggle/working/run_results.json", "w"), indent=1)
    print(f"[{i}/{len(game_ids)}] {gid} {rec.get('minutes', 0)} min DONE: "
          f"{(rec.get('state') or rec.get('error', ''))[:60]}", flush=True)

_HB["stop"] = True
print(f"\nLoop complete — total wall time: {(time.time() - _WALL0)/3600:.1f}h", flush=True)

# ---------- Scorecard (API guard) ----------

if not hasattr(arc, "get_scorecard"):
    print("WARN Arcade.get_scorecard() nahi hai — scorecard skip")
else:
    try:
        sc = arc.get_scorecard()
        score = sc.score if hasattr(sc, "score") else sc
        print("\nComposite scorecard:", score)
        json.dump({"score": str(score)}, open("/kaggle/working/scorecard.json", "w"))
    except Exception as e:
        print(f"WARN get_scorecard() fail: {type(e).__name__}: {e}")

print("Run complete — results: /kaggle/working/run_results.json")


In [ ]:
# Coordination summary v2: thinker+coder evidence (defensive)
import json, os

_RESULTS = "/kaggle/working/run_results.json"
if not os.path.exists(_RESULTS):
    raise SystemExit(f"{_RESULTS} missing — Cell 16 (THE RUN) pehle chalao")

try:
    with open(_RESULTS) as f:
        results = json.load(f)
except json.JSONDecodeError as e:
    raise SystemExit(f"run_results.json corrupted: {e}")

ok = [r for r in results if "error" not in r]
err = [r for r in results if "error" in r]

print(f"Games: {len(results)} | clean: {len(ok)} | errors: {len(err)}")
print(f"total levels: {sum(r.get('levels', 0) for r in ok)} | "
      f"total actions: {sum(r.get('actions', 0) for r in ok)}")
print(f"reasoner calls: fast={sum(r.get('reasoner_fast',0) for r in ok)} "
      f"deep={sum(r.get('reasoner_deep',0) for r in ok)} | "
      f"coder calls: {sum(r.get('coder_calls',0) for r in ok)}")
print(f"total IL calls: {sum(r.get('il_calls',0) for r in ok)}")
print(f"finish: length={sum(r.get('finish_length',0) for r in ok)} | "
      f"other={sum(r.get('finish_other',0) for r in ok)} | "
      f"il_calls={sum(r.get('il_calls',0) for r in ok)}")
print(f"tokens: in={sum(r.get('tokens_in',0) for r in ok):,} "
      f"out={sum(r.get('tokens_out',0) for r in ok):,}")
print(f"stuck fires: {sum(r.get('stuck',0) for r in ok)} | "
      f"deadlock fires: {sum(r.get('deadlock',0) for r in ok)}")

print("\nper-game:")
for r in results:
    _gid = r.get("game", "?")
    _mins = r.get("minutes", 0) or 0
    if "error" in r:
        print(f"  {_gid:8} ERROR: {str(r['error'])[:80]}")
    else:
        print(f"  {_gid:8} state={str(r.get('state', '?')):10} "
              f"levels={r.get('levels', 0):3} "
              f"acts={r.get('actions', 0):4} "
              f"{_mins:5.1f}min "
              f"R:{r.get('reasoner_fast',0)}+{r.get('reasoner_deep',0)} "
              f"C:{r.get('coder_calls',0)}")

# scorecard display
_SC_PATH = "/kaggle/working/scorecard.json"
if os.path.exists(_SC_PATH):
    try:
        with open(_SC_PATH) as f:
            sc = json.load(f)
        print(f"\nSCORECARD: {sc}")
    except Exception as e:
        print(f"WARN scorecard load fail: {e}")
else:
    print("\nWARN scorecard.json missing (Cell 16 scorecard step fail hua?)")


In [ ]:
# Backup v2: md5 manifest + selective zip (pycache/checkpoints/games-cache/.git/nested zips excluded — logs INCLUDED as run evidence)
import hashlib, os, json, shutil, zipfile

_ARCHIVE_ROOT = "/kaggle/working"
_TARGET_ZIP = f"{_ARCHIVE_ROOT}/goa_rtx_run_results.zip"
_MANIFEST = f"{_ARCHIVE_ROOT}/md5_manifest.json"

# ---------- purana zip delete (nested-zip self-inclusion prevent) ----------

if os.path.exists(_TARGET_ZIP):
    _old_size = os.path.getsize(_TARGET_ZIP) / 1e6
    os.remove(_TARGET_ZIP)
    print(f"OK   pichla backup hata ({_old_size:.1f} MB)")

# ---------- Exclude list ----------

_EXCLUDE_DIRS = {"__pycache__", ".ipynb_checkpoints", "environment_files", ".git"}
_EXCLUDE_FILES = {os.path.basename(_TARGET_ZIP), os.path.basename(_MANIFEST)}

# ---------- Streaming md5 ----------

def _md5_stream(path, chunk=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# ---------- Walk + hash (relative paths) ----------

recs = {}
_hash_fail = []
for root, dirs, files in os.walk(_ARCHIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in _EXCLUDE_DIRS]
    for fn in files:
        if fn in _EXCLUDE_FILES or fn.endswith(".zip"):
            continue
        p = os.path.join(root, fn)
        rel = os.path.relpath(p, _ARCHIVE_ROOT)
        try:
            recs[rel] = _md5_stream(p)
        except Exception as e:
            _hash_fail.append((rel, str(e)[:100]))

with open(_MANIFEST, "w") as f:
    json.dump({"root": _ARCHIVE_ROOT, "files": recs, "hash_fail": _hash_fail}, f, indent=1)
print(f"OK   manifest: {len(recs)} files hashed, {len(_hash_fail)} fail")

# ---------- Selective stage (only what belongs in backup) ----------

_STAGE = "/tmp/goa_backup_stage"
if os.path.exists(_STAGE):
    shutil.rmtree(_STAGE)
os.makedirs(_STAGE)

for root, dirs, files in os.walk(_ARCHIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in _EXCLUDE_DIRS]
    for fn in files:
        if fn in _EXCLUDE_FILES or fn.endswith(".zip"):
            continue
        src = os.path.join(root, fn)
        rel = os.path.relpath(src, _ARCHIVE_ROOT)
        dst = os.path.join(_STAGE, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        try:
            shutil.copy2(src, dst)
        except Exception as e:
            print(f"WARN copy fail {rel}: {e}")

# ---------- Archive ----------

try:
    _arch = shutil.make_archive("/tmp/goa_rtx_run_results", "zip", root_dir=_STAGE)
except Exception as e:
    raise SystemExit(f"make_archive fail: {e}")

# ---------- Move + integrity verify ----------

shutil.move(_arch, _TARGET_ZIP)

try:
    with zipfile.ZipFile(_TARGET_ZIP) as z:
        _bad = z.testzip()
        if _bad:
            raise SystemExit(f"zip corrupt at {_bad}")
        _n = len(z.namelist())
    _sz = os.path.getsize(_TARGET_ZIP) / 1e6
    print(f"backup ready: {_n} entries, {_sz:.1f} MB")
    if _sz > 500:
        print(f"WARN zip {_sz:.0f} MB bada — unexpected content check karo")
except Exception as e:
    raise SystemExit(f"zip validation fail: {e}")

# ---------- Cleanup ----------

shutil.rmtree(_STAGE, ignore_errors=True)


## CODER-SOLVER CONFORMANCE — moved before THE RUN (FIX-V, audit round-4)
Ye cell runbook ke baad, FRAME PROBE se pehle chalta hai.
Purana position (backup ke baad) ab sirf audit-trail hai.
Template-conference cell pehle se THE RUN se pehle tha (audit
round-4 ka wala claim galat numbering par tha — adjudicated FALSE).


In [ ]:
# md5 register (v10.21.0) — Run-21 certified + coder-solver set
import hashlib, os
_FILES = [
    ("context_manager",  "18b8dfa5be82c8224671e0fa92ee8135"),
    ("inference_layer",  "8ec5288b371b0e15f3f4d8ed49b0fa46"),
    ("grid_text",        "bdbc329a990d82c19bf4b4e22b838bdd"),
    ("action_pipeline",  "826cf5f96000f615e57f1a04233102e9"),
    ("my_agent",         "1c1151af8c0dcf5931af02138c2bb590"),
]
for _n, _e in _FILES:
    _p = f"/kaggle/working/goa/agent/{_n}.py"
    _a = hashlib.md5(open(_p, "rb").read()).hexdigest() if os.path.exists(_p) else "MISSING"
    print(f"{_n:18} {'OK' if _a == _e else 'MISMATCH ' + _a}  (expected {_e[:8]})")
_CS_FILES = ["sandbox_v6", "replay_verifier", "cegis_taxonomy", "cegis_router",
             "adversarial_critic", "inference_layer_v1_6", "inference_layer_v1_6_additions",
             "coder_solver"]
for _n in _CS_FILES:
    _p = f"/kaggle/working/goa/agent/{_n}.py"
    print(f"{_n:30} {hashlib.md5(open(_p, 'rb').read()).hexdigest()[:12] if os.path.exists(_p) else 'MISSING'}")
print("Run-20 baseline md5: 6536295199ce0508750e4a4fda6a0f93")


In [ ]:
# md5 register source-of-truth (v10.21.0) — Run-21
# certified: CM v1.6 18b8dfa5 | IL v1.6 8ec5288b | GT bdbc329a | AP 826cf5f9 | my_agent v1.0.21 (Object Vision) 1c1151af
# coder-solver: sandbox_v6, replay_verifier, cegis_taxonomy, cegis_router, adversarial_critic,
#               additions f9bab0b0 | coder_solver b79ca7e8 | lora 9744896a
# schema: 97073721699cf92d53bb4909322ad212 | R-13 artifact 001ce125 | Run-20 baseline 6536295199ce0508750e4a4fda6a0f93